In [ ]:
# Copyright (c) 2026 Daniyar Kuzekov, Li Yang, Ercan Engin Kuruoğlu, Wai Kin (Victor) Chan
# Licensed under the GNU Affero General Public License v3.0 (AGPL‑3.0)

In [1]:
# ============================================================
# OFFLINE DeepSeek-MoE Ws Builder (no HF access required)
# - Discovers MoE experts from model.safetensors.index.json
# - Loads expert weights from local shards with progress bar
# - Builds effective matrices W_eff = down @ up (or down @ gate)
# - Optional int8 quant error sanity check
# - Saves outputs to OUTPUT_DIR
# ============================================================

import os, re, json, math, gc, sys
from pathlib import Path
from typing import Dict, Tuple, List, Optional
import numpy as np

# ---------- USER CONFIG ----------
MODEL_DIR   = os.path.expanduser("~/deepseek-model")     # <-- set to your server path
OUTPUT_DIR  = os.path.expanduser("~/moe_ws_outputs")     # where to write Ws outputs
LAYER       = None  # e.g. 1, or None = auto-pick first layer with MoE experts
MAX_EXPERTS = 8     # keep modest for quick tests; raise if you want more
PREFER_UP   = "up_proj"   # "up_proj" or "gate_proj"
DTYPE_OUT   = "float16"   # "float16" or "float32"
USE_TORCH_MATMUL = True   # torch matmul can be faster; falls back to numpy
RUN_INT8_SANITY = True    # quick symmetric int8 quantization error stats
# --------------------------------

# ---------- lightweight deps ----------
def _pip_install(pkgs: List[str]) -> None:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU"] + pkgs)

try:
    from tqdm import tqdm
except Exception:
    _pip_install(["tqdm>=4.66"])
    from tqdm import tqdm

try:
    from safetensors import safe_open as st_safe_open
except Exception:
    _pip_install(["safetensors>=0.4.5"])
    from safetensors import safe_open as st_safe_open

TORCH_OK = False
torch = None
if USE_TORCH_MATMUL:
    try:
        import torch
        TORCH_OK = True
    except Exception:
        TORCH_OK = False

np.set_printoptions(suppress=True, linewidth=160)

# ---------- helpers ----------
def die(msg: str):
    raise RuntimeError(msg)

def human_gb(nbytes: int) -> str:
    return f"{nbytes / (1024**3):.2f} GB"

def load_index(model_dir: str) -> Dict[str, str]:
    idx_path = Path(model_dir) / "model.safetensors.index.json"
    if not idx_path.exists():
        die(f"Missing index file: {idx_path}\nMake sure you copied model.safetensors.index.json into MODEL_DIR.")
    with open(idx_path, "r", encoding="utf-8") as f:
        j = json.load(f)
    wm = j.get("weight_map", {})
    if not wm:
        die(f"Index has no weight_map entries: {idx_path}")
    return wm

def list_local_shards(model_dir: str) -> Dict[str, str]:
    """Map shard filename -> absolute path"""
    model_dir = Path(model_dir)
    out = {}
    for p in sorted(model_dir.glob("model-*-of-*.safetensors")):
        out[p.name] = str(p)
    # also allow single-shard
    for p in sorted(model_dir.glob("model.safetensors")):
        out[p.name] = str(p)
    return out

# DeepSeek expert MLP has gate_proj, up_proj, down_proj weights (module is DeepseekMLP inside DeepseekMoE) :contentReference[oaicite:1]{index=1}
EXPERT_RE = re.compile(
    r"^model\.layers\.(\d+)\.mlp\.experts\.(\d+)\.(gate_proj|up_proj|down_proj)\.weight$"
)

def discover_experts_from_weight_map(weight_map: Dict[str, str]) -> Dict[int, Dict[int, Dict[str, str]]]:
    """
    Returns:
      layers[layer_idx][expert_id][proj_name] = tensor_key
    where proj_name in {"gate_proj","up_proj","down_proj"} and tensor_key is the exact key.
    """
    layers: Dict[int, Dict[int, Dict[str, str]]] = {}
    for k in weight_map.keys():
        m = EXPERT_RE.match(k)
        if not m:
            continue
        L = int(m.group(1))
        eid = int(m.group(2))
        proj = m.group(3)
        layers.setdefault(L, {}).setdefault(eid, {})[proj] = k
    return layers

def pick_layer(layers: Dict[int, Dict[int, Dict[str, str]]], preferred: Optional[int]) -> Tuple[int, List[int]]:
    """Pick a layer with at least 1 expert having down + (prefer_up or gate)."""
    if preferred is not None:
        if preferred not in layers:
            die(f"LAYER={preferred} not found in index (no experts matched pattern). Available layers: {sorted(layers)}")
        eids = sorted(layers[preferred].keys())
        return preferred, eids

    # auto-pick first valid
    for L in sorted(layers.keys()):
        eids = sorted(layers[L].keys())
        # ensure overlap down + something
        ok = []
        for eid in eids:
            d = layers[L][eid]
            if "down_proj" in d and ("up_proj" in d or "gate_proj" in d):
                ok.append(eid)
        if ok:
            return L, ok
    die("No MoE expert weights discovered in index (pattern mismatch).")

def resolve_shards_needed(weight_map: Dict[str, str], keys: List[str]) -> List[str]:
    needed = sorted({weight_map[k] for k in keys})
    return needed

class ShardStore:
    """Open shards once, reuse handles."""
    def __init__(self, shard_paths: Dict[str, str]):
        self.shard_paths = shard_paths
        self.handles_np = {}
        self.handles_pt = {}

    def get_tensor_np32(self, shard_fn: str, key: str) -> np.ndarray:
        # NumPy path first
        if shard_fn not in self.handles_np:
            self.handles_np[shard_fn] = st_safe_open(self.shard_paths[shard_fn], framework="np")
        try:
            arr = self.handles_np[shard_fn].get_tensor(key)
            return arr.astype(np.float32, copy=False)
        except TypeError:
            # some envs might not like bf16 -> fallback torch
            return self.get_tensor_torch_fp32(shard_fn, key).cpu().numpy()

    def get_tensor_torch_fp32(self, shard_fn: str, key: str):
        if not TORCH_OK:
            die("Torch fallback requested but torch is not available.")
        if shard_fn not in self.handles_pt:
            self.handles_pt[shard_fn] = st_safe_open(self.shard_paths[shard_fn], framework="pt")
        t = self.handles_pt[shard_fn].get_tensor(key)
        return t.detach().to(dtype=torch.float32, device="cpu")

    def close(self):
        # safe_open objects close on GC; explicitly drop refs
        self.handles_np.clear()
        self.handles_pt.clear()

def matmul_down_up(down: np.ndarray, up: np.ndarray) -> np.ndarray:
    """
    down: (hidden, inter)
    up:   (inter, hidden)
    returns (hidden, hidden)
    """
    if TORCH_OK and USE_TORCH_MATMUL:
        td = torch.from_numpy(down)
        tu = torch.from_numpy(up)
        out = (td @ tu).contiguous()
        return out.cpu().numpy()
    else:
        return down @ up

def to_dtype(arr: np.ndarray, dtype_out: str) -> np.ndarray:
    if dtype_out == "float16":
        return arr.astype(np.float16, copy=False)
    if dtype_out == "float32":
        return arr.astype(np.float32, copy=False)
    die(f"Unsupported DTYPE_OUT={dtype_out}")

def int8_symmetric_quant_error(x: np.ndarray) -> Dict[str, float]:
    """Simple per-tensor symmetric int8 quant error summary."""
    x = x.astype(np.float32, copy=False)
    maxabs = float(np.max(np.abs(x)))
    if maxabs == 0.0:
        return {"maxabs": 0.0, "scale": 1.0, "mse": 0.0, "rmse": 0.0, "rel_rmse": 0.0}
    scale = maxabs / 127.0
    q = np.clip(np.round(x / scale), -127, 127).astype(np.int8)
    xhat = (q.astype(np.float32) * scale)
    err = xhat - x
    mse = float(np.mean(err * err))
    rmse = float(math.sqrt(mse))
    rel = float(rmse / (np.sqrt(float(np.mean(x * x))) + 1e-12))
    return {"maxabs": maxabs, "scale": float(scale), "mse": mse, "rmse": rmse, "rel_rmse": rel}

# ---------- main ----------
print("== OFFLINE DeepSeek-MoE Ws Builder ==")
print(f"MODEL_DIR:  {MODEL_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Torch available: {TORCH_OK} | USE_TORCH_MATMUL={USE_TORCH_MATMUL}")

model_dir = Path(MODEL_DIR)
if not model_dir.exists():
    die(f"MODEL_DIR does not exist: {MODEL_DIR}")

weight_map = load_index(MODEL_DIR)
layers = discover_experts_from_weight_map(weight_map)
layer_ok, eids_all = pick_layer(layers, LAYER)

# Choose experts that have required projections
valid_eids = []
for eid in sorted(eids_all):
    d = layers[layer_ok][eid]
    if "down_proj" not in d:
        continue
    if PREFER_UP in d:
        valid_eids.append(eid)
    elif PREFER_UP == "up_proj" and "gate_proj" in d:
        valid_eids.append(eid)  # fallback will happen per expert
    elif PREFER_UP == "gate_proj" and "up_proj" in d:
        valid_eids.append(eid)
if not valid_eids:
    die(f"No valid experts found at layer {layer_ok} with down_proj + ({PREFER_UP} or fallback).")

picked_eids = valid_eids[:MAX_EXPERTS]
print(f"\n[FOUND] layer={layer_ok} total_experts={len(valid_eids)} using_first={len(picked_eids)} eids={picked_eids}")

# Collect tensor keys to load
expert_specs = []
all_keys = []
for eid in picked_eids:
    d = layers[layer_ok][eid]
    down_k = d["down_proj"]
    up_k = d.get(PREFER_UP, None)
    used_up = PREFER_UP
    if up_k is None:
        # fallback
        alt = "gate_proj" if PREFER_UP == "up_proj" else "up_proj"
        up_k = d.get(alt, None)
        used_up = alt
    if up_k is None:
        continue
    expert_specs.append((eid, used_up, down_k, up_k))
    all_keys += [down_k, up_k]

if not expert_specs:
    die("After fallback logic, no experts remain to load.")

# Make sure shards exist locally
local_shards = list_local_shards(MODEL_DIR)
if not local_shards:
    die(f"No shard files found in MODEL_DIR.\nExpected model-00001-of-00007.safetensors ...\nGot empty.\nMODEL_DIR={MODEL_DIR}")

needed_shards = resolve_shards_needed(weight_map, all_keys)
missing = [fn for fn in needed_shards if fn not in local_shards]
if missing:
    die("Missing shard(s) in MODEL_DIR:\n" + "\n".join("  - " + m for m in missing) +
        f"\n\nPut the missing shards into: {MODEL_DIR}")

# Open shard handles once
store = ShardStore({fn: local_shards[fn] for fn in needed_shards})

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
out_npz = Path(OUTPUT_DIR) / f"deepseek_layer{layer_ok}_Ws_first{len(expert_specs)}.npz"

stats_rows = []
Ws = {}  # keep small set in memory; still saved to disk

print("\n=== Loading experts & building W_eff (with progress) ===")
for (eid, used_up, down_k, up_k) in tqdm(expert_specs, desc="experts"):
    down_shard = weight_map[down_k]
    up_shard   = weight_map[up_k]

    down = store.get_tensor_np32(down_shard, down_k)  # (hidden, inter)
    up   = store.get_tensor_np32(up_shard, up_k)      # (inter, hidden)

    # Build effective
    weff = matmul_down_up(down, up)

    # Convert dtype for saving
    down_s = to_dtype(down, DTYPE_OUT)
    up_s   = to_dtype(up, DTYPE_OUT)
    weff_s = to_dtype(weff, DTYPE_OUT)

    # Basic norms
    row = {
        "expert": eid,
        "used_up": used_up,
        "down_shape": tuple(down.shape),
        "up_shape": tuple(up.shape),
        "weff_shape": tuple(weff.shape),
        "down_frob": float(np.linalg.norm(down)),
        "up_frob": float(np.linalg.norm(up)),
        "weff_frob": float(np.linalg.norm(weff)),
    }

    # Optional int8 sanity
    if RUN_INT8_SANITY:
        q = int8_symmetric_quant_error(weff)
        row.update({
            "weff_int8_rmse": q["rmse"],
            "weff_int8_rel_rmse": q["rel_rmse"],
            "weff_int8_scale": q["scale"],
        })

    stats_rows.append(row)

    # Store arrays under unique names
    Ws[f"expert{eid}.down"] = down_s
    Ws[f"expert{eid}.up_{used_up}"] = up_s
    Ws[f"expert{eid}.W_eff"] = weff_s

    # free big temporaries
    del down, up, weff
    gc.collect()

store.close()

# Save packed result
np.savez_compressed(out_npz, **Ws)

# Print summary table
print("\n=== SUMMARY ===")
print(f"Saved: {out_npz}  ({out_npz.stat().st_size/1024/1024:.2f} MB)")
print(f"Layer: {layer_ok} | Experts saved: {len(expert_specs)} | DTYPE_OUT={DTYPE_OUT}")
print()

# Pretty print a few rows
cols = ["expert","used_up","down_shape","up_shape","weff_shape","weff_frob"]
if RUN_INT8_SANITY:
    cols += ["weff_int8_rel_rmse","weff_int8_scale"]

for r in stats_rows[:min(12, len(stats_rows))]:
    line = " | ".join([f"{c}={r[c]}" for c in cols])
    print(line)

print("\n✅ Done. You can now load the .npz and feed `expert*.W_eff` (or `down/up`) into your quantization pipeline.")
print("Tip: If you want a different layer, set LAYER=<int> and rerun.")


== OFFLINE DeepSeek-MoE Ws Builder ==
MODEL_DIR:  /home/daniyar/deepseek-model
OUTPUT_DIR: /home/daniyar/moe_ws_outputs
Torch available: True | USE_TORCH_MATMUL=True

[FOUND] layer=1 total_experts=64 using_first=8 eids=[0, 1, 2, 3, 4, 5, 6, 7]

=== Loading experts & building W_eff (with progress) ===


experts: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:13<00:00,  1.71s/it]



=== SUMMARY ===
Saved: /home/daniyar/moe_ws_outputs/deepseek_layer1_Ws_first8.npz  (126.82 MB)
Layer: 1 | Experts saved: 8 | DTYPE_OUT=float16

expert=0 | used_up=up_proj | down_shape=(2048, 1408) | up_shape=(1408, 2048) | weff_shape=(2048, 2048) | weff_frob=81.77774047851562 | weff_int8_rel_rmse=0.019156362185399702 | weff_int8_scale=0.002649810839825728
expert=1 | used_up=up_proj | down_shape=(2048, 1408) | up_shape=(1408, 2048) | weff_shape=(2048, 2048) | weff_frob=79.79796600341797 | weff_int8_rel_rmse=0.013997053062172226 | weff_int8_scale=0.001889506193596547
expert=2 | used_up=up_proj | down_shape=(2048, 1408) | up_shape=(1408, 2048) | weff_shape=(2048, 2048) | weff_frob=68.22835540771484 | weff_int8_rel_rmse=0.01573123310351635 | weff_int8_scale=0.0018155933834436372
expert=3 | used_up=up_proj | down_shape=(2048, 1408) | up_shape=(1408, 2048) | weff_shape=(2048, 2048) | weff_frob=87.83383178710938 | weff_int8_rel_rmse=0.012141378951348071 | weff_int8_scale=0.001803937507426644

In [4]:
# ============================================================
# OFFLINE DeepSeek-MoE Ws (W_eff) BUILDER + SAFE BRIDGE FOR KT++-X
#   - No HuggingFace download calls
#   - Reads local sharded safetensors using model.safetensors.index.json weight_map
#   - Builds W_eff = down_proj @ up_proj per expert (square [hidden,hidden])
#   - Avoids the "Ws is dict" + shape-mismatch torch.stack crash
# ============================================================

import os, re, json, glob
import numpy as np
from tqdm import tqdm

# ---- torch + safetensors ----
try:
    import torch
except Exception as e:
    raise RuntimeError("This cell requires torch (CPU is fine). Install torch in the venv.") from e

try:
    from safetensors import safe_open as st_safe_open
except Exception as e:
    raise RuntimeError("This cell requires safetensors. Install: pip install safetensors") from e


# =========================
# USER SETTINGS
# =========================
MODEL_DIR   = os.environ.get("DEESEEK_MODEL_DIR", "/home/daniyar/deepseek-model")  # folder containing model-000xx-of-000yy.safetensors + model.safetensors.index.json
OUTPUT_DIR  = os.environ.get("DEESEEK_OUT_DIR",  "/home/daniyar/moe_ws_outputs")

LAYER       = int(os.environ.get("DEESEEK_LAYER", "1"))
MAX_EXPERTS = int(os.environ.get("DEESEEK_MAX_EXPERTS", "8"))

# If you have a stale globals()["Ws"] dict, this will ignore it unless it is already a clean [E,n,n].
PREFER_REBUILD_FROM_SHARDS = os.environ.get("DEESEEK_FORCE_REBUILD", "1") == "1"

DTYPE_COMPUTE = torch.float32   # matmul compute dtype
DTYPE_SAVE    = torch.float16   # npz save dtype (smaller)

os.makedirs(OUTPUT_DIR, exist_ok=True)
NPZ_PATH = os.path.join(OUTPUT_DIR, f"deepseek_layer{LAYER}_Ws_first{MAX_EXPERTS}.npz")

print("= OFFLINE DeepSeek-MoE Ws Builder ==")
print(f"MODEL_DIR:   {MODEL_DIR}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")
print(f"LAYER:       {LAYER}")
print(f"MAX_EXPERTS: {MAX_EXPERTS}")
print(f"DTYPE_SAVE:  {str(DTYPE_SAVE).replace('torch.', '')}")
print(f"Torch:       {torch.__version__} | threads={torch.get_num_threads()}")
print()


# =========================
# HELPERS
# =========================
def _must_exist(path: str, what: str):
    if not os.path.exists(path):
        raise RuntimeError(f"Missing {what}: {path}")

def _load_index(model_dir: str) -> dict:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    _must_exist(idx_path, "model.safetensors.index.json")
    with open(idx_path, "r") as f:
        return json.load(f)

def _available_shards(model_dir: str):
    return sorted(glob.glob(os.path.join(model_dir, "model-*.safetensors")))

def _open_shard_cached(shard_path: str, cache: dict):
    h = cache.get(shard_path)
    if h is None:
        h = st_safe_open(shard_path, framework="pt")  # torch tensors
        cache[shard_path] = h
    return h

def _close_all(cache: dict):
    # safetensors safe_open returns a context manager-like object with .__exit__
    for _, h in list(cache.items()):
        try:
            h.__exit__(None, None, None)
        except Exception:
            pass
    cache.clear()

def _to_f32(t: torch.Tensor) -> torch.Tensor:
    if t.dtype != torch.float32:
        t = t.to(dtype=torch.float32)
    return t

def _find_deepseek_expert_keys(weight_map: dict, layer: int):
    """
    DeepSeek uses keys like:
      model.layers.{L}.mlp.experts.{eid}.up_proj.weight
      model.layers.{L}.mlp.experts.{eid}.down_proj.weight
    We'll also allow w1/w2 (Mixtral-like) as fallback.
    """
    layer_tag = f"model.layers.{layer}.mlp.experts."
    # candidate regexes
    up_re   = re.compile(rf"^{re.escape(layer_tag)}(\d+)\.(up_proj|w1)\.weight$")
    down_re = re.compile(rf"^{re.escape(layer_tag)}(\d+)\.(down_proj|w2)\.weight$")

    ups = {}
    downs = {}
    for k in weight_map.keys():
        m = up_re.match(k)
        if m:
            eid = int(m.group(1))
            ups.setdefault(eid, []).append(k)
            continue
        m = down_re.match(k)
        if m:
            eid = int(m.group(1))
            downs.setdefault(eid, []).append(k)
            continue

    eids = sorted(set(ups.keys()) & set(downs.keys()))
    return eids, ups, downs

def _pick_key(keys: list, prefer: list):
    for p in prefer:
        for k in keys:
            if f".{p}.weight" in k:
                return k
    return keys[0]

def _load_tensor_from_weight_map(model_dir: str, weight_map: dict, key: str, shard_cache: dict) -> torch.Tensor:
    fn = weight_map.get(key, None)
    if fn is None:
        raise KeyError(f"Key not in weight_map: {key}")
    shard_path = os.path.join(model_dir, fn)
    _must_exist(shard_path, f"shard file for {key}")
    h = _open_shard_cached(shard_path, shard_cache)
    t = h.get_tensor(key)
    return t

def _try_use_clean_global_Ws() -> torch.Tensor | None:
    """
    Accept globals()['Ws'] only if it is already a clean [E,n,n] tensor/ndarray
    with consistent square shapes. Otherwise return None.
    """
    if "Ws" not in globals():
        return None

    Ws_any = globals()["Ws"]
    if isinstance(Ws_any, np.ndarray):
        if Ws_any.ndim == 3 and Ws_any.shape[1] == Ws_any.shape[2]:
            return torch.from_numpy(Ws_any).to(dtype=torch.float32).contiguous()
        return None

    if torch.is_tensor(Ws_any):
        if Ws_any.ndim == 3 and Ws_any.shape[1] == Ws_any.shape[2]:
            return Ws_any.to(dtype=torch.float32).contiguous()
        return None

    if isinstance(Ws_any, dict):
        # Only accept dict if ALL entries are square and same shape
        mats = []
        for _, v in Ws_any.items():
            if isinstance(v, np.ndarray):
                t = torch.from_numpy(v)
            elif torch.is_tensor(v):
                t = v
            else:
                return None
            if t.ndim != 2 or t.shape[0] != t.shape[1]:
                return None
            mats.append(t)
        if not mats:
            return None
        n0 = mats[0].shape[0]
        if any(m.shape != (n0, n0) for m in mats):
            return None
        return torch.stack([m.to(dtype=torch.float32) for m in mats], dim=0).contiguous()

    return None


# =========================
# MAIN: BUILD Ws_t
# =========================
Ws_t = None

if not PREFER_REBUILD_FROM_SHARDS:
    Ws_t = _try_use_clean_global_Ws()
    if Ws_t is not None:
        print("[ok] Using existing clean globals()['Ws'] as square [E,n,n].")
        print(f"[ok] Ws_t: {tuple(Ws_t.shape)} dtype={Ws_t.dtype}")
        print()

# If not using global Ws, rebuild from shards (offline)
if Ws_t is None:
    # sanity: required files
    idx = _load_index(MODEL_DIR)
    weight_map = idx.get("weight_map", None)
    if weight_map is None:
        raise RuntimeError("Index json missing 'weight_map' field.")

    shard_files = _available_shards(MODEL_DIR)
    if not shard_files:
        raise RuntimeError(f"No model-*.safetensors found in {MODEL_DIR}")

    # discover experts for layer
    eids, ups, downs = _find_deepseek_expert_keys(weight_map, LAYER)
    if not eids:
        raise RuntimeError(f"No expert up/down pairs found for layer={LAYER} in weight_map.")

    use_eids = eids[:MAX_EXPERTS]
    print(f"[FOUND] layer={LAYER} total_experts={len(eids)} using_first={len(use_eids)} eids={use_eids}")
    print()

    # build W_eff for each expert
    shard_cache = {}
    W_eff_list = []
    meta = []

    try:
        for eid in tqdm(use_eids, desc="experts"):
            up_key   = _pick_key(ups[eid],   prefer=["up_proj", "w1"])
            down_key = _pick_key(downs[eid], prefer=["down_proj", "w2"])

            up   = _load_tensor_from_weight_map(MODEL_DIR, weight_map, up_key, shard_cache)
            down = _load_tensor_from_weight_map(MODEL_DIR, weight_map, down_key, shard_cache)

            up   = _to_f32(up)
            down = _to_f32(down)

            # DeepSeek typical: up [intermediate, hidden], down [hidden, intermediate]
            if down.shape[1] != up.shape[0]:
                raise RuntimeError(
                    f"Shape mismatch for eid={eid}:\n"
                    f"  up  {up_key}   shape={tuple(up.shape)}\n"
                    f"  down{down_key} shape={tuple(down.shape)}\n"
                    f"Expected down.shape[1] == up.shape[0]."
                )

            W_eff = (down @ up).to(dtype=DTYPE_COMPUTE).contiguous()  # [hidden, hidden]
            if W_eff.shape[0] != W_eff.shape[1]:
                raise RuntimeError(f"W_eff not square for eid={eid}: shape={tuple(W_eff.shape)}")

            W_eff_list.append(W_eff)
            meta.append((eid, up_key, down_key, tuple(up.shape), tuple(down.shape), tuple(W_eff.shape)))
    finally:
        _close_all(shard_cache)

    # stack to Ws_t
    n = W_eff_list[0].shape[0]
    if any(W.shape != (n, n) for W in W_eff_list):
        shapes = [tuple(W.shape) for W in W_eff_list]
        raise RuntimeError(f"Inconsistent W_eff shapes: {shapes}")

    Ws_t = torch.stack(W_eff_list, dim=0).to(dtype=torch.float32).contiguous()
    print("\n=== SUMMARY ===")
    print(f"Ws_t:  {tuple(Ws_t.shape)} dtype={Ws_t.dtype}")
    print(f"n:     {n}")
    print()

    # Save NPZ (float16 by default)
    Ws_np = Ws_t.to(dtype=DTYPE_SAVE).cpu().numpy()
    np.savez_compressed(
        NPZ_PATH,
        Ws=Ws_np,
        layer=np.array([LAYER], dtype=np.int32),
        expert_ids=np.array(use_eids, dtype=np.int32),
    )
    print(f"Saved: {NPZ_PATH}  ({os.path.getsize(NPZ_PATH)/1024/1024:.2f} MB)")
    print()
    for (eid, up_key, down_key, up_sh, down_sh, weff_sh) in meta[:min(8, len(meta))]:
        print(f"expert={eid} | up={up_sh} down={down_sh} -> W_eff={weff_sh}")
    print()

# Export legacy globals expected by other cells
Ws = Ws_t.cpu().numpy().astype(np.float32, copy=False)  # [E,n,n] float32
print("✅ Ready for KT++-X")
print(f"   Ws_t: torch.Tensor {tuple(Ws_t.shape)} dtype={Ws_t.dtype}")
print(f"   Ws:   np.ndarray   {tuple(Ws.shape)} dtype={Ws.dtype}")
print(f"   NPZ:  {NPZ_PATH}")
print()

# Optional: quick self-check that your old assertion will pass
assert Ws_t.ndim == 3 and Ws_t.shape[1] == Ws_t.shape[2], "Ws_t must be [E,n,n]"
assert isinstance(Ws, np.ndarray) and Ws.ndim == 3 and Ws.shape[1] == Ws.shape[2], "Ws must be [E,n,n]"
print("[check] Ws is clean square stack. Your `.float().contiguous()` line will work now.")


= OFFLINE DeepSeek-MoE Ws Builder ==
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS: 8
DTYPE_SAVE:  float16
Torch:       2.4.1+cpu | threads=8

[FOUND] layer=1 total_experts=64 using_first=8 eids=[0, 1, 2, 3, 4, 5, 6, 7]



experts: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:04<00:00,  1.73it/s]



=== SUMMARY ===
Ws_t:  (8, 2048, 2048) dtype=torch.float32
n:     2048

Saved: /home/daniyar/moe_ws_outputs/deepseek_layer1_Ws_first8.npz  (59.04 MB)

expert=0 | up=(1408, 2048) down=(2048, 1408) -> W_eff=(2048, 2048)
expert=1 | up=(1408, 2048) down=(2048, 1408) -> W_eff=(2048, 2048)
expert=2 | up=(1408, 2048) down=(2048, 1408) -> W_eff=(2048, 2048)
expert=3 | up=(1408, 2048) down=(2048, 1408) -> W_eff=(2048, 2048)
expert=4 | up=(1408, 2048) down=(2048, 1408) -> W_eff=(2048, 2048)
expert=5 | up=(1408, 2048) down=(2048, 1408) -> W_eff=(2048, 2048)
expert=6 | up=(1408, 2048) down=(2048, 1408) -> W_eff=(2048, 2048)
expert=7 | up=(1408, 2048) down=(2048, 1408) -> W_eff=(2048, 2048)

✅ Ready for KT++-X
   Ws_t: torch.Tensor (8, 2048, 2048) dtype=torch.float32
   Ws:   np.ndarray   (8, 2048, 2048) dtype=float32
   NPZ:  /home/daniyar/moe_ws_outputs/deepseek_layer1_Ws_first8.npz

[check] Ws is clean square stack. Your `.float().contiguous()` line will work now.


In [5]:
# ============================================================
# KT++-X BACK TO BASICS - Using Original Working Approach
# ============================================================

import os, math, time, random
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict
import numpy as np
import torch
import torch.nn as nn

# ------------ safe threading ------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try: torch.set_num_threads(NTHREADS)
except Exception: pass

SEED = 1234
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
DEVICE = torch.device("cpu")

# ------------ expect Ws ------------
assert 'Ws' in globals(), "Expected `Ws` as [E, n, n] in memory."
Ws_t = torch.from_numpy(Ws) if isinstance(Ws, np.ndarray) else Ws
Ws_t = Ws_t.float().contiguous()
E, n, _ = Ws_t.shape
print(f"[KT++-X ORIGINAL APPROACH] Using Ws: {tuple(Ws_t.shape)} on {DEVICE}")

# =============== ORIGINAL WORKING APPROACH ===============

def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord='fro')

@torch.no_grad()
def hadamard_like_Q(n: int) -> torch.Tensor:
    """Original working orthogonal initialization"""
    H = torch.eye(n, dtype=torch.float32)
    steps = int(math.log2(max(n,2))) + 1
    for _ in range(steps):
        v = torch.randn(n); v /= (v.norm()+1e-12)
        H = H - 2.0*(H@v).unsqueeze(1)@v.unsqueeze(0)
    Q, _ = torch.linalg.qr(H)
    return Q.contiguous()

@torch.no_grad()
def pooled_svd_for_subset_fast(Ws_subset: torch.Tensor):
    n = Ws_subset.shape[1]
    return hadamard_like_Q(n), hadamard_like_Q(n)

def random_projection_features(Ws, d=64, seed=SEED):
    rng = np.random.default_rng(seed)
    E, n, _ = Ws.shape
    R = torch.from_numpy(rng.choice([-1.0,1.0], size=(n, d)).astype(np.float32))
    feats=[]
    for e in range(E):
        W = Ws[e]
        row_e = torch.diag(W @ W.t())
        col_e = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row_e@R).unsqueeze(0),(col_e@R).unsqueeze(0)],dim=1))
    return torch.cat(feats, dim=0)

def kmeans_torch(X, k=6, iters=60, seed=SEED):
    g = torch.Generator().manual_seed(int(seed))
    idx = torch.randperm(X.shape[0], generator=g)[:k]
    C = X[idx].clone()
    for _ in range(iters):
        dist = torch.cdist(X, C)
        lab = dist.argmin(dim=1)
        for j in range(k):
            m = (lab==j)
            if m.any(): C[j] = X[m].mean(dim=0)
    return lab, C

# =============== ORIGINAL SLICE OPERATIONS ===============

def diag_slice_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor):
    V_S = V[:, S]                         
    T = torch.matmul(Ws_batch, V_S)       
    U_S = U[:, S]                         
    Xs = torch.matmul(T.transpose(1,2), U_S)  
    return torch.diagonal(Xs, dim1=1, dim2=2)

def offdiag_l1_slice_batch(Ws_sub: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor):
    V_S = V[:, S]                         
    T = torch.matmul(Ws_sub, V_S)         
    U_S = U[:, S]                         
    Xs = torch.matmul(T.transpose(1,2), U_S)  
    D  = torch.diagonal(Xs, dim1=1, dim2=2)   
    X_off = Xs - torch.diag_embed(D)
    return X_off.abs().mean()

@torch.no_grad()
def smart_k_from_pool(pool: torch.Tensor, target: float):
    order = torch.argsort(pool, descending=True)
    csum = torch.cumsum(pool[order], dim=0)
    frac = csum/(pool.sum()+1e-12)
    if (frac>=target).any():
        K = int(torch.nonzero(frac>=target, as_tuple=False)[0].item()+1)
    else:
        K = pool.numel()
    return K, order

# =============== ORIGINAL CONFIG FROM WORKING VERSION ===============

@dataclass
class OriginalConfig:
    M: int = 6
    target_energy: float = 0.995
    bands_B: int = 8
    
    # Phase 1 (UV warmup) - FROM WORKING VERSION
    p1_steps: int = 12
    p1_subm: int = 128
    p1_batchE: int = 3
    p1_lr: float = 1e-1
    
    # Phase 2 (hybrid: worst clusters get weight updates) - FROM WORKING VERSION
    p2_steps: int = 40
    p2_subm: int = 160
    p2_batchE: int = 4
    p2_lr_W: float = 5e-4
    worst_ratio: float = 0.5
    
    # Loss coefficients - FROM WORKING VERSION
    lam_off: float = 2e-3
    lam_grp: float = 5e-4
    lam_prox: float = 5e-4
    
    report_every: int = 4
    reortho_every: int = 4
    ema_beta: float = 0.9
    bands_update_every: int = 8

OC = OriginalConfig()

# =============== ORIGINAL CLUSTERING ===============

print(f"[ORIGINAL] Clustering to M={OC.M} ...")
Xfeat = random_projection_features(Ws_t, d=64, seed=SEED)
labels, _ = kmeans_torch(Xfeat, k=OC.M, iters=60, seed=SEED)
clusters = [torch.nonzero(labels==m, as_tuple=False).flatten().tolist() for m in range(OC.M)]
print("[ORIGINAL] cluster sizes:", [len(c) for c in clusters])

# =============== ORIGINAL ORTHO PARAM ===============

class OrthoParam(nn.Module):
    def __init__(self, M_init): 
        super().__init__(); 
        self.M = nn.Parameter(M_init.clone())
    def orthogonal(self): 
        Q,_ = torch.linalg.qr(self.M); 
        return Q

U_par, V_par, pool_ema = [], [], []
for m, idx in enumerate(clusters):
    if len(idx)==0: 
        U_par.append(None); V_par.append(None); pool_ema.append(None); 
        continue
    U0, V0 = pooled_svd_for_subset_fast(Ws_t[idx])
    U_par.append(OrthoParam(U0)); V_par.append(OrthoParam(V0))
    pool_ema.append(torch.zeros(n))

# =============== ORIGINAL TRAINING - PROVEN TO WORK ===============

def original_phase_train(steps, subm, batchE, lr_UV, lam_off, lam_grp, allow_W=False, W_params=None, lr_W=0.0, title="P1"):
    # optimizer
    params = [uv.M for uv in U_par+V_par if uv is not None]
    if allow_W:
        params += list(W_params)
    opt = torch.optim.Adam(params, lr=(lr_W if allow_W else lr_UV))
    t0=time.perf_counter()

    for step in range(1, steps+1):
        S = torch.randperm(n)[:subm]
        L_total = 0.0
        for m, idx in enumerate(clusters):
            if len(idx)==0: continue
            Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
            # choose experts
            if batchE and batchE < len(idx):
                pick = torch.randperm(len(idx))[:batchE]
                idx_step = [idx[i.item()] for i in pick]
            else:
                idx_step = idx
            # Ws batch
            if allow_W:
                Ws_batch = torch.stack([W_params[e] for e in idx_step], dim=0)
            else:
                Ws_batch = torch.stack([Ws_t[e] for e in idx_step], dim=0)
            # diag slice
            D = diag_slice_batch(Ws_batch, Uo, Vo, S)     # (E_b, s)
            with torch.no_grad():
                pooled = D.abs().mean(0)
                pool_ema[m][S] = OC.ema_beta*pool_ema[m][S] + (1-OC.ema_beta)*pooled
            # offdiag loss
            L_off = offdiag_l1_slice_batch(Ws_batch, Uo, Vo, S)
            # group-support (slice)
            L_grp = 0.0
            if (step % OC.bands_update_every)==0 and pool_ema[m].abs().sum().item()>0:
                K, order = smart_k_from_pool(pool_ema[m].clamp_min(0), OC.target_energy)
                top = order[:K]
                bands = [c for c in torch.chunk(top, OC.bands_B) if len(c)>0]
            else:
                order = torch.argsort(D.abs().mean(0), descending=True)
                Kloc = min(len(order), int(max(1, subm*0.9)))
                top = S[order[:Kloc]]
                bands = [c for c in torch.chunk(top, OC.bands_B) if len(c)>0]
            for g_idx in bands:
                mask = torch.isin(S, g_idx)
                if not mask.any(): continue
                D_band = D[:, mask]
                L_grp = L_grp + (D_band.pow(2).sum(dim=0).sqrt().sum() / (D_band.shape[1] + 1e-12))
            if isinstance(L_grp, float): L_grp = torch.tensor(0.0)
            L_total = L_total + lam_off*L_off + lam_grp*L_grp

        opt.zero_grad(); L_total.backward()
        if (step % OC.reortho_every)==0 or step==steps:
            with torch.no_grad():
                for uv in U_par+V_par:
                    if uv is not None: uv.M.copy_(uv.orthogonal())
        opt.step()

        if (step==1) or (step%OC.report_every)==0 or step==steps:
            # sample diag metric
            diag_fracs=[]
            for m, idx in enumerate(clusters):
                if len(idx)==0: continue
                Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                U_S, V_S = Uo[:,S], Vo[:,S]
                e_list = idx[:min(2,len(idx))]
                dsum=0.0; tsum=0.0
                for e in e_list:
                    We = (W_params[e] if allow_W and isinstance(W_params[e], nn.Parameter) else Ws_t[e])
                    Xs = U_S.t() @ (We @ V_S)
                    dsum += torch.abs(torch.diag(Xs)).sum().item()
                    tsum += torch.abs(Xs).sum().item()
                diag_fracs.append(dsum/(tsum+1e-12) if tsum>0 else 0.0)
            t1=time.perf_counter()
            print(f"[{title}] step {step:3d}/{steps} | diag_frac(sample)≈{float(np.mean(diag_fracs)):.4f}  (+{t1-t0:.1f}s)")
            t0=t1

# Phase 1 run (UV-only)
print("\n=== PHASE 1: UV WARMUP ===")
original_phase_train(OC.p1_steps, OC.p1_subm, OC.p1_batchE, OC.p1_lr, OC.lam_off, OC.lam_grp, allow_W=False, title="P1-UV")

# =============== ORIGINAL CLUSTER SELECTION ===============

with torch.no_grad():
    S_probe = torch.randperm(n)[:max(128, OC.p1_subm)]
    scores=[]
    for m, idx in enumerate(clusters):
        if len(idx)==0: scores.append((m, 1e9)); continue
        Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
        U_S, V_S = Uo[:,S_probe], Vo[:,S_probe]
        # mean L1 diag fraction over a couple experts
        e_list = idx[:min(3,len(idx))]
        dsum=0.0; tsum=0.0
        for e in e_list:
            Xs = U_S.t() @ (Ws_t[e] @ V_S)
            dsum += torch.abs(torch.diag(Xs)).sum().item()
            tsum += torch.abs(Xs).sum().item()
        frac = dsum/(tsum+1e-12) if tsum>0 else 0.0
        scores.append((m, frac))
    worst_sorted = sorted(scores, key=lambda kv: kv[1])  # low diag -> worse
    keep_worst = set([m for m,_ in worst_sorted[:max(1, int(OC.worst_ratio*len(clusters)))]])
    print("[ORIGINAL] worst clusters selected for weight updates:", sorted(list(keep_worst)))

# =============== ORIGINAL WEIGHT UPDATES ===============

# Wrap Ws into parameters only for experts in worst clusters
W_params = []
for e in range(E):
    m_of_e = None
    for m, idx in enumerate(clusters):
        if e in idx: m_of_e=m; break
    if m_of_e in keep_worst:
        W_params.append(nn.Parameter(Ws_t[e].clone()))
    else:
        # frozen wrapper
        class FrozenWrap:
            def __init__(self, T): self.data=T
        W_params.append(FrozenWrap(Ws_t[e]))
W0_snap = [ (p.detach().clone() if isinstance(p, nn.Parameter) else p.data.clone()) for p in W_params ]

def original_phase2_train():
    params = [uv.M for uv in U_par+V_par if uv is not None]
    # add only nn.Parameter weights (worst clusters)
    params += [p for p in W_params if isinstance(p, nn.Parameter)]
    opt = torch.optim.Adam(params, lr=OC.p2_lr_W)
    t0=time.perf_counter()
    for step in range(1, OC.p2_steps+1):
        S = torch.randperm(n)[:OC.p2_subm]
        L_total = 0.0
        for m, idx in enumerate(clusters):
            if len(idx)==0: continue
            Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
            # choose experts
            if OC.p2_batchE and OC.p2_batchE < len(idx):
                pick = torch.randperm(len(idx))[:OC.p2_batchE]
                idx_step = [idx[i.item()] for i in pick]
            else:
                idx_step = idx
            # Ws batch (params or frozen)
            Ws_batch = torch.stack([(W_params[e] if isinstance(W_params[e], torch.Tensor) and W_params[e].requires_grad else W_params[e].data) for e in idx_step], dim=0)
            # diag slice
            D = diag_slice_batch(Ws_batch, Uo, Vo, S)     # (E_b, s)
            with torch.no_grad():
                pooled = D.abs().mean(0)
                pool_ema[m][S] = OC.ema_beta*pool_ema[m][S] + (1-OC.ema_beta)*pooled
            # offdiag
            L_off = offdiag_l1_slice_batch(Ws_batch, Uo, Vo, S)
            # group-support (slice)
            L_grp = 0.0
            if pool_ema[m].abs().sum().item()>0:
                K, order = smart_k_from_pool(pool_ema[m].clamp_min(0), OC.target_energy)
                top = order[:K]
                bands = [c for c in torch.chunk(top, OC.bands_B) if len(c)>0]
            else:
                order = torch.argsort(D.abs().mean(0), descending=True)
                Kloc = min(len(order), int(max(1, OC.p2_subm*0.9)))
                top = S[order[:Kloc]]
                bands = [c for c in torch.chunk(top, OC.bands_B) if len(c)>0]
            for g_idx in bands:
                mask = torch.isin(S, g_idx)
                if not mask.any(): continue
                D_band = D[:, mask]
                L_grp = L_grp + (D_band.pow(2).sum(dim=0).sqrt().sum() / (D_band.shape[1] + 1e-12))
            if isinstance(L_grp, float): L_grp = torch.tensor(0.0)

            # sum (only adds gradients to U,V everywhere; weights get grads only if they are nn.Parameter)
            L_total = L_total + OC.lam_off*L_off + OC.lam_grp*L_grp

        # proximity regularizer for trainable weights
        prox = 0.0
        for P,Z in zip(W_params, W0_snap):
            if isinstance(P, nn.Parameter):
                prox = prox + ((P - Z)**2).mean()
        L_total = L_total + OC.lam_prox*prox

        opt.zero_grad(); L_total.backward()
        # re-orthogonalize
        if (step % OC.reortho_every)==0 or step==OC.p2_steps:
            with torch.no_grad():
                for uv in U_par+V_par:
                    if uv is not None: uv.M.copy_(uv.orthogonal())
        opt.step()

        if (step==1) or (step%OC.report_every)==0 or step==OC.p2_steps:
            # sample diag metric
            diag_fracs=[]
            for m, idx in enumerate(clusters):
                if len(idx)==0: continue
                Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                U_S, V_S = Uo[:,S], Vo[:,S]
                e_list = idx[:min(3,len(idx))]
                dsum=0.0; tsum=0.0
                for e in e_list:
                    We = (W_params[e].data if isinstance(W_params[e], nn.Parameter) else W_params[e].data)
                    Xs = U_S.t() @ (We @ V_S)
                    dsum += torch.abs(torch.diag(Xs)).sum().item()
                    tsum += torch.abs(Xs).sum().item()
                diag_fracs.append(dsum/(tsum+1e-12) if tsum>0 else 0.0)
            t1=time.perf_counter()
            print(f"[P2-HYBRID] step {step:3d}/{OC.p2_steps} | diag_frac(sample)≈{float(np.mean(diag_fracs)):.4f}  (+{t1-t0:.1f}s)")
            t0=t1

print("\n=== PHASE 2: SELECTIVE WEIGHT UPDATES ===")
original_phase2_train()

# =============== ORIGINAL PAYLOAD BUILDING ===============

@torch.no_grad()
def diag_in_basis(Ws, U, V):
    E, n, _ = Ws.shape
    out = torch.empty(E, n)
    for e in range(E):
        out[e] = torch.diag(U.t() @ Ws[e] @ V)
    return out

def greedy_blocks_from_residual(R: torch.Tensor, t: int, b: int):
    n = R.shape[0]; blocks=[]; used=torch.zeros(n,n,dtype=torch.bool)
    if t<=0 or b<=0: return blocks
    for _ in range(t):
        best=None; bestE=-1.0
        for i0 in range(0,n,b):
            for j0 in range(0,n,b):
                i1=min(n,i0+b); j1=min(n,j0+b)
                if used[i0:i1,j0:j1].any(): continue
                Eblk = (R[i0:i1,j0:j1]**2).sum().item()
                if Eblk>bestE: bestE=Eblk; best=(i0,j0,i1,j1)
        if best is None: break
        i0,j0,i1,j1=best; used[i0:i1,j0:j1]=True
        blocks.append((i0,j0,R[i0:i1,j0:j1].clone()))
    return blocks

RANK_R, T_BLOCKS, B_SIZE = 96, 12, 32
basis_payloads=[]; basis_of_e={}
print("[ORIGINAL] Building per-basis payloads …")
for m, idx in enumerate(clusters):
    if len(idx)==0:
        basis_payloads.append(None); continue
    Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
    # K from EMA if available; else fresh diag
    if pool_ema[m] is not None and pool_ema[m].abs().sum().item()>0:
        K, order = smart_k_from_pool(pool_ema[m].clamp_min(0), OC.target_energy)
        topK_idx = order[:K]
    else:
        sub = torch.stack([Ws_t[e] for e in idx])
        diag_vals = diag_in_basis(sub, Uo, Vo)
        pool = diag_vals.abs().mean(0)
        K, order = smart_k_from_pool(pool, OC.target_energy)
        topK_idx = order[:K]

    # residuals
    X_list, R_list = [], []
    for j,e in enumerate(idx):
        # use updated param if it exists else original
        We = (W_params[e].data if isinstance(W_params[e], nn.Parameter) else Ws_t[e])
        X = Uo.t() @ We @ Vo
        X_list.append(X)
        Xk = torch.zeros_like(X); d = torch.diag(X); Xk[topK_idx, topK_idx] = d[topK_idx]
        R_list.append(X - Xk)

    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    U_s, s_s, V_s = torch.linalg.svd(Rmean, full_matrices=False)
    DL = U_s[:, :RANK_R].contiguous(); DR = V_s[:, :RANK_R].contiguous()

    codes_c = torch.empty(len(idx), RANK_R); codes_d = torch.empty(len(idx), RANK_R)
    blocks_per_e=[]
    for j in range(len(idx)):
        R = R_list[j]
        gam = torch.sum(DL * (R @ DR), dim=0)
        mag = torch.sqrt(torch.clamp(gam.abs(), min=0.0)); sign = torch.sign(gam + 1e-12)
        c = sign*mag; d = mag
        codes_c[j], codes_d[j] = c, d
        R2 = R - (DL * (c*d).view(1,-1)) @ DR.t()
        blocks_per_e.append(greedy_blocks_from_residual(R2, T_BLOCKS, B_SIZE))

    basis_payloads.append(dict(U=Uo, V=Vo, topK_idx=topK_idx, DL=DL, DR=DR,
                               X_list=X_list, codes_c=codes_c, codes_d=codes_d, blocks=blocks_per_e, K=len(topK_idx)))
    for j,e in enumerate(idx): basis_of_e[e]=(m,j)
    print(f"  - basis {m}: E={len(idx)} | K={len(topK_idx)}/{n} ({len(topK_idx)/n:.1%})")

# =============== ORIGINAL RUNTIME & EVALUATION ===============

class KTXRuntimeMB(nn.Module):
    def __init__(self, basis_payloads, basis_of_e):
        super().__init__(); self.bp=basis_payloads; self.map=basis_of_e
        for b in basis_payloads:
            if b is not None: self.n=b['U'].shape[0]; break
    def forward(self, x, routed, gates):
        y=torch.zeros_like(x); used={}
        for a,e in zip(gates, routed):
            if e in self.map:
                m,j=self.map[e]; used.setdefault(m,[]).append((a,j))
        for m, items in used.items():
            B=self.bp[m]; U,V,DL,DR,top=B['U'],B['V'],B['DL'],B['DR'],B['topK_idx']
            z=x@U; u_acc=torch.zeros_like(z)
            for (a,j) in items:
                d_full=torch.zeros(self.n); d_full[top]=torch.diag(B['X_list'][j])[top]
                u=z*d_full
                gam=B['codes_c'][j]*B['codes_d'][j]
                u = u + ((z@DL)*gam.view(1,-1))@DR.t()
                for (i0,j0,Bb) in B['blocks'][j]:
                    h,w=Bb.shape; u[:, j0:j0+w]+= z[:, i0:i0+h] @ Bb
                u_acc += a*u
            y += u_acc @ V.t()
        return y

def dense_apply(Ws_like, x, routed, gates):
    n = (Ws_like[0].shape[0] if isinstance(Ws_like[0], torch.Tensor) and Ws_like[0].requires_grad else Ws_like[0].data.shape[0])
    Wsum=torch.zeros(n,n)
    for a,e in zip(gates, routed):
        We = (Ws_like[e] if isinstance(Ws_like[e], torch.Tensor) and Ws_like[e].requires_grad else Ws_like[e].data)
        Wsum += a*We
    return x @ Wsum

def eval_runtime(rt_module, W_like, trials=3, routed_k=8, batch=2):
    errs=[]
    for _ in range(trials):
        x=torch.randn(batch, n)
        routed=random.sample(range(E), k=min(routed_k,E))
        gates=torch.rand(len(routed)); gates=gates/gates.sum()
        with torch.no_grad():
            y_hat=rt_module(x, routed, gates)
            y_ref=dense_apply(W_like, x, routed, gates)
            err=float((frob(y_hat - y_ref)/(frob(y_ref)+1e-12)).item())
        errs.append(err)
    print(f"[Eval MB] forward rel-error = {np.mean(errs):.4f} ± {np.std(errs):.4f}")

rt = KTXRuntimeMB(basis_payloads, basis_of_e)
W_like = [ (p if isinstance(p, nn.Parameter) else p) for p in W_params ]
eval_runtime(rt, W_like, trials=3, routed_k=8, batch=2)

print("\n[ORIGINAL APPROACH COMPLETE]")

[KT++-X ORIGINAL APPROACH] Using Ws: (8, 2048, 2048) on cpu
[ORIGINAL] Clustering to M=6 ...
[ORIGINAL] cluster sizes: [1, 3, 1, 1, 1, 1]

=== PHASE 1: UV WARMUP ===
[P1-UV] step   1/12 | diag_frac(sample)≈0.0096  (+10.6s)
[P1-UV] step   4/12 | diag_frac(sample)≈0.0821  (+20.5s)
[P1-UV] step   8/12 | diag_frac(sample)≈0.1033  (+25.6s)
[P1-UV] step  12/12 | diag_frac(sample)≈0.0697  (+26.0s)
[ORIGINAL] worst clusters selected for weight updates: [1, 3, 4]

=== PHASE 2: SELECTIVE WEIGHT UPDATES ===
[P2-HYBRID] step   1/40 | diag_frac(sample)≈0.0358  (+8.2s)
[P2-HYBRID] step   4/40 | diag_frac(sample)≈0.0260  (+20.8s)
[P2-HYBRID] step   8/40 | diag_frac(sample)≈0.0268  (+25.9s)
[P2-HYBRID] step  12/40 | diag_frac(sample)≈0.0256  (+25.9s)
[P2-HYBRID] step  16/40 | diag_frac(sample)≈0.0266  (+26.1s)
[P2-HYBRID] step  20/40 | diag_frac(sample)≈0.0240  (+25.9s)
[P2-HYBRID] step  24/40 | diag_frac(sample)≈0.0283  (+26.0s)
[P2-HYBRID] step  28/40 | diag_frac(sample)≈0.0311  (+26.1s)
[P2-HYBRID]

In [6]:
# ============================================================
# OFFLINE DeepSeek-MoE → build Ws (W_eff = down @ up) → KT++-X
# Single-cell / single-script end-to-end runner
# ============================================================

import os, re, json, math, time, random, glob
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np

# ------------------- threading / determinism -------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))

SEED = int(os.environ.get("KTXX_SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)

# ------------------- torch (optional but recommended) -------------------
try:
    import torch
    import torch.nn as nn
    torch.manual_seed(SEED)
    try:
        torch.set_num_threads(NTHREADS)
    except Exception:
        pass
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    raise RuntimeError("This script requires torch. Install torch first.") from e

DEVICE = torch.device("cpu")

# ------------------- safetensors -------------------
try:
    from safetensors.torch import safe_open
except Exception as e:
    raise RuntimeError("Missing safetensors. Install: pip install safetensors") from e

# ------------------- tqdm progress bar (optional) -------------------
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # fallback
        return x

# ============================================================
# CONFIG (edit these)
# ============================================================
MODEL_DIR  = os.environ.get("MODEL_DIR",  "/home/daniyar/deepseek-model")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

LAYER      = int(os.environ.get("LAYER", "1"))
MAX_EXPERTS = int(os.environ.get("MAX_EXPERTS", "8"))  # how many experts to load
DTYPE_SAVE = os.environ.get("DTYPE_SAVE", "float16")   # float16 recommended for cache

# KT++ config defaults (override via env if you want)
DEFAULT_M_CLUSTERS = int(os.environ.get("KTXX_M", "2"))     # IMPORTANT: avoid many singleton clusters
TARGET_ENERGY = float(os.environ.get("KTXX_TARGET_ENERGY", "0.995"))
BANDS_B = int(os.environ.get("KTXX_BANDS_B", "8"))

# Payload config (increase if rel-error is huge)
RANK_R   = int(os.environ.get("KTXX_RANK_R", "192"))
T_BLOCKS = int(os.environ.get("KTXX_T_BLOCKS", "24"))
B_SIZE   = int(os.environ.get("KTXX_B_SIZE", "32"))

# ============================================================
# Utilities
# ============================================================
def _mb(x_bytes: int) -> float:
    return x_bytes / (1024**2)

def _ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def _load_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _pick_dtype(dtype_str: str):
    s = dtype_str.lower().strip()
    if s in ("fp16", "float16", "half"):
        return np.float16
    if s in ("fp32", "float32", "float"):
        return np.float32
    raise ValueError(f"Unsupported DTYPE_SAVE={dtype_str}")

def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def hadamard_like_Q(n: int) -> torch.Tensor:
    """Stable-ish random orthogonal init via Householder + QR."""
    H = torch.eye(n, dtype=torch.float32)
    steps = int(math.log2(max(n, 2))) + 1
    for _ in range(steps):
        v = torch.randn(n)
        v = v / (v.norm() + 1e-12)
        H = H - 2.0 * (H @ v).unsqueeze(1) @ v.unsqueeze(0)
    Q, _ = torch.linalg.qr(H)
    return Q.contiguous()

# ============================================================
# OFFLINE shard/index loader
# ============================================================
def _find_index_json(model_dir: str) -> str:
    cand = os.path.join(model_dir, "model.safetensors.index.json")
    if os.path.exists(cand):
        return cand
    # fallback: any *.index.json
    xs = glob.glob(os.path.join(model_dir, "*.index.json"))
    if xs:
        return xs[0]
    raise FileNotFoundError(f"Cannot find safetensors index json in {model_dir}")

def _resolve_weight_map(model_dir: str) -> Dict[str, str]:
    idx_path = _find_index_json(model_dir)
    idx = _load_json(idx_path)
    if "weight_map" not in idx:
        raise RuntimeError(f"Index missing weight_map: {idx_path}")
    return idx["weight_map"]

def _open_tensor_from_shard(model_dir: str, shard_name: str, key: str) -> torch.Tensor:
    shard_path = os.path.join(model_dir, shard_name)
    if not os.path.exists(shard_path):
        raise FileNotFoundError(f"Missing shard file: {shard_path}")
    with safe_open(shard_path, framework="pt", device="cpu") as f:
        if key not in f.keys():
            raise KeyError(f"Key not found in shard: {key} (shard={shard_name})")
        return f.get_tensor(key)

def _keys_for_layer_expert(weight_map: Dict[str, str], layer: int, expert: int) -> List[str]:
    prefix = f"model.layers.{layer}.mlp.experts.{expert}."
    return [k for k in weight_map.keys() if k.startswith(prefix)]

def _match_key(cands: List[str], patterns: List[str]) -> Optional[str]:
    """
    Choose the first candidate whose name matches any regex in patterns (in order).
    """
    for pat in patterns:
        rx = re.compile(pat)
        for k in cands:
            if rx.search(k):
                return k
    return None

def _infer_up_down_keys(weight_map: Dict[str, str], layer: int, expert: int) -> Tuple[str, str]:
    """
    Robustly find:
      up  : up_proj.weight or w1.weight or gate_up_proj.weight (packed)
      down: down_proj.weight or w2.weight

    Returns (up_key, down_key).
    """
    cands = _keys_for_layer_expert(weight_map, layer, expert)

    # Typical HF DeepSeek-style
    up_key = _match_key(cands, [
        r"\.up_proj\.weight$",
        r"\.w1\.weight$",
        r"\.gate_up_proj\.weight$",
    ])
    down_key = _match_key(cands, [
        r"\.down_proj\.weight$",
        r"\.w2\.weight$",
    ])

    if up_key is None or down_key is None:
        # Print what we saw to help debugging
        raise RuntimeError(
            f"Could not infer up/down keys for layer={layer} expert={expert}.\n"
            f"Candidates:\n  - " + "\n  - ".join(cands[:50])
        )
    return up_key, down_key

def build_Ws_from_model_dir(
    model_dir: str,
    out_dir: str,
    layer: int,
    max_experts: int,
    dtype_save: str = "float16",
) -> Tuple[torch.Tensor, List[int], str]:
    """
    Build Ws = stack_e (W_eff_e), where W_eff = down @ up.
    Saves an .npz cache and returns (Ws_t, expert_ids, cache_path).
    """
    _ensure_dir(out_dir)
    cache_path = os.path.join(out_dir, f"deepseek_layer{layer}_Ws_first{max_experts}.npz")

    if os.path.exists(cache_path):
        print(f"[cache] Loading existing Ws cache: {cache_path} ({_mb(os.path.getsize(cache_path)):.2f} MB)")
        data = np.load(cache_path, allow_pickle=False)
        Ws_np = data["Ws"]
        expert_ids = data["expert_ids"].tolist()
        Ws_t = torch.from_numpy(Ws_np).to(dtype=torch.float32).contiguous()
        return Ws_t, expert_ids, cache_path

    print(f"[build] MODEL_DIR:  {model_dir}")
    print(f"[build] OUTPUT_DIR: {out_dir}")
    print(f"[build] LAYER={layer} | MAX_EXPERTS={max_experts} | DTYPE_SAVE={dtype_save}")
    print("[build] Reading index + resolving weight map...")

    weight_map = _resolve_weight_map(model_dir)

    # Determine total experts available by scanning keys
    rx = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    experts_found = sorted({int(rx.match(k).group(1)) for k in weight_map.keys() if rx.match(k)})
    if not experts_found:
        raise RuntimeError(f"No experts found for layer={layer}. Check layer index or model format.")

    total_experts = max(experts_found) + 1
    expert_ids = list(range(min(max_experts, total_experts)))

    print(f"[build] Found experts for layer {layer}: total_experts={total_experts} | using={len(expert_ids)}")
    print("[build] Loading up/down weights and computing W_eff = down @ up ...")

    Ws_list: List[torch.Tensor] = []
    diag_stats = []

    for e in tqdm(expert_ids, desc="experts", total=len(expert_ids)):
        up_key, down_key = _infer_up_down_keys(weight_map, layer, e)
        up_shard = weight_map[up_key]
        down_shard = weight_map[down_key]

        W_up = _open_tensor_from_shard(model_dir, up_shard, up_key).to(dtype=torch.float32)
        W_down = _open_tensor_from_shard(model_dir, down_shard, down_key).to(dtype=torch.float32)

        # Expect shapes like:
        #   up:   (intermediate, hidden)  e.g. (1408, 2048)
        #   down: (hidden, intermediate)  e.g. (2048, 1408)
        if W_up.ndim != 2 or W_down.ndim != 2:
            raise RuntimeError(f"Expected 2D tensors for up/down. Got: up={W_up.shape}, down={W_down.shape}")

        if W_down.shape[1] != W_up.shape[0]:
            raise RuntimeError(
                f"Shape mismatch for expert {e}: down={tuple(W_down.shape)} up={tuple(W_up.shape)} "
                f"(need down.cols == up.rows)"
            )

        W_eff = (W_down @ W_up).contiguous()  # (hidden, hidden)
        if W_eff.shape[0] != W_eff.shape[1]:
            raise RuntimeError(f"W_eff not square for expert {e}: {tuple(W_eff.shape)}")

        # quick diagnostics
        with torch.no_grad():
            d = torch.diag(W_eff)
            diag_stats.append((float(d.abs().mean().item()), float(W_eff.abs().mean().item())))

        Ws_list.append(W_eff)

    Ws_t = torch.stack(Ws_list, dim=0).contiguous()  # [E,n,n]
    E, n, _ = Ws_t.shape
    print(f"[build] Built Ws_t: {tuple(Ws_t.shape)} (float32 in memory)")

    # save cache
    np_dtype = _pick_dtype(dtype_save)
    Ws_np = Ws_t.to(dtype=torch.float16 if np_dtype==np.float16 else torch.float32).cpu().numpy()
    np.savez_compressed(cache_path, Ws=Ws_np, expert_ids=np.array(expert_ids, dtype=np.int32),
                        layer=np.array([layer], dtype=np.int32), n=np.array([n], dtype=np.int32))
    print(f"[build] Saved cache: {cache_path} ({_mb(os.path.getsize(cache_path)):.2f} MB)")

    # print a couple stats
    m_diag = float(np.mean([a for a,_ in diag_stats]))
    m_all  = float(np.mean([b for _,b in diag_stats]))
    print(f"[build] Mean(|diag|)≈{m_diag:.4f} | Mean(|W|)≈{m_all:.4f} | diag/abs≈{(m_diag/(m_all+1e-12)):.4f}")

    return Ws_t, expert_ids, cache_path

# ============================================================
# KT++-X "Original" with some practical fixes
# ============================================================
def random_projection_features(Ws: torch.Tensor, d: int = 64, seed: int = SEED) -> torch.Tensor:
    """
    Feature for clustering experts.
    Uses fast random sign projection of row/col energy profiles.
    """
    rng = np.random.default_rng(seed)
    E, n, _ = Ws.shape
    R = torch.from_numpy(rng.choice([-1.0, 1.0], size=(n, d)).astype(np.float32))
    feats = []
    for e in range(E):
        W = Ws[e]
        row_e = torch.diag(W @ W.t())
        col_e = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row_e @ R).unsqueeze(0), (col_e @ R).unsqueeze(0)], dim=1))
    return torch.cat(feats, dim=0)

def kmeans_torch(X: torch.Tensor, k: int, iters: int = 60, seed: int = SEED) -> Tuple[torch.Tensor, torch.Tensor]:
    g = torch.Generator().manual_seed(int(seed))
    idx = torch.randperm(X.shape[0], generator=g)[:k]
    C = X[idx].clone()
    for _ in range(iters):
        dist = torch.cdist(X, C)
        lab = dist.argmin(dim=1)
        for j in range(k):
            m = (lab == j)
            if m.any():
                C[j] = X[m].mean(dim=0)
    return lab, C

class OrthoParam(nn.Module):
    def __init__(self, M_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(M_init.clone())

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def smart_k_from_pool(pool: torch.Tensor, target: float) -> Tuple[int, torch.Tensor]:
    order = torch.argsort(pool, descending=True)
    csum = torch.cumsum(pool[order], dim=0)
    frac = csum / (pool.sum() + 1e-12)
    if (frac >= target).any():
        K = int(torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1)
    else:
        K = pool.numel()
    return K, order

def diag_slice_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    V_S = V[:, S]
    T = torch.matmul(Ws_batch, V_S)
    U_S = U[:, S]
    Xs = torch.matmul(T.transpose(1, 2), U_S)
    return torch.diagonal(Xs, dim1=1, dim2=2)

def offdiag_l1_slice_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    V_S = V[:, S]
    T = torch.matmul(Ws_batch, V_S)
    U_S = U[:, S]
    Xs = torch.matmul(T.transpose(1, 2), U_S)
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    X_off = Xs - torch.diag_embed(D)
    return X_off.abs().mean()

def diag_frac_sample(Ws_t: torch.Tensor, clusters: List[List[int]], U_par: List[Optional[OrthoParam]], V_par: List[Optional[OrthoParam]], S: torch.Tensor, samples_per_cluster: int = 2) -> float:
    fracs = []
    for m, idx in enumerate(clusters):
        if not idx:
            continue
        Uo = U_par[m].orthogonal()
        Vo = V_par[m].orthogonal()
        U_S, V_S = Uo[:, S], Vo[:, S]
        e_list = idx[:min(samples_per_cluster, len(idx))]
        dsum, tsum = 0.0, 0.0
        for e in e_list:
            Xs = U_S.t() @ (Ws_t[e] @ V_S)
            dsum += torch.abs(torch.diag(Xs)).sum().item()
            tsum += torch.abs(Xs).sum().item()
        fracs.append(dsum / (tsum + 1e-12))
    return float(np.mean(fracs)) if fracs else 0.0

@dataclass
class KTConfig:
    M: int = DEFAULT_M_CLUSTERS
    target_energy: float = TARGET_ENERGY
    bands_B: int = BANDS_B

    # Phase 1 (UV warmup)
    p1_steps: int = int(os.environ.get("KTXX_P1_STEPS", "16"))
    p1_subm: int = int(os.environ.get("KTXX_P1_SUBM", "192"))
    p1_batchE: int = int(os.environ.get("KTXX_P1_BATCHE", "4"))
    p1_lr: float = float(os.environ.get("KTXX_P1_LR", "5e-2"))

    # Phase 2 (optional)
    p2_steps: int = int(os.environ.get("KTXX_P2_STEPS", "0"))   # set >0 if you really want it
    p2_subm: int = int(os.environ.get("KTXX_P2_SUBM", "256"))
    p2_batchE: int = int(os.environ.get("KTXX_P2_BATCHE", "4"))
    p2_lr_UV: float = float(os.environ.get("KTXX_P2_LR_UV", "1e-2"))
    p2_lr_W:  float = float(os.environ.get("KTXX_P2_LR_W", "5e-4"))
    worst_ratio: float = float(os.environ.get("KTXX_WORST_RATIO", "0.5"))

    # Loss weights
    lam_off: float = float(os.environ.get("KTXX_LAM_OFF", "2e-3"))
    lam_grp: float = float(os.environ.get("KTXX_LAM_GRP", "5e-4"))
    lam_prox: float = float(os.environ.get("KTXX_LAM_PROX", "5e-4"))

    report_every: int = int(os.environ.get("KTXX_REPORT_EVERY", "4"))
    reortho_every: int = int(os.environ.get("KTXX_REORTHO_EVERY", "2"))
    ema_beta: float = float(os.environ.get("KTXX_EMA_BETA", "0.9"))
    bands_update_every: int = int(os.environ.get("KTXX_BANDS_UPDATE_EVERY", "8"))

OC = KTConfig()

# ============================================================
# Payload + runtime (your original form, but parameterized)
# ============================================================
@torch.no_grad()
def diag_in_basis(Ws: torch.Tensor, U: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    E, n, _ = Ws.shape
    out = torch.empty(E, n)
    for e in range(E):
        out[e] = torch.diag(U.t() @ Ws[e] @ V)
    return out

def greedy_blocks_from_residual(R: torch.Tensor, t: int, b: int) -> List[Tuple[int,int,torch.Tensor]]:
    n = R.shape[0]
    blocks = []
    used = torch.zeros(n, n, dtype=torch.bool)
    if t <= 0 or b <= 0:
        return blocks
    for _ in range(t):
        best = None
        bestE = -1.0
        for i0 in range(0, n, b):
            for j0 in range(0, n, b):
                i1 = min(n, i0 + b)
                j1 = min(n, j0 + b)
                if used[i0:i1, j0:j1].any():
                    continue
                Eblk = (R[i0:i1, j0:j1] ** 2).sum().item()
                if Eblk > bestE:
                    bestE = Eblk
                    best = (i0, j0, i1, j1)
        if best is None:
            break
        i0, j0, i1, j1 = best
        used[i0:i1, j0:j1] = True
        blocks.append((i0, j0, R[i0:i1, j0:j1].clone()))
    return blocks

class KTXRuntimeMB(nn.Module):
    def __init__(self, basis_payloads, basis_of_e):
        super().__init__()
        self.bp = basis_payloads
        self.map = basis_of_e
        self.n = None
        for b in basis_payloads:
            if b is not None:
                self.n = b["U"].shape[0]
                break
        if self.n is None:
            raise RuntimeError("No basis payloads were built.")

    def forward(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        used: Dict[int, List[Tuple[float,int]]] = {}
        for a, e in zip(gates.tolist(), routed):
            if e in self.map:
                m, j = self.map[e]
                used.setdefault(m, []).append((a, j))

        for m, items in used.items():
            B = self.bp[m]
            U, V, DL, DR, top = B["U"], B["V"], B["DL"], B["DR"], B["topK_idx"]
            z = x @ U
            u_acc = torch.zeros_like(z)
            for (a, j) in items:
                # diag part (only top-K)
                d_full = torch.zeros(self.n)
                d_full[top] = torch.diag(B["X_list"][j])[top]
                u = z * d_full

                # low-rank part
                gam = B["codes_c"][j] * B["codes_d"][j]
                u = u + ((z @ DL) * gam.view(1, -1)) @ DR.t()

                # sparse blocks
                for (i0, j0, Bb) in B["blocks"][j]:
                    h, w = Bb.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb

                u_acc += a * u

            y += u_acc @ V.t()

        return y

def dense_apply(Ws: torch.Tensor, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
    n = Ws.shape[1]
    Wsum = torch.zeros(n, n)
    for a, e in zip(gates.tolist(), routed):
        Wsum += a * Ws[e]
    return x @ Wsum

def eval_runtime(rt_module: nn.Module, Ws: torch.Tensor, trials: int = 3, routed_k: int = 8, batch: int = 2) -> Tuple[float,float]:
    E = Ws.shape[0]
    n = Ws.shape[1]
    errs = []
    for _ in range(trials):
        x = torch.randn(batch, n)
        routed = random.sample(range(E), k=min(routed_k, E))
        gates = torch.rand(len(routed))
        gates = gates / gates.sum()
        with torch.no_grad():
            y_hat = rt_module(x, routed, gates)
            y_ref = dense_apply(Ws, x, routed, gates)
            err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)
    return float(np.mean(errs)), float(np.std(errs))

# ============================================================
# MAIN
# ============================================================
print("== OFFLINE DeepSeek-MoE KT++-X Runner ==")
print(f"MODEL_DIR:  {MODEL_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"LAYER:      {LAYER}")
print(f"MAX_EXPERTS:{MAX_EXPERTS}")
print(f"TORCH:      {torch.__version__} | threads={NTHREADS} | device={DEVICE}")

_ensure_dir(OUTPUT_DIR)

# 1) Build/load Ws_t
Ws_t, expert_ids, cache_path = build_Ws_from_model_dir(
    model_dir=MODEL_DIR,
    out_dir=OUTPUT_DIR,
    layer=LAYER,
    max_experts=MAX_EXPERTS,
    dtype_save=DTYPE_SAVE,
)
Ws_t = Ws_t.to(dtype=torch.float32).contiguous()
E, n, _ = Ws_t.shape
print(f"[KT++-X] Using Ws_t: {tuple(Ws_t.shape)}")

# 2) Clustering (avoid many singleton clusters)
M = min(max(1, OC.M), E)  # can't exceed E
# If M is too big, force down to reduce singletons
if M > max(2, E // 2) and E >= 4:
    M = max(2, E // 2)
print(f"[KT++-X] Clustering experts: E={E} → M={M}")

Xfeat = random_projection_features(Ws_t, d=64, seed=SEED)
labels, _ = kmeans_torch(Xfeat, k=M, iters=60, seed=SEED)
clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
print("[KT++-X] cluster sizes:", [len(c) for c in clusters])

# 3) Initialize U/V per cluster + EMA pools
U_par: List[Optional[OrthoParam]] = []
V_par: List[Optional[OrthoParam]] = []
pool_ema: List[Optional[torch.Tensor]] = []

for m, idx in enumerate(clusters):
    if not idx:
        U_par.append(None); V_par.append(None); pool_ema.append(None)
        continue
    U0 = hadamard_like_Q(n)
    V0 = hadamard_like_Q(n)
    U_par.append(OrthoParam(U0))
    V_par.append(OrthoParam(V0))
    pool_ema.append(torch.zeros(n))

# 4) Phase 1 training (UV only) — with LR group fix and projection timing fix
def phase1_train():
    params_uv = [p.M for p in (U_par + V_par) if p is not None]
    opt = torch.optim.Adam([{"params": params_uv, "lr": OC.p1_lr}])

    t0 = time.perf_counter()
    for step in range(1, OC.p1_steps + 1):
        S = torch.randperm(n)[:min(OC.p1_subm, n)]
        L_total = torch.tensor(0.0)

        for m, idx in enumerate(clusters):
            if not idx:
                continue
            Uo = U_par[m].orthogonal()
            Vo = V_par[m].orthogonal()

            if OC.p1_batchE and OC.p1_batchE < len(idx):
                pick = torch.randperm(len(idx))[:OC.p1_batchE]
                idx_step = [idx[i.item()] for i in pick]
            else:
                idx_step = idx

            Ws_batch = torch.stack([Ws_t[e] for e in idx_step], dim=0)

            D = diag_slice_batch(Ws_batch, Uo, Vo, S)
            with torch.no_grad():
                pooled = D.abs().mean(0)
                pool_ema[m][S] = OC.ema_beta * pool_ema[m][S] + (1 - OC.ema_beta) * pooled

            L_off = offdiag_l1_slice_batch(Ws_batch, Uo, Vo, S)

            # group-support slice
            if (step % OC.bands_update_every) == 0 and pool_ema[m].abs().sum().item() > 0:
                K, order = smart_k_from_pool(pool_ema[m].clamp_min(0), OC.target_energy)
                top = order[:K]
                bands = [c for c in torch.chunk(top, OC.bands_B) if len(c) > 0]
            else:
                order = torch.argsort(D.abs().mean(0), descending=True)
                Kloc = min(len(order), int(max(1, len(S) * 0.9)))
                top = S[order[:Kloc]]
                bands = [c for c in torch.chunk(top, OC.bands_B) if len(c) > 0]

            L_grp = torch.tensor(0.0)
            for g_idx in bands:
                mask = torch.isin(S, g_idx)
                if not mask.any():
                    continue
                D_band = D[:, mask]
                L_grp = L_grp + (D_band.pow(2).sum(dim=0).sqrt().sum() / (D_band.shape[1] + 1e-12))

            L_total = L_total + OC.lam_off * L_off + OC.lam_grp * L_grp

        opt.zero_grad()
        L_total.backward()
        opt.step()

        # projection after step (important)
        if (step % OC.reortho_every) == 0 or step == OC.p1_steps:
            with torch.no_grad():
                for uv in (U_par + V_par):
                    if uv is not None:
                        uv.M.copy_(uv.orthogonal())

        if step == 1 or (step % OC.report_every) == 0 or step == OC.p1_steps:
            t1 = time.perf_counter()
            diag_frac = diag_frac_sample(Ws_t, clusters, U_par, V_par, S, samples_per_cluster=2)
            print(f"[P1] step {step:3d}/{OC.p1_steps} | diag_frac(sample)≈{diag_frac:.4f} | (+{t1-t0:.1f}s)")
            t0 = t1

print("\n=== PHASE 1: UV WARMUP ===")
phase1_train()

# 5) (Optional) Phase 2 selective weight updates — disabled by default (p2_steps=0)
#    Keeping your structure, but with proper LR groups if you enable it.
W_params = [Ws_t[e].detach().clone() for e in range(E)]  # default: frozen copy

if OC.p2_steps > 0:
    # choose worst clusters
    with torch.no_grad():
        S_probe = torch.randperm(n)[:max(128, min(OC.p1_subm, n))]
        scores = []
        for m, idx in enumerate(clusters):
            if not idx:
                scores.append((m, 1e9))
                continue
            Uo = U_par[m].orthogonal()
            Vo = V_par[m].orthogonal()
            U_S, V_S = Uo[:, S_probe], Vo[:, S_probe]
            e_list = idx[:min(3, len(idx))]
            dsum, tsum = 0.0, 0.0
            for e in e_list:
                Xs = U_S.t() @ (Ws_t[e] @ V_S)
                dsum += torch.abs(torch.diag(Xs)).sum().item()
                tsum += torch.abs(Xs).sum().item()
            frac = dsum / (tsum + 1e-12)
            scores.append((m, frac))
        worst_sorted = sorted(scores, key=lambda kv: kv[1])
        keep_worst = set([m for m, _ in worst_sorted[:max(1, int(OC.worst_ratio * len(clusters)))]])
        print("[P2] worst clusters selected:", sorted(list(keep_worst)))

    # make params
    trainable = []
    frozen = []
    for e in range(E):
        m_of_e = None
        for m, idx in enumerate(clusters):
            if e in idx:
                m_of_e = m
                break
        if m_of_e in keep_worst:
            trainable.append(e)
        else:
            frozen.append(e)

    W_param_list: List[Any] = []
    for e in range(E):
        if e in trainable:
            W_param_list.append(nn.Parameter(Ws_t[e].detach().clone()))
        else:
            W_param_list.append(Ws_t[e].detach().clone())  # frozen tensor

    W0_snap = [p.detach().clone() if isinstance(p, nn.Parameter) else p.clone() for p in W_param_list]

    params_uv = [p.M for p in (U_par + V_par) if p is not None]
    params_w  = [p for p in W_param_list if isinstance(p, nn.Parameter)]

    opt = torch.optim.Adam([
        {"params": params_uv, "lr": OC.p2_lr_UV},
        {"params": params_w,  "lr": OC.p2_lr_W},
    ])

    print("\n=== PHASE 2: SELECTIVE WEIGHT UPDATES ===")
    t0 = time.perf_counter()
    for step in range(1, OC.p2_steps + 1):
        S = torch.randperm(n)[:min(OC.p2_subm, n)]
        L_total = torch.tensor(0.0)

        for m, idx in enumerate(clusters):
            if not idx:
                continue
            Uo = U_par[m].orthogonal()
            Vo = V_par[m].orthogonal()

            if OC.p2_batchE and OC.p2_batchE < len(idx):
                pick = torch.randperm(len(idx))[:OC.p2_batchE]
                idx_step = [idx[i.item()] for i in pick]
            else:
                idx_step = idx

            Ws_batch = torch.stack(
                [(W_param_list[e] if isinstance(W_param_list[e], torch.Tensor) else W_param_list[e].data) for e in idx_step],
                dim=0
            )

            D = diag_slice_batch(Ws_batch, Uo, Vo, S)
            with torch.no_grad():
                pooled = D.abs().mean(0)
                pool_ema[m][S] = OC.ema_beta * pool_ema[m][S] + (1 - OC.ema_beta) * pooled

            L_off = offdiag_l1_slice_batch(Ws_batch, Uo, Vo, S)

            if pool_ema[m].abs().sum().item() > 0:
                K, order = smart_k_from_pool(pool_ema[m].clamp_min(0), OC.target_energy)
                top = order[:K]
                bands = [c for c in torch.chunk(top, OC.bands_B) if len(c) > 0]
            else:
                order = torch.argsort(D.abs().mean(0), descending=True)
                Kloc = min(len(order), int(max(1, len(S) * 0.9)))
                top = S[order[:Kloc]]
                bands = [c for c in torch.chunk(top, OC.bands_B) if len(c) > 0]

            L_grp = torch.tensor(0.0)
            for g_idx in bands:
                mask = torch.isin(S, g_idx)
                if not mask.any():
                    continue
                D_band = D[:, mask]
                L_grp = L_grp + (D_band.pow(2).sum(dim=0).sqrt().sum() / (D_band.shape[1] + 1e-12))

            L_total = L_total + OC.lam_off * L_off + OC.lam_grp * L_grp

        # prox on trainable weights
        prox = torch.tensor(0.0)
        for P, Z in zip(W_param_list, W0_snap):
            if isinstance(P, nn.Parameter):
                prox = prox + ((P - Z) ** 2).mean()
        L_total = L_total + OC.lam_prox * prox

        opt.zero_grad()
        L_total.backward()
        opt.step()

        if (step % OC.reortho_every) == 0 or step == OC.p2_steps:
            with torch.no_grad():
                for uv in (U_par + V_par):
                    if uv is not None:
                        uv.M.copy_(uv.orthogonal())

        if step == 1 or (step % OC.report_every) == 0 or step == OC.p2_steps:
            t1 = time.perf_counter()
            diag_frac = diag_frac_sample(Ws_t, clusters, U_par, V_par, S, samples_per_cluster=2)
            print(f"[P2] step {step:3d}/{OC.p2_steps} | diag_frac(sample)≈{diag_frac:.4f} | (+{t1-t0:.1f}s)")
            t0 = t1

    # finalize Ws_t (use updated weights where applicable)
    Ws_new = []
    for e in range(E):
        if isinstance(W_param_list[e], nn.Parameter):
            Ws_new.append(W_param_list[e].detach().clone())
        else:
            Ws_new.append(W_param_list[e].detach().clone())
    Ws_t = torch.stack(Ws_new, dim=0).contiguous()

# 6) Diagnostics: how diagonalizable is it?
print("\n=== DIAGNOSTICS (full-matrix, per-cluster samples) ===")
with torch.no_grad():
    for m, idx in enumerate(clusters):
        if not idx:
            continue
        Uo = U_par[m].orthogonal()
        Vo = V_par[m].orthogonal()
        e = idx[0]
        X = Uo.t() @ Ws_t[e] @ Vo
        diag_energy = float((torch.diag(X) ** 2).sum().item())
        tot_energy  = float((X ** 2).sum().item())
        frac = diag_energy / (tot_energy + 1e-12)
        off_frac = 1.0 - frac
        print(f"  cluster {m}: size={len(idx)} | sample expert={e} | diag_energy_frac={frac:.4f} | offdiag_frac={off_frac:.4f}")

# 7) Build payloads and runtime evaluation
print("\n=== PAYLOAD BUILDING (diag + lowrank + blocks) ===")
basis_payloads = []
basis_of_e: Dict[int, Tuple[int,int]] = {}

for m, idx in enumerate(clusters):
    if not idx:
        basis_payloads.append(None)
        continue

    Uo = U_par[m].orthogonal()
    Vo = V_par[m].orthogonal()

    # select K based on pooled diagonal magnitudes (same as your original)
    if pool_ema[m] is not None and pool_ema[m].abs().sum().item() > 0:
        K, order = smart_k_from_pool(pool_ema[m].clamp_min(0), OC.target_energy)
        topK_idx = order[:K]
    else:
        sub = torch.stack([Ws_t[e] for e in idx])
        diag_vals = diag_in_basis(sub, Uo, Vo)
        pool = diag_vals.abs().mean(0)
        K, order = smart_k_from_pool(pool, OC.target_energy)
        topK_idx = order[:K]

    X_list, R_list = [], []
    for e in idx:
        X = Uo.t() @ Ws_t[e] @ Vo
        X_list.append(X)
        Xk = torch.zeros_like(X)
        d = torch.diag(X)
        Xk[topK_idx, topK_idx] = d[topK_idx]
        R_list.append(X - Xk)

    Rmean = torch.stack(R_list, dim=0).mean(dim=0)

    # low-rank basis from mean residual
    U_s, s_s, V_s = torch.linalg.svd(Rmean, full_matrices=False)
    DL = U_s[:, :RANK_R].contiguous()
    DR = V_s[:, :RANK_R].contiguous()

    codes_c = torch.empty(len(idx), RANK_R)
    codes_d = torch.empty(len(idx), RANK_R)
    blocks_per_e = []

    for j in range(len(idx)):
        R = R_list[j]
        gam = torch.sum(DL * (R @ DR), dim=0)
        mag = torch.sqrt(torch.clamp(gam.abs(), min=0.0))
        sign = torch.sign(gam + 1e-12)
        c = sign * mag
        d = mag
        codes_c[j], codes_d[j] = c, d

        R2 = R - (DL * (c * d).view(1, -1)) @ DR.t()
        blocks_per_e.append(greedy_blocks_from_residual(R2, T_BLOCKS, B_SIZE))

    basis_payloads.append(dict(
        U=Uo, V=Vo, topK_idx=topK_idx, DL=DL, DR=DR,
        X_list=X_list, codes_c=codes_c, codes_d=codes_d,
        blocks=blocks_per_e, K=len(topK_idx),
    ))

    for j, e in enumerate(idx):
        basis_of_e[e] = (m, j)

    print(f"  - basis {m}: E={len(idx)} | K={len(topK_idx)}/{n} ({len(topK_idx)/n:.1%}) | RANK_R={RANK_R} | T_BLOCKS={T_BLOCKS}")

rt = KTXRuntimeMB(basis_payloads, basis_of_e)
mean_err, std_err = eval_runtime(rt, Ws_t, trials=3, routed_k=min(8, E), batch=2)
print(f"\n[Eval MB] forward rel-error = {mean_err:.4f} ± {std_err:.4f}")

# 8) Save a compact artifact (optional)
artifact_path = os.path.join(OUTPUT_DIR, f"ktxx_layer{LAYER}_E{E}_M{M}_payload.npz")
# Note: saving all tensors in payloads can be huge; instead store U/V + topK indices for now
to_save = {}
for m, B in enumerate(basis_payloads):
    if B is None:
        continue
    to_save[f"U_{m}"] = B["U"].cpu().numpy().astype(np.float16)
    to_save[f"V_{m}"] = B["V"].cpu().numpy().astype(np.float16)
    to_save[f"topK_{m}"] = B["topK_idx"].cpu().numpy().astype(np.int32)
np.savez_compressed(artifact_path, **to_save)
print(f"[save] Saved compact (U,V,topK) payload: {artifact_path} ({_mb(os.path.getsize(artifact_path)):.2f} MB)")

print("\n✅ DONE: Offline Ws built + KT++-X run complete.")
print("Tips:")
print("  • If rel-error stays ~1.0, your diagonal+lowrank+blocks approximation is too weak for these Ws.")
print("    Try increasing: KTXX_RANK_R, KTXX_T_BLOCKS, or rethink the target structure (not-diagonal).")
print("  • Avoid M=6 when E=8; it creates many singletons and hurts shared-basis learning.")


== OFFLINE DeepSeek-MoE KT++-X Runner ==
MODEL_DIR:  /home/daniyar/deepseek-model
OUTPUT_DIR: /home/daniyar/moe_ws_outputs
LAYER:      1
MAX_EXPERTS:8
TORCH:      2.4.1+cpu | threads=8 | device=cpu
[cache] Loading existing Ws cache: /home/daniyar/moe_ws_outputs/deepseek_layer1_Ws_first8.npz (59.04 MB)
[KT++-X] Using Ws_t: (8, 2048, 2048)
[KT++-X] Clustering experts: E=8 → M=2
[KT++-X] cluster sizes: [3, 5]

=== PHASE 1: UV WARMUP ===
[P1] step   1/16 | diag_frac(sample)≈0.0104 | (+2.1s)
[P1] step   4/16 | diag_frac(sample)≈0.0403 | (+8.1s)
[P1] step   8/16 | diag_frac(sample)≈0.0349 | (+10.0s)
[P1] step  12/16 | diag_frac(sample)≈0.0277 | (+10.2s)
[P1] step  16/16 | diag_frac(sample)≈0.0273 | (+9.9s)

=== DIAGNOSTICS (full-matrix, per-cluster samples) ===
  cluster 0: size=3 | sample expert=2 | diag_energy_frac=0.0115 | offdiag_frac=0.9885
  cluster 1: size=5 | sample expert=0 | diag_energy_frac=0.0080 | offdiag_frac=0.9920

=== PAYLOAD BUILDING (diag + lowrank + blocks) ===
  - basis 

RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

In [7]:
# ============================================================
# OFFLINE DeepSeek-MoE: Ws Builder + Quantization Research Runner
#   - No HuggingFace download required
#   - Builds W_eff per expert from local shards
#   - Evaluates strong PTQ baselines + Hadamard rotation (QuaRot-like)
#   - Optional: SmoothQuant-like scaling, GPTQ-lite (needs real calib X)
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np

# ---- torch is optional but recommended ----
import torch

# ---- safetensors for shard loading ----
try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError(
        "Missing dependency: safetensors. Install via:\n"
        "  pip install safetensors\n"
        f"Original import error: {e}"
    )

# ---- progress bar (optional) ----
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

# ----------------------------
# Config (edit or use env vars)
# ----------------------------
MODEL_DIR   = os.environ.get("MODEL_DIR",   "/home/daniyar/deepseek-model")
OUTPUT_DIR  = os.environ.get("OUTPUT_DIR",  "/home/daniyar/moe_ws_outputs")

LAYER       = int(os.environ.get("LAYER", "1"))
MAX_EXPERTS = int(os.environ.get("MAX_EXPERTS", "8"))

# Build choice:
#   - "down@up" (default): W_eff = down @ up
#   - "down@gate": W_eff = down @ gate
#   - "down@avg":  W_eff = down @ ((up+gate)/2) if both exist else down@up
W_EFF_MODE  = os.environ.get("W_EFF_MODE", "down@up")

# Cache file (npz)
WS_CACHE_PATH = os.path.join(OUTPUT_DIR, f"deepseek_layer{LAYER}_Ws_first{MAX_EXPERTS}.npz")

# Calibration:
# If you have REAL activations X (shape [N, hidden]) saved as npz with key "X",
# set CALIB_PATH=/path/to/calib_X.npz. Otherwise random X will be used (weaker for GPTQ/AWQ/SmoothQuant).
CALIB_PATH   = os.environ.get("CALIB_PATH", "").strip()
CALIB_SAMPLES = int(os.environ.get("CALIB_SAMPLES", "512"))
CALIB_SEED    = int(os.environ.get("CALIB_SEED", "1234"))

# Quantization sweeps
BITS_TO_TEST   = [8, 4, 3, 2]  # symmetric signed
GROUP_SIZE     = int(os.environ.get("GROUP_SIZE", "128"))  # groupwise along columns

# Rotation (QuaRot-like)
RUN_HADAMARD_ROT = os.environ.get("RUN_HADAMARD_ROT", "1") == "1"

# SmoothQuant-like scaling (needs calibration stats to be meaningful)
RUN_SMOOTHQUANT  = os.environ.get("RUN_SMOOTHQUANT", "1") == "1"
SMOOTHQUANT_ALPHA = float(os.environ.get("SMOOTHQUANT_ALPHA", "0.5"))  # 0..1

# GPTQ-lite (needs real calibration X to be meaningful)
RUN_GPTQ_LITE  = os.environ.get("RUN_GPTQ_LITE", "0") == "1"
GPTQ_BLOCK     = int(os.environ.get("GPTQ_BLOCK", "128"))     # columns per block
GPTQ_DAMP      = float(os.environ.get("GPTQ_DAMP", "0.01"))   # relative damp

# Evaluation
EVAL_BATCH   = int(os.environ.get("EVAL_BATCH", "16"))  # random x batch
MOE_ROUTE_K  = int(os.environ.get("MOE_ROUTE_K", "4"))  # mixture size
DEVICE       = torch.device(os.environ.get("DEVICE", "cpu"))

# Threading safety
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

# Repro
random.seed(CALIB_SEED)
np.random.seed(CALIB_SEED)
torch.manual_seed(CALIB_SEED)

# ----------------------------
# Helpers
# ----------------------------
def _pbar(it, **kwargs):
    if tqdm is None:
        return it
    return tqdm(it, **kwargs)

def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def human_bytes(n: int) -> str:
    for unit in ["B","KB","MB","GB","TB"]:
        if n < 1024:
            return f"{n:.2f}{unit}"
        n /= 1024
    return f"{n:.2f}PB"

def print_header():
    print("== OFFLINE DeepSeek-MoE Quant Runner ==")
    print(f"MODEL_DIR:  {MODEL_DIR}")
    print(f"OUTPUT_DIR: {OUTPUT_DIR}")
    print(f"LAYER:      {LAYER}")
    print(f"MAX_EXPERTS:{MAX_EXPERTS}")
    print(f"Torch:      {torch.__version__} | device={DEVICE} | threads={NTHREADS}")
    print(f"W_EFF_MODE: {W_EFF_MODE}")
    print(f"CALIB_PATH: {CALIB_PATH or '(none -> random X)'} | CALIB_SAMPLES={CALIB_SAMPLES}")
    print(f"GROUP_SIZE: {GROUP_SIZE} | BITS: {BITS_TO_TEST}")
    print(f"RUN_HADAMARD_ROT={RUN_HADAMARD_ROT} | RUN_SMOOTHQUANT={RUN_SMOOTHQUANT} | RUN_GPTQ_LITE={RUN_GPTQ_LITE}")
    print()

def check_model_dir():
    idx_path = os.path.join(MODEL_DIR, "model.safetensors.index.json")
    if not os.path.exists(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        idx = json.load(f)
    weight_map = idx.get("weight_map", {})
    if not weight_map:
        raise RuntimeError("Index has no weight_map entries.")
    # check that at least one shard exists
    shards = set(weight_map.values())
    missing = [s for s in shards if not os.path.exists(os.path.join(MODEL_DIR, s))]
    if missing:
        print("[WARN] Missing shard files:")
        for m in missing[:20]:
            print("  -", m)
        raise FileNotFoundError("Some shard files are missing in MODEL_DIR.")
    return idx_path, weight_map

def load_tensors_from_shards(weight_map: Dict[str,str], keys: List[str]) -> Dict[str, torch.Tensor]:
    """Loads a list of tensor keys, grouping by shard for efficiency."""
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard, ks in by_shard.items():
        path = os.path.join(MODEL_DIR, shard)
        with safe_open(path, framework="pt", device=str(DEVICE)) as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

def discover_experts_in_layer(weight_map: Dict[str,str], layer: int) -> Dict[int, List[str]]:
    """
    Returns mapping: expert_id -> list of keys for that expert at this layer.
    Looks for keys like: model.layers.{L}.mlp.experts.{EID}.*.weight
    """
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.(.+)\.weight$")
    ex: Dict[int, List[str]] = {}
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            eid = int(m.group(1))
            ex.setdefault(eid, []).append(k)
    return ex

def pick_up_gate_down(keys_for_expert: List[str], tensors: Dict[str, torch.Tensor]) -> Tuple[str, Optional[str], str]:
    """
    Heuristics:
      - down: prefer name containing 'w2' or 'down'
      - up/gate: among (intermediate, hidden) shaped weights:
          if two candidates exist, prefer 'w3'/'up' as up and 'w1'/'gate' as gate
          else treat single candidate as up
    Returns: (up_key, gate_key_or_None, down_key)
    """
    # candidates by name
    down_cands = [k for k in keys_for_expert if (".w2." in k or "down" in k)]
    if not down_cands:
        # fallback: anything whose shape[0] == hidden and shape[1] == intermediate
        down_cands = []
        for k in keys_for_expert:
            t = tensors[k]
            if t.ndim == 2 and t.shape[0] > t.shape[1]:
                down_cands.append(k)
    if not down_cands:
        raise RuntimeError("Could not find a down-proj candidate for expert.")
    down_key = sorted(down_cands)[0]

    # up/gate candidates: (intermediate, hidden)
    upgate = []
    for k in keys_for_expert:
        t = tensors[k]
        if t.ndim == 2 and t.shape[1] == tensors[down_key].shape[1] and t.shape[0] == tensors[down_key].shape[0] and False:
            pass
        # For DeepSeek style, down is (hidden, inter), up/gate are (inter, hidden)
        if t.ndim == 2 and t.shape[1] == tensors[down_key].shape[0]:
            upgate.append(k)

    if not upgate:
        raise RuntimeError("Could not find any (intermediate, hidden) up/gate candidate for expert.")

    # If there are two (common for SwiGLU), choose by name
    def score_up(k: str) -> int:
        s = 0
        if ".w3." in k or "up" in k: s += 10
        if ".w1." in k or "gate" in k: s -= 2
        return -s  # lower is better in sort

    def score_gate(k: str) -> int:
        s = 0
        if ".w1." in k or "gate" in k: s += 10
        if ".w3." in k or "up" in k: s -= 2
        return -s

    up_key = sorted(upgate, key=score_up)[0]
    gate_key = None
    if len(upgate) >= 2:
        # pick a different key as gate if possible
        cand_gate = [k for k in upgate if k != up_key]
        gate_key = sorted(cand_gate, key=score_gate)[0]

    return up_key, gate_key, down_key

def build_W_eff_for_expert(t_up: torch.Tensor, t_gate: Optional[torch.Tensor], t_down: torch.Tensor, mode: str) -> torch.Tensor:
    """
    Returns W_eff (hidden x hidden) in float32.
    Assumes:
      - down: (hidden, inter)
      - up/gate: (inter, hidden)
    """
    down = t_down.float()
    up   = t_up.float()
    if mode == "down@up":
        return down @ up
    if mode == "down@gate":
        if t_gate is None:
            return down @ up
        gate = t_gate.float()
        return down @ gate
    if mode == "down@avg":
        if t_gate is None:
            return down @ up
        gate = t_gate.float()
        return down @ (0.5*(up + gate))
    # default
    return down @ up

# ----------------------------
# Hadamard rotation (fast, structured, QuaRot-like)
# ----------------------------
def fht_lastdim(x: torch.Tensor) -> torch.Tensor:
    """
    Fast Hadamard Transform along last dimension.
    Unnormalized: H^T H = n I. Use /sqrt(n) to make orthonormal.
    Requires length power of 2.
    """
    n = x.shape[-1]
    if n & (n-1) != 0:
        raise ValueError("Hadamard transform requires power-of-2 length.")
    y = x
    h = 1
    while h < n:
        y = y.view(*y.shape[:-1], -1, 2*h)
        a = y[..., :h]
        b = y[..., h:2*h]
        y = torch.cat([a + b, a - b], dim=-1)
        y = y.view(*y.shape[:-2], -1)
        h *= 2
    return y

@dataclass
class RandomHadamardRotation:
    n: int
    seed: int = 1234

    def __post_init__(self):
        g = torch.Generator(device="cpu").manual_seed(int(self.seed))
        self.signs = torch.where(torch.rand(self.n, generator=g) > 0.5, 1.0, -1.0).float().to(DEVICE)
        perm = torch.randperm(self.n, generator=g)
        self.perm = perm.to(DEVICE)
        inv = torch.empty_like(self.perm)
        inv[self.perm] = torch.arange(self.n, device=DEVICE)
        self.inv_perm = inv
        self.norm = math.sqrt(self.n)

    def apply_x(self, x: torch.Tensor) -> torch.Tensor:
        # x @ R where R = (1/sqrt(n)) * H * D * P
        y = fht_lastdim(x) / self.norm
        y = y * self.signs
        y = y[:, self.perm]
        return y

    def apply_W_left(self, W: torch.Tensor) -> torch.Tensor:
        # W_rot = R^T @ W where R^T = (1/sqrt(n)) * P^T * D * H
        # Step1: H @ W  => hadamard along rows -> do fht on W.T lastdim then transpose back
        Wh = fht_lastdim(W.T).T / self.norm
        Wh = Wh * self.signs.view(-1, 1)
        Wrot = Wh[self.inv_perm, :]
        return Wrot

# ----------------------------
# Quantization primitives
# ----------------------------
def qmax_for_bits(bits: int) -> int:
    # symmetric signed
    return (2 ** (bits - 1)) - 1

@torch.no_grad()
def quant_dequant_groupwise_sym(W: torch.Tensor, bits: int, group_size: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Groupwise symmetric quant along columns, per-row scales.
    W: [out, in]
    Returns: (W_hat, scales) where scales is [out, n_groups]
    """
    assert W.ndim == 2
    out, inn = W.shape
    qmax = qmax_for_bits(bits)
    n_groups = (inn + group_size - 1) // group_size
    W_hat = torch.empty_like(W, dtype=torch.float32, device=W.device)
    scales = torch.empty((out, n_groups), dtype=torch.float32, device=W.device)

    eps = 1e-12
    for gi in range(n_groups):
        j0 = gi * group_size
        j1 = min(inn, (gi + 1) * group_size)
        block = W[:, j0:j1].float()
        s = block.abs().amax(dim=1, keepdim=True) / max(qmax, 1)  # [out,1]
        s = torch.clamp(s, min=eps)
        q = torch.round(block / s).clamp(-qmax, qmax)
        W_hat[:, j0:j1] = q * s
        scales[:, gi] = s.squeeze(1)
    return W_hat, scales

@torch.no_grad()
def rel_frob_err(A: torch.Tensor, B: torch.Tensor) -> float:
    num = torch.linalg.norm((A - B).float(), ord="fro")
    den = torch.linalg.norm(B.float(), ord="fro") + 1e-12
    return float((num / den).item())

@torch.no_grad()
def rel_out_err(W_hat: torch.Tensor, W: torch.Tensor, batch: int, seed: int) -> float:
    g = torch.Generator(device="cpu").manual_seed(int(seed))
    x = torch.randn(batch, W.shape[0], generator=g, device=W.device, dtype=torch.float32)
    y_hat = x @ W_hat.float()
    y_ref = x @ W.float()
    return rel_frob_err(y_hat, y_ref)

@torch.no_grad()
def moe_mix_out_err(Ws_hat: torch.Tensor, Ws: torch.Tensor, k: int, batch: int, seed: int) -> float:
    """
    Ws: [E, n, n]
    Sample k routed experts with random gates, compare x @ sum(a_i W_i)
    """
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(int(seed))
    x = torch.randn(batch, n, generator=g, device=Ws.device, dtype=torch.float32)

    pick = torch.randperm(E, generator=g)[:min(k, E)].tolist()
    gates = torch.rand(len(pick), generator=g, device=Ws.device)
    gates = gates / (gates.sum() + 1e-12)

    Wsum = torch.zeros(n, n, device=Ws.device, dtype=torch.float32)
    Wsum_hat = torch.zeros_like(Wsum)
    for a, e in zip(gates, pick):
        Wsum += a * Ws[e].float()
        Wsum_hat += a * Ws_hat[e].float()

    y_ref = x @ Wsum
    y_hat = x @ Wsum_hat
    return rel_frob_err(y_hat, y_ref)

# ----------------------------
# SmoothQuant-like scaling (column-wise)
# ----------------------------
@torch.no_grad()
def smoothquant_scales(X: torch.Tensor, W: torch.Tensor, alpha: float) -> torch.Tensor:
    """
    SmoothQuant chooses diagonal scale s for input channels (columns of W / dims of X).
    Simplified version:
      s_j = (amax(|X_j|)^alpha) / (amax(|W_:,j|)^(1-alpha))
    Returns s [in]
    """
    eps = 1e-12
    ax = X.abs().amax(dim=0).float().clamp_min(eps)              # [in]
    aw = W.abs().amax(dim=0).float().clamp_min(eps)              # [in]
    s = (ax ** alpha) / (aw ** (1.0 - alpha))
    # normalize for numerical stability
    s = s / (torch.median(s).clamp_min(eps))
    return s

@torch.no_grad()
def apply_smoothquant(W: torch.Tensor, X: torch.Tensor, alpha: float) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Scale activations and weights:
      x' = x / s
      W' = W * s   (scales columns)
    so that xW == x'W' exactly in fp32.
    """
    s = smoothquant_scales(X, W, alpha=alpha)  # [in]
    Wp = W * s.view(1, -1)
    return Wp, s

# ----------------------------
# GPTQ-lite (blockwise, needs real X to matter)
# ----------------------------
@torch.no_grad()
def gptq_lite_blockwise(W: torch.Tensor, X: torch.Tensor, bits: int, block: int, group_size: int, damp: float) -> torch.Tensor:
    """
    Blockwise GPTQ-like second-order quantization approximation.
    Practical compromise:
      - Compute H = X^T X / N + damp*I
      - Use Hinv blocks of size 'block'
      - Quantize each block with error feedback inside block (vectorized over rows)
    W: [out, in], X: [N, in]
    Returns W_hat [out, in] float32
    """
    out, inn = W.shape
    qmax = qmax_for_bits(bits)
    eps = 1e-12

    # Hessian
    H = (X.float().T @ X.float()) / max(1, X.shape[0])  # [in,in]
    d = torch.mean(torch.diag(H)).clamp_min(eps)
    H = H + (damp * d) * torch.eye(inn, device=W.device, dtype=torch.float32)

    # Full inverse is expensive but ok for inn=2048
    # Cholesky inverse is stable
    L = torch.linalg.cholesky(H)
    Hinv = torch.cholesky_inverse(L)  # [in,in]

    W_hat = torch.empty_like(W, dtype=torch.float32)

    n_blocks = (inn + block - 1) // block
    for bi in _pbar(range(n_blocks), desc=f"GPTQ-lite {bits}b", total=n_blocks):
        i0 = bi * block
        i1 = min(inn, (bi + 1) * block)
        bsz = i1 - i0

        Hinv_blk = Hinv[i0:i1, i0:i1].contiguous()  # [b,b]
        diagH = torch.diag(Hinv_blk).clamp_min(eps)  # [b]

        # take working block
        Wblk = W[:, i0:i1].float().clone()           # [out,b]
        # groupwise scale across columns inside the block (per-row)
        # If block > group_size, we still quant by group_size inside block
        # but error feedback uses full block.
        Qblk = torch.empty_like(Wblk)

        # Precompute group assignment
        n_groups = (bsz + group_size - 1) // group_size
        # scales per-row per-group
        scales = torch.empty((out, n_groups), device=W.device, dtype=torch.float32)

        # init scales
        for gi in range(n_groups):
            j0 = gi * group_size
            j1 = min(bsz, (gi + 1) * group_size)
            part = Wblk[:, j0:j1]
            s = part.abs().amax(dim=1, keepdim=True) / max(qmax, 1)
            s = s.clamp_min(eps)
            scales[:, gi] = s.squeeze(1)

        # sequential within block with vectorized row updates
        for j in range(bsz):
            gi = min(j // group_size, n_groups - 1)
            s = scales[:, gi].view(-1, 1)  # [out,1]
            wj = Wblk[:, j:j+1]            # [out,1]
            qj = torch.round(wj / s).clamp(-qmax, qmax) * s
            Qblk[:, j:j+1] = qj

            err = (wj - qj) / diagH[j]     # [out,1]
            # error feedback to remaining cols in block
            if j + 1 < bsz:
                Wblk[:, j+1:] -= err @ Hinv_blk[j:j+1, j+1:]

        W_hat[:, i0:i1] = Qblk

    return W_hat

# ----------------------------
# Main pipeline
# ----------------------------
def load_or_build_Ws() -> Tuple[torch.Tensor, List[int]]:
    ensure_dir(OUTPUT_DIR)

    if os.path.exists(WS_CACHE_PATH):
        print(f"[cache] Loading existing Ws cache: {WS_CACHE_PATH} ({human_bytes(os.path.getsize(WS_CACHE_PATH))})")
        z = np.load(WS_CACHE_PATH, allow_pickle=True)
        Ws = torch.from_numpy(z["Ws"]).to(DEVICE).float()  # store eval in fp32
        expert_ids = z["expert_ids"].tolist()
        return Ws, expert_ids

    print("[build] No cache found, building Ws from local shards...")
    _, weight_map = check_model_dir()
    experts = discover_experts_in_layer(weight_map, LAYER)
    if not experts:
        raise RuntimeError(f"No experts found at layer={LAYER}. Check that this model is MoE and layer index is correct.")

    expert_ids = sorted(experts.keys())[:MAX_EXPERTS]
    print(f"[build] Found {len(experts)} experts at layer {LAYER}; using first {len(expert_ids)}: {expert_ids}")

    # load required keys for chosen experts
    all_keys = []
    for eid in expert_ids:
        all_keys.extend(experts[eid])
    tensors = load_tensors_from_shards(weight_map, all_keys)

    Ws_list = []
    meta = []
    for eid in _pbar(expert_ids, desc="Building W_eff", total=len(expert_ids)):
        keys = experts[eid]
        up_key, gate_key, down_key = pick_up_gate_down(keys, tensors)

        up = tensors[up_key]
        gate = tensors[gate_key] if gate_key is not None else None
        down = tensors[down_key]

        W_eff = build_W_eff_for_expert(up, gate, down, mode=W_EFF_MODE)  # [hidden, hidden]
        Ws_list.append(W_eff.unsqueeze(0))
        meta.append((eid, up_key, gate_key, down_key, tuple(up.shape), tuple(down.shape)))

    Ws = torch.cat(Ws_list, dim=0).contiguous()  # [E,n,n]
    expert_ids_out = expert_ids

    # save cache as float16 to keep small
    Ws_np = Ws.detach().cpu().to(torch.float16).numpy()
    np.savez_compressed(
        WS_CACHE_PATH,
        Ws=Ws_np,
        expert_ids=np.array(expert_ids_out, dtype=np.int32),
        layer=np.int32(LAYER),
        w_eff_mode=np.array([W_EFF_MODE], dtype=object),
        meta=np.array(meta, dtype=object),
    )
    print(f"[cache] Saved Ws cache: {WS_CACHE_PATH} ({human_bytes(os.path.getsize(WS_CACHE_PATH))})")

    return Ws.to(DEVICE).float(), expert_ids_out

def load_calibration_X(n: int) -> torch.Tensor:
    if CALIB_PATH and os.path.exists(CALIB_PATH):
        z = np.load(CALIB_PATH, allow_pickle=True)
        if "X" not in z:
            raise KeyError(f"{CALIB_PATH} must contain array 'X' with shape [N, {n}]")
        X = torch.from_numpy(z["X"]).to(DEVICE).float()
        if X.ndim != 2 or X.shape[1] != n:
            raise ValueError(f"Bad X shape: {tuple(X.shape)} expected [N, {n}]")
        print(f"[calib] Loaded X from {CALIB_PATH}: {tuple(X.shape)}")
        return X
    # fallback random
    g = torch.Generator(device="cpu").manual_seed(int(CALIB_SEED))
    X = torch.randn(CALIB_SAMPLES, n, generator=g, device=DEVICE, dtype=torch.float32)
    print(f"[calib] Using RANDOM X: {tuple(X.shape)} (set CALIB_PATH for real activations)")
    return X

def run_eval(Ws: torch.Tensor, expert_ids: List[int]):
    E, n, _ = Ws.shape
    X = load_calibration_X(n)

    # precompute rotation
    rot = None
    if RUN_HADAMARD_ROT:
        rot = RandomHadamardRotation(n=n, seed=CALIB_SEED)

    results = {
        "ts": time.time(),
        "model_dir": MODEL_DIR,
        "layer": LAYER,
        "expert_ids": expert_ids,
        "E": E,
        "n": n,
        "group_size": GROUP_SIZE,
        "bits": BITS_TO_TEST,
        "hadamard_rot": RUN_HADAMARD_ROT,
        "smoothquant": RUN_SMOOTHQUANT,
        "smoothquant_alpha": SMOOTHQUANT_ALPHA,
        "gptq_lite": RUN_GPTQ_LITE,
        "gptq_block": GPTQ_BLOCK,
        "gptq_damp": GPTQ_DAMP,
        "notes": "For strongest results, provide real CALIB_PATH with activation samples X from the model.",
        "per_expert": {},
        "moe_mix": {},
    }

    # Evaluate per expert
    for ei in _pbar(range(E), desc="Per-expert quant eval", total=E):
        eid = expert_ids[ei]
        W = Ws[ei].float()

        per = {}
        # optional smoothquant preprocessing
        W_sq = None
        s_sq = None
        if RUN_SMOOTHQUANT:
            W_sq, s_sq = apply_smoothquant(W, X, alpha=SMOOTHQUANT_ALPHA)

        # optional hadamard rotation preprocessing (rotate rows of W via R^T W, and rotate x via xR)
        # We'll evaluate output error properly for each method.
        for bits in BITS_TO_TEST:
            tag_base = f"g{GROUP_SIZE}"
            # -------- baseline: direct quant --------
            W_hat, _ = quant_dequant_groupwise_sym(W, bits=bits, group_size=GROUP_SIZE)
            per[f"base_{bits}b_{tag_base}"] = {
                "w_rel_frob": rel_frob_err(W_hat, W),
                "y_rel_frob": rel_out_err(W_hat, W, batch=EVAL_BATCH, seed=CALIB_SEED + 10 + bits),
            }

            # -------- smoothquant: quantize W' = W*s, evaluate with x' = x/s --------
            if RUN_SMOOTHQUANT and W_sq is not None and s_sq is not None:
                Wsq_hat, _ = quant_dequant_groupwise_sym(W_sq, bits=bits, group_size=GROUP_SIZE)

                # output error for exact equivalence path:
                # y = xW, with smoothquant we compute y_hat = (x/s) @ Q(W*s)
                g = torch.Generator(device="cpu").manual_seed(int(CALIB_SEED + 100 + bits))
                x = torch.randn(EVAL_BATCH, n, generator=g, device=DEVICE, dtype=torch.float32)
                x2 = x / s_sq.view(1, -1)
                y_hat = x2 @ Wsq_hat
                y_ref = x @ W
                per[f"smoothq_{bits}b_{tag_base}"] = {
                    "w_rel_frob": rel_frob_err(Wsq_hat, W_sq),
                    "y_rel_frob": rel_frob_err(y_hat, y_ref),
                }

            # -------- hadamard rotation (QuaRot-like): quantize W_rot = R^T W, eval with x_rot = x R --------
            if rot is not None:
                Wrot = rot.apply_W_left(W)
                Wrot_hat, _ = quant_dequant_groupwise_sym(Wrot, bits=bits, group_size=GROUP_SIZE)

                g = torch.Generator(device="cpu").manual_seed(int(CALIB_SEED + 200 + bits))
                x = torch.randn(EVAL_BATCH, n, generator=g, device=DEVICE, dtype=torch.float32)
                xrot = rot.apply_x(x)
                y_hat = xrot @ Wrot_hat
                y_ref = x @ W
                per[f"hadam_{bits}b_{tag_base}"] = {
                    "w_rel_frob": rel_frob_err(Wrot_hat, Wrot),
                    "y_rel_frob": rel_frob_err(y_hat, y_ref),
                }

            # -------- smoothquant + hadamard --------
            if (rot is not None) and RUN_SMOOTHQUANT and (W_sq is not None) and (s_sq is not None):
                # Rotate rows after smoothquant: W' = W*s, then Wrot = R^T W'
                Wrot2 = rot.apply_W_left(W_sq)
                Wrot2_hat, _ = quant_dequant_groupwise_sym(Wrot2, bits=bits, group_size=GROUP_SIZE)

                g = torch.Generator(device="cpu").manual_seed(int(CALIB_SEED + 300 + bits))
                x = torch.randn(EVAL_BATCH, n, generator=g, device=DEVICE, dtype=torch.float32)
                x2 = x / s_sq.view(1, -1)
                xrot = rot.apply_x(x2)
                y_hat = xrot @ Wrot2_hat
                y_ref = x @ W
                per[f"smoothq+hadam_{bits}b_{tag_base}"] = {
                    "w_rel_frob": rel_frob_err(Wrot2_hat, Wrot2),
                    "y_rel_frob": rel_frob_err(y_hat, y_ref),
                }

            # -------- GPTQ-lite (blockwise) --------
            if RUN_GPTQ_LITE:
                # GPTQ-lite should be applied to the representation used at inference.
                # We'll do it for baseline (no rotation) and optionally hadamard rotated.
                Wg = gptq_lite_blockwise(W, X, bits=bits, block=GPTQ_BLOCK, group_size=GROUP_SIZE, damp=GPTQ_DAMP)
                per[f"gptqLite_{bits}b_b{GPTQ_BLOCK}_{tag_base}"] = {
                    "w_rel_frob": rel_frob_err(Wg, W),
                    "y_rel_frob": rel_out_err(Wg, W, batch=EVAL_BATCH, seed=CALIB_SEED + 400 + bits),
                }
                if rot is not None:
                    Wrot = rot.apply_W_left(W)
                    Xrot = rot.apply_x(X)  # because inference uses x_rot; Hessian should match rotated activations
                    Wrot_g = gptq_lite_blockwise(Wrot, Xrot, bits=bits, block=GPTQ_BLOCK, group_size=GROUP_SIZE, damp=GPTQ_DAMP)

                    g = torch.Generator(device="cpu").manual_seed(int(CALIB_SEED + 500 + bits))
                    x = torch.randn(EVAL_BATCH, n, generator=g, device=DEVICE, dtype=torch.float32)
                    xrot = rot.apply_x(x)
                    y_hat = xrot @ Wrot_g
                    y_ref = x @ W
                    per[f"hadam+gptqLite_{bits}b_b{GPTQ_BLOCK}_{tag_base}"] = {
                        "w_rel_frob": rel_frob_err(Wrot_g, Wrot),
                        "y_rel_frob": rel_frob_err(y_hat, y_ref),
                    }

        results["per_expert"][str(eid)] = per

    # Evaluate MoE mixture error for a couple representative methods (4-bit, 3-bit)
    # We'll build Ws_hat for those methods and compare random routed sums.
    def build_Ws_hat(method: str) -> torch.Tensor:
        Ws_hat = torch.empty_like(Ws, dtype=torch.float32)
        for ei in range(E):
            W = Ws[ei].float()
            if method.startswith("base_"):
                bits = int(method.split("_")[1].replace("b",""))
                Wh, _ = quant_dequant_groupwise_sym(W, bits=bits, group_size=GROUP_SIZE)
                Ws_hat[ei] = Wh
            elif method.startswith("hadam_") and rot is not None:
                bits = int(method.split("_")[1].replace("b",""))
                Wrot = rot.apply_W_left(W)
                Wrot_hat, _ = quant_dequant_groupwise_sym(Wrot, bits=bits, group_size=GROUP_SIZE)
                # Store rotated version since mixture test will apply the right input transform per compare below.
                # We'll handle evaluation separately.
                Ws_hat[ei] = Wrot_hat
            else:
                raise ValueError(method)
        return Ws_hat

    for bits in [4, 3]:
        # baseline mixture
        Ws_hat = torch.empty_like(Ws, dtype=torch.float32)
        for ei in range(E):
            Wh, _ = quant_dequant_groupwise_sym(Ws[ei].float(), bits=bits, group_size=GROUP_SIZE)
            Ws_hat[ei] = Wh
        results["moe_mix"][f"base_{bits}b"] = {
            "rel_out_err": moe_mix_out_err(Ws_hat, Ws, k=MOE_ROUTE_K, batch=EVAL_BATCH, seed=CALIB_SEED + 1000 + bits),
            "note": "y = x @ sum(a_i W_i)"
        }

        # hadamard mixture (compare in original space: y_ref = xW, y_hat = (xR) @ sum(a_i (R^T W)_hat)
        if rot is not None:
            Ws_rot_hat = torch.empty_like(Ws, dtype=torch.float32)
            Ws_rot = torch.empty_like(Ws, dtype=torch.float32)
            for ei in range(E):
                W = Ws[ei].float()
                Wrot = rot.apply_W_left(W)
                Wrot_hat, _ = quant_dequant_groupwise_sym(Wrot, bits=bits, group_size=GROUP_SIZE)
                Ws_rot[ei] = Wrot
                Ws_rot_hat[ei] = Wrot_hat

            # Evaluate mixture properly
            g = torch.Generator(device="cpu").manual_seed(int(CALIB_SEED + 2000 + bits))
            x = torch.randn(EVAL_BATCH, n, generator=g, device=DEVICE, dtype=torch.float32)

            pick = torch.randperm(E, generator=g)[:min(MOE_ROUTE_K, E)].tolist()
            gates = torch.rand(len(pick), generator=g, device=DEVICE)
            gates = gates / (gates.sum() + 1e-12)

            Wsum_ref = torch.zeros(n, n, device=DEVICE, dtype=torch.float32)
            Wsum_hat = torch.zeros_like(Wsum_ref)
            for a, e in zip(gates, pick):
                Wsum_ref += a * Ws[e].float()
                Wsum_hat += a * Ws_rot_hat[e].float()  # rotated-space weights

            y_ref = x @ Wsum_ref
            xrot = rot.apply_x(x)
            y_hat = xrot @ Wsum_hat

            results["moe_mix"][f"hadam_{bits}b"] = {
                "rel_out_err": rel_frob_err(y_hat, y_ref),
                "note": "y_hat = (xR) @ sum(a_i Q(R^T W_i))"
            }

    # Save report
    ensure_dir(OUTPUT_DIR)
    out_json = os.path.join(OUTPUT_DIR, f"quant_report_layer{LAYER}_E{E}_g{GROUP_SIZE}.json")
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(f"\n[done] Wrote report: {out_json}")

    # Print a short “best-of” summary for 4-bit by avg y error
    def avg_yerr(prefix: str) -> float:
        vals = []
        for eid in expert_ids:
            per = results["per_expert"][str(eid)]
            for k, v in per.items():
                if k == prefix:
                    vals.append(v["y_rel_frob"])
        return float(np.mean(vals)) if vals else float("nan")

    # pick representative keys if they exist
    keys_to_show = []
    for k in [f"base_4b_g{GROUP_SIZE}", f"hadam_4b_g{GROUP_SIZE}", f"smoothq_4b_g{GROUP_SIZE}", f"smoothq+hadam_4b_g{GROUP_SIZE}"]:
        keys_to_show.append(k)

    print("\n== Quick 4-bit snapshot (per-expert avg y_rel_frob) ==")
    for k in keys_to_show:
        vals = []
        for eid in expert_ids:
            per = results["per_expert"][str(eid)]
            if k in per:
                vals.append(per[k]["y_rel_frob"])
        if vals:
            print(f"{k:28s}: {float(np.mean(vals)):.6f}  (±{float(np.std(vals)):.6f})")
        else:
            print(f"{k:28s}: (not run)")

def main():
    print_header()
    ensure_dir(OUTPUT_DIR)
    # Validate model dir early
    check_model_dir()
    # Load/build Ws
    Ws, expert_ids = load_or_build_Ws()
    print(f"[ok] Ws: {tuple(Ws.shape)} | experts={expert_ids[:min(10,len(expert_ids))]} ...")
    # Run eval
    run_eval(Ws, expert_ids)

if __name__ == "__main__":
    main()


== OFFLINE DeepSeek-MoE Quant Runner ==
MODEL_DIR:  /home/daniyar/deepseek-model
OUTPUT_DIR: /home/daniyar/moe_ws_outputs
LAYER:      1
MAX_EXPERTS:8
Torch:      2.4.1+cpu | device=cpu | threads=8
W_EFF_MODE: down@up
CALIB_PATH: (none -> random X) | CALIB_SAMPLES=512
GROUP_SIZE: 128 | BITS: [8, 4, 3, 2]
RUN_HADAMARD_ROT=True | RUN_SMOOTHQUANT=True | RUN_GPTQ_LITE=False

[cache] Loading existing Ws cache: /home/daniyar/moe_ws_outputs/deepseek_layer1_Ws_first8.npz (59.04MB)
[ok] Ws: (8, 2048, 2048) | experts=[0, 1, 2, 3, 4, 5, 6, 7] ...
[calib] Using RANDOM X: (512, 2048) (set CALIB_PATH for real activations)


Per-expert quant eval:   0%|          | 0/8 [00:00<?, ?it/s]


[done] Wrote report: /home/daniyar/moe_ws_outputs/quant_report_layer1_E8_g128.json

== Quick 4-bit snapshot (per-expert avg y_rel_frob) ==
base_4b_g128                : 0.118324  (±0.001128)
hadam_4b_g128               : 1.418151  (±0.012841)
smoothq_4b_g128             : 0.171671  (±0.011489)
smoothq+hadam_4b_g128       : 1.428965  (±0.011562)


In [8]:
# ============================================================
# OFFLINE DeepSeek-MoE Quant Runner (GIANT SINGLE CELL) v3
# - Loads MoE expert weights from LOCAL model.safetensors shards (no HF needed)
# - Can build:
#     (A) W_eff = down @ up  (your current proxy)
#     (B) Full FFN simulation if gate/up/down exist: y = (act(x@Wg^T) * (x@Wu^T)) @ Wd^T
# - Quant methods:
#     * baseline groupwise symmetric (per-row, groups over columns)
#     * SmoothQuant-like per-input-channel scaling (alpha grid search)
#     * QuaRot-style structured Hadamard rotation (tries multiple seeds; keeps best)
#     * SmoothQuant + Hadamard combo (keeps best)
#     * GPTQ-lite (optional; slower; blockwise per-row with covariance)
# - Outputs:
#     * JSON report with per-expert errors for bits list
#     * NPZ payload (best method params per expert) for later application
#
# Requirements: torch, numpy, tqdm, safetensors
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np

# ---- torch ----
import torch
torch.set_grad_enabled(False)

# ---- progress ----
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kw: x

# ---- safetensors ----
SAFE_OPEN = None
try:
    from safetensors.torch import safe_open as SAFE_OPEN
except Exception:
    try:
        from safetensors import safe_open as SAFE_OPEN
    except Exception:
        SAFE_OPEN = None

# ----------------------------
# CONFIG (edit these)
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = "/home/daniyar/deepseek-model"
    OUTPUT_DIR: str = "/home/daniyar/moe_ws_outputs"

    LAYER: int = 1
    MAX_EXPERTS: int = 8

    # Choose what you quant-evaluate:
    #   "weff" -> W_eff = down@up (matches your current pipeline)
    #   "ffn"  -> simulate FFN if (gate, up, down) exist (better proxy of real behavior)
    EVAL_MODE: str = "weff"  # "weff" or "ffn"

    # Calibration:
    # - If you can dump real activations, put npz path with array "X" (shape [N, hidden]).
    CALIB_PATH: Optional[str] = None
    CALIB_SAMPLES: int = 512

    # Quant grid:
    GROUP_SIZE: int = 128
    BITS_LIST: Tuple[int, ...] = (8, 4, 3, 2)

    # Rotation / scaling
    SMOOTHQUANT_ALPHA_GRID: Tuple[float, ...] = (0.0, 0.25, 0.5, 0.75, 1.0)
    TRY_HADAMARD: bool = True
    HADAMARD_TRIES: int = 6  # number of random (perm,sign) seeds to try per expert per bit
    HADAMARD_SEED0: int = 1234

    # GPTQ-lite (optional; can be slow on CPU)
    RUN_GPTQ_LITE: bool = False
    GPTQ_BLOCK: int = 128   # columns per block
    GPTQ_DAMP: float = 1e-4

    # Runtime
    DEVICE: str = "cpu"
    THREADS: int = 8
    SEED: int = 1234

CFG = Cfg()

# ----------------------------
# Threading / seeds
# ----------------------------
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
os.environ.setdefault("OMP_NUM_THREADS", str(CFG.THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(CFG.THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(CFG.THREADS))
try:
    torch.set_num_threads(CFG.THREADS)
except Exception:
    pass

random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)

DEVICE = torch.device(CFG.DEVICE)

print("== OFFLINE DeepSeek-MoE Quant Runner v3 ==")
print(f"MODEL_DIR:   {CFG.MODEL_DIR}")
print(f"OUTPUT_DIR:  {CFG.OUTPUT_DIR}")
print(f"LAYER:       {CFG.LAYER}")
print(f"MAX_EXPERTS: {CFG.MAX_EXPERTS}")
print(f"EVAL_MODE:   {CFG.EVAL_MODE}")
print(f"Torch:       {torch.__version__} | device={DEVICE} | threads={CFG.THREADS}")
if SAFE_OPEN is None:
    raise RuntimeError("safetensors is not available. Try: pip install safetensors")

# ============================================================
# Helpers: IO, model index, tensor loading
# ============================================================

def _read_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _human_mb(nbytes: int) -> float:
    return float(nbytes) / (1024.0 * 1024.0)

def _ensure_exists(path: str, what: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {what}: {path}")

INDEX_PATH = os.path.join(CFG.MODEL_DIR, "model.safetensors.index.json")
_ensure_exists(INDEX_PATH, "model.safetensors.index.json")
index = _read_json(INDEX_PATH)
weight_map: Dict[str, str] = index.get("weight_map", {})
if not weight_map:
    raise RuntimeError("index json has no weight_map")

# cache open handles per shard
_shard_handles: Dict[str, Any] = {}

def load_tensor(key: str, dtype: torch.dtype = torch.float32) -> torch.Tensor:
    """Load a single tensor from the right shard using safetensors safe_open."""
    if key not in weight_map:
        raise KeyError(f"Key not found in weight_map: {key}")
    shard = weight_map[key]
    shard_path = os.path.join(CFG.MODEL_DIR, shard)
    _ensure_exists(shard_path, f"shard for {key}")
    if shard_path not in _shard_handles:
        _shard_handles[shard_path] = SAFE_OPEN(shard_path, framework="pt", device="cpu")
    h = _shard_handles[shard_path]
    t = h.get_tensor(key)
    return t.to(dtype=dtype)

def find_expert_keys_for_layer(layer: int) -> Dict[str, List[str]]:
    """
    Tries to discover MoE expert weight key patterns.
    Supports common patterns:
      - model.layers.L.mlp.experts.E.w1.weight / w2.weight / w3.weight
      - model.layers.L.mlp.experts.E.up_proj.weight / down_proj.weight / gate_proj.weight
    Returns dict with lists for 'gate','up','down' (some may be empty).
    """
    prefix = f"model.layers.{layer}.mlp.experts."
    gate_keys, up_keys, down_keys = [], [], []

    # candidate suffixes
    for k in weight_map.keys():
        if not k.startswith(prefix):
            continue
        # normalize "expert id" extraction
        # e.g. model.layers.1.mlp.experts.12.w1.weight
        #      model.layers.1.mlp.experts.12.up_proj.weight
        if k.endswith(".w1.weight") or k.endswith(".gate_proj.weight"):
            gate_keys.append(k)
        elif k.endswith(".w3.weight") or k.endswith(".up_proj.weight"):
            up_keys.append(k)
        elif k.endswith(".w2.weight") or k.endswith(".down_proj.weight"):
            down_keys.append(k)

    # Sort by expert id if possible
    def _eid_of(key: str) -> int:
        m = re.search(r"\.experts\.(\d+)\.", key)
        return int(m.group(1)) if m else 10**9

    gate_keys.sort(key=_eid_of)
    up_keys.sort(key=_eid_of)
    down_keys.sort(key=_eid_of)

    return {"gate": gate_keys, "up": up_keys, "down": down_keys}

keys = find_expert_keys_for_layer(CFG.LAYER)
if len(keys["up"]) == 0 or len(keys["down"]) == 0:
    raise RuntimeError(
        f"Could not find expert up/down keys for layer={CFG.LAYER}. "
        f"Found: gate={len(keys['gate'])}, up={len(keys['up'])}, down={len(keys['down'])}."
    )

def _expert_ids_from_keys(ks: List[str]) -> List[int]:
    out = []
    for k in ks:
        m = re.search(r"\.experts\.(\d+)\.", k)
        out.append(int(m.group(1)) if m else -1)
    return out

up_eids = _expert_ids_from_keys(keys["up"])
down_eids = _expert_ids_from_keys(keys["down"])
common = sorted(set(up_eids).intersection(set(down_eids)))
if not common:
    raise RuntimeError("No overlapping expert ids between up and down keys")

expert_ids = common[:CFG.MAX_EXPERTS]
print(f"[found] experts for layer {CFG.LAYER}: {expert_ids[:10]} (using {len(expert_ids)})")

# ============================================================
# Build / load cache: expert weights
# ============================================================

CACHE_NPZ = os.path.join(CFG.OUTPUT_DIR, f"deepseek_layer{CFG.LAYER}_E{len(expert_ids)}_cache.npz")

def build_cache(npz_path: str):
    """
    Saves:
      - up_e{i}, down_e{i}, (optional gate_e{i})
      - also stores shapes and expert ids
      - does NOT stack mismatched shapes into one tensor
    """
    print("[cache] building expert cache (this reads shards; can take a bit) ...")
    to_save: Dict[str, np.ndarray] = {}
    to_save["expert_ids"] = np.array(expert_ids, dtype=np.int32)

    # map eid -> key
    gate_map = {int(re.search(r"\.experts\.(\d+)\.", k).group(1)): k for k in keys["gate"]} if keys["gate"] else {}
    up_map   = {int(re.search(r"\.experts\.(\d+)\.", k).group(1)): k for k in keys["up"]}
    down_map = {int(re.search(r"\.experts\.(\d+)\.", k).group(1)): k for k in keys["down"]}

    for eid in tqdm(expert_ids, desc="load experts"):
        up_k = up_map[eid]
        dn_k = down_map[eid]
        up = load_tensor(up_k, dtype=torch.float16)
        dn = load_tensor(dn_k, dtype=torch.float16)
        to_save[f"up_e{eid}"] = up.cpu().numpy()
        to_save[f"down_e{eid}"] = dn.cpu().numpy()
        if eid in gate_map:
            gt = load_tensor(gate_map[eid], dtype=torch.float16)
            to_save[f"gate_e{eid}"] = gt.cpu().numpy()

    np.savez_compressed(npz_path, **to_save)
    print(f"[cache] wrote: {npz_path} ({_human_mb(os.path.getsize(npz_path)):.2f} MB)")

def load_cache(npz_path: str) -> Dict[str, Any]:
    d = np.load(npz_path, allow_pickle=False)
    out = {k: d[k] for k in d.files}
    return out

if os.path.exists(CACHE_NPZ):
    print(f"[cache] loading: {CACHE_NPZ} ({_human_mb(os.path.getsize(CACHE_NPZ)):.2f} MB)")
    cache = load_cache(CACHE_NPZ)
else:
    build_cache(CACHE_NPZ)
    cache = load_cache(CACHE_NPZ)

expert_ids = [int(x) for x in cache["expert_ids"].tolist()]

def get_expert_weights(eid: int):
    up = torch.from_numpy(cache[f"up_e{eid}"]).to(device=DEVICE, dtype=torch.float32)
    dn = torch.from_numpy(cache[f"down_e{eid}"]).to(device=DEVICE, dtype=torch.float32)
    gate = None
    gk = f"gate_e{eid}"
    if gk in cache:
        gate = torch.from_numpy(cache[gk]).to(device=DEVICE, dtype=torch.float32)
    return gate, up, dn

# infer hidden size from up/down
_, up0, dn0 = get_expert_weights(expert_ids[0])

# Heuristic for orientation:
# - If dn is [hidden, inter] and up is [inter, hidden], then W_eff = dn @ up -> [hidden, hidden]
# - If dn is [inter, hidden] and up is [hidden, inter], then W_eff = up @ dn -> [hidden, hidden]
def build_weff(up: torch.Tensor, dn: torch.Tensor) -> torch.Tensor:
    a, b = dn.shape
    c, d = up.shape
    # try dn@up first
    if b == c:
        return (dn @ up).contiguous()
    # else try up@dn
    if d == a:
        return (up @ dn).contiguous()
    raise RuntimeError(f"Cannot multiply for W_eff: dn{tuple(dn.shape)} up{tuple(up.shape)}")

W_eff0 = build_weff(up0, dn0)
H = W_eff0.shape[0]
print(f"[ok] inferred hidden size H={H} from W_eff shape={tuple(W_eff0.shape)}")
if CFG.EVAL_MODE not in ("weff", "ffn"):
    raise ValueError("EVAL_MODE must be 'weff' or 'ffn'")

# ============================================================
# Calibration X
# ============================================================

def load_or_make_X() -> torch.Tensor:
    if CFG.CALIB_PATH is None:
        X = torch.randn(CFG.CALIB_SAMPLES, H, device=DEVICE, dtype=torch.float32)
        print(f"[calib] Using RANDOM X: {tuple(X.shape)} (best is real activations npz with key 'X')")
        return X
    if not os.path.exists(CFG.CALIB_PATH):
        raise FileNotFoundError(f"CALIB_PATH not found: {CFG.CALIB_PATH}")
    data = np.load(CFG.CALIB_PATH, allow_pickle=False)
    if "X" in data.files:
        Xn = data["X"]
    else:
        Xn = data[data.files[0]]
    X = torch.from_numpy(Xn).to(device=DEVICE, dtype=torch.float32)
    if X.ndim != 2:
        raise ValueError(f"Expected X to be 2D [N,H], got {X.shape}")
    if X.shape[1] != H:
        raise ValueError(f"X has H={X.shape[1]} but model hidden H={H}")
    if X.shape[0] > CFG.CALIB_SAMPLES:
        X = X[:CFG.CALIB_SAMPLES]
    print(f"[calib] Loaded X: {tuple(X.shape)} from {CFG.CALIB_PATH}")
    return X

X = load_or_make_X()

# ============================================================
# Core math: quantization, SmoothQuant scaling, Hadamard rotation
# ============================================================

def rel_frob(A: torch.Tensor, B: torch.Tensor, eps=1e-12) -> float:
    num = torch.linalg.norm(A - B)
    den = torch.linalg.norm(B) + eps
    return float((num / den).item())

def groupwise_sym_quant(W: torch.Tensor, bits: int, group_size: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Symmetric per-row, grouped over columns.
    Returns: (Wq_float, scales [rows, n_groups])
    """
    assert W.ndim == 2
    n_in, n_out = W.shape
    if n_out % group_size != 0:
        # pad columns for grouping
        pad = group_size - (n_out % group_size)
        Wp = torch.cat([W, torch.zeros(n_in, pad, device=W.device, dtype=W.dtype)], dim=1)
    else:
        Wp = W
        pad = 0
    _, n_out2 = Wp.shape
    ng = n_out2 // group_size
    qmax = (1 << (bits - 1)) - 1

    Wv = Wp.view(n_in, ng, group_size)
    amax = Wv.abs().amax(dim=2).clamp_min(1e-8)
    scale = amax / qmax
    Q = torch.round(Wv / scale.unsqueeze(-1)).clamp(-qmax, qmax)
    Wq = (Q * scale.unsqueeze(-1)).view(n_in, n_out2)
    if pad:
        Wq = Wq[:, :n_out]
    return Wq, scale

# ---- Fast Hadamard Transform (unnormalized) ----
def fht_lastdim(x: torch.Tensor) -> torch.Tensor:
    """
    Unnormalized fast Hadamard transform along last dim.
    Requires power-of-2 last dim.
    """
    n = x.shape[-1]
    if n & (n - 1) != 0:
        raise ValueError(f"Hadamard requires power-of-2, got n={n}")
    y = x
    h = 1
    while h < n:
        y = y.view(*y.shape[:-1], -1, 2 * h)
        a = y[..., :, :h]
        b = y[..., :, h:2*h]
        y = torch.cat([a + b, a - b], dim=-1)
        y = y.view(*y.shape[:-2], -1)
        h *= 2
    return y

@dataclass
class HadamardParams:
    perm: np.ndarray   # shape [H]
    signs: np.ndarray  # shape [H], entries in {-1,+1}

def apply_hadamard_to_X(X: torch.Tensor, hp: HadamardParams) -> torch.Tensor:
    perm = torch.from_numpy(hp.perm).to(device=X.device)
    signs = torch.from_numpy(hp.signs).to(device=X.device, dtype=X.dtype)
    n = X.shape[-1]
    norm = math.sqrt(n)
    Y = fht_lastdim(X) / norm
    Y = Y * signs
    Y = Y[:, perm]
    return Y

def apply_hadamard_to_W_left(W: torch.Tensor, hp: HadamardParams) -> torch.Tensor:
    """
    For row-vector convention y = X @ W.
    If X' = X @ R, then W' = R^T @ W.
    Here R = (1/sqrt(n)) H D P  ->  R^T = P^T D H / sqrt(n)
    """
    perm = torch.from_numpy(hp.perm).to(device=W.device)
    signs = torch.from_numpy(hp.signs).to(device=W.device, dtype=W.dtype)
    inv_perm = torch.empty_like(perm)
    inv_perm[perm] = torch.arange(perm.numel(), device=W.device)
    n = W.shape[0]
    norm = math.sqrt(n)
    Wh = fht_lastdim(W.T).T / norm  # H @ W / sqrt(n)
    Wh = Wh * signs.view(-1, 1)     # D @ ...
    Wrot = Wh[inv_perm, :]          # P^T @ ...
    return Wrot.contiguous()

def make_hadamard_params(n: int, seed: int) -> HadamardParams:
    rng = np.random.default_rng(seed)
    perm = np.arange(n, dtype=np.int64)
    rng.shuffle(perm)
    signs = rng.choice([-1.0, 1.0], size=(n,)).astype(np.float32)
    return HadamardParams(perm=perm, signs=signs)

# ---- SmoothQuant-like scaling (row-vector convention) ----
@dataclass
class SmoothQuantParams:
    alpha: float
    s: np.ndarray  # shape [H] positive

def smoothquant_scale(X: torch.Tensor, W: torch.Tensor, alpha: float, eps=1e-6) -> Tuple[torch.Tensor, torch.Tensor, SmoothQuantParams]:
    """
    Row-vector convention y = X @ W, with X shape [N,H], W [H,Out]
    Scale per input channel j:
       X' = X / s
       W' = diag(s) @ W   (scales rows of W)
    Choose s_j ~ (act_scale_j^alpha) / (w_scale_j^(1-alpha)).
    """
    # activation scale per channel
    act = X.abs().amax(dim=0).clamp_min(eps)          # [H]
    wsc = W.abs().amax(dim=1).clamp_min(eps)          # [H] (row scale)
    s = (act ** alpha) / (wsc ** (1.0 - alpha))
    s = s.clamp_min(eps)

    Xs = X / s.view(1, -1)
    Ws = W * s.view(-1, 1)
    params = SmoothQuantParams(alpha=float(alpha), s=s.detach().cpu().numpy().astype(np.float32))
    return Xs, Ws, params

# ---- GPTQ-lite (blockwise, per-row) ----
def gptq_lite_rowwise(W: torch.Tensor, X: torch.Tensor, bits: int, group_size: int, block: int, damp: float) -> torch.Tensor:
    """
    Very simplified GPTQ-like: uses covariance H = X^T X / N + damp*I, and quantizes each row with blockwise error feedback.
    This is not a full reproduction of GPTQ, but often beats plain RTN on hard rows.
    """
    H = (X.T @ X) / X.shape[0]
    H = H + damp * torch.eye(H.shape[0], device=H.device, dtype=H.dtype)

    # precompute block inverses (cheap-ish if block small)
    n = H.shape[0]
    Wq = W.clone()

    qmax = (1 << (bits - 1)) - 1

    for i in range(W.shape[0]):
        w = Wq[i].clone()
        err = torch.zeros_like(w)

        for j0 in range(0, n, block):
            j1 = min(n, j0 + block)
            Hb = H[j0:j1, j0:j1]
            # cholesky
            L = torch.linalg.cholesky(Hb)
            # block weights with accumulated error
            wb = (w[j0:j1] + err[j0:j1]).contiguous()

            # groupwise within the block (still grouped over columns)
            # here "columns" are the same axis because we're in a row vector.
            # approximate by per-group scale inside the block
            # (good enough for a lite implementation)
            # pad to group size
            m = wb.numel()
            if m % group_size != 0:
                pad = group_size - (m % group_size)
                wb2 = torch.cat([wb, torch.zeros(pad, device=W.device, dtype=W.dtype)])
            else:
                wb2 = wb
                pad = 0
            wbv = wb2.view(-1, group_size)
            amax = wbv.abs().amax(dim=1).clamp_min(1e-8)
            scale = amax / qmax
            qb = torch.round(wbv / scale.unsqueeze(-1)).clamp(-qmax, qmax)
            w_hat = (qb * scale.unsqueeze(-1)).view(-1)
            if pad:
                w_hat = w_hat[:m]

            # write back quantized block
            w[j0:j1] = w_hat

            # update error via solving Hb * delta ≈ (w_hat - wb)
            # delta = Hb^{-1}(wb - w_hat)
            rhs = (wb - w_hat)
            # solve Hb^{-1} rhs using cholesky
            delta = torch.cholesky_solve(rhs.view(-1, 1), L).view(-1)
            err[j0:j1] = err[j0:j1] + delta

        Wq[i] = w

    return Wq

# ============================================================
# Evaluation functions for EVAL_MODE
# ============================================================

def eval_weff_error(X: torch.Tensor, W: torch.Tensor, Wq: torch.Tensor) -> float:
    y_ref = X @ W
    y_hat = X @ Wq
    return rel_frob(y_hat, y_ref)

def silu(x: torch.Tensor) -> torch.Tensor:
    return x * torch.sigmoid(x)

def eval_ffn_error(X: torch.Tensor, gate: torch.Tensor, up: torch.Tensor, down: torch.Tensor,
                   gate_q: torch.Tensor, up_q: torch.Tensor, down_q: torch.Tensor) -> float:
    """
    Tries a Llama-style SwiGLU:
      h = silu(X @ gate^T) * (X @ up^T)
      y = h @ down^T
    We infer orientation by matching hidden size H on the last dimension of X.
    """
    def matmul_infer(X, W):
        # If W is [out,in], use X @ W.T
        if W.shape[1] == X.shape[1]:
            return X @ W.T
        # If W is [in,out], use X @ W
        if W.shape[0] == X.shape[1]:
            return X @ W
        raise RuntimeError(f"Cannot align X{tuple(X.shape)} with W{tuple(W.shape)}")

    h_ref = silu(matmul_infer(X, gate)) * matmul_infer(X, up)
    y_ref = matmul_infer(h_ref, down)

    h_hat = silu(matmul_infer(X, gate_q)) * matmul_infer(X, up_q)
    y_hat = matmul_infer(h_hat, down_q)

    return rel_frob(y_hat, y_ref)

# ============================================================
# Per-expert method runner: baseline / smoothquant / hadamard / combos / gptq-lite
# ============================================================

def run_methods_for_expert(eid: int) -> Dict[str, Any]:
    gate, up, dn = get_expert_weights(eid)

    # Build the operator to quantize for weff mode
    W = build_weff(up, dn)  # [H,H] usually
    if W.shape[0] != H or W.shape[1] != H:
        raise RuntimeError(f"W_eff is not [H,H]: got {tuple(W.shape)}")

    out: Dict[str, Any] = {
        "expert_id": eid,
        "has_gate": gate is not None,
        "results": {}  # method -> bits -> error
    }

    def eval_method_bits(name: str, X_use: torch.Tensor, W_use: torch.Tensor,
                         hp: Optional[HadamardParams]=None,
                         sq: Optional[SmoothQuantParams]=None,
                         use_gptq: bool=False) -> Dict[int, float]:
        errs = {}
        for b in CFG.BITS_LIST:
            if use_gptq:
                Wq = gptq_lite_rowwise(W_use, X_use, bits=b, group_size=CFG.GROUP_SIZE,
                                       block=CFG.GPTQ_BLOCK, damp=CFG.GPTQ_DAMP)
            else:
                Wq, _ = groupwise_sym_quant(W_use, bits=b, group_size=CFG.GROUP_SIZE)

            if CFG.EVAL_MODE == "weff":
                err = eval_weff_error(X_use, W_use, Wq)
            else:
                # ffn mode: quantize gate/up/down separately (more realistic)
                if gate is None:
                    # fallback to weff if gate missing
                    err = eval_weff_error(X_use, W_use, Wq)
                else:
                    # For ffn mode, we quantize *separately* (do NOT use W_eff)
                    # We'll produce gate_q/up_q/down_q using the same method:
                    # - hadamard is applied on X only if we also rotate the matching weight side (handled below)
                    # - smoothquant is applied via scaling on X and scaling rows of each weight (consistent)
                    # Here we simply quant each matrix with same groupwise scheme.
                    gate_use, up_use, dn_use = gate, up, dn

                    # If SmoothQuant params exist, apply consistent scaling on each input-channel:
                    # This requires H to match the input dim of each matrix. We apply only when compatible.
                    if sq is not None:
                        s = torch.from_numpy(sq.s).to(device=DEVICE, dtype=torch.float32)
                        def apply_sq_to_W_for_X(Wm):
                            # row-vector convention scaling expects W as [H, out] (rows correspond to input channels).
                            # In ffn we allow both [out,H] and [H,out]. We adapt:
                            if Wm.shape[0] == H:
                                return Wm * s.view(-1,1)
                            if Wm.shape[1] == H:
                                return Wm * s.view(1,-1)  # effectively scaling the input channels
                            return Wm

                        gate_use = apply_sq_to_W_for_X(gate_use)
                        up_use   = apply_sq_to_W_for_X(up_use)
                        dn_use   = dn_use  # dn takes hidden->?; scaling depends on its input; skip unless matched

                    # Quantize each
                    gate_q, _ = groupwise_sym_quant(
                        gate_use if gate_use.shape[0] == H else gate_use.T, bits=b, group_size=CFG.GROUP_SIZE
                    )
                    up_q, _ = groupwise_sym_quant(
                        up_use if up_use.shape[0] == H else up_use.T, bits=b, group_size=CFG.GROUP_SIZE
                    )
                    # down quant: treat its input dim as hidden; transpose if needed
                    dn_q, _ = groupwise_sym_quant(
                        dn_use.T if dn_use.shape[1] == H else dn_use, bits=b, group_size=CFG.GROUP_SIZE
                    )

                    # restore original orientations
                    if gate_use.shape[0] != H: gate_q = gate_q.T
                    if up_use.shape[0] != H:   up_q = up_q.T
                    if dn_use.shape[1] == H:   dn_q = dn_q.T

                    err = eval_ffn_error(X_use, gate_use, up_use, dn_use, gate_q, up_q, dn_q)

            errs[int(b)] = float(err)
        out["results"][name] = {"errs": errs}
        if hp is not None:
            out["results"][name]["hadamard"] = {"perm": hp.perm.tolist(), "signs": hp.signs.tolist()}
        if sq is not None:
            out["results"][name]["smoothquant"] = {"alpha": sq.alpha}  # store s in payload file, not in JSON
        return errs

    # ---------------- baseline ----------------
    eval_method_bits("base", X, W)

    # ---------------- SmoothQuant grid search ----------------
    best_sq = None
    best_sq_err4 = 1e9
    best_sq_params = None
    for a in CFG.SMOOTHQUANT_ALPHA_GRID:
        Xs, Ws, params = smoothquant_scale(X, W, alpha=a)
        errs = eval_method_bits(f"smoothq_a{a:.2f}", Xs, Ws, sq=params)
        # pick by 4-bit if present, else smallest bit
        keyb = 4 if 4 in errs else min(errs.keys())
        if errs[keyb] < best_sq_err4:
            best_sq_err4 = errs[keyb]
            best_sq = (Xs, Ws)
            best_sq_params = params

    out["best_smoothq"] = {"alpha": float(best_sq_params.alpha), "best_keybit": 4, "err_at_4b": float(best_sq_err4)}

    # ---------------- Hadamard rotation search (keep best) ----------------
    if CFG.TRY_HADAMARD:
        best_h_err4 = 1e9
        best_hp = None

        # We search on baseline (no SQ) first
        for t in range(CFG.HADAMARD_TRIES):
            seed = CFG.HADAMARD_SEED0 + 1000*eid + t
            hp = make_hadamard_params(H, seed)
            Xh = apply_hadamard_to_X(X, hp)
            Wh = apply_hadamard_to_W_left(W, hp)
            errs = eval_method_bits(f"hadam_t{t}", Xh, Wh, hp=hp)
            keyb = 4 if 4 in errs else min(errs.keys())
            if errs[keyb] < best_h_err4:
                best_h_err4 = errs[keyb]
                best_hp = hp

        out["best_hadam"] = {"best_keybit": 4, "err_at_4b": float(best_h_err4)}

        # Combine best SmoothQuant + Hadamard (search hadam again but on SQ-scaled)
        if best_sq is not None and best_sq_params is not None:
            Xs, Ws = best_sq
            best_sh_err4 = 1e9
            best_sh_hp = None
            for t in range(CFG.HADAMARD_TRIES):
                seed = CFG.HADAMARD_SEED0 + 50000 + 1000*eid + t
                hp = make_hadamard_params(H, seed)
                Xsh = apply_hadamard_to_X(Xs, hp)
                Wsh = apply_hadamard_to_W_left(Ws, hp)
                errs = eval_method_bits(f"smoothq+hadam_t{t}", Xsh, Wsh, hp=hp, sq=best_sq_params)
                keyb = 4 if 4 in errs else min(errs.keys())
                if errs[keyb] < best_sh_err4:
                    best_sh_err4 = errs[keyb]
                    best_sh_hp = hp
            out["best_smoothq_hadam"] = {"best_keybit": 4, "err_at_4b": float(best_sh_err4)}

    # ---------------- GPTQ-lite (optional) ----------------
    if CFG.RUN_GPTQ_LITE:
        eval_method_bits("gptq_lite", X, W, use_gptq=True)

    return out, best_sq_params

# ============================================================
# Run all experts
# ============================================================

all_results = []
best_sq_params_per_expert: Dict[int, SmoothQuantParams] = {}

for eid in tqdm(expert_ids, desc="Per-expert quant eval"):
    r, sqp = run_methods_for_expert(eid)
    all_results.append(r)
    if sqp is not None:
        best_sq_params_per_expert[eid] = sqp

# ============================================================
# Summaries + payload export
# ============================================================

# Utility: choose best method per expert for a target bit (default 4)
TARGET_BIT = 4

def best_method_for_expert(r: Dict[str, Any], target_bit=TARGET_BIT) -> Tuple[str, float]:
    best_name, best_err = None, 1e9
    for mname, md in r["results"].items():
        errs = md["errs"]
        if str(target_bit) in errs:
            e = float(errs[str(target_bit)])
        elif target_bit in errs:
            e = float(errs[target_bit])
        else:
            # fallback to min bit
            e = float(errs[min(errs.keys())])
        if e < best_err:
            best_err = e
            best_name = mname
    return best_name, best_err

best_picks = {}
for r in all_results:
    name, err = best_method_for_expert(r, TARGET_BIT)
    best_picks[int(r["expert_id"])] = {"method": name, "err": float(err)}

# Save JSON report
report = {
    "cfg": asdict(CFG),
    "ts": time.time(),
    "hidden": H,
    "experts": expert_ids,
    "target_bit": TARGET_BIT,
    "best_picks": best_picks,
    "per_expert": all_results,
}

report_path = os.path.join(CFG.OUTPUT_DIR, f"quant_report_layer{CFG.LAYER}_E{len(expert_ids)}_g{CFG.GROUP_SIZE}.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print(f"\n[done] Wrote report: {report_path}")

# Save NPZ payload: store best SmoothQuant s vectors per expert (and optionally best hadam perms/signs if present)
payload = {}
payload["expert_ids"] = np.array(expert_ids, dtype=np.int32)

for r in all_results:
    eid = int(r["expert_id"])
    # store SmoothQuant s (best alpha across grid)
    if eid in best_sq_params_per_expert:
        payload[f"sq_s_e{eid}"] = best_sq_params_per_expert[eid].s.astype(np.float32)
        payload[f"sq_alpha_e{eid}"] = np.array([best_sq_params_per_expert[eid].alpha], dtype=np.float32)

# also store the BEST hadam params we found (if any were tried) by scanning methods that start with "hadam_"
# NOTE: We stored perms/signs per method in JSON only; NPZ payload stays small; you can extend if needed.
payload_path = os.path.join(CFG.OUTPUT_DIR, f"quant_payload_layer{CFG.LAYER}_E{len(expert_ids)}.npz")
np.savez_compressed(payload_path, **payload)
print(f"[done] Wrote payload: {payload_path} ({_human_mb(os.path.getsize(payload_path)):.2f} MB)")

# Quick snapshot at 4-bit: base vs best smoothq vs best hadam (if present)
def avg_err(method_prefix: str, bit=4) -> Tuple[float, float]:
    vals = []
    for r in all_results:
        for mname, md in r["results"].items():
            if mname == method_prefix:
                errs = md["errs"]
                v = errs.get(bit, errs.get(str(bit), None))
                if v is not None:
                    vals.append(float(v))
    if not vals:
        return float("nan"), float("nan")
    return float(np.mean(vals)), float(np.std(vals))

base_m, base_s = avg_err("base", 4)
print("\n== Quick 4-bit snapshot (avg y_rel_frob) ==")
print(f"base_4b_g{CFG.GROUP_SIZE:3d} : {base_m:.6f} (±{base_s:.6f})")
print("\nTip: If SmoothQuant/Hadamard look bad with RANDOM X, dump real X activations and rerun.")


== OFFLINE DeepSeek-MoE Quant Runner v3 ==
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS: 8
EVAL_MODE:   weff
Torch:       2.4.1+cpu | device=cpu | threads=8
[found] experts for layer 1: [0, 1, 2, 3, 4, 5, 6, 7] (using 8)
[cache] building expert cache (this reads shards; can take a bit) ...


load experts:   0%|          | 0/8 [00:00<?, ?it/s]

[cache] wrote: /home/daniyar/moe_ws_outputs/deepseek_layer1_E8_cache.npz (101.69 MB)
[ok] inferred hidden size H=2048 from W_eff shape=(2048, 2048)
[calib] Using RANDOM X: (512, 2048) (best is real activations npz with key 'X')


Per-expert quant eval:   0%|          | 0/8 [00:00<?, ?it/s]


[done] Wrote report: /home/daniyar/moe_ws_outputs/quant_report_layer1_E8_g128.json
[done] Wrote payload: /home/daniyar/moe_ws_outputs/quant_payload_layer1_E8.npz (0.06 MB)

== Quick 4-bit snapshot (avg y_rel_frob) ==
base_4b_g128 : 0.118365 (±0.000753)

Tip: If SmoothQuant/Hadamard look bad with RANDOM X, dump real X activations and rerun.


In [9]:
# ============================================================
# OFFLINE DeepSeek-MoE Quant Runner (single-file, terminal-first)
# - No HuggingFace downloads
# - Reads local shards via model.safetensors.index.json
# - Builds/loads expert cache for a given layer
# - Evaluates groupwise quant (base / hadamard / smoothquant / smooth+hadam)
# - Prints all results to terminal (no separate report files)
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np

# ---- torch is required for speed; keep CPU-friendly ----
import torch

# ---- safetensors required to read shards ----
try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError(
        "Missing dependency: safetensors. Install with: pip install safetensors"
    ) from e

# ---- tqdm for progress ----
try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    tqdm = None
    _HAS_TQDM = False

# ----------------------------
# Terminal-safe logger
# ----------------------------
def log(msg: str):
    if _HAS_TQDM:
        try:
            tqdm.write(str(msg))
            return
        except Exception:
            pass
    print(msg, flush=True)

# ----------------------------
# Config (override via env vars)
# ----------------------------
MODEL_DIR   = os.environ.get("MODEL_DIR",  "/home/daniyar/deepseek-model")
OUTPUT_DIR  = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")  # only for optional cache
LAYER       = int(os.environ.get("LAYER", "1"))
MAX_EXPERTS = int(os.environ.get("MAX_EXPERTS", "8"))

# cache behavior:
#  - "off": never read/write caches
#  - "read": read if exists, else build but do NOT write
#  - "write": always build and write
#  - "readwrite": read if exists else build+write (recommended)
CACHE_MODE  = os.environ.get("CACHE_MODE", "readwrite").lower()

# calibration:
CALIB_PATH     = os.environ.get("CALIB_PATH", "").strip()  # optional npz with key 'X'
CALIB_SAMPLES  = int(os.environ.get("CALIB_SAMPLES", "512"))

# quant:
GROUP_SIZE   = int(os.environ.get("GROUP_SIZE", "128"))
BITS_LIST    = os.environ.get("BITS", "8,4,3,2")
BITS         = [int(x) for x in BITS_LIST.split(",") if x.strip()]

# smoothquant:
RUN_SMOOTHQUANT = os.environ.get("RUN_SMOOTHQUANT", "1") == "1"
SQ_ALPHA        = float(os.environ.get("SQ_ALPHA", "0.5"))

# hadamard rotation:
RUN_HADAMARD    = os.environ.get("RUN_HADAMARD", "0") == "1"  # default off until you trust calib X
HADAMARD_SEED   = int(os.environ.get("HADAMARD_SEED", "1234"))

# eval mode:
#  - "weff": evaluate W_eff (down@up) mapping hidden->hidden
EVAL_MODE   = os.environ.get("EVAL_MODE", "weff").lower()

# threading
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

# ----------------------------
# Helpers
# ----------------------------
def _must_exist(path: str, what: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{what} not found: {path}")

def _read_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _human_gb(nbytes: int) -> str:
    return f"{nbytes / (1024**3):.2f} GB"

def _np_load_any(path: str) -> dict:
    z = np.load(path, allow_pickle=False)
    return {k: z[k] for k in z.files}

# ----------------------------
# Hadamard / FWHT (fast Walsh-Hadamard transform)
# Works on last dimension, expects n=power of 2.
# Returns normalized orthogonal transform (divide by sqrt(n)).
# ----------------------------
@torch.no_grad()
def fwht_(x: torch.Tensor) -> torch.Tensor:
    # in-place-like transform using views; returns new tensor for safety
    # x: (..., n)
    n = x.shape[-1]
    if n & (n - 1) != 0:
        raise ValueError(f"FWHT requires power-of-2 length, got n={n}")
    y = x.contiguous()
    h = 1
    while h < n:
        # reshape to (..., n/(2h), 2h)
        y = y.view(*y.shape[:-1], n // (2 * h), 2 * h)
        a = y[..., :, :h]
        b = y[..., :, h:2*h]
        # compute (a+b, a-b)
        y = torch.cat([a + b, a - b], dim=-1)
        # flatten back
        y = y.view(*y.shape[:-2], n)
        h *= 2
    y = y / math.sqrt(n)
    return y

@torch.no_grad()
def make_hadamard_recipe(n: int, seed: int = 1234) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Build sign vector d (+/-1) and permutation inv_perm for applying R^T = D H P^T
    to row-vectors (x @ R^T).
    We'll implement:
      apply_Rt(M):  M = (M * d) -> FWHT -> M[:, inv_perm]
    """
    g = torch.Generator(device="cpu").manual_seed(seed)
    d = torch.randint(low=0, high=2, size=(n,), generator=g, dtype=torch.int8)
    d = (2 * d - 1).to(torch.float32)  # +/-1
    perm = torch.randperm(n, generator=g)
    inv_perm = torch.empty_like(perm)
    inv_perm[perm] = torch.arange(n)
    return d, inv_perm

@torch.no_grad()
def apply_Rt_rowmat(M: torch.Tensor, d: torch.Tensor, inv_perm: torch.Tensor) -> torch.Tensor:
    # M: (..., n), row-vector convention
    # step1: multiply by d (signs)
    X = M * d
    # step2: hadamard
    X = fwht_(X)
    # step3: permute columns by inv_perm (P^T)
    X = X.index_select(dim=-1, index=inv_perm)
    return X

# ----------------------------
# Groupwise symmetric quant-dequant
# Convention: signed range [-2^(b-1), 2^(b-1)-1]  (2^b levels)
# ----------------------------
@torch.no_grad()
def quant_dequant_groupwise(W: torch.Tensor, bits: int, group_size: int) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    W: (out, in) float32/float16
    Returns:
      Wdq: dequantized float32
      stats: dict (scale_mean, w_rel_frob, etc.)
    """
    Wf = W.to(torch.float32)
    out, inn = Wf.shape
    if inn % group_size != 0:
        raise ValueError(f"in_features={inn} not divisible by group_size={group_size}")
    ng = inn // group_size
    Wg = Wf.view(out, ng, group_size)

    denom = float(2 ** (bits - 1))  # max abs of integer range
    qmin = -int(2 ** (bits - 1))
    qmax = int(2 ** (bits - 1) - 1)

    maxabs = Wg.abs().amax(dim=-1, keepdim=True)  # (out, ng, 1)
    scale = (maxabs / denom).clamp_min(1e-8)      # avoid div0
    Q = torch.round(Wg / scale).clamp(qmin, qmax)
    Wdq = (Q * scale).view(out, inn)

    # stats
    w_frob = torch.linalg.norm(Wf, ord="fro")
    err_frob = torch.linalg.norm(Wdq - Wf, ord="fro")
    w_rel = float((err_frob / (w_frob + 1e-12)).item())
    stats = {
        "bits": float(bits),
        "group_size": float(group_size),
        "scale_mean": float(scale.mean().item()),
        "scale_max": float(scale.max().item()),
        "w_rel_frob": w_rel,
    }
    return Wdq, stats

# ----------------------------
# SmoothQuant pre-scaling (activation-aware)
# x' = x * s ; W' = W / s  (column scaling)
# s_i = (amax_i^alpha) / (wmax_i^(1-alpha))
# ----------------------------
@torch.no_grad()
def smoothquant_scale(X: torch.Tensor, W: torch.Tensor, alpha: float) -> torch.Tensor:
    # X: (N, in), W: (out, in)
    eps = 1e-8
    amax = X.abs().amax(dim=0).to(torch.float32) + eps
    wmax = W.abs().amax(dim=0).to(torch.float32) + eps
    s = (amax ** alpha) / (wmax ** (1.0 - alpha))
    s = s.clamp_min(1e-6).clamp_max(1e6)
    return s

# ----------------------------
# Load DeepSeek shards offline
# ----------------------------
@dataclass
class DeepSeekIndex:
    weight_map: Dict[str, str]
    shards: List[str]

def load_index(model_dir: str) -> DeepSeekIndex:
    index_path = os.path.join(model_dir, "model.safetensors.index.json")
    _must_exist(index_path, "index json")
    idx = _read_json(index_path)
    weight_map = idx.get("weight_map", {})
    if not weight_map:
        raise ValueError("index json has empty weight_map")
    shards = sorted(set(weight_map.values()))
    return DeepSeekIndex(weight_map=weight_map, shards=shards)

def find_layer_experts(weight_map: Dict[str, str], layer: int) -> Dict[int, Dict[str, str]]:
    """
    Return mapping:
      expert_id -> dict with keys: up, down, gate (if found)
    Supports naming variants:
      ...experts.{e}.w1.weight / w2.weight / w3.weight
      ...experts.{e}.up_proj.weight / down_proj.weight / gate_proj.weight
    """
    layer_pat = rf"(?:^|\.)(?:model\.)?layers\.{layer}\.mlp\.experts\.(\d+)\."
    re_layer = re.compile(layer_pat)

    # collect all keys for this layer experts
    per_expert_keys: Dict[int, List[str]] = {}
    for k in weight_map.keys():
        m = re_layer.search(k)
        if not m:
            continue
        e = int(m.group(1))
        per_expert_keys.setdefault(e, []).append(k)

    def pick_key(keys: List[str], suffixes: List[str]) -> Optional[str]:
        for s in suffixes:
            for k in keys:
                if k.endswith(s):
                    return k
        return None

    out: Dict[int, Dict[str, str]] = {}
    for e, keys in per_expert_keys.items():
        # candidates
        up_key = pick_key(keys, ["up_proj.weight", "w1.weight"])
        down_key = pick_key(keys, ["down_proj.weight", "w2.weight"])
        gate_key = pick_key(keys, ["gate_proj.weight", "w3.weight"])  # may not exist

        if up_key is None or down_key is None:
            # skip incomplete expert
            continue
        out[e] = {"up": up_key, "down": down_key}
        if gate_key is not None:
            out[e]["gate"] = gate_key

    return out

class ShardLoader:
    def __init__(self, model_dir: str, weight_map: Dict[str, str]):
        self.model_dir = model_dir
        self.weight_map = weight_map
        self._handles: Dict[str, object] = {}

    def _open(self, shard_name: str):
        if shard_name in self._handles:
            return self._handles[shard_name]
        path = os.path.join(self.model_dir, shard_name)
        _must_exist(path, f"shard {shard_name}")
        h = safe_open(path, framework="pt", device="cpu")
        self._handles[shard_name] = h
        return h

    def get_tensor(self, key: str) -> torch.Tensor:
        shard = self.weight_map.get(key, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {key}")
        h = self._open(shard)
        return h.get_tensor(key)

# ----------------------------
# Expert cache (optional)
# ----------------------------
def cache_path(layer: int, E: int) -> str:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    return os.path.join(OUTPUT_DIR, f"deepseek_layer{layer}_E{E}_cache.npz")

@torch.no_grad()
def build_expert_cache(model_dir: str, layer: int, max_experts: int, cache_mode: str) -> Tuple[Dict[str, np.ndarray], List[int]]:
    """
    Returns:
      cache dict with keys:
        - W_eff: (E,H,H) float16
        - up:    (E,I,H) float16
        - down:  (E,H,I) float16
        - gate:  (E,I,H) float16  (optional)
      expert_ids: list of expert ids
    """
    idx = load_index(model_dir)
    ex_map = find_layer_experts(idx.weight_map, layer)
    if not ex_map:
        raise RuntimeError(f"No experts found for layer {layer}. Check naming / layer id.")

    expert_ids = sorted(ex_map.keys())[:max_experts]
    E = len(expert_ids)
    cpath = cache_path(layer, E)

    # read cache?
    can_read = (cache_mode in ("read", "readwrite")) and os.path.exists(cpath)
    if can_read:
        log(f"[cache] Loading existing expert cache: {cpath} ({os.path.getsize(cpath)/1e6:.2f} MB)")
        z = _np_load_any(cpath)
        return z, expert_ids

    # build
    log("[cache] building expert cache (reads shards) ...")
    loader = ShardLoader(model_dir, idx.weight_map)

    ups, downs, gates, weffs = [], [], [], []
    it = expert_ids
    if _HAS_TQDM:
        it = tqdm(expert_ids, desc="load experts", total=len(expert_ids))
    for e in it:
        keys = ex_map[e]
        up = loader.get_tensor(keys["up"]).to(torch.float32)     # (I,H) or (I,H) depending
        down = loader.get_tensor(keys["down"]).to(torch.float32) # (H,I)
        gate = None
        if "gate" in keys:
            gate = loader.get_tensor(keys["gate"]).to(torch.float32)

        # Ensure shapes are (out,in) convention for applying y = x @ W^T
        # up is usually (I,H) (out=I, in=H), down is (H,I) (out=H, in=I)
        if up.ndim != 2 or down.ndim != 2:
            raise ValueError(f"Unexpected tensor rank for expert {e}: up {up.shape}, down {down.shape}")
        I, H = up.shape
        H2, I2 = down.shape
        if H2 != H or I2 != I:
            raise ValueError(f"Shape mismatch expert {e}: up {up.shape}, down {down.shape}")

        # W_eff = down @ up  => (H,H)
        weff = down @ up

        ups.append(up.to(torch.float16).cpu().numpy())
        downs.append(down.to(torch.float16).cpu().numpy())
        weffs.append(weff.to(torch.float16).cpu().numpy())
        if gate is not None:
            if gate.shape != up.shape:
                # tolerate mismatch, but warn
                log(f"[warn] expert {e}: gate shape {tuple(gate.shape)} != up shape {tuple(up.shape)} (still caching gate)")
            gates.append(gate.to(torch.float16).cpu().numpy())

    cache: Dict[str, np.ndarray] = {
        "W_eff": np.stack(weffs, axis=0),
        "up":    np.stack(ups, axis=0),
        "down":  np.stack(downs, axis=0),
    }
    if len(gates) == E:
        cache["gate"] = np.stack(gates, axis=0)

    can_write = cache_mode in ("write", "readwrite")
    if can_write:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        np.savez(cpath, **cache)
        log(f"[cache] wrote: {cpath} ({os.path.getsize(cpath)/1e6:.2f} MB)")
    else:
        log("[cache] not writing cache (CACHE_MODE does not allow writes).")

    return cache, expert_ids

# ----------------------------
# Calibration loader
# ----------------------------
def load_calib_X(H: int, calib_path: str, calib_samples: int) -> torch.Tensor:
    if calib_path and os.path.exists(calib_path):
        z = _np_load_any(calib_path)
        if "X" not in z:
            raise KeyError(f"CALIB_PATH npz must contain key 'X'. Keys: {list(z.keys())}")
        X = z["X"]
        if X.ndim != 2 or X.shape[1] != H:
            raise ValueError(f"Bad X shape: {X.shape}, expected (N,{H})")
        X = X[:calib_samples].astype(np.float32, copy=False)
        log(f"[calib] Using REAL X from {calib_path}: {X.shape}")
        return torch.from_numpy(X).to(torch.float32)
    # random fallback
    X = torch.randn(calib_samples, H, dtype=torch.float32)
    log(f"[calib] Using RANDOM X: {tuple(X.shape)} (best is real activations npz with key 'X')")
    return X

# ----------------------------
# Evaluation
# ----------------------------
@torch.no_grad()
def y_rel_frob(y_hat: torch.Tensor, y_ref: torch.Tensor) -> float:
    num = torch.linalg.norm(y_hat - y_ref, ord="fro")
    den = torch.linalg.norm(y_ref, ord="fro") + 1e-12
    return float((num / den).item())

@torch.no_grad()
def eval_one_expert(W: torch.Tensor, X: torch.Tensor,
                   bits: int, group_size: int,
                   do_hadamard: bool, do_smoothquant: bool, alpha: float,
                   had_d: Optional[torch.Tensor] = None,
                   had_invperm: Optional[torch.Tensor] = None) -> Dict[str, float]:
    """
    W: (H,H) float16/float32
    X: (N,H) float32
    """
    Wf = W.to(torch.float32)
    y_ref = X @ Wf.t()

    out: Dict[str, float] = {}

    # ---- base quant ----
    Wdq, wstats = quant_dequant_groupwise(Wf, bits=bits, group_size=group_size)
    y_hat = X @ Wdq.t()
    out[f"base_y_rel"] = y_rel_frob(y_hat, y_ref)
    out[f"base_w_rel"] = float(wstats["w_rel_frob"])

    # ---- hadamard rotation (correctly applied to BOTH X and W on input dimension) ----
    if do_hadamard:
        assert had_d is not None and had_invperm is not None
        Xr = apply_Rt_rowmat(X, had_d, had_invperm)
        Wr = apply_Rt_rowmat(Wf, had_d, had_invperm)  # rotate columns
        Wdq_r, wstats_r = quant_dequant_groupwise(Wr, bits=bits, group_size=group_size)
        y_hat_r = Xr @ Wdq_r.t()
        out[f"hadam_y_rel"] = y_rel_frob(y_hat_r, y_ref)
        out[f"hadam_w_rel"] = float(wstats_r["w_rel_frob"])

    # ---- smoothquant ----
    if do_smoothquant:
        s = smoothquant_scale(X, Wf, alpha=alpha)          # (H,)
        Xs = X * s
        Ws = Wf / s.unsqueeze(0)
        Wdq_s, wstats_s = quant_dequant_groupwise(Ws, bits=bits, group_size=group_size)
        y_hat_s = Xs @ Wdq_s.t()
        out[f"smoothq_y_rel"] = y_rel_frob(y_hat_s, y_ref)
        out[f"smoothq_w_rel"] = float(wstats_s["w_rel_frob"])

    # ---- smoothquant + hadamard (apply hadamard first, then compute SQ on rotated X) ----
    if do_hadamard and do_smoothquant:
        Xr = apply_Rt_rowmat(X, had_d, had_invperm)
        Wr = apply_Rt_rowmat(Wf, had_d, had_invperm)
        s2 = smoothquant_scale(Xr, Wr, alpha=alpha)
        Xrs = Xr * s2
        Wrs = Wr / s2.unsqueeze(0)
        Wdq_rs, wstats_rs = quant_dequant_groupwise(Wrs, bits=bits, group_size=group_size)
        y_hat_rs = Xrs @ Wdq_rs.t()
        out[f"smoothq+hadam_y_rel"] = y_rel_frob(y_hat_rs, y_ref)
        out[f"smoothq+hadam_w_rel"] = float(wstats_rs["w_rel_frob"])

    return out

def _agg(vals: List[float]) -> Tuple[float, float]:
    if not vals:
        return float("nan"), float("nan")
    m = float(np.mean(vals))
    s = float(np.std(vals))
    return m, s

def run():
    t_start = time.time()
    log("== OFFLINE DeepSeek-MoE Quant Runner (terminal-first) ==")
    log(f"MODEL_DIR:  {MODEL_DIR}")
    log(f"OUTPUT_DIR: {OUTPUT_DIR}")
    log(f"LAYER:      {LAYER}")
    log(f"MAX_EXPERTS:{MAX_EXPERTS}")
    log(f"Torch:      {torch.__version__} | device={DEVICE} | threads={NTHREADS}")
    log(f"EVAL_MODE:  {EVAL_MODE}")
    log(f"CACHE_MODE: {CACHE_MODE}")
    log(f"GROUP_SIZE: {GROUP_SIZE} | BITS={BITS}")
    log(f"RUN_HADAMARD={RUN_HADAMARD} | RUN_SMOOTHQUANT={RUN_SMOOTHQUANT} (alpha={SQ_ALPHA})")
    if CALIB_PATH:
        log(f"CALIB_PATH: {CALIB_PATH}")
    else:
        log("CALIB_PATH: (none -> random X)")

    # sanity
    _must_exist(MODEL_DIR, "MODEL_DIR")
    _must_exist(os.path.join(MODEL_DIR, "model.safetensors.index.json"), "model.safetensors.index.json")

    # load/build cache
    cache, expert_ids = build_expert_cache(MODEL_DIR, LAYER, MAX_EXPERTS, CACHE_MODE)
    if EVAL_MODE != "weff":
        raise ValueError("Only EVAL_MODE=weff supported in this runner.")
    Ws = torch.from_numpy(cache["W_eff"]).to(torch.float16)  # (E,H,H)

    E, H, H2 = Ws.shape
    assert H == H2
    log(f"[ok] Ws: {tuple(Ws.shape)} | experts={expert_ids}")

    # calib X
    X = load_calib_X(H, CALIB_PATH, CALIB_SAMPLES).to(torch.float32)

    # hadamard recipe
    had_d = had_invperm = None
    if RUN_HADAMARD:
        had_d, had_invperm = make_hadamard_recipe(H, seed=HADAMARD_SEED)
        had_d = had_d.to(torch.float32)
        had_invperm = had_invperm.to(torch.long)

    # run evaluation
    results = {}  # method_bit -> list of per-expert vals
    # structure: results[(bits, key)] = [values...]
    keys = ["base"]
    if RUN_HADAMARD:
        keys.append("hadam")
    if RUN_SMOOTHQUANT:
        keys.append("smoothq")
    if RUN_HADAMARD and RUN_SMOOTHQUANT:
        keys.append("smoothq+hadam")

    # init containers
    for b in BITS:
        for k in keys:
            results[(b, f"{k}_y_rel")] = []
            results[(b, f"{k}_w_rel")] = []

    it = range(E)
    if _HAS_TQDM:
        it = tqdm(it, desc="Per-expert quant eval", total=E)

    for ei in it:
        W = Ws[ei]
        for b in BITS:
            out = eval_one_expert(
                W=W, X=X,
                bits=b, group_size=GROUP_SIZE,
                do_hadamard=RUN_HADAMARD,
                do_smoothquant=RUN_SMOOTHQUANT,
                alpha=SQ_ALPHA,
                had_d=had_d, had_invperm=had_invperm,
            )
            # collect
            results[(b, "base_y_rel")].append(out["base_y_rel"])
            results[(b, "base_w_rel")].append(out["base_w_rel"])
            if RUN_HADAMARD:
                results[(b, "hadam_y_rel")].append(out["hadam_y_rel"])
                results[(b, "hadam_w_rel")].append(out["hadam_w_rel"])
            if RUN_SMOOTHQUANT:
                results[(b, "smoothq_y_rel")].append(out["smoothq_y_rel"])
                results[(b, "smoothq_w_rel")].append(out["smoothq_w_rel"])
            if RUN_HADAMARD and RUN_SMOOTHQUANT:
                results[(b, "smoothq+hadam_y_rel")].append(out["smoothq+hadam_y_rel"])
                results[(b, "smoothq+hadam_w_rel")].append(out["smoothq+hadam_w_rel"])

    # print summary (terminal)
    log("\n==================== SUMMARY (avg ± std over experts) ====================")
    for b in BITS:
        log(f"\n--- bits={b} group={GROUP_SIZE} ---")
        for k in keys:
            y_key = f"{k}_y_rel"
            w_key = f"{k}_w_rel"
            y_m, y_s = _agg(results[(b, y_key)])
            w_m, w_s = _agg(results[(b, w_key)])
            log(f"{k:14s} | y_rel_frob={y_m:.6f} ± {y_s:.6f} | w_rel_frob={w_m:.6f} ± {w_s:.6f}")

    # quick “4-bit snapshot” like you asked (if 4 is present)
    if 4 in BITS:
        y_m, y_s = _agg(results[(4, "base_y_rel")])
        log("\n== Quick 4-bit snapshot (avg y_rel_frob) ==")
        log(f"base_4b_g{GROUP_SIZE}: {y_m:.6f} (±{y_s:.6f})")

    # warning if hadamard is clearly broken (common sign of wrong rotation or random-X mismatch)
    if RUN_HADAMARD and 4 in BITS:
        base_m, _ = _agg(results[(4, "base_y_rel")])
        had_m, _  = _agg(results[(4, "hadam_y_rel")])
        if had_m > 3.0 * (base_m + 1e-9):
            log("\n[warn] Hadamard looks much worse than base. This usually means:")
            log("  - You are using RANDOM X (not representative), OR")
            log("  - You rotate W but not X (or vice versa), OR")
            log("  - You rotate the wrong dimension (must be input/hidden dim).")

    log("\nDone.")
    log(f"Wall time: {time.time() - t_start:.1f}s")

# ----------------------------
# Execute
# ----------------------------
if __name__ == "__main__" or True:
    run()


== OFFLINE DeepSeek-MoE Quant Runner (terminal-first) ==
MODEL_DIR:  /home/daniyar/deepseek-model
OUTPUT_DIR: /home/daniyar/moe_ws_outputs
LAYER:      1
MAX_EXPERTS:8
Torch:      2.4.1+cpu | device=cpu | threads=8
EVAL_MODE:  weff
CACHE_MODE: readwrite
GROUP_SIZE: 128 | BITS=[8, 4, 3, 2]
RUN_HADAMARD=False | RUN_SMOOTHQUANT=True (alpha=0.5)
CALIB_PATH: (none -> random X)
[cache] Loading existing expert cache: /home/daniyar/moe_ws_outputs/deepseek_layer1_E8_cache.npz (106.63 MB)


KeyError: 'W_eff'

In [10]:
# ============================================================
# OFFLINE DeepSeek-MoE Expert Quant Runner (terminal-first)
# - Loads MoE expert weights from local safetensors shards
# - Optional cache .npz (no JSON reports; all results printed)
# - Correct invariance for SmoothQuant and Hadamard rotations
# - Supports EVAL_MODE=mlp (recommended) or EVAL_MODE=weff (proxy)
# ============================================================

import os, re, json, math, time, random
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn.functional as F

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("Missing dependency: safetensors. Install with: pip install safetensors") from e

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # minimal fallback
        return x

# ----------------------------
# Config (override via env vars)
# ----------------------------
MODEL_DIR     = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
OUTPUT_DIR    = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")
LAYER         = int(os.environ.get("LAYER", "1"))
MAX_EXPERTS   = int(os.environ.get("MAX_EXPERTS", "8"))
EVAL_MODE     = os.environ.get("EVAL_MODE", "mlp").strip().lower()  # "mlp" or "weff"
CACHE_MODE    = os.environ.get("CACHE_MODE", "readwrite").strip().lower()  # off|read|write|readwrite
CACHE_PATH    = os.path.join(OUTPUT_DIR, os.environ.get("CACHE_NAME", f"deepseek_layer{LAYER}_E{MAX_EXPERTS}_{EVAL_MODE}_cache_v5.npz"))

# Quant setup
GROUP_SIZE    = int(os.environ.get("GROUP_SIZE", "128"))
BITS_LIST     = [int(x) for x in os.environ.get("BITS", "8,4,3,2").split(",")]

# Calibration
CALIB_PATH    = os.environ.get("CALIB_PATH", "").strip() or None  # npz with key 'X'
CALIB_SAMPLES = int(os.environ.get("CALIB_SAMPLES", "512"))

# SmoothQuant
RUN_SMOOTHQUANT = os.environ.get("RUN_SMOOTHQUANT", "1").strip() not in ("0", "false", "False")
SQ_ALPHA        = float(os.environ.get("SQ_ALPHA", "0.5"))

# Hadamard rotation (input-dim only)
RUN_HADAMARD    = os.environ.get("RUN_HADAMARD", "0").strip() not in ("0", "false", "False")
HAD_SEED        = int(os.environ.get("HAD_SEED", "1234"))

# Torch setup
THREADS      = int(os.environ.get("KTXX_THREADS", "8"))
torch.set_num_threads(THREADS)
DTYPE_W      = torch.float16
DTYPE_ACC    = torch.float32
DEVICE       = torch.device("cpu")

os.makedirs(OUTPUT_DIR, exist_ok=True)

def banner():
    print("== OFFLINE DeepSeek-MoE Quant Runner (terminal-first) ==")
    print(f"MODEL_DIR:   {MODEL_DIR}")
    print(f"OUTPUT_DIR:  {OUTPUT_DIR}")
    print(f"LAYER:       {LAYER}")
    print(f"MAX_EXPERTS:  {MAX_EXPERTS}")
    print(f"EVAL_MODE:   {EVAL_MODE}")
    print(f"Torch:       {torch.__version__} | device={DEVICE} | threads={THREADS}")
    print(f"GROUP_SIZE:  {GROUP_SIZE} | BITS={BITS_LIST}")
    print(f"SmoothQuant: {RUN_SMOOTHQUANT} (alpha={SQ_ALPHA})")
    print(f"Hadamard:    {RUN_HADAMARD} (seed={HAD_SEED})")
    print(f"CALIB_PATH:  {CALIB_PATH or '(none -> random X)'} | CALIB_SAMPLES={CALIB_SAMPLES}")
    print(f"CACHE_MODE:  {CACHE_MODE} | CACHE_PATH={CACHE_PATH}")
    print()

# ----------------------------
# Helpers: NPZ cache I/O (robust)
# ----------------------------
def npz_list_keys(path: str) -> List[str]:
    z = np.load(path, allow_pickle=False)
    keys = list(z.files)  # documented behavior :contentReference[oaicite:2]{index=2}
    z.close()
    return keys

def safe_np_savez(path: str, arrays: Dict[str, np.ndarray]):
    # ensure parent exists
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

def try_load_cache(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

# ----------------------------
# Helpers: Hadamard rotation (orthonormal)
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

@torch.no_grad()
def hadamard_orthonormal(n: int, seed: int = 0) -> torch.Tensor:
    """
    Builds an orthonormal Hadamard-based rotation R of shape [n,n]:
    R = H / sqrt(n) with random column sign flips.
    """
    if not is_power_of_two(n):
        raise ValueError(f"Hadamard requires power-of-two n, got {n}")
    # Sylvester construction
    H = torch.tensor([[1.0]], dtype=DTYPE_ACC, device=DEVICE)
    while H.shape[0] < n:
        H = torch.cat([torch.cat([H,  H], dim=1),
                       torch.cat([H, -H], dim=1)], dim=0)
    H = H / math.sqrt(n)
    g = torch.Generator(device="cpu").manual_seed(seed)
    signs = torch.randint(0, 2, (n,), generator=g, dtype=torch.int8) * 2 - 1
    signs = signs.to(DTYPE_ACC)
    # Flip columns: H * signs
    R = H * signs.view(1, n)
    return R.contiguous()

# ----------------------------
# Helpers: SmoothQuant scaling (invariance-preserving)
# ----------------------------
@torch.no_grad()
def smoothquant_scales(X: torch.Tensor, W_list: List[torch.Tensor], alpha: float = 0.5, eps: float = 1e-5) -> torch.Tensor:
    """
    Returns per-input-channel scales s (shape [H]) for invariance:
      X' = X / s
      W' = W * s  (scale columns)
    This keeps X @ W^T unchanged in exact arithmetic.
    """
    X = X.to(DTYPE_ACC)
    act = X.abs().amax(dim=0) + eps
    # combine multiple matrices that see same X (e.g., gate+up)
    wmax = torch.stack([W.to(DTYPE_ACC).abs().amax(dim=0) for W in W_list], dim=0).amax(dim=0) + eps
    s = (act.pow(alpha) / wmax.pow(1.0 - alpha)).clamp_min(eps)
    return s

@torch.no_grad()
def apply_column_scale(W: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
    # W: [out, in], s: [in]
    return (W.to(DTYPE_ACC) * s.view(1, -1)).to(W.dtype)

@torch.no_grad()
def apply_activation_scale(X: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
    return (X.to(DTYPE_ACC) / s.view(1, -1)).to(X.dtype)

# ----------------------------
# Quantizer: per-row, per-group symmetric quant-dequant
# ----------------------------
@torch.no_grad()
def quant_dequant_groupwise(W: torch.Tensor, bits: int, group_size: int) -> torch.Tensor:
    """
    Simulate groupwise symmetric quantization by dequantizing back to fp16.
    Quantizes along input dimension (columns) in groups for each output row.

    W: [out, in] float16/float32
    Returns: W_qdq float16 with same shape.
    """
    Wf = W.to(DTYPE_ACC)
    out, inn = Wf.shape
    g = group_size
    n_groups = (inn + g - 1) // g
    pad = n_groups * g - inn
    if pad:
        Wf = torch.cat([Wf, torch.zeros(out, pad, dtype=Wf.dtype, device=Wf.device)], dim=1)

    Wg = Wf.view(out, n_groups, g)

    if bits == 8:
        qmax = 127
        qmin = -128
    else:
        qmax = 2 ** (bits - 1) - 1
        qmin = -2 ** (bits - 1)

    # scale per (row, group)
    amax = Wg.abs().amax(dim=2, keepdim=True).clamp_min(1e-8)
    scale = amax / float(qmax)

    q = torch.round(Wg / scale).clamp(qmin, qmax)
    Wdq = (q * scale).view(out, n_groups * g)
    Wdq = Wdq[:, :inn].contiguous()
    return Wdq.to(DTYPE_W)

# ----------------------------
# DeepSeek MoE weight discovery & loading (offline)
# ----------------------------
def read_index(model_dir: str) -> Tuple[Dict[str, str], Dict]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    weight_map = obj.get("weight_map", {})
    if not weight_map:
        raise RuntimeError("Index JSON has empty weight_map.")
    return weight_map, obj

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    """
    Return dict with keys for 'up','gate','down' if found.
    Supports multiple naming conventions.
    """
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    keys = [k for k in weight_map.keys() if k.startswith(prefix)]

    def pick(suffix_candidates: List[str]) -> Optional[str]:
        for suf in suffix_candidates:
            want = prefix + suf
            if want in weight_map:
                return want
        # fallback: search contains
        for k in keys:
            for suf in suffix_candidates:
                if k.endswith(suf):
                    return k
        return None

    up_key   = pick(["up_proj.weight", "w1.weight", "fc1.weight"])
    gate_key = pick(["gate_proj.weight", "w3.weight"])
    down_key = pick(["down_proj.weight", "w2.weight", "fc2.weight"])

    found = {}
    if up_key: found["up"] = up_key
    if gate_key: found["gate"] = gate_key
    if down_key: found["down"] = down_key
    return found

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], tensor_keys: List[str]) -> Dict[str, torch.Tensor]:
    """
    Loads a set of tensors from safetensors shards efficiently by grouping by shard.
    Uses safe_open(...) to read tensors. :contentReference[oaicite:3]{index=3}
    """
    by_shard: Dict[str, List[str]] = {}
    for k in tensor_keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Tensor key not found in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, keys in by_shard.items():
        shard_path = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(shard_path):
            raise FileNotFoundError(f"Missing shard file: {shard_path}")
        with safe_open(shard_path, framework="pt", device="cpu") as f:
            for k in keys:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Build / load expert cache (mlp or weff)
# ----------------------------
@torch.no_grad()
def build_expert_cache() -> Dict[str, np.ndarray]:
    weight_map, _ = read_index(MODEL_DIR)
    all_eids = find_layer_expert_ids(weight_map, LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer {LAYER}. Check model dir.")
    eids = all_eids[:MAX_EXPERTS]
    print(f"[found] experts for layer {LAYER}: {all_eids} (using {len(eids)})")

    # discover tensor keys
    per_e_keys = {}
    all_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(weight_map, LAYER, eid)
        if "up" not in kk or "down" not in kk:
            raise RuntimeError(f"Expert {eid} missing up/down keys. Found: {kk}")
        # gate optional
        per_e_keys[eid] = kk
        all_keys.extend(list(kk.values()))

    # load tensors
    print("[load] reading expert tensors from shards ...")
    tensors = load_tensors_from_shards(MODEL_DIR, weight_map, sorted(set(all_keys)))

    # stack
    W_up_list, W_gate_list, W_down_list = [], [], []
    has_gate = True
    d_ff = None
    H = None
    for eid in eids:
        kk = per_e_keys[eid]
        W_up = tensors[kk["up"]].to(dtype=DTYPE_W)
        W_down = tensors[kk["down"]].to(dtype=DTYPE_W)
        if "gate" in kk:
            W_gate = tensors[kk["gate"]].to(dtype=DTYPE_W)
        else:
            has_gate = False
            W_gate = None

        # shapes sanity
        # Expect: W_up [d_ff, H], W_gate [d_ff, H], W_down [H, d_ff]
        if H is None:
            d_ff, H = W_up.shape
        if W_up.shape[1] != H:
            raise RuntimeError(f"Unexpected W_up shape for eid={eid}: {tuple(W_up.shape)} (H={H})")
        if W_down.shape[0] != H:
            raise RuntimeError(f"Unexpected W_down shape for eid={eid}: {tuple(W_down.shape)} (H={H})")
        if d_ff is None:
            d_ff = W_up.shape[0]

        W_up_list.append(W_up)
        W_down_list.append(W_down)
        if W_gate is not None:
            W_gate_list.append(W_gate)

    W_up = torch.stack(W_up_list, dim=0)         # [E, d_ff, H]
    W_down = torch.stack(W_down_list, dim=0)     # [E, H, d_ff]
    if has_gate:
        W_gate = torch.stack(W_gate_list, dim=0) # [E, d_ff, H]
    else:
        W_gate = None

    cache = {
        "expert_ids": np.array(eids, dtype=np.int32),
        "W_up":   W_up.cpu().numpy().astype(np.float16),
        "W_down": W_down.cpu().numpy().astype(np.float16),
        "has_gate": np.array([1 if has_gate else 0], dtype=np.int32),
    }
    if has_gate and W_gate is not None:
        cache["W_gate"] = W_gate.cpu().numpy().astype(np.float16)

    # weff proxy cache
    if EVAL_MODE == "weff":
        # W_eff = W_down @ W_up  => [E, H, H]
        print("[build] computing W_eff = W_down @ W_up ...")
        W_eff = torch.matmul(W_down.to(DTYPE_ACC), W_up.to(DTYPE_ACC)).to(DTYPE_W)
        cache["W_eff"] = W_eff.cpu().numpy().astype(np.float16)

    print(f"[ok] built cache tensors: E={len(eids)} | H={H} | d_ff={d_ff} | gate={has_gate}")
    return cache

def load_or_build_cache() -> Dict[str, np.ndarray]:
    do_read = CACHE_MODE in ("read", "readwrite")
    do_write = CACHE_MODE in ("write", "readwrite")

    if do_read and os.path.isfile(CACHE_PATH):
        keys = npz_list_keys(CACHE_PATH)
        print(f"[cache] found: {CACHE_PATH} ({os.path.getsize(CACHE_PATH)/1e6:.2f} MB)")
        print(f"[cache] keys: {keys}")
        cache = try_load_cache(CACHE_PATH)
        if cache is None:
            raise RuntimeError("Cache load failed unexpectedly.")
        # Validate minimum needed keys
        if "expert_ids" not in cache:
            print("[cache] missing 'expert_ids' -> rebuilding.")
        else:
            if EVAL_MODE == "weff" and ("W_eff" in cache):
                return cache
            if EVAL_MODE == "mlp" and ("W_up" in cache) and ("W_down" in cache):
                # gate optional
                return cache
            # fallthrough
            print("[cache] not compatible with requested EVAL_MODE -> rebuilding.")

    cache = build_expert_cache()
    if do_write:
        # detach/np already done since cache is numpy arrays
        safe_np_savez(CACHE_PATH, cache)
        print(f"[cache] wrote: {CACHE_PATH} ({os.path.getsize(CACHE_PATH)/1e6:.2f} MB)")
    return cache

# ----------------------------
# Calibration activations
# ----------------------------
def load_calib_X(H: int) -> torch.Tensor:
    if CALIB_PATH is None:
        X = torch.randn(CALIB_SAMPLES, H, dtype=DTYPE_W)
        print(f"[calib] Using RANDOM X: {tuple(X.shape)} (best is real activations npz with key 'X')")
        return X
    if not os.path.isfile(CALIB_PATH):
        raise FileNotFoundError(f"CALIB_PATH not found: {CALIB_PATH}")
    z = np.load(CALIB_PATH, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Has keys: {list(z.files)}")
    X = torch.from_numpy(z["X"]).to(DTYPE_W)
    z.close()
    if X.ndim != 2 or X.shape[1] != H:
        raise RuntimeError(f"Bad X shape in calib file: {tuple(X.shape)} expected (*,{H})")
    if X.shape[0] > CALIB_SAMPLES:
        X = X[:CALIB_SAMPLES]
    print(f"[calib] Loaded X from file: {tuple(X.shape)}")
    return X

# ----------------------------
# Forward definitions
# ----------------------------
@torch.no_grad()
def forward_weff(X: torch.Tensor, W_eff: torch.Tensor) -> torch.Tensor:
    # X [N,H], W_eff [H,H] (stored as [out,in] == [H,H])
    return X.to(DTYPE_ACC) @ W_eff.to(DTYPE_ACC).t()

@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: Optional[torch.Tensor], W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    # X [N,H]
    # W_up   [d_ff,H]
    # W_gate [d_ff,H] optional
    # W_down [H,d_ff]
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    if W_gate is not None:
        gate = Xf @ W_gate.to(DTYPE_ACC).t()
        hid = F.silu(gate) * up
    else:
        hid = F.silu(up)
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

@torch.no_grad()
def rel_frob(y_hat: torch.Tensor, y_ref: torch.Tensor, eps: float = 1e-12) -> float:
    num = torch.linalg.norm((y_hat - y_ref).reshape(-1), ord=2)
    den = torch.linalg.norm(y_ref.reshape(-1), ord=2).clamp_min(eps)
    return float((num / den).item())

# ----------------------------
# Evaluation runners (base / smoothquant / hadamard / both)
# ----------------------------
@torch.no_grad()
def eval_expert_weff(X: torch.Tensor, W_eff: torch.Tensor, bits: int) -> Dict[str, float]:
    res = {}

    # reference
    y_ref = forward_weff(X, W_eff)

    # base quant
    Wq = quant_dequant_groupwise(W_eff, bits=bits, group_size=GROUP_SIZE)
    y_q = forward_weff(X, Wq)
    res["base"] = rel_frob(y_q, y_ref)

    # smoothquant (invariance: X' = X/s, W' = W*s (columns))
    if RUN_SMOOTHQUANT:
        s = smoothquant_scales(X, [W_eff], alpha=SQ_ALPHA)
        Xs = apply_activation_scale(X, s)
        Ws = apply_column_scale(W_eff, s)
        Wsq = quant_dequant_groupwise(Ws, bits=bits, group_size=GROUP_SIZE)
        y_sq = forward_weff(Xs, Wsq)  # compare to y_ref (invariance holds in exact arithmetic)
        res["smoothq"] = rel_frob(y_sq, y_ref)

    # hadamard (invariance: X' = X R, W' = W R)
    if RUN_HADAMARD:
        H = W_eff.shape[1]
        R = hadamard_orthonormal(H, seed=HAD_SEED)
        Xr = (X.to(DTYPE_ACC) @ R).to(DTYPE_W)
        Wr = (W_eff.to(DTYPE_ACC) @ R).to(DTYPE_W)
        Wrq = quant_dequant_groupwise(Wr, bits=bits, group_size=GROUP_SIZE)
        y_rq = forward_weff(Xr, Wrq)
        res["hadam"] = rel_frob(y_rq, y_ref)

        # smoothq + hadam (do smoothquant in rotated domain)
        if RUN_SMOOTHQUANT:
            s2 = smoothquant_scales(Xr, [Wr], alpha=SQ_ALPHA)
            Xrs = apply_activation_scale(Xr, s2)
            Wrs = apply_column_scale(Wr, s2)
            Wrsq = quant_dequant_groupwise(Wrs, bits=bits, group_size=GROUP_SIZE)
            y_combo = forward_weff(Xrs, Wrsq)
            res["smoothq+hadam"] = rel_frob(y_combo, y_ref)

    return res

@torch.no_grad()
def eval_expert_mlp(X: torch.Tensor, W_gate: Optional[torch.Tensor], W_up: torch.Tensor, W_down: torch.Tensor, bits: int) -> Dict[str, float]:
    res = {}

    # reference
    y_ref = forward_mlp(X, W_gate, W_up, W_down)

    # base quant (quantize all involved weights)
    W_up_q = quant_dequant_groupwise(W_up, bits=bits, group_size=GROUP_SIZE)
    W_down_q = quant_dequant_groupwise(W_down, bits=bits, group_size=GROUP_SIZE)
    W_gate_q = quant_dequant_groupwise(W_gate, bits=bits, group_size=GROUP_SIZE) if (W_gate is not None) else None
    y_q = forward_mlp(X, W_gate_q, W_up_q, W_down_q)
    res["base"] = rel_frob(y_q, y_ref)

    # smoothquant on the shared input X for (gate, up) only (invariance-preserving)
    if RUN_SMOOTHQUANT:
        mats = [W_up] + ([W_gate] if (W_gate is not None) else [])
        s = smoothquant_scales(X, mats, alpha=SQ_ALPHA)
        Xs = apply_activation_scale(X, s)
        W_up_s = apply_column_scale(W_up, s)
        W_gate_s = apply_column_scale(W_gate, s) if (W_gate is not None) else None

        W_up_sq = quant_dequant_groupwise(W_up_s, bits=bits, group_size=GROUP_SIZE)
        W_down_sq = quant_dequant_groupwise(W_down, bits=bits, group_size=GROUP_SIZE)
        W_gate_sq = quant_dequant_groupwise(W_gate_s, bits=bits, group_size=GROUP_SIZE) if (W_gate_s is not None) else None
        y_sq = forward_mlp(Xs, W_gate_sq, W_up_sq, W_down_sq)
        res["smoothq"] = rel_frob(y_sq, y_ref)

    # hadamard rotation on input channels only (invariance-preserving for the two first mats)
    if RUN_HADAMARD:
        H = W_up.shape[1]
        R = hadamard_orthonormal(H, seed=HAD_SEED)
        Xr = (X.to(DTYPE_ACC) @ R).to(DTYPE_W)
        W_up_r = (W_up.to(DTYPE_ACC) @ R).to(DTYPE_W)
        W_gate_r = (W_gate.to(DTYPE_ACC) @ R).to(DTYPE_W) if (W_gate is not None) else None

        W_up_rq = quant_dequant_groupwise(W_up_r, bits=bits, group_size=GROUP_SIZE)
        W_down_rq = quant_dequant_groupwise(W_down, bits=bits, group_size=GROUP_SIZE)  # down unchanged by input rotation
        W_gate_rq = quant_dequant_groupwise(W_gate_r, bits=bits, group_size=GROUP_SIZE) if (W_gate_r is not None) else None
        y_rq = forward_mlp(Xr, W_gate_rq, W_up_rq, W_down_rq)
        res["hadam"] = rel_frob(y_rq, y_ref)

        if RUN_SMOOTHQUANT:
            mats2 = [W_up_r] + ([W_gate_r] if (W_gate_r is not None) else [])
            s2 = smoothquant_scales(Xr, mats2, alpha=SQ_ALPHA)
            Xrs = apply_activation_scale(Xr, s2)
            W_up_rs = apply_column_scale(W_up_r, s2)
            W_gate_rs = apply_column_scale(W_gate_r, s2) if (W_gate_r is not None) else None

            W_up_rsq = quant_dequant_groupwise(W_up_rs, bits=bits, group_size=GROUP_SIZE)
            W_down_rsq = quant_dequant_groupwise(W_down, bits=bits, group_size=GROUP_SIZE)
            W_gate_rsq = quant_dequant_groupwise(W_gate_rs, bits=bits, group_size=GROUP_SIZE) if (W_gate_rs is not None) else None
            y_combo = forward_mlp(Xrs, W_gate_rsq, W_up_rsq, W_down_rsq)
            res["smoothq+hadam"] = rel_frob(y_combo, y_ref)

    return res

# ----------------------------
# Main
# ----------------------------
def run():
    banner()
    cache = load_or_build_cache()

    eids = cache["expert_ids"].astype(np.int32).tolist()
    E = len(eids)

    if EVAL_MODE == "weff":
        if "W_eff" not in cache:
            # backward compatible: build W_eff from W_up/W_down if present
            if "W_up" in cache and "W_down" in cache:
                W_up = torch.from_numpy(cache["W_up"]).to(DTYPE_W)
                W_down = torch.from_numpy(cache["W_down"]).to(DTYPE_W)
                W_eff = torch.matmul(W_down.to(DTYPE_ACC), W_up.to(DTYPE_ACC)).to(DTYPE_W)
                print("[cache] 'W_eff' missing -> computed from W_down@W_up")
            else:
                raise KeyError(f"Cache missing 'W_eff'. Available keys: {list(cache.keys())}")
        else:
            W_eff = torch.from_numpy(cache["W_eff"]).to(DTYPE_W)

        H = int(W_eff.shape[-1])
        X = load_calib_X(H)

        print(f"\n[ok] Using W_eff: {tuple(W_eff.shape)} | experts={eids[:8]}{'...' if E>8 else ''}")
        for bits in BITS_LIST:
            print(f"\n== bits={bits} (group={GROUP_SIZE}) ==")
            rows = []
            for i in tqdm(range(E), desc="Per-expert eval"):
                res = eval_expert_weff(X, W_eff[i], bits=bits)
                rows.append(res)

            # aggregate keys
            keys = sorted({k for r in rows for k in r.keys()})
            for k in keys:
                vals = [r[k] for r in rows if k in r]
                print(f"{k:<14s}: mean={float(np.mean(vals)):.6f}  std={float(np.std(vals)):.6f}  (n={len(vals)})")

    elif EVAL_MODE == "mlp":
        W_up = torch.from_numpy(cache["W_up"]).to(DTYPE_W)
        W_down = torch.from_numpy(cache["W_down"]).to(DTYPE_W)
        has_gate = bool(int(cache.get("has_gate", np.array([0], dtype=np.int32))[0]))
        W_gate = torch.from_numpy(cache["W_gate"]).to(DTYPE_W) if (has_gate and "W_gate" in cache) else None

        H = int(W_up.shape[-1])
        X = load_calib_X(H)

        print(f"\n[ok] Using MLP weights:")
        print(f"  W_up:   {tuple(W_up.shape)}")
        print(f"  W_down: {tuple(W_down.shape)}")
        print(f"  W_gate: {tuple(W_gate.shape) if W_gate is not None else None}")
        print(f"  experts={eids[:8]}{'...' if E>8 else ''}")

        for bits in BITS_LIST:
            print(f"\n== bits={bits} (group={GROUP_SIZE}) ==")
            rows = []
            for i in tqdm(range(E), desc="Per-expert eval"):
                res = eval_expert_mlp(X, (W_gate[i] if W_gate is not None else None), W_up[i], W_down[i], bits=bits)
                rows.append(res)

            keys = sorted({k for r in rows for k in r.keys()})
            for k in keys:
                vals = [r[k] for r in rows if k in r]
                print(f"{k:<14s}: mean={float(np.mean(vals)):.6f}  std={float(np.std(vals)):.6f}  (n={len(vals)})")
    else:
        raise ValueError("EVAL_MODE must be 'mlp' or 'weff'")

    print("\n✅ Done (terminal-first).")
    print("Tip: For realistic numbers, set CALIB_PATH to real activations (npz with key 'X').")
    print("Tip: If Hadamard is slow, set RUN_HADAMARD=0; the main win usually comes from good calibration + GPTQ/AWQ-style PTQ.")

if __name__ == "__main__" or True:
    run()


== OFFLINE DeepSeek-MoE Quant Runner (terminal-first) ==
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  8
EVAL_MODE:   mlp
Torch:       2.4.1+cpu | device=cpu | threads=8
GROUP_SIZE:  128 | BITS=[8, 4, 3, 2]
SmoothQuant: True (alpha=0.5)
Hadamard:    False (seed=1234)
CALIB_PATH:  (none -> random X) | CALIB_SAMPLES=512
CACHE_MODE:  readwrite | CACHE_PATH=/home/daniyar/moe_ws_outputs/deepseek_layer1_E8_mlp_cache_v5.npz

[found] experts for layer 1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63] (using 8)
[load] reading expert tensors from shards ...
[ok] built cache tensors: E=8 | H=2048 | d_ff=1408 | gate=True
[cache] wrote: /home/daniyar/moe_ws_outputs/deepseek_layer1_E8_mlp_cache_v5.npz (138.41 MB)
[calib] Using RANDOM X: 

Per-expert eval:   0%|          | 0/8 [00:00<?, ?it/s]

base          : mean=0.010784  std=0.000226  (n=8)
smoothq       : mean=0.010885  std=0.000221  (n=8)

== bits=4 (group=128) ==


Per-expert eval:   0%|          | 0/8 [00:00<?, ?it/s]

base          : mean=0.197135  std=0.004200  (n=8)
smoothq       : mean=0.198931  std=0.004060  (n=8)

== bits=3 (group=128) ==


Per-expert eval:   0%|          | 0/8 [00:00<?, ?it/s]

base          : mean=0.477184  std=0.010472  (n=8)
smoothq       : mean=0.481270  std=0.010039  (n=8)

== bits=2 (group=128) ==


Per-expert eval:   0%|          | 0/8 [00:00<?, ?it/s]

base          : mean=1.260940  std=0.031442  (n=8)
smoothq       : mean=1.260752  std=0.031033  (n=8)

✅ Done (terminal-first).
Tip: For realistic numbers, set CALIB_PATH to real activations (npz with key 'X').
Tip: If Hadamard is slow, set RUN_HADAMARD=0; the main win usually comes from good calibration + GPTQ/AWQ-style PTQ.


In [11]:
# ============================================================
# OFFLINE DeepSeek-MoE "EQuAL-MoE" Quant Runner (single file)
#   - terminal-first (prints everything)
#   - offline: reads local safetensors shards via index.json
#   - MoE-aware calibration (optional router weights)
#   - SmoothQuant (shared gate+up), optional consistent Hadamard
#   - mixed-precision bit allocation per group under avg-bit budget
#   - GroupGPTQ-lite (within-group error feedback)
#   - optional outlier columns kept fp16
#   - optional low-rank residual compensation (LoRC)
# ============================================================

import os, re, json, math, time
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np

try:
    import torch
    import torch.nn.functional as F
except Exception as e:
    raise RuntimeError("This script requires PyTorch.") from e

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("This script requires safetensors.") from e

try:
    from tqdm import tqdm
except Exception:
    tqdm = None


# ----------------------------
# Threading / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))  # CPU by default


# ----------------------------
# User knobs (env vars)
# ----------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")
    CACHE_PATH: str = os.environ.get("CACHE_PATH", "")  # optional override

    # Which layer / how many experts to test
    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "8"))

    # Calibration
    CALIB_PATH: str = os.environ.get("CALIB_PATH", "")   # npz with key 'X' (N,H)
    ROUTER_PATH: str = os.environ.get("ROUTER_PATH", "") # npz with key 'P' (N,E_total) optional
    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "512"))

    # Quantization
    GROUP_SIZE: int = int(os.environ.get("GROUP_SIZE", "128"))
    TARGET_AVG_BITS: float = float(os.environ.get("TARGET_AVG_BITS", "3.0"))  # across quantized weights
    BITS: List[int] = None  # set below

    # Techniques
    RUN_SMOOTHQUANT: bool = os.environ.get("RUN_SMOOTHQUANT", "1") == "1"
    SQ_ALPHA: float = float(os.environ.get("SQ_ALPHA", "0.5"))

    RUN_HADAMARD: bool = os.environ.get("RUN_HADAMARD", "0") == "1"
    HADAMARD_SIGN_FLIP: bool = os.environ.get("HADAMARD_SIGN_FLIP", "1") == "1"

    RUN_GROUPGPTQ: bool = os.environ.get("RUN_GROUPGPTQ", "1") == "1"
    GPTQ_DAMP: float = float(os.environ.get("GPTQ_DAMP", "1e-3"))  # relative to trace(H)/dim

    OUTLIER_RATIO: float = float(os.environ.get("OUTLIER_RATIO", "0.005"))  # keep top cols fp16
    OUTLIER_MIN_COLS: int = int(os.environ.get("OUTLIER_MIN_COLS", "8"))

    # LoRC low-rank residual (0 disables)
    LORC_RANK: int = int(os.environ.get("LORC_RANK", "0"))

    # Save payload (optional)
    SAVE_PAYLOAD: bool = os.environ.get("SAVE_PAYLOAD", "0") == "1"

cfg = Cfg()
cfg.BITS = [int(x) for x in os.environ.get("BITS", "2,3,4").split(",")]

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)


# ----------------------------
# Pretty printing helpers
# ----------------------------
def hsize(n_bytes: int) -> str:
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n_bytes < 1024:
            return f"{n_bytes:.2f}{unit}"
        n_bytes /= 1024
    return f"{n_bytes:.2f}PB"

def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

def pbar(it, **kwargs):
    if tqdm is None:
        return it
    return tqdm(it, **kwargs)


# ----------------------------
# Model index + offline tensor loader
# ----------------------------
def load_index(model_dir: str) -> Dict:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.exists(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r") as f:
        return json.load(f)

def shard_path(model_dir: str, shard_name: str) -> str:
    p = os.path.join(model_dir, shard_name)
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing shard file: {p}")
    return p

def infer_experts_for_layer(weight_map: Dict[str, str], layer: int) -> List[int]:
    # Handles both:
    #   model.layers.{L}.mlp.experts.{E}.w1.weight / w2 / w3
    #   model.layers.{L}.mlp.experts.{E}.up_proj.weight / down_proj / gate_proj
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.(.+)\.weight$")
    eids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            eids.add(int(m.group(1)))
    return sorted(eids)

def detect_mlp_names_for_layer(weight_map: Dict[str, str], layer: int) -> Tuple[str, str, str]:
    # returns (gate_name_suffix, up_name_suffix, down_name_suffix) without ".weight"
    # Try common conventions.
    candidates = [
        ("w1", "w3", "w2"),               # LLaMA-style FFN naming
        ("gate_proj", "up_proj", "down_proj"),
        ("gate", "up", "down"),
    ]
    # Probe for expert 0 (or any)
    eids = infer_experts_for_layer(weight_map, layer)
    if not eids:
        raise RuntimeError(f"No experts found for layer={layer}. Check MODEL_DIR and index.")
    e0 = eids[0]
    keys = set(weight_map.keys())
    for gate_s, up_s, down_s in candidates:
        kg = f"model.layers.{layer}.mlp.experts.{e0}.{gate_s}.weight"
        ku = f"model.layers.{layer}.mlp.experts.{e0}.{up_s}.weight"
        kd = f"model.layers.{layer}.mlp.experts.{e0}.{down_s}.weight"
        if kg in keys and ku in keys and kd in keys:
            return gate_s, up_s, down_s
    # fallback: scan any suffixes
    raise RuntimeError(
        f"Could not detect MLP expert tensor names for layer={layer}. "
        f"Expected patterns like w1/w2/w3 or gate_proj/up_proj/down_proj."
    )

def load_tensors_offline(model_dir: str, weight_map: Dict[str, str], tensor_names: List[str]) -> Dict[str, torch.Tensor]:
    # Load multiple tensors from local shards efficiently: group by shard then safe_open once.
    by_shard: Dict[str, List[str]] = {}
    for name in tensor_names:
        if name not in weight_map:
            raise KeyError(f"Tensor not in weight_map: {name}")
        by_shard.setdefault(weight_map[name], []).append(name)

    out: Dict[str, torch.Tensor] = {}
    for shard, names in by_shard.items():
        sp = shard_path(model_dir, shard)
        with safe_open(sp, framework="pt", device="cpu") as f:
            for n in names:
                out[n] = f.get_tensor(n)
    return out


# ----------------------------
# Cache: build/load expert MLP tensors (gate/up/down)
# ----------------------------
def default_cache_path() -> str:
    # stable cache name
    return os.path.join(cfg.OUTPUT_DIR, f"deepseek_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_mlp_cache.npz")

def load_or_build_mlp_cache() -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[int]]:
    cache_path = cfg.CACHE_PATH.strip() or default_cache_path()

    if os.path.exists(cache_path):
        z = np.load(cache_path, allow_pickle=False)
        if not all(k in z for k in ["W_gate", "W_up", "W_down", "experts"]):
            raise RuntimeError(f"Cache missing required keys: {cache_path}")
        Wg = z["W_gate"]
        Wu = z["W_up"]
        Wd = z["W_down"]
        experts = [int(x) for x in z["experts"].tolist()]
        return Wg, Wu, Wd, experts

    log(f"[{now()}] [cache] building MLP cache from shards


SyntaxError: unterminated f-string literal (detected at line 217) (2566913198.py, line 217)

In [14]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek-Optimized OFFLINE KT++-X Runner (single file) v2
#   - offline: reads local safetensors shards via index.json
#   - builds square expert mats:
#       * LIN_MODE=ridge (recommended; uses calib activations)
#       * LIN_MODE=weff_gate (fast proxy)
#       * LIN_MODE=weff (baseline)
#   - KT++ payload:
#       * CORE_MODE=blockdiag (default)
#       * shared low-rank residual + greedy sparse blocks
#   - fixes autograd breakages:
#       * training wrapped in torch.enable_grad()
#       * asserts L_total.requires_grad
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np

try:
    import torch
    import torch.nn.functional as F
    import torch.nn as nn
except Exception as e:
    raise RuntimeError("This script requires PyTorch.") from e

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("This script requires safetensors (pip install safetensors).") from e

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

# ----------------------------
# Threading / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))

DTYPE_W   = torch.float16
DTYPE_ACC = torch.float32

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config (env overrides)
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "16"))

    # Calibration (npz with key 'X': (N,H))
    CALIB_PATH: str = os.environ.get("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "512"))

    # Optional router probs (npz with key 'P': (N,E_total))
    ROUTER_PATH: str = os.environ.get("ROUTER_PATH", "").strip()
    ROUTER_EIDS_ARE_GLOBAL: bool = os.environ.get("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    # Build square matrices
    LIN_MODE: str = os.environ.get("LIN_MODE", "ridge").strip().lower()  # ridge | weff_gate | weff
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))      # relative to trace(XtX)/H
    RIDGE_WEIGHTED: bool = os.environ.get("RIDGE_WEIGHTED", "0") == "1"  # weight by router P if provided

    # KT++ clustering / training
    M_CLUSTERS: int = int(os.environ.get("M_CLUSTERS", "0"))  # 0 => auto
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "24"))
    SUBM: int = int(os.environ.get("SUBM", "256"))
    BATCH_E: int = int(os.environ.get("BATCH_E", "4"))
    LR_UV: float = float(os.environ.get("LR_UV", "5e-2"))
    REORTHO_EVERY: int = int(os.environ.get("REORTHO_EVERY", "4"))
    REPORT_EVERY: int = int(os.environ.get("REPORT_EVERY", "4"))

    # Payload core
    CORE_MODE: str = os.environ.get("CORE_MODE", "blockdiag").strip().lower()  # blockdiag | none
    CORE_BLOCK: int = int(os.environ.get("CORE_BLOCK", "64"))                 # for blockdiag
    CORE_TARGET: float = float(os.environ.get("CORE_TARGET", "0.50"))         # fraction of Fro energy to keep (approx)

    # Residual model
    RES_RANK: int = int(os.environ.get("RES_RANK", "512"))
    RES_BLOCKS: int = int(os.environ.get("RES_BLOCKS", "48"))
    RES_BSIZE: int = int(os.environ.get("RES_BSIZE", "64"))

    # Optional similarity Hadamard rotation on square Ws
    USE_HADAMARD: bool = os.environ.get("USE_HADAMARD", "0") == "1"
    HAD_SEED: int = int(os.environ.get("HAD_SEED", "1234"))

    # Cache
    CACHE_MODE: str = os.environ.get("CACHE_MODE", "readwrite").strip().lower()  # off|read|write|readwrite
    CACHE_NAME: str = os.environ.get("CACHE_NAME", "").strip()  # override

    # Eval
    EVAL_TRIALS: int = int(os.environ.get("EVAL_TRIALS", "5"))
    ROUTED_K: int = int(os.environ.get("ROUTED_K", "8"))
    EVAL_BATCH: int = int(os.environ.get("EVAL_BATCH", "2"))

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def cache_path():
    if cfg.CACHE_NAME:
        return os.path.join(cfg.OUTPUT_DIR, cfg.CACHE_NAME)
    tag = f"deepseek_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_{cfg.LIN_MODE}_Ws_cache.npz"
    return os.path.join(cfg.OUTPUT_DIR, tag)

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."

    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None

    # NOTE: DeepSeek commonly uses gate_proj/up_proj/down_proj.
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    out = {}
    if up: out["up"] = up
    if gate: out["gate"] = gate
    if down: out["down"] = down
    return out

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration loader
# ----------------------------
def load_calib_X(H: int) -> torch.Tensor:
    if not cfg.CALIB_PATH:
        X = torch.randn(cfg.CALIB_SAMPLES, H, dtype=DTYPE_W, device=DEVICE)
        log(f"[calib] RANDOM X: {tuple(X.shape)}  (set CALIB_PATH npz key 'X' for real activations)")
        return X
    if not os.path.isfile(cfg.CALIB_PATH):
        raise FileNotFoundError(f"CALIB_PATH not found: {cfg.CALIB_PATH}")
    z = np.load(cfg.CALIB_PATH, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Has keys: {list(z.files)}")
    X = torch.from_numpy(z["X"]).to(DTYPE_W).to(DEVICE)
    z.close()
    if X.ndim != 2 or X.shape[0] < 2:
        raise RuntimeError(f"Bad X shape: {tuple(X.shape)}")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    log(f"[calib] Loaded X: {tuple(X.shape)}")
    return X

def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    if not os.path.isfile(cfg.ROUTER_PATH):
        raise FileNotFoundError(f"ROUTER_PATH not found: {cfg.ROUTER_PATH}")
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Has keys: {list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    log(f"[router] Loaded P: {tuple(P.shape)}")
    return P

# ----------------------------
# Hadamard (optional)
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

@torch.no_grad()
def hadamard_orth(n: int, seed: int) -> torch.Tensor:
    if not is_power_of_two(n):
        raise ValueError(f"Hadamard requires power-of-two n, got {n}")
    H = torch.tensor([[1.0]], dtype=DTYPE_ACC, device=DEVICE)
    while H.shape[0] < n:
        H = torch.cat([torch.cat([H,  H], dim=1),
                       torch.cat([H, -H], dim=1)], dim=0)
    H = H / math.sqrt(n)
    g = torch.Generator(device="cpu").manual_seed(seed)
    signs = (torch.randint(0, 2, (n,), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    R = (H * signs.view(1, n)).contiguous()
    return R

# ----------------------------
# Build expert square Ws
# ----------------------------
@torch.no_grad()
def forward_mlp_no_grad(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

@torch.no_grad()
def build_Ws_square() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer {cfg.LAYER} experts: {all_eids}  (using {len(eids)})")

    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if "up" not in kk or "down" not in kk or "gate" not in kk:
            raise RuntimeError(f"Expert {eid} missing keys: {kk}")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]].to(DTYPE_W)
    dff, H = W_up0.shape
    log(f"[shape] H={H} d_ff={dff}")

    X = load_calib_X(H)
    P = load_router_P()

    R = hadamard_orth(H, cfg.HAD_SEED) if cfg.USE_HADAMARD else None

    # Shared ridge (unweighted)
    Xf = X.to(DTYPE_ACC)
    XtX = (Xf.t() @ Xf)
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    cholG = None
    if cfg.LIN_MODE == "ridge" and not (cfg.RIDGE_WEIGHTED and (P is not None)):
        G = XtX + lam * I
        cholG = torch.linalg.cholesky(G)

    Ws = []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws")):
        W_up   = T[per_e[eid]["up"]].to(DTYPE_W).to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DTYPE_W).to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DTYPE_W).to(DEVICE)

        if cfg.LIN_MODE == "weff":
            W = (W_down.to(DTYPE_ACC) @ W_up.to(DTYPE_ACC))

        elif cfg.LIN_MODE == "weff_gate":
            gate_act = (Xf @ W_gate.to(DTYPE_ACC).t())
            m = F.silu(gate_act).mean(dim=0)  # (dff,)
            W = (W_down.to(DTYPE_ACC) * m.view(1, -1)) @ W_up.to(DTYPE_ACC)

        elif cfg.LIN_MODE == "ridge":
            Y = forward_mlp_no_grad(X, W_gate, W_up, W_down).to(DTYPE_ACC)  # (N,H)

            if cfg.RIDGE_WEIGHTED and (P is not None):
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                else:
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw
                XtX_e = Xw.t() @ Xw
                lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
                G_e = XtX_e + lam_e * I
                chol = torch.linalg.cholesky(G_e)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)
                W = Wt.t().contiguous()
            else:
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)
                W = Wt.t().contiguous()
        else:
            raise ValueError("LIN_MODE must be ridge|weff_gate|weff")

        if R is not None:
            W = (R.t() @ W @ R).contiguous()

        # NOTE: For KT++ structure discovery, per-expert normalization is OK;
        # if you later want true scale, store fn separately.
        fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
        W = (W / fn).contiguous()
        Ws.append(W)

    Ws = torch.stack(Ws, dim=0)  # (E,H,H)
    return eids, Ws

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    p = cache_path()
    do_read = cfg.CACHE_MODE in ("read", "readwrite")
    do_write = cfg.CACHE_MODE in ("write", "readwrite")

    if do_read and os.path.isfile(p):
        z = try_load_npz(p)
        if z and ("expert_ids" in z) and ("Ws" in z):
            eids = [int(x) for x in z["expert_ids"].tolist()]
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
            log(f"[cache] loaded Ws: {p}  Ws={tuple(Ws.shape)}")
            return eids, Ws
        log("[cache] invalid cache -> rebuilding.")

    eids, Ws = build_Ws_square()
    if do_write:
        save_npz(p, {
            "expert_ids": np.array(eids, dtype=np.int32),
            "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        })
        log(f"[cache] wrote: {p}  ({os.path.getsize(p)/1e6:.2f} MB)")
    return eids, Ws

# ----------------------------
# Clustering features + kmeans
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int = 64) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R).unsqueeze(0), (col @ R).unsqueeze(0)], dim=1))
    return torch.cat(feats, dim=0)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int = 60) -> Tuple[torch.Tensor, torch.Tensor]:
    g = torch.Generator(device="cpu").manual_seed(SEED)
    idx = torch.randperm(X.shape[0], generator=g)[:k]
    C = X[idx].clone()
    for _ in range(iters):
        dist = torch.cdist(X, C)
        lab = dist.argmin(dim=1)
        for j in range(k):
            m = (lab == j)
            if m.any():
                C[j] = X[m].mean(dim=0)
    return lab, C

# ----------------------------
# Ortho params + training helpers (GRAD ENABLED)
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, M_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(M_init.clone().to(DEVICE))

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)  # differentiable
        return Q

@torch.no_grad()
def svd_init(W_mean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(W_mean, full_matrices=False)
    V = Vh.t().contiguous()
    return U.contiguous(), V.contiguous()

# IMPORTANT: no @torch.no_grad() here
def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    # Ws_batch: (Eb,n,n), U/V: (n,n), S: (s,)
    U_S = U[:, S]  # (n,s)
    V_S = V[:, S]  # (n,s)
    T = Ws_batch @ V_S                      # (Eb,n,s)
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)  # (1,s,n)@(Eb,n,s)->(Eb,s,s)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

# ----------------------------
# Payload building (no_grad)
# ----------------------------
@torch.no_grad()
def greedy_blocks(R: torch.Tensor, t: int, b: int) -> List[Tuple[int,int,torch.Tensor]]:
    n = R.shape[0]
    used = torch.zeros(n, n, dtype=torch.bool, device=R.device)
    out = []
    if t <= 0:
        return out
    for _ in range(t):
        best = None
        bestE = -1.0
        for i0 in range(0, n, b):
            for j0 in range(0, n, b):
                i1 = min(n, i0 + b)
                j1 = min(n, j0 + b)
                if used[i0:i1, j0:j1].any():
                    continue
                Eblk = float((R[i0:i1, j0:j1] ** 2).sum().item())
                if Eblk > bestE:
                    bestE = Eblk
                    best = (i0, j0, i1, j1)
        if best is None:
            break
        i0, j0, i1, j1 = best
        used[i0:i1, j0:j1] = True
        out.append((i0, j0, R[i0:i1, j0:j1].clone()))
    return out

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor], mode: str, block: int, target: float):
    n = X_list[0].shape[0]
    E = len(X_list)

    if mode == "none":
        return {"mode": "none"}

    if mode != "blockdiag":
        raise ValueError("CORE_MODE must be blockdiag|none")

    b = block
    nb = (n + b - 1) // b

    tot = 0.0
    blkE = torch.zeros(nb, dtype=DTYPE_ACC, device=X_list[0].device)
    for X in X_list:
        tot += float((X ** 2).sum().item())
        for bi in range(nb):
            i0 = bi * b
            i1 = min(n, i0 + b)
            blkE[bi] += float((X[i0:i1, i0:i1] ** 2).sum().item())
    tot /= max(1, E)
    blkE /= max(1, E)

    order = torch.argsort(blkE, descending=True)
    csum = torch.cumsum(blkE[order], dim=0)
    frac = csum / max(tot, 1e-12)
    Kb = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else nb)
    keep = order[:Kb].tolist()

    keep_ranges = []
    for bi in keep:
        i0 = bi * b
        i1 = min(n, i0 + b)
        keep_ranges.append((i0, i1))
    keep_ranges.sort()

    return {"mode": "blockdiag", "block": b, "keep_ranges": keep_ranges}

@torch.no_grad()
def build_payload_for_cluster(Ws: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor):
    X_list = []
    for e in idx:
        X = (U.t() @ Ws[e] @ V).contiguous()
        X_list.append(X)

    core = choose_core_blocks(X_list, cfg.CORE_MODE, cfg.CORE_BLOCK, cfg.CORE_TARGET)

    core_blocks = []
    for X in X_list:
        lst = []
        if core["mode"] == "blockdiag":
            for (i0, i1) in core["keep_ranges"]:
                lst.append((i0, i1, X[i0:i1, i0:i1].clone()))
        core_blocks.append(lst)

    R_list = []
    for X, cb in zip(X_list, core_blocks):
        Xc = torch.zeros_like(X)
        for (i0, i1, Bc) in cb:
            Xc[i0:i1, i0:i1] = Bc
        R_list.append((X - Xc).contiguous())

    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    U_s, _, Vh_s = torch.linalg.svd(Rmean, full_matrices=False)
    r = min(cfg.RES_RANK, U_s.shape[1])
    DL = U_s[:, :r].contiguous()
    DR = Vh_s.t()[:, :r].contiguous()

    gam_list = []
    blocks_list = []
    for Rm in R_list:
        g = torch.sum(DL * (Rm @ DR), dim=0)  # (r,)
        gam_list.append(g.contiguous())
        R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        blks = greedy_blocks(R2, cfg.RES_BLOCKS, cfg.RES_BSIZE)
        blocks_list.append(blks)

    gam = torch.stack(gam_list, dim=0)

    return {
        "U": U, "V": V,
        "core": core,
        "core_blocks": core_blocks,
        "DL": DL, "DR": DR,
        "gam": gam,
        "blocks": blocks_list,
        "idx": idx,
    }

class KTXRuntime(nn.Module):
    def __init__(self, payloads: List[Optional[dict]], basis_of_e: Dict[int, Tuple[int,int]]):
        super().__init__()
        self.payloads = payloads
        self.map = basis_of_e
        self.n = None
        for p in payloads:
            if p is not None:
                self.n = int(p["U"].shape[0])
                break
        if self.n is None:
            raise RuntimeError("No payloads provided.")

    @torch.no_grad()
    def forward(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        used: Dict[int, List[Tuple[float,int]]] = {}
        for a, e in zip(gates.tolist(), routed):
            if e in self.map:
                m, j = self.map[e]
                used.setdefault(m, []).append((a, j))

        for m, items in used.items():
            P = self.payloads[m]
            if P is None:
                continue
            U = P["U"]; V = P["V"]
            DL = P["DL"]; DR = P["DR"]
            z = x @ U
            u_acc = torch.zeros_like(z)

            for (a, j) in items:
                u = torch.zeros_like(z)

                # core blocks (diagonal blocks)
                for (i0, i1, Bc) in P["core_blocks"][j]:
                    u[:, i0:i1] += z[:, i0:i1] @ Bc

                # low-rank residual
                g = P["gam"][j]
                u += ((z @ DL) * g.view(1, -1)) @ DR.t()

                # sparse blocks
                for (i0, j0, Bb) in P["blocks"][j]:
                    h, w = Bb.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb

                u_acc += float(a) * u

            y += u_acc @ V.t()

        return y

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def dense_apply(Ws: torch.Tensor, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
    n = Ws.shape[-1]
    Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=x.device)
    for a, e in zip(gates, routed):
        Wsum += float(a.item()) * Ws[e]
    return x @ Wsum

@torch.no_grad()
def eval_runtime(rt: KTXRuntime, Ws: torch.Tensor, trials: int, routed_k: int, batch: int):
    E = Ws.shape[0]
    errs = []
    for _ in range(trials):
        x = torch.randn(batch, rt.n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(routed_k, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)

        y_hat = rt(x, routed, gates)
        y_ref = dense_apply(Ws, x, routed, gates)
        err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)

    log(f"[eval] forward rel-error = {float(np.mean(errs)):.4f} ± {float(np.std(errs)):.4f}  (trials={trials})")

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X Optimized (offline) v2 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  (RIDGE_DAMP={cfg.RIDGE_DAMP} weighted={cfg.RIDGE_WEIGHTED})")
    log(f"CORE_MODE:   {cfg.CORE_MODE}  CORE_BLOCK={cfg.CORE_BLOCK} CORE_TARGET={cfg.CORE_TARGET}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK}  blocks={cfg.RES_BLOCKS}  bsize={cfg.RES_BSIZE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} subm={cfg.SUBM} batchE={cfg.BATCH_E} lr={cfg.LR_UV}")
    log(f"Hadamard:    {cfg.USE_HADAMARD} seed={cfg.HAD_SEED}")
    log(f"Threads:     {NTHREADS}  Torch={torch.__version__} device={DEVICE}")
    log(f"Cache:       {cfg.CACHE_MODE}  path={cache_path()}")
    log("")

def main():
    banner()

    eids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} experts={eids[:8]}{'...' if len(eids)>8 else ''}")

    # clusters: better auto for E=16 (defaults ~8)
    if cfg.M_CLUSTERS > 0:
        M = min(cfg.M_CLUSTERS, E)
    else:
        # good default: ~2*sqrt(E), clamped
        M = int(round(2.0 * math.sqrt(E)))
        M = max(4, min(M, E))
    log(f"[cluster] M={M}")

    Xfeat = random_proj_features(Ws, d=64)
    labels, _ = kmeans_torch(Xfeat, k=M, iters=60)
    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    log(f"[cluster] sizes: {[len(c) for c in clusters]}")

    # init U,V
    U_par: List[Optional[OrthoParam]] = []
    V_par: List[Optional[OrthoParam]] = []
    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            U_par.append(None); V_par.append(None)
            continue
        Wm = Ws[idx].mean(dim=0)
        U0, V0 = svd_init(Wm)
        U_par.append(OrthoParam(U0))
        V_par.append(OrthoParam(V0))

    # -------- TRAIN (grad must be enabled) --------
    if cfg.TRAIN_STEPS > 0:
        params = [uv.M for uv in (U_par + V_par) if uv is not None]
        if len(params) == 0:
            raise RuntimeError("No trainable U/V params found.")
        opt = torch.optim.Adam(params, lr=cfg.LR_UV)

        t0 = time.perf_counter()

        # In notebooks, grad can be disabled globally; force-enable here.
        with torch.enable_grad():
            for step in range(1, cfg.TRAIN_STEPS + 1):
                S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]
                L_total = None
                n_terms = 0

                for m, idx in enumerate(clusters):
                    if len(idx) == 0:
                        continue

                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()

                    if cfg.BATCH_E > 0 and cfg.BATCH_E < len(idx):
                        pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                        idx_step = [idx[p] for p in pick]
                    else:
                        idx_step = idx

                    Ws_batch = Ws[idx_step]  # (Eb,n,n) constant
                    Xs = slice_X_batch(Ws_batch, Uo, Vo, S)

                    off = offdiag_abs_mean(Xs)
                    diag = diag_abs_mean(Xs)
                    loss = off / (diag + 1e-6)

                    L_total = loss if (L_total is None) else (L_total + loss)
                    n_terms += 1

                if L_total is None:
                    raise RuntimeError("No clusters contributed to loss (all empty?)")

                L_total = L_total / max(1, n_terms)

                # Hard fail-fast if the graph is broken
                if not L_total.requires_grad:
                    raise RuntimeError(
                        "Loss is not connected to U/V params (no grad_fn). "
                        "This usually means something ran under torch.no_grad(), "
                        "or U/V were detached. Check your environment/global contexts."
                    )

                opt.zero_grad(set_to_none=True)
                L_total.backward()
                opt.step()

                # Project back to orthonormal after the step (projected GD)
                if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    with torch.no_grad():
                        for uv in (U_par + V_par):
                            if uv is not None:
                                uv.M.copy_(uv.orthogonal())

                if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    t1 = time.perf_counter()
                    log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} | loss={float(L_total.item()):.4f}  (+{t1-t0:.1f}s)")
                    t0 = t1

    # build payloads
    payloads: List[Optional[dict]] = []
    basis_of_e: Dict[int, Tuple[int,int]] = {}
    log("[build] payloads ...")
    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            payloads.append(None)
            continue
        Uo = U_par[m].orthogonal().detach()
        Vo = V_par[m].orthogonal().detach()
        P = build_payload_for_cluster(Ws, idx, Uo, Vo)
        payloads.append(P)

        for j, e in enumerate(idx):
            basis_of_e[e] = (m, j)

        core = P["core"]
        kept = sum((i1 - i0) for (i0, i1) in core["keep_ranges"]) if core["mode"] == "blockdiag" else 0
        log(f"  - basis {m}: E={len(idx)} kept_dim≈{kept}/{n} ({kept/n:.1%}) rank={P['DL'].shape[1]} blocks={cfg.RES_BLOCKS}")

    rt = KTXRuntime(payloads, basis_of_e)
    eval_runtime(rt, Ws, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    log("\n✅ Done.")
    log("DeepSeek tips:")
    log(" - Use real CALIB_PATH (layer input activations) for LIN_MODE=ridge.")
    log(" - If you have ROUTER_PATH, set RIDGE_WEIGHTED=1.")
    log(" - If error stays high: increase RES_RANK and RES_BLOCKS, and consider M_CLUSTERS=8 or 12.")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X Optimized (offline) v2 ==
Time:        2026-01-01 16:35:48
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
LIN_MODE:    ridge  (RIDGE_DAMP=0.001 weighted=False)
CORE_MODE:   blockdiag  CORE_BLOCK=64 CORE_TARGET=0.5
RESIDUAL:    rank=512  blocks=48  bsize=64
TRAIN:       steps=24 subm=256 batchE=4 lr=0.05
Hadamard:    False seed=1234
Threads:     8  Torch=2.4.1+cpu device=cpu
Cache:       readwrite  path=/home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache.npz

[cache] loaded Ws: /home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache.npz  Ws=(16, 2048, 2048)
[Ws] shape=(16, 2048, 2048) experts=[0, 1, 2, 3, 4, 5, 6, 7]...
[cluster] M=8
[cluster] sizes: [2, 1, 1, 1, 1, 1, 7, 2]
[train] step   1/24 | loss=0.0102  (+7.7s)
[train] step   4/24 | loss=0.1779  (+24.6s)
[train] step   8/24 | loss=0.2276  (+30.3s)
[train] step  12/24 | loss=0.2007  (+30.4s)
[train] step  16/24 | loss=0.1769 

In [15]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek-Optimized OFFLINE KT++-X Runner (single file) v3
#   - offline safetensors via index.json
#   - square expert mats (ridge/weff_gate/weff)
#   - KT++ payload with:
#       * CORE_MODE=blocktopk (NEW, off-diagonal blocks allowed)
#       * CORE_MODE=blockdiag (old)
#       * shared low-rank residual + greedy blocks
#   - training fixes: stable loss, skip singleton clusters, grad clip
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np

try:
    import torch
    import torch.nn.functional as F
    import torch.nn as nn
except Exception as e:
    raise RuntimeError("This script requires PyTorch.") from e

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("This script requires safetensors (pip install safetensors).") from e

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

# ----------------------------
# Threading / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))

DTYPE_W   = torch.float16
DTYPE_ACC = torch.float32

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config (env overrides)
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "16"))

    # Calibration (npz with key 'X': (N,H))
    CALIB_PATH: str = os.environ.get("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "512"))

    # Optional router probs (npz with key 'P': (N,E_total))
    ROUTER_PATH: str = os.environ.get("ROUTER_PATH", "").strip()
    ROUTER_EIDS_ARE_GLOBAL: bool = os.environ.get("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    # Build square matrices
    LIN_MODE: str = os.environ.get("LIN_MODE", "ridge").strip().lower()  # ridge | weff_gate | weff
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))      # relative to trace(XtX)/H
    RIDGE_WEIGHTED: bool = os.environ.get("RIDGE_WEIGHTED", "0") == "1"  # weight by router P if provided

    # Clustering
    M_CLUSTERS: int = int(os.environ.get("M_CLUSTERS", "0"))  # 0 => auto

    # Training
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "24"))
    SUBM: int = int(os.environ.get("SUBM", "256"))
    BATCH_E: int = int(os.environ.get("BATCH_E", "4"))
    LR_UV: float = float(os.environ.get("LR_UV", "5e-2"))
    REORTHO_EVERY: int = int(os.environ.get("REORTHO_EVERY", "4"))
    REPORT_EVERY: int = int(os.environ.get("REPORT_EVERY", "4"))
    TRAIN_MIN_CLUSTER: int = int(os.environ.get("TRAIN_MIN_CLUSTER", "2"))  # skip clusters smaller than this
    GRAD_CLIP: float = float(os.environ.get("GRAD_CLIP", "1.0"))            # 0 disables

    # Core model
    CORE_MODE: str = os.environ.get("CORE_MODE", "blocktopk").strip().lower()  # blocktopk|blockdiag|none
    CORE_BLOCK: int = int(os.environ.get("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(os.environ.get("CORE_TARGET", "0.60"))         # energy fraction target
    CORE_MAX_BLOCKS: int = int(os.environ.get("CORE_MAX_BLOCKS", "192"))       # cap for blocktopk

    # Residual model
    RES_RANK: int = int(os.environ.get("RES_RANK", "1024"))
    RES_BLOCKS: int = int(os.environ.get("RES_BLOCKS", "96"))
    RES_BSIZE: int = int(os.environ.get("RES_BSIZE", "64"))

    # Optional Hadamard similarity rotation on Ws
    USE_HADAMARD: bool = os.environ.get("USE_HADAMARD", "0") == "1"
    HAD_SEED: int = int(os.environ.get("HAD_SEED", "1234"))

    # Cache
    CACHE_MODE: str = os.environ.get("CACHE_MODE", "readwrite").strip().lower()  # off|read|write|readwrite
    CACHE_NAME: str = os.environ.get("CACHE_NAME", "").strip()
    FORCE_REBUILD_WS: bool = os.environ.get("FORCE_REBUILD_WS", "0") == "1"

    # Eval
    EVAL_TRIALS: int = int(os.environ.get("EVAL_TRIALS", "5"))
    ROUTED_K: int = int(os.environ.get("ROUTED_K", "8"))
    EVAL_BATCH: int = int(os.environ.get("EVAL_BATCH", "2"))

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def cache_path():
    if cfg.CACHE_NAME:
        return os.path.join(cfg.OUTPUT_DIR, cfg.CACHE_NAME)
    tag = f"deepseek_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_{cfg.LIN_MODE}_Ws_cache_v3.npz"
    return os.path.join(cfg.OUTPUT_DIR, tag)

def _meta_dict():
    # minimal cache validation; avoids silently reusing Ws built with random X
    return dict(
        layer=cfg.LAYER,
        max_experts=cfg.MAX_EXPERTS,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES,
        router_path=cfg.ROUTER_PATH or "",
        hadamard=cfg.USE_HADAMARD,
        had_seed=cfg.HAD_SEED,
    )

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None

    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    out = {}
    if up: out["up"] = up
    if gate: out["gate"] = gate
    if down: out["down"] = down
    return out

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration / router loading
# ----------------------------
def load_calib_X(H: int) -> torch.Tensor:
    if not cfg.CALIB_PATH:
        X = torch.randn(cfg.CALIB_SAMPLES, H, dtype=DTYPE_W, device=DEVICE)
        log(f"[calib] RANDOM X: {tuple(X.shape)}  (set CALIB_PATH npz key 'X' for realistic ridge)")
        return X
    if not os.path.isfile(cfg.CALIB_PATH):
        raise FileNotFoundError(f"CALIB_PATH not found: {cfg.CALIB_PATH}")
    z = np.load(cfg.CALIB_PATH, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Has keys: {list(z.files)}")
    X = torch.from_numpy(z["X"]).to(DTYPE_W).to(DEVICE)
    z.close()
    if X.ndim != 2 or X.shape[0] < 2:
        raise RuntimeError(f"Bad X shape: {tuple(X.shape)}")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    log(f"[calib] Loaded X: {tuple(X.shape)}")
    return X

def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    if not os.path.isfile(cfg.ROUTER_PATH):
        raise FileNotFoundError(f"ROUTER_PATH not found: {cfg.ROUTER_PATH}")
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Has keys: {list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    log(f"[router] Loaded P: {tuple(P.shape)}")
    return P

# ----------------------------
# Hadamard (optional)
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

@torch.no_grad()
def hadamard_orth(n: int, seed: int) -> torch.Tensor:
    if not is_power_of_two(n):
        raise ValueError(f"Hadamard requires power-of-two n, got {n}")
    H = torch.tensor([[1.0]], dtype=DTYPE_ACC, device=DEVICE)
    while H.shape[0] < n:
        H = torch.cat([torch.cat([H,  H], dim=1),
                       torch.cat([H, -H], dim=1)], dim=0)
    H = H / math.sqrt(n)
    g = torch.Generator(device="cpu").manual_seed(seed)
    signs = (torch.randint(0, 2, (n,), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    R = (H * signs.view(1, n)).contiguous()
    return R

# ----------------------------
# Build expert square Ws
# ----------------------------
@torch.no_grad()
def forward_mlp_no_grad(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

@torch.no_grad()
def build_Ws_square() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer {cfg.LAYER} experts: {all_eids}  (using {len(eids)})")

    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if "up" not in kk or "down" not in kk or "gate" not in kk:
            raise RuntimeError(f"Expert {eid} missing keys: {kk}")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]].to(DTYPE_W)
    dff, H = W_up0.shape
    log(f"[shape] H={H} d_ff={dff}")

    X = load_calib_X(H)
    P = load_router_P()

    R = hadamard_orth(H, cfg.HAD_SEED) if cfg.USE_HADAMARD else None

    Xf = X.to(DTYPE_ACC)
    XtX = (Xf.t() @ Xf)
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    cholG = None
    if cfg.LIN_MODE == "ridge" and not (cfg.RIDGE_WEIGHTED and (P is not None)):
        G = XtX + lam * I
        cholG = torch.linalg.cholesky(G)

    Ws = []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws")):
        W_up   = T[per_e[eid]["up"]].to(DTYPE_W).to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DTYPE_W).to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DTYPE_W).to(DEVICE)

        if cfg.LIN_MODE == "weff":
            W = (W_down.to(DTYPE_ACC) @ W_up.to(DTYPE_ACC))

        elif cfg.LIN_MODE == "weff_gate":
            gate_act = (Xf @ W_gate.to(DTYPE_ACC).t())
            m = F.silu(gate_act).mean(dim=0)  # (dff,)
            W = (W_down.to(DTYPE_ACC) * m.view(1, -1)) @ W_up.to(DTYPE_ACC)

        elif cfg.LIN_MODE == "ridge":
            Y = forward_mlp_no_grad(X, W_gate, W_up, W_down).to(DTYPE_ACC)  # (N,H)

            if cfg.RIDGE_WEIGHTED and (P is not None):
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                else:
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw
                XtX_e = Xw.t() @ Xw
                lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
                G_e = XtX_e + lam_e * I
                chol = torch.linalg.cholesky(G_e)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)
                W = Wt.t().contiguous()
            else:
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)
                W = Wt.t().contiguous()
        else:
            raise ValueError("LIN_MODE must be ridge|weff_gate|weff")

        if R is not None:
            W = (R.t() @ W @ R).contiguous()

        fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
        W = (W / fn).contiguous()
        Ws.append(W)

    Ws = torch.stack(Ws, dim=0)  # (E,H,H)
    return eids, Ws

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    p = cache_path()
    do_read = cfg.CACHE_MODE in ("read", "readwrite")
    do_write = cfg.CACHE_MODE in ("write", "readwrite")

    if do_read and (not cfg.FORCE_REBUILD_WS) and os.path.isfile(p):
        z = try_load_npz(p)
        if z and ("expert_ids" in z) and ("Ws" in z):
            ok = True
            if "meta" in z:
                meta = _decode_meta(z["meta"])
                ok = (meta == _meta_dict())
            if ok:
                eids = [int(x) for x in z["expert_ids"].tolist()]
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws: {p}  Ws={tuple(Ws.shape)}")
                return eids, Ws
            else:
                log("[cache] meta mismatch (likely CALIB/ROUTER changed) -> rebuilding Ws.")
        else:
            log("[cache] invalid cache -> rebuilding Ws.")

    eids, Ws = build_Ws_square()
    if do_write:
        save_npz(p, {
            "meta": _encode_meta(_meta_dict()),
            "expert_ids": np.array(eids, dtype=np.int32),
            "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        })
        log(f"[cache] wrote: {p}  ({os.path.getsize(p)/1e6:.2f} MB)")
    return eids, Ws

# ----------------------------
# Clustering features + kmeans
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int = 64) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R).unsqueeze(0), (col @ R).unsqueeze(0)], dim=1))
    return torch.cat(feats, dim=0)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int = 60) -> Tuple[torch.Tensor, torch.Tensor]:
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(X.shape[0], generator=g)[:k]
    C = X[idx].clone()
    for _ in range(iters):
        dist = torch.cdist(X, C)
        lab = dist.argmin(dim=1)
        for j in range(k):
            m = (lab == j)
            if m.any():
                C[j] = X[m].mean(dim=0)
    return lab, C

# ----------------------------
# Ortho params + training helpers
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, M_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(M_init.clone().to(DEVICE))

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def svd_init(W_mean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(W_mean, full_matrices=False)
    V = Vh.t().contiguous()
    return U.contiguous(), V.contiguous()

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    # Xs = U_S^T * (Ws_batch * V_S)  => (Eb, s, s)
    U_S = U[:, S]  # (n,s)
    V_S = V[:, S]  # (n,s)
    T = Ws_batch @ V_S
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

# ----------------------------
# Payload building
# ----------------------------
@torch.no_grad()
def greedy_blocks(R: torch.Tensor, t: int, b: int) -> List[Tuple[int,int,torch.Tensor]]:
    n = R.shape[0]
    used = torch.zeros(n, n, dtype=torch.bool, device=R.device)
    out = []
    if t <= 0:
        return out
    for _ in range(t):
        best = None
        bestE = -1.0
        for i0 in range(0, n, b):
            for j0 in range(0, n, b):
                i1 = min(n, i0 + b)
                j1 = min(n, j0 + b)
                if used[i0:i1, j0:j1].any():
                    continue
                Eblk = float((R[i0:i1, j0:j1] ** 2).sum().item())
                if Eblk > bestE:
                    bestE = Eblk
                    best = (i0, j0, i1, j1)
        if best is None:
            break
        i0, j0, i1, j1 = best
        used[i0:i1, j0:j1] = True
        out.append((i0, j0, R[i0:i1, j0:j1].clone()))
    return out

@torch.no_grad()
def _block_energy_grid(X: torch.Tensor, b: int) -> torch.Tensor:
    # assumes n divisible by b for speed (DeepSeek H=2048, b=64 OK)
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        # pad to nb*b
        pad = nb * b - n
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
        n = nb*b
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()  # (nb,nb,b,b)
    E = (Xb * Xb).sum(dim=(2,3))  # (nb,nb)
    return E

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor], mode: str, block: int, target: float, max_blocks: int):
    n = X_list[0].shape[0]
    E = len(X_list)
    b = block
    nb = (n + b - 1) // b

    if mode == "none":
        return {"mode": "none", "energy_frac": 0.0, "num_blocks": 0, "blocks": []}

    # total energy (mean over experts)
    tot = 0.0
    Eg = torch.zeros(nb, nb, dtype=DTYPE_ACC, device=X_list[0].device)
    for X in X_list:
        tot += float((X * X).sum().item())
        Eg += _block_energy_grid(X, b).to(DTYPE_ACC)
    tot /= max(1, E)
    Eg /= max(1, E)

    blocks = []

    if mode == "blockdiag":
        # only diagonal blocks
        diagE = torch.diagonal(Eg, 0)
        order = torch.argsort(diagE, descending=True)
        csum = torch.cumsum(diagE[order], dim=0)
        frac = csum / max(tot, 1e-12)
        K = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else nb)
        keep = order[:K].tolist()
        kept_energy = float(diagE[order[:K]].sum().item())
        for bi in keep:
            i0 = bi * b
            i1 = min(n, i0 + b)
            blocks.append((i0, i0, i1 - i0, i1 - i0))  # (i0, j0, h, w) where j0=i0
        return {"mode": "blockdiag", "block": b, "energy_frac": kept_energy / max(tot, 1e-12),
                "num_blocks": len(blocks), "blocks": blocks}

    if mode == "blocktopk":
        # top blocks anywhere in grid
        flat = Eg.reshape(-1)
        order = torch.argsort(flat, descending=True)
        csum = torch.cumsum(flat[order], dim=0)
        frac = csum / max(tot, 1e-12)

        need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
        K = min(need, max_blocks, flat.numel())
        pick = order[:K]

        kept_energy = float(flat[pick].sum().item())
        for idx in pick.tolist():
            bi = idx // nb
            bj = idx % nb
            i0 = bi * b
            j0 = bj * b
            h = min(b, n - i0)
            w = min(b, n - j0)
            blocks.append((i0, j0, h, w))
        return {"mode": "blocktopk", "block": b, "energy_frac": kept_energy / max(tot, 1e-12),
                "num_blocks": len(blocks), "blocks": blocks}

    raise ValueError("CORE_MODE must be blocktopk|blockdiag|none")

@torch.no_grad()
def build_payload_for_cluster(Ws: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor):
    X_list = []
    for e in idx:
        X = (U.t() @ Ws[e] @ V).contiguous()
        X_list.append(X)

    core = choose_core_blocks(
        X_list,
        cfg.CORE_MODE,
        cfg.CORE_BLOCK,
        cfg.CORE_TARGET,
        cfg.CORE_MAX_BLOCKS
    )

    # per-expert core blocks
    core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
    for X in X_list:
        lst = []
        for (i0, j0, h, w) in core["blocks"]:
            lst.append((i0, j0, X[i0:i0+h, j0:j0+w].clone()))
        core_blocks.append(lst)

    # residuals: R = X - Xcore
    R_list = []
    for X, cb in zip(X_list, core_blocks):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    U_s, _, Vh_s = torch.linalg.svd(Rmean, full_matrices=False)
    r = min(cfg.RES_RANK, U_s.shape[1])
    DL = U_s[:, :r].contiguous()
    DR = Vh_s.t()[:, :r].contiguous()

    gam_list = []
    blocks_list = []
    for Rm in R_list:
        g = torch.sum(DL * (Rm @ DR), dim=0)  # (r,)
        gam_list.append(g.contiguous())
        R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        blks = greedy_blocks(R2, cfg.RES_BLOCKS, cfg.RES_BSIZE)
        blocks_list.append(blks)

    gam = torch.stack(gam_list, dim=0)

    return {
        "U": U, "V": V,
        "core": core,
        "core_blocks": core_blocks,
        "DL": DL, "DR": DR,
        "gam": gam,
        "blocks": blocks_list,
        "idx": idx,
    }

class KTXRuntime(nn.Module):
    def __init__(self, payloads: List[Optional[dict]], basis_of_e: Dict[int, Tuple[int,int]]):
        super().__init__()
        self.payloads = payloads
        self.map = basis_of_e
        self.n = None
        for p in payloads:
            if p is not None:
                self.n = int(p["U"].shape[0])
                break
        if self.n is None:
            raise RuntimeError("No payloads provided.")

    @torch.no_grad()
    def forward(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        used: Dict[int, List[Tuple[float,int]]] = {}
        for a, e in zip(gates.tolist(), routed):
            if e in self.map:
                m, j = self.map[e]
                used.setdefault(m, []).append((a, j))

        for m, items in used.items():
            P = self.payloads[m]
            if P is None:
                continue
            U = P["U"]; V = P["V"]
            DL = P["DL"]; DR = P["DR"]
            z = x @ U
            u_acc = torch.zeros_like(z)

            for (a, j) in items:
                u = torch.zeros_like(z)

                # core blocks (possibly off-diagonal)
                for (i0, j0, Bc) in P["core_blocks"][j]:
                    h, w = Bc.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bc

                # low-rank residual
                g = P["gam"][j]
                u += ((z @ DL) * g.view(1, -1)) @ DR.t()

                # greedy residual blocks
                for (i0, j0, Bb) in P["blocks"][j]:
                    h, w = Bb.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb

                u_acc += float(a) * u

            y += u_acc @ V.t()

        return y

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def dense_apply(Ws: torch.Tensor, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
    n = Ws.shape[-1]
    Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=x.device)
    for a, e in zip(gates, routed):
        Wsum += float(a.item()) * Ws[e]
    return x @ Wsum

@torch.no_grad()
def eval_runtime(rt: KTXRuntime, Ws: torch.Tensor, trials: int, routed_k: int, batch: int):
    E = Ws.shape[0]
    errs = []
    for _ in range(trials):
        x = torch.randn(batch, rt.n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(routed_k, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)

        y_hat = rt(x, routed, gates)
        y_ref = dense_apply(Ws, x, routed, gates)
        err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)

    log(f"[eval] forward rel-error = {float(np.mean(errs)):.4f} ± {float(np.std(errs)):.4f}  (trials={trials})")

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X Optimized (offline) v3 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  (RIDGE_DAMP={cfg.RIDGE_DAMP})")
    log(f"CORE_MODE:   {cfg.CORE_MODE}  block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK}  blocks={cfg.RES_BLOCKS}  bsize={cfg.RES_BSIZE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} subm={cfg.SUBM} batchE={cfg.BATCH_E} lr={cfg.LR_UV} min_cluster={cfg.TRAIN_MIN_CLUSTER}")
    log(f"Hadamard:    {cfg.USE_HADAMARD} seed={cfg.HAD_SEED}")
    log(f"Threads:     {NTHREADS}  Torch={torch.__version__} device={DEVICE}")
    log(f"Cache:       {cfg.CACHE_MODE}  path={cache_path()}  force_rebuild={cfg.FORCE_REBUILD_WS}")
    log("")

def main():
    banner()

    eids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} experts={eids[:8]}{'...' if len(eids)>8 else ''}")

    # clusters
    if cfg.M_CLUSTERS > 0:
        M = min(cfg.M_CLUSTERS, E)
    else:
        M = int(round(2.0 * math.sqrt(E)))
        M = max(6, min(M, E))
    log(f"[cluster] M={M}")

    Xfeat = random_proj_features(Ws, d=64)
    labels, _ = kmeans_torch(Xfeat, k=M, iters=60)
    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    log(f"[cluster] sizes: {[len(c) for c in clusters]}")

    # init U,V
    U_par: List[Optional[OrthoParam]] = []
    V_par: List[Optional[OrthoParam]] = []
    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            U_par.append(None); V_par.append(None)
            continue
        Wm = Ws[idx].mean(dim=0)
        U0, V0 = svd_init(Wm)
        U_par.append(OrthoParam(U0))
        V_par.append(OrthoParam(V0))

    # -------- TRAIN --------
    if cfg.TRAIN_STEPS > 0:
        params = [uv.M for uv in (U_par + V_par) if uv is not None]
        opt = torch.optim.Adam(params, lr=cfg.LR_UV)
        t0 = time.perf_counter()

        with torch.enable_grad():
            for step in range(1, cfg.TRAIN_STEPS + 1):
                S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]
                L_total = None
                n_terms = 0

                for m, idx in enumerate(clusters):
                    if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                        continue
                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()

                    if cfg.BATCH_E > 0 and cfg.BATCH_E < len(idx):
                        pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                        idx_step = [idx[p] for p in pick]
                    else:
                        idx_step = idx

                    Ws_batch = Ws[idx_step]
                    Xs = slice_X_batch(Ws_batch, Uo, Vo, S)

                    off = offdiag_abs_mean(Xs)
                    diag = diag_abs_mean(Xs)

                    # stable: minimize log(off/diag)
                    loss = torch.log(off + 1e-6) - torch.log(diag + 1e-6)

                    L_total = loss if (L_total is None) else (L_total + loss)
                    n_terms += 1

                if L_total is None:
                    log("[train] skipped (no clusters >= TRAIN_MIN_CLUSTER)")
                    break

                L_total = L_total / max(1, n_terms)

                if not L_total.requires_grad:
                    raise RuntimeError("Training graph broken: loss has no grad_fn.")

                opt.zero_grad(set_to_none=True)
                L_total.backward()

                if cfg.GRAD_CLIP > 0:
                    torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)

                opt.step()

                if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    with torch.no_grad():
                        for uv in (U_par + V_par):
                            if uv is not None:
                                uv.M.copy_(uv.orthogonal())

                if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    t1 = time.perf_counter()
                    log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} | loss={float(L_total.item()):.4f}  (+{t1-t0:.1f}s)")
                    t0 = t1

    # build payloads
    payloads: List[Optional[dict]] = []
    basis_of_e: Dict[int, Tuple[int,int]] = {}
    log("[build] payloads ...")
    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            payloads.append(None)
            continue
        Uo = U_par[m].orthogonal().detach()
        Vo = V_par[m].orthogonal().detach()
        P = build_payload_for_cluster(Ws, idx, Uo, Vo)
        payloads.append(P)

        for j, e in enumerate(idx):
            basis_of_e[e] = (m, j)

        c = P["core"]
        log(f"  - basis {m}: E={len(idx)} core={c['mode']} blocks={c['num_blocks']} "
            f"core_energy≈{c['energy_frac']:.3f} rank={P['DL'].shape[1]} res_blocks={cfg.RES_BLOCKS}")

    rt = KTXRuntime(payloads, basis_of_e)
    eval_runtime(rt, Ws, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    log("\n✅ Done.")
    log("If DeepSeek error is still high:")
    log(" - CORE_MODE=blocktopk and raise CORE_TARGET (0.7-0.85) with CORE_MAX_BLOCKS cap.")
    log(" - Increase RES_RANK and RES_BLOCKS.")
    log(" - Increase M_CLUSTERS (12 or 16) to avoid mega-clusters.")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X Optimized (offline) v3 ==
Time:        2026-01-01 16:45:11
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES=512
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
LIN_MODE:    ridge  (RIDGE_DAMP=0.001)
CORE_MODE:   blocktopk  block=64 target=0.6 max_blocks=192
RESIDUAL:    rank=1024  blocks=96  bsize=64
TRAIN:       steps=24 subm=256 batchE=4 lr=0.05 min_cluster=2
Hadamard:    False seed=1234
Threads:     8  Torch=2.4.1+cpu device=cpu
Cache:       readwrite  path=/home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v3.npz  force_rebuild=False

[found] layer 1 experts: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]  (using 16)
[load] reading tensors from shards ...
[sha

Build Ws:   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote: /home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v3.npz  (268.44 MB)
[Ws] shape=(16, 2048, 2048) experts=[0, 1, 2, 3, 4, 5, 6, 7]...
[cluster] M=8
[cluster] sizes: [2, 1, 1, 1, 1, 1, 7, 2]
[train] step   1/24 | loss=-3.7251  (+2.7s)
[train] step   4/24 | loss=-1.1787  (+11.7s)
[train] step   8/24 | loss=-1.3783  (+13.7s)
[train] step  12/24 | loss=-1.5551  (+13.7s)
[train] step  16/24 | loss=-1.7924  (+13.7s)
[train] step  20/24 | loss=-1.6597  (+13.9s)
[train] step  24/24 | loss=-1.7647  (+13.7s)
[build] payloads ...
  - basis 0: E=2 core=blocktopk blocks=192 core_energy≈0.513 rank=1024 res_blocks=96
  - basis 1: E=1 core=blocktopk blocks=1 core_energy≈0.837 rank=1024 res_blocks=96
  - basis 2: E=1 core=blocktopk blocks=2 core_energy≈0.692 rank=1024 res_blocks=96
  - basis 3: E=1 core=blocktopk blocks=2 core_energy≈0.700 rank=1024 res_blocks=96
  - basis 4: E=1 core=blocktopk blocks=2 core_energy≈0.669 rank=1024 res_blocks=96
  - basis 5: E=1 core=blockto

In [16]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek-Optimized OFFLINE KT++-X Runner (single file) v4
#
# Key upgrade vs v3:
#   - CORE_MODE=blocktopk_perexpert (per-expert block selection)
#     fixes "cluster mean smears energy" -> big win for DeepSeek.
#
# CORE_MODE options:
#   - blocktopk_perexpert  (NEW, recommended for DeepSeek)
#   - blocktopk            (shared blocks across cluster)
#   - blockdiag            (shared diagonal blocks only)
#   - none
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np

try:
    import torch
    import torch.nn.functional as F
    import torch.nn as nn
except Exception as e:
    raise RuntimeError("This script requires PyTorch.") from e

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("This script requires safetensors (pip install safetensors).") from e

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

# ----------------------------
# Threading / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))

DTYPE_W   = torch.float16
DTYPE_ACC = torch.float32

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "16"))

    # Calibration
    CALIB_PATH: str = os.environ.get("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "512"))

    # Router probs optional
    ROUTER_PATH: str = os.environ.get("ROUTER_PATH", "").strip()
    ROUTER_EIDS_ARE_GLOBAL: bool = os.environ.get("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    # Build square matrices
    LIN_MODE: str = os.environ.get("LIN_MODE", "ridge").strip().lower()  # ridge | weff_gate | weff
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))
    RIDGE_WEIGHTED: bool = os.environ.get("RIDGE_WEIGHTED", "0") == "1"

    # Clustering
    M_CLUSTERS: int = int(os.environ.get("M_CLUSTERS", "0"))  # 0 => auto

    # Training
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "24"))
    SUBM: int = int(os.environ.get("SUBM", "256"))
    BATCH_E: int = int(os.environ.get("BATCH_E", "4"))
    LR_UV: float = float(os.environ.get("LR_UV", "5e-2"))
    REORTHO_EVERY: int = int(os.environ.get("REORTHO_EVERY", "4"))
    REPORT_EVERY: int = int(os.environ.get("REPORT_EVERY", "4"))
    TRAIN_MIN_CLUSTER: int = int(os.environ.get("TRAIN_MIN_CLUSTER", "2"))
    GRAD_CLIP: float = float(os.environ.get("GRAD_CLIP", "1.0"))

    # Core model
    CORE_MODE: str = os.environ.get("CORE_MODE", "blocktopk_perexpert").strip().lower()
    # for shared blocktopk: how to aggregate per-expert block energies to choose shared blocks
    CORE_AGG: str = os.environ.get("CORE_AGG", "mean").strip().lower()  # mean|max
    CORE_BLOCK: int = int(os.environ.get("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(os.environ.get("CORE_TARGET", "0.75"))
    CORE_MAX_BLOCKS: int = int(os.environ.get("CORE_MAX_BLOCKS", "256"))

    # Residual
    RES_RANK: int = int(os.environ.get("RES_RANK", "1024"))
    RES_BLOCKS: int = int(os.environ.get("RES_BLOCKS", "128"))
    RES_BSIZE: int = int(os.environ.get("RES_BSIZE", "64"))

    # Optional similarity rotation
    USE_HADAMARD: bool = os.environ.get("USE_HADAMARD", "0") == "1"
    HAD_SEED: int = int(os.environ.get("HAD_SEED", "1234"))

    # Cache
    CACHE_MODE: str = os.environ.get("CACHE_MODE", "readwrite").strip().lower()
    CACHE_NAME: str = os.environ.get("CACHE_NAME", "").strip()
    FORCE_REBUILD_WS: bool = os.environ.get("FORCE_REBUILD_WS", "0") == "1"

    # Eval
    EVAL_TRIALS: int = int(os.environ.get("EVAL_TRIALS", "5"))
    ROUTED_K: int = int(os.environ.get("ROUTED_K", "8"))
    EVAL_BATCH: int = int(os.environ.get("EVAL_BATCH", "2"))

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def cache_path():
    if cfg.CACHE_NAME:
        return os.path.join(cfg.OUTPUT_DIR, cfg.CACHE_NAME)
    tag = f"deepseek_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_{cfg.LIN_MODE}_Ws_cache_v4.npz"
    return os.path.join(cfg.OUTPUT_DIR, tag)

def _meta_dict():
    return dict(
        layer=cfg.LAYER,
        max_experts=cfg.MAX_EXPERTS,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES,
        router_path=cfg.ROUTER_PATH or "",
        hadamard=cfg.USE_HADAMARD,
        had_seed=cfg.HAD_SEED,
    )

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None

    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    out = {}
    if up: out["up"] = up
    if gate: out["gate"] = gate
    if down: out["down"] = down
    return out

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration / router loading
# ----------------------------
def load_calib_X(H: int) -> torch.Tensor:
    if not cfg.CALIB_PATH:
        X = torch.randn(cfg.CALIB_SAMPLES, H, dtype=DTYPE_W, device=DEVICE)
        log(f"[calib] RANDOM X: {tuple(X.shape)}  (set CALIB_PATH npz key 'X' for realistic ridge)")
        return X
    if not os.path.isfile(cfg.CALIB_PATH):
        raise FileNotFoundError(f"CALIB_PATH not found: {cfg.CALIB_PATH}")
    z = np.load(cfg.CALIB_PATH, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Has keys: {list(z.files)}")
    X = torch.from_numpy(z["X"]).to(DTYPE_W).to(DEVICE)
    z.close()
    if X.ndim != 2 or X.shape[0] < 2:
        raise RuntimeError(f"Bad X shape: {tuple(X.shape)}")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    log(f"[calib] Loaded X: {tuple(X.shape)}")
    return X

def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    if not os.path.isfile(cfg.ROUTER_PATH):
        raise FileNotFoundError(f"ROUTER_PATH not found: {cfg.ROUTER_PATH}")
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Has keys: {list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    log(f"[router] Loaded P: {tuple(P.shape)}")
    return P

# ----------------------------
# Hadamard (optional)
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

@torch.no_grad()
def hadamard_orth(n: int, seed: int) -> torch.Tensor:
    if not is_power_of_two(n):
        raise ValueError(f"Hadamard requires power-of-two n, got {n}")
    H = torch.tensor([[1.0]], dtype=DTYPE_ACC, device=DEVICE)
    while H.shape[0] < n:
        H = torch.cat([torch.cat([H,  H], dim=1),
                       torch.cat([H, -H], dim=1)], dim=0)
    H = H / math.sqrt(n)
    g = torch.Generator(device="cpu").manual_seed(seed)
    signs = (torch.randint(0, 2, (n,), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    R = (H * signs.view(1, n)).contiguous()
    return R

# ----------------------------
# Build expert square Ws
# ----------------------------
@torch.no_grad()
def forward_mlp_no_grad(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

@torch.no_grad()
def build_Ws_square() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer {cfg.LAYER} experts: {all_eids}  (using {len(eids)})")

    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if "up" not in kk or "down" not in kk or "gate" not in kk:
            raise RuntimeError(f"Expert {eid} missing keys: {kk}")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]].to(DTYPE_W)
    dff, H = W_up0.shape
    log(f"[shape] H={H} d_ff={dff}")

    X = load_calib_X(H)
    P = load_router_P()

    R = hadamard_orth(H, cfg.HAD_SEED) if cfg.USE_HADAMARD else None

    Xf = X.to(DTYPE_ACC)
    XtX = (Xf.t() @ Xf)
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    cholG = None
    if cfg.LIN_MODE == "ridge" and not (cfg.RIDGE_WEIGHTED and (P is not None)):
        G = XtX + lam * I
        cholG = torch.linalg.cholesky(G)

    Ws = []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws")):
        W_up   = T[per_e[eid]["up"]].to(DTYPE_W).to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DTYPE_W).to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DTYPE_W).to(DEVICE)

        if cfg.LIN_MODE == "weff":
            W = (W_down.to(DTYPE_ACC) @ W_up.to(DTYPE_ACC))

        elif cfg.LIN_MODE == "weff_gate":
            gate_act = (Xf @ W_gate.to(DTYPE_ACC).t())
            m = F.silu(gate_act).mean(dim=0)
            W = (W_down.to(DTYPE_ACC) * m.view(1, -1)) @ W_up.to(DTYPE_ACC)

        elif cfg.LIN_MODE == "ridge":
            Y = forward_mlp_no_grad(X, W_gate, W_up, W_down).to(DTYPE_ACC)

            if cfg.RIDGE_WEIGHTED and (P is not None):
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                else:
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw
                XtX_e = Xw.t() @ Xw
                lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
                G_e = XtX_e + lam_e * I
                chol = torch.linalg.cholesky(G_e)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)
                W = Wt.t().contiguous()
            else:
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)
                W = Wt.t().contiguous()
        else:
            raise ValueError("LIN_MODE must be ridge|weff_gate|weff")

        if R is not None:
            W = (R.t() @ W @ R).contiguous()

        fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
        Ws.append((W / fn).contiguous())

    Ws = torch.stack(Ws, dim=0)  # (E,H,H)
    return eids, Ws

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    p = cache_path()
    do_read = cfg.CACHE_MODE in ("read", "readwrite")
    do_write = cfg.CACHE_MODE in ("write", "readwrite")

    if do_read and (not cfg.FORCE_REBUILD_WS) and os.path.isfile(p):
        z = try_load_npz(p)
        if z and ("expert_ids" in z) and ("Ws" in z):
            ok = True
            if "meta" in z:
                meta = _decode_meta(z["meta"])
                ok = (meta == _meta_dict())
            if ok:
                eids = [int(x) for x in z["expert_ids"].tolist()]
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws: {p}  Ws={tuple(Ws.shape)}")
                return eids, Ws
            else:
                log("[cache] meta mismatch -> rebuilding Ws.")
        else:
            log("[cache] invalid cache -> rebuilding Ws.")

    eids, Ws = build_Ws_square()
    if do_write:
        save_npz(p, {
            "meta": _encode_meta(_meta_dict()),
            "expert_ids": np.array(eids, dtype=np.int32),
            "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        })
        log(f"[cache] wrote: {p}  ({os.path.getsize(p)/1e6:.2f} MB)")
    return eids, Ws

# ----------------------------
# Clustering features + kmeans
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int = 64) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R).unsqueeze(0), (col @ R).unsqueeze(0)], dim=1))
    return torch.cat(feats, dim=0)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int = 60) -> Tuple[torch.Tensor, torch.Tensor]:
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(X.shape[0], generator=g)[:k]
    C = X[idx].clone()
    for _ in range(iters):
        dist = torch.cdist(X, C)
        lab = dist.argmin(dim=1)
        for j in range(k):
            m = (lab == j)
            if m.any():
                C[j] = X[m].mean(dim=0)
    return lab, C

# ----------------------------
# Ortho params + training helpers
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, M_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(M_init.clone().to(DEVICE))

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def svd_init(W_mean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(W_mean, full_matrices=False)
    V = Vh.t().contiguous()
    return U.contiguous(), V.contiguous()

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S = U[:, S]  # (n,s)
    V_S = V[:, S]  # (n,s)
    T = Ws_batch @ V_S
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

# ----------------------------
# Payload building
# ----------------------------
@torch.no_grad()
def greedy_blocks(R: torch.Tensor, t: int, b: int) -> List[Tuple[int,int,torch.Tensor]]:
    n = R.shape[0]
    used = torch.zeros(n, n, dtype=torch.bool, device=R.device)
    out = []
    if t <= 0:
        return out
    for _ in range(t):
        best = None
        bestE = -1.0
        for i0 in range(0, n, b):
            for j0 in range(0, n, b):
                i1 = min(n, i0 + b)
                j1 = min(n, j0 + b)
                if used[i0:i1, j0:j1].any():
                    continue
                Eblk = float((R[i0:i1, j0:j1] ** 2).sum().item())
                if Eblk > bestE:
                    bestE = Eblk
                    best = (i0, j0, i1, j1)
        if best is None:
            break
        i0, j0, i1, j1 = best
        used[i0:i1, j0:j1] = True
        out.append((i0, j0, R[i0:i1, j0:j1].clone()))
    return out

@torch.no_grad()
def _block_energy_grid(X: torch.Tensor, b: int) -> torch.Tensor:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        pad = nb * b - n
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
        n = nb*b
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()  # (nb,nb,b,b)
    E = (Xb * Xb).sum(dim=(2,3))  # (nb,nb)
    return E

@torch.no_grad()
def _pick_top_blocks_from_energy(Eg: torch.Tensor, tot_energy: float, b: int, target: float, max_blocks: int) -> Tuple[List[Tuple[int,int,int,int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())

    pick = order[:K].tolist()
    kept = float(flat[order[:K]].sum().item())

    blocks = []
    for idx in pick:
        bi = idx // nb
        bj = idx % nb
        i0 = bi * b
        j0 = bj * b
        blocks.append((i0, j0, b, b))
    return blocks, kept / max(tot_energy, 1e-12)

@torch.no_grad()
def choose_core_blocks(
    X_list: List[torch.Tensor],
    mode: str,
    agg: str,
    block: int,
    target: float,
    max_blocks: int
):
    n = X_list[0].shape[0]
    b = block
    nb = (n + b - 1) // b

    if mode == "none":
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}

    if mode == "blockdiag":
        # shared diag blocks by mean energy
        diagE = torch.zeros(nb, dtype=DTYPE_ACC, device=X_list[0].device)
        tot = 0.0
        for X in X_list:
            tot += float((X*X).sum().item())
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            diagE += torch.diagonal(Eg, 0)
        tot /= max(1, len(X_list))
        diagE /= max(1, len(X_list))

        order = torch.argsort(diagE, descending=True)
        csum = torch.cumsum(diagE[order], dim=0)
        frac = csum / max(tot, 1e-12)
        need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else nb)
        K = min(need, max_blocks, nb)
        keep = order[:K].tolist()

        blocks = []
        for bi in keep:
            i0 = bi * b
            blocks.append((i0, i0, b, b))
        return {"mode": "blockdiag", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [float(frac[K-1].item()) if K>0 else 0.0]}

    if mode == "blocktopk":
        # shared top blocks using mean or max aggregation over experts
        Eg_all = []
        tots = []
        for X in X_list:
            tots.append(float((X*X).sum().item()))
            Eg_all.append(_block_energy_grid(X, b).to(DTYPE_ACC))
        tot = float(np.mean(tots))

        if agg == "max":
            Eg = torch.stack(Eg_all, dim=0).amax(dim=0)
        else:
            Eg = torch.stack(Eg_all, dim=0).mean(dim=0)

        blocks, ef = _pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
        return {"mode": "blocktopk", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    if mode == "blocktopk_perexpert":
        # NEW: each expert picks its own blocks to reach target energy
        blocks_per = []
        efracs = []
        for X in X_list:
            tot = float((X*X).sum().item())
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            blocks, ef = _pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
            blocks_per.append(blocks)
            efracs.append(ef)
        return {"mode": "blocktopk_perexpert", "blocks_shared": [], "blocks_per_expert": blocks_per, "energy_fracs": efracs}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

@torch.no_grad()
def build_payload_for_cluster(Ws: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor):
    X_list = [(U.t() @ Ws[e] @ V).contiguous() for e in idx]

    core = choose_core_blocks(
        X_list,
        cfg.CORE_MODE,
        cfg.CORE_AGG,
        cfg.CORE_BLOCK,
        cfg.CORE_TARGET,
        cfg.CORE_MAX_BLOCKS
    )

    # per-expert core blocks (always materialize actual tensors here)
    core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
    if core["blocks_per_expert"] is not None:
        for X, blocks in zip(X_list, core["blocks_per_expert"]):
            lst = []
            for (i0, j0, h, w) in blocks:
                h = min(h, X.shape[0] - i0)
                w = min(w, X.shape[1] - j0)
                lst.append((i0, j0, X[i0:i0+h, j0:j0+w].clone()))
            core_blocks.append(lst)
    else:
        blocks = core["blocks_shared"]
        for X in X_list:
            lst = []
            for (i0, j0, h, w) in blocks:
                h = min(h, X.shape[0] - i0)
                w = min(w, X.shape[1] - j0)
                lst.append((i0, j0, X[i0:i0+h, j0:j0+w].clone()))
            core_blocks.append(lst)

    # residuals
    R_list = []
    for X, cb in zip(X_list, core_blocks):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    # shared low-rank basis from mean residual
    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    U_s, _, Vh_s = torch.linalg.svd(Rmean, full_matrices=False)
    r = min(cfg.RES_RANK, U_s.shape[1])
    DL = U_s[:, :r].contiguous()
    DR = Vh_s.t()[:, :r].contiguous()

    gam_list = []
    blocks_list = []
    for Rm in R_list:
        g = torch.sum(DL * (Rm @ DR), dim=0)  # (r,)
        gam_list.append(g.contiguous())
        R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        blks = greedy_blocks(R2, cfg.RES_BLOCKS, cfg.RES_BSIZE)
        blocks_list.append(blks)

    gam = torch.stack(gam_list, dim=0)

    return {
        "U": U, "V": V,
        "core": core,
        "core_blocks": core_blocks,
        "DL": DL, "DR": DR,
        "gam": gam,
        "blocks": blocks_list,
        "idx": idx,
    }

class KTXRuntime(nn.Module):
    def __init__(self, payloads: List[Optional[dict]], basis_of_e: Dict[int, Tuple[int,int]]):
        super().__init__()
        self.payloads = payloads
        self.map = basis_of_e
        self.n = None
        for p in payloads:
            if p is not None:
                self.n = int(p["U"].shape[0])
                break
        if self.n is None:
            raise RuntimeError("No payloads provided.")

    @torch.no_grad()
    def forward(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        used: Dict[int, List[Tuple[float,int]]] = {}
        for a, e in zip(gates.tolist(), routed):
            if e in self.map:
                m, j = self.map[e]
                used.setdefault(m, []).append((a, j))

        for m, items in used.items():
            P = self.payloads[m]
            if P is None:
                continue
            U = P["U"]; V = P["V"]
            DL = P["DL"]; DR = P["DR"]
            z = x @ U
            u_acc = torch.zeros_like(z)

            for (a, j) in items:
                u = torch.zeros_like(z)

                # core blocks (may be per-expert)
                for (i0, j0, Bc) in P["core_blocks"][j]:
                    h, w = Bc.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bc

                # low-rank residual
                g = P["gam"][j]
                u += ((z @ DL) * g.view(1, -1)) @ DR.t()

                # greedy residual blocks
                for (i0, j0, Bb) in P["blocks"][j]:
                    h, w = Bb.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb

                u_acc += float(a) * u

            y += u_acc @ V.t()
        return y

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def dense_apply(Ws: torch.Tensor, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
    n = Ws.shape[-1]
    Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=x.device)
    for a, e in zip(gates, routed):
        Wsum += float(a.item()) * Ws[e]
    return x @ Wsum

@torch.no_grad()
def eval_runtime(rt: KTXRuntime, Ws: torch.Tensor, trials: int, routed_k: int, batch: int):
    E = Ws.shape[0]
    errs = []
    for _ in range(trials):
        x = torch.randn(batch, rt.n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(routed_k, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)
        y_hat = rt(x, routed, gates)
        y_ref = dense_apply(Ws, x, routed, gates)
        err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)
    log(f"[eval] forward rel-error = {float(np.mean(errs)):.4f} ± {float(np.std(errs)):.4f}  (trials={trials})")

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X Optimized (offline) v4 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  (RIDGE_DAMP={cfg.RIDGE_DAMP})")
    log(f"CORE_MODE:   {cfg.CORE_MODE}  agg={cfg.CORE_AGG} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK}  blocks={cfg.RES_BLOCKS}  bsize={cfg.RES_BSIZE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} subm={cfg.SUBM} batchE={cfg.BATCH_E} lr={cfg.LR_UV} min_cluster={cfg.TRAIN_MIN_CLUSTER}")
    log(f"Hadamard:    {cfg.USE_HADAMARD} seed={cfg.HAD_SEED}")
    log(f"Threads:     {NTHREADS}  Torch={torch.__version__} device={DEVICE}")
    log(f"Cache:       {cfg.CACHE_MODE}  path={cache_path()}  force_rebuild={cfg.FORCE_REBUILD_WS}")
    log("")

def main():
    banner()

    eids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} experts={eids[:8]}{'...' if len(eids)>8 else ''}")

    # clusters
    if cfg.M_CLUSTERS > 0:
        M = min(cfg.M_CLUSTERS, E)
    else:
        M = int(round(2.0 * math.sqrt(E)))
        M = max(6, min(M, E))
    log(f"[cluster] M={M}")

    Xfeat = random_proj_features(Ws, d=64)
    labels, _ = kmeans_torch(Xfeat, k=M, iters=60)
    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    log(f"[cluster] sizes: {[len(c) for c in clusters]}")

    # init U,V
    U_par: List[Optional[OrthoParam]] = []
    V_par: List[Optional[OrthoParam]] = []
    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            U_par.append(None); V_par.append(None)
            continue
        Wm = Ws[idx].mean(dim=0)
        U0, V0 = svd_init(Wm)
        U_par.append(OrthoParam(U0))
        V_par.append(OrthoParam(V0))

    # train
    if cfg.TRAIN_STEPS > 0:
        params = [uv.M for uv in (U_par + V_par) if uv is not None]
        opt = torch.optim.Adam(params, lr=cfg.LR_UV)
        t0 = time.perf_counter()

        with torch.enable_grad():
            for step in range(1, cfg.TRAIN_STEPS + 1):
                S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]
                L_total = None
                n_terms = 0

                for m, idx in enumerate(clusters):
                    if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                        continue
                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()

                    if cfg.BATCH_E > 0 and cfg.BATCH_E < len(idx):
                        pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                        idx_step = [idx[p] for p in pick]
                    else:
                        idx_step = idx

                    Ws_batch = Ws[idx_step]
                    Xs = slice_X_batch(Ws_batch, Uo, Vo, S)

                    off = offdiag_abs_mean(Xs)
                    diag = diag_abs_mean(Xs)
                    loss = torch.log(off + 1e-6) - torch.log(diag + 1e-6)

                    L_total = loss if (L_total is None) else (L_total + loss)
                    n_terms += 1

                if L_total is None:
                    log("[train] skipped (no clusters >= TRAIN_MIN_CLUSTER)")
                    break

                L_total = L_total / max(1, n_terms)
                if not L_total.requires_grad:
                    raise RuntimeError("Training graph broken: loss has no grad_fn.")

                opt.zero_grad(set_to_none=True)
                L_total.backward()
                if cfg.GRAD_CLIP > 0:
                    torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                opt.step()

                if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    with torch.no_grad():
                        for uv in (U_par + V_par):
                            if uv is not None:
                                uv.M.copy_(uv.orthogonal())

                if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    t1 = time.perf_counter()
                    log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} | loss={float(L_total.item()):.4f}  (+{t1-t0:.1f}s)")
                    t0 = t1

    # build payloads
    payloads: List[Optional[dict]] = []
    basis_of_e: Dict[int, Tuple[int,int]] = {}
    log("[build] payloads ...")
    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            payloads.append(None)
            continue
        Uo = U_par[m].orthogonal().detach()
        Vo = V_par[m].orthogonal().detach()
        P = build_payload_for_cluster(Ws, idx, Uo, Vo)
        payloads.append(P)

        for j, e in enumerate(idx):
            basis_of_e[e] = (m, j)

        efr = P["core"]["energy_fracs"]
        if len(efr) == 0:
            ef_mean = ef_min = 0.0
        else:
            ef_mean = float(np.mean(efr))
            ef_min = float(np.min(efr))
        nb_mean = float(np.mean([len(x) for x in P["core_blocks"]])) if len(P["core_blocks"]) else 0.0

        log(f"  - basis {m}: E={len(idx)} core={P['core']['mode']} "
            f"core_energy(mean/min)≈{ef_mean:.3f}/{ef_min:.3f} "
            f"core_blocks(mean)≈{nb_mean:.1f} rank={P['DL'].shape[1]} res_blocks={cfg.RES_BLOCKS}")

    rt = KTXRuntime(payloads, basis_of_e)
    eval_runtime(rt, Ws, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    log("\n✅ Done.")
    log("Next levers for DeepSeek:")
    log(" - CORE_TARGET up (0.75→0.85) and/or CORE_MAX_BLOCKS up (256→384).")
    log(" - M_CLUSTERS up (12/16) if one basis dominates routing.")
    log(" - Real CALIB_PATH + ROUTER_PATH (RIDGE_WEIGHTED=1) is the real unlock for ridge.")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X Optimized (offline) v4 ==
Time:        2026-01-01 16:53:09
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES=512
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
LIN_MODE:    ridge  (RIDGE_DAMP=0.001)
CORE_MODE:   blocktopk_perexpert  agg=mean block=64 target=0.75 max_blocks=256
RESIDUAL:    rank=1024  blocks=128  bsize=64
TRAIN:       steps=24 subm=256 batchE=4 lr=0.05 min_cluster=2
Hadamard:    False seed=1234
Threads:     8  Torch=2.4.1+cpu device=cpu
Cache:       readwrite  path=/home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v4.npz  force_rebuild=False

[found] layer 1 experts: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]  (using 16)
[load] reading tensors

Build Ws:   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote: /home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v4.npz  (268.44 MB)
[Ws] shape=(16, 2048, 2048) experts=[0, 1, 2, 3, 4, 5, 6, 7]...
[cluster] M=8
[cluster] sizes: [2, 1, 1, 1, 1, 1, 7, 2]
[train] step   1/24 | loss=-3.7251  (+2.7s)
[train] step   4/24 | loss=-1.1787  (+11.3s)
[train] step   8/24 | loss=-1.3783  (+14.0s)
[train] step  12/24 | loss=-1.5551  (+13.8s)
[train] step  16/24 | loss=-1.7924  (+13.7s)
[train] step  20/24 | loss=-1.6597  (+13.6s)
[train] step  24/24 | loss=-1.7647  (+13.7s)
[build] payloads ...
  - basis 0: E=2 core=blocktopk_perexpert core_energy(mean/min)≈0.583/0.583 core_blocks(mean)≈256.0 rank=1024 res_blocks=128
  - basis 1: E=1 core=blocktopk_perexpert core_energy(mean/min)≈0.837/0.837 core_blocks(mean)≈1.0 rank=1024 res_blocks=128
  - basis 2: E=1 core=blocktopk_perexpert core_energy(mean/min)≈0.814/0.814 core_blocks(mean)≈3.0 rank=1024 res_blocks=128
  - basis 3: E=1 core=blocktopk_perexpert core_energy(mean/min)≈0.820/0.820

In [17]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X Optimized (offline) — "everything included" v5
#
# Offline:
#   - Loads DeepSeek MoE expert tensors from local safetensors shards via model.safetensors.index.json
#   - Builds square expert mats Ws[e] (H x H) using:
#       LIN_MODE = ridge        (best proxy if CALIB_PATH is real)
#                = weff_gate    (fast proxy using gate mean)
#                = weff         (baseline W_down @ W_up)
#   - Optional ROUTER_PATH weighting for ridge (RIDGE_WEIGHTED=1)
#
# KT++ payload:
#   - Clusters experts -> per-cluster orthogonal bases U,V
#   - Trains U,V (autograd) to improve compressibility
#     + optional BLOCK-SPARSITY penalty aligned with block-core payload
#   - Core options:
#       CORE_MODE = blocktopk_perexpert  (recommended for DeepSeek)
#                = blocktopk            (shared blocks per cluster)
#                = blockdiag            (diag-only blocks)
#                = none
#   - Residual options:
#       shared low-rank (rank RES_RANK) + greedy sparse blocks (RES_BLOCKS, RES_BSIZE)
#
# Terminal-first:
#   - Prints everything, produces one eval number (random routed mixture)
#
# Dependencies:
#   - torch, safetensors, numpy (tqdm optional)
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise RuntimeError("This script requires PyTorch.") from e

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("This script requires safetensors (pip install safetensors).") from e

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

# ----------------------------
# Threading / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))

DTYPE_W   = torch.float16
DTYPE_ACC = torch.float32

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    # Model slice
    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "16"))

    # Calibration: npz with key 'X' shape (N,H)
    CALIB_PATH: str = os.environ.get("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "512"))

    # Router: npz with key 'P' shape (N, E_total) or (N, E_used)
    ROUTER_PATH: str = os.environ.get("ROUTER_PATH", "").strip()
    ROUTER_EIDS_ARE_GLOBAL: bool = os.environ.get("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    # Build square mats
    LIN_MODE: str = os.environ.get("LIN_MODE", "ridge").strip().lower()  # ridge | weff_gate | weff
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))      # relative to trace(XtX)/H
    RIDGE_WEIGHTED: bool = os.environ.get("RIDGE_WEIGHTED", "0") == "1"  # weight by router P

    # Optional similarity rotation (applies after Ws build)
    USE_HADAMARD: bool = os.environ.get("USE_HADAMARD", "0") == "1"
    HAD_SEED: int = int(os.environ.get("HAD_SEED", "1234"))

    # Cache
    CACHE_MODE: str = os.environ.get("CACHE_MODE", "readwrite").strip().lower()  # off|read|write|readwrite
    CACHE_NAME: str = os.environ.get("CACHE_NAME", "").strip()
    FORCE_REBUILD_WS: bool = os.environ.get("FORCE_REBUILD_WS", "0") == "1"

    # Clustering
    M_CLUSTERS: int = int(os.environ.get("M_CLUSTERS", "0"))  # 0 => auto
    CLUSTER_ITERS: int = int(os.environ.get("CLUSTER_ITERS", "60"))
    CLUSTER_FEAT_D: int = int(os.environ.get("CLUSTER_FEAT_D", "64"))
    CLUSTER_MIN_SIZE: int = int(os.environ.get("CLUSTER_MIN_SIZE", "1"))  # reassign small clusters to large ones if >1

    # Training (U,V)
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "24"))
    SUBM: int = int(os.environ.get("SUBM", "256"))
    BATCH_E: int = int(os.environ.get("BATCH_E", "4"))
    LR_UV: float = float(os.environ.get("LR_UV", "5e-2"))
    REORTHO_EVERY: int = int(os.environ.get("REORTHO_EVERY", "4"))
    REPORT_EVERY: int = int(os.environ.get("REPORT_EVERY", "4"))
    TRAIN_MIN_CLUSTER: int = int(os.environ.get("TRAIN_MIN_CLUSTER", "2"))
    GRAD_CLIP: float = float(os.environ.get("GRAD_CLIP", "1.0"))

    # Training objective knobs
    TRAIN_OBJ: str = os.environ.get("TRAIN_OBJ", "logratio").strip().lower()  # logratio|ratio
    TRAIN_LAM_BLOCK: float = float(os.environ.get("TRAIN_LAM_BLOCK", "0.2")) # block-sparsity penalty strength
    TRAIN_BLOCK_USE_CORE_BLOCK: bool = os.environ.get("TRAIN_BLOCK_USE_CORE_BLOCK", "1") == "1"

    # Core model
    CORE_MODE: str = os.environ.get("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_AGG: str = os.environ.get("CORE_AGG", "mean").strip().lower()  # mean|max for shared blocktopk
    CORE_BLOCK: int = int(os.environ.get("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(os.environ.get("CORE_TARGET", "0.75"))
    CORE_MAX_BLOCKS: int = int(os.environ.get("CORE_MAX_BLOCKS", "256"))

    # Residual
    RES_RANK: int = int(os.environ.get("RES_RANK", "1024"))
    RES_BLOCKS: int = int(os.environ.get("RES_BLOCKS", "128"))
    RES_BSIZE: int = int(os.environ.get("RES_BSIZE", "64"))

    # Eval
    EVAL_TRIALS: int = int(os.environ.get("EVAL_TRIALS", "5"))
    ROUTED_K: int = int(os.environ.get("ROUTED_K", "8"))
    EVAL_BATCH: int = int(os.environ.get("EVAL_BATCH", "2"))

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def cache_path():
    if cfg.CACHE_NAME:
        return os.path.join(cfg.OUTPUT_DIR, cfg.CACHE_NAME)
    tag = f"deepseek_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_{cfg.LIN_MODE}_Ws_cache_v5.npz"
    return os.path.join(cfg.OUTPUT_DIR, tag)

def _meta_dict():
    return dict(
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        max_experts=cfg.MAX_EXPERTS,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES,
        router_path=cfg.ROUTER_PATH or "",
        router_global=cfg.ROUTER_EIDS_ARE_GLOBAL,
        hadamard=cfg.USE_HADAMARD,
        had_seed=cfg.HAD_SEED,
    )

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    # DeepSeek typically: up_proj, gate_proj, down_proj
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    out = {}
    if up: out["up"] = up
    if gate: out["gate"] = gate
    if down: out["down"] = down
    return out

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration / router
# ----------------------------
def load_calib_X(H: int) -> torch.Tensor:
    if not cfg.CALIB_PATH:
        X = torch.randn(cfg.CALIB_SAMPLES, H, dtype=DTYPE_W, device=DEVICE)
        log(f"[calib] RANDOM X: {tuple(X.shape)}  (set CALIB_PATH npz key 'X' for realistic ridge)")
        return X
    if not os.path.isfile(cfg.CALIB_PATH):
        raise FileNotFoundError(f"CALIB_PATH not found: {cfg.CALIB_PATH}")
    z = np.load(cfg.CALIB_PATH, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Has keys: {list(z.files)}")
    X = torch.from_numpy(z["X"]).to(DTYPE_W).to(DEVICE)
    z.close()
    if X.ndim != 2 or X.shape[0] < 2:
        raise RuntimeError(f"Bad X shape: {tuple(X.shape)}")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    log(f"[calib] Loaded X: {tuple(X.shape)}")
    return X

def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    if not os.path.isfile(cfg.ROUTER_PATH):
        raise FileNotFoundError(f"ROUTER_PATH not found: {cfg.ROUTER_PATH}")
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Has keys: {list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    log(f"[router] Loaded P: {tuple(P.shape)}")
    return P

# ----------------------------
# Hadamard rotation (optional)
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

@torch.no_grad()
def hadamard_orth(n: int, seed: int) -> torch.Tensor:
    if not is_power_of_two(n):
        raise ValueError(f"Hadamard requires power-of-two n, got {n}")
    H = torch.tensor([[1.0]], dtype=DTYPE_ACC, device=DEVICE)
    while H.shape[0] < n:
        H = torch.cat([torch.cat([H,  H], dim=1),
                       torch.cat([H, -H], dim=1)], dim=0)
    H = H / math.sqrt(n)
    g = torch.Generator(device="cpu").manual_seed(seed)
    signs = (torch.randint(0, 2, (n,), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    return (H * signs.view(1, n)).contiguous()

# ----------------------------
# Build expert square Ws
# ----------------------------
@torch.no_grad()
def forward_mlp_no_grad(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

@torch.no_grad()
def build_Ws_square() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer {cfg.LAYER} experts: {all_eids}  (using {len(eids)})")

    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if "up" not in kk or "down" not in kk or "gate" not in kk:
            raise RuntimeError(f"Expert {eid} missing keys: {kk}")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]].to(DTYPE_W)
    dff, H = W_up0.shape
    log(f"[shape] H={H} d_ff={dff}")

    X = load_calib_X(H)
    P = load_router_P()

    R = hadamard_orth(H, cfg.HAD_SEED) if cfg.USE_HADAMARD else None

    Xf = X.to(DTYPE_ACC)
    XtX = (Xf.t() @ Xf)
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)

    cholG = None
    if cfg.LIN_MODE == "ridge" and not (cfg.RIDGE_WEIGHTED and (P is not None)):
        G = XtX + lam * I
        cholG = torch.linalg.cholesky(G)

    Ws = []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws")):
        W_up   = T[per_e[eid]["up"]].to(DTYPE_W).to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DTYPE_W).to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DTYPE_W).to(DEVICE)

        if cfg.LIN_MODE == "weff":
            W = (W_down.to(DTYPE_ACC) @ W_up.to(DTYPE_ACC))

        elif cfg.LIN_MODE == "weff_gate":
            gate_act = (Xf @ W_gate.to(DTYPE_ACC).t())
            m = F.silu(gate_act).mean(dim=0)  # (dff,)
            W = (W_down.to(DTYPE_ACC) * m.view(1, -1)) @ W_up.to(DTYPE_ACC)

        elif cfg.LIN_MODE == "ridge":
            Y = forward_mlp_no_grad(X, W_gate, W_up, W_down).to(DTYPE_ACC)  # (N,H)

            if cfg.RIDGE_WEIGHTED and (P is not None):
                # weights per sample
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                else:
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw
                XtX_e = Xw.t() @ Xw
                lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
                G_e = XtX_e + lam_e * I
                chol = torch.linalg.cholesky(G_e)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)  # (H,H)
                W = Wt.t().contiguous()
            else:
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)  # (H,H)
                W = Wt.t().contiguous()
        else:
            raise ValueError("LIN_MODE must be ridge|weff_gate|weff")

        if R is not None:
            W = (R.t() @ W @ R).contiguous()

        fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
        Ws.append((W / fn).contiguous())

    Ws = torch.stack(Ws, dim=0).to(DTYPE_ACC)  # (E,H,H)
    return eids, Ws

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    p = cache_path()
    do_read = cfg.CACHE_MODE in ("read", "readwrite")
    do_write = cfg.CACHE_MODE in ("write", "readwrite")

    if do_read and (not cfg.FORCE_REBUILD_WS) and os.path.isfile(p):
        z = try_load_npz(p)
        if z and ("expert_ids" in z) and ("Ws" in z):
            ok = True
            if "meta" in z:
                meta = _decode_meta(z["meta"])
                ok = (meta == _meta_dict())
            if ok:
                eids = [int(x) for x in z["expert_ids"].tolist()]
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws: {p}  Ws={tuple(Ws.shape)}")
                return eids, Ws
            log("[cache] meta mismatch -> rebuilding Ws.")
        else:
            log("[cache] invalid cache -> rebuilding Ws.")

    eids, Ws = build_Ws_square()
    if do_write:
        save_npz(p, {
            "meta": _encode_meta(_meta_dict()),
            "expert_ids": np.array(eids, dtype=np.int32),
            "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        })
        log(f"[cache] wrote: {p}  ({os.path.getsize(p)/1e6:.2f} MB)")
    return eids, Ws

# ----------------------------
# Clustering
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R).unsqueeze(0), (col @ R).unsqueeze(0)], dim=1))
    return torch.cat(feats, dim=0)  # (E, 2d)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int) -> Tuple[torch.Tensor, torch.Tensor]:
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(X.shape[0], generator=g)[:k]
    C = X[idx].clone()
    for _ in range(iters):
        dist = torch.cdist(X, C)
        lab = dist.argmin(dim=1)
        for j in range(k):
            m = (lab == j)
            if m.any():
                C[j] = X[m].mean(dim=0)
    return lab, C

@torch.no_grad()
def enforce_min_cluster_size(labels: torch.Tensor, centers: torch.Tensor, X: torch.Tensor, min_size: int) -> torch.Tensor:
    # Reassign points from clusters smaller than min_size into nearest "big" center.
    if min_size <= 1:
        return labels
    k = centers.shape[0]
    counts = torch.bincount(labels, minlength=k)
    big = (counts >= min_size).nonzero(as_tuple=False).flatten()
    if big.numel() == 0:
        return labels  # can't fix
    small = (counts < min_size).nonzero(as_tuple=False).flatten()
    if small.numel() == 0:
        return labels

    big_centers = centers[big]
    for c in small.tolist():
        idxs = (labels == c).nonzero(as_tuple=False).flatten()
        if idxs.numel() == 0:
            continue
        d = torch.cdist(X[idxs], big_centers)  # (m, |big|)
        nn = d.argmin(dim=1)
        labels[idxs] = big[nn]
    return labels

# ----------------------------
# Ortho params + training
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, M_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(M_init.clone().to(DEVICE))

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def svd_init(W_mean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(W_mean, full_matrices=False)
    V = Vh.t().contiguous()
    return U.contiguous(), V.contiguous()

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    # Ws_batch: (Eb,n,n), U/V: (n,n), S: (s,)
    U_S = U[:, S]                # (n,s)
    V_S = V[:, S]                # (n,s)
    T = Ws_batch @ V_S           # (Eb,n,s)
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)  # (1,s,n) @ (Eb,n,s) -> (Eb,s,s)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    """
    Align training with block-core compressibility:
      - compute block energy on the SUBM x SUBM slice
      - group-lasso style penalty => prefer energy concentrated in fewer blocks
    """
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    nb = s // b
    if nb <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()  # (Eb,s2,s2)
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()  # (Eb,nb,nb,b,b)
    Eblk = (Xb * Xb).sum(dim=(3,4))  # (Eb,nb,nb)
    P = Eblk.mean(dim=0)             # (nb,nb)
    # normalized group-lasso: sum sqrt(P) / sum(P)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

# ----------------------------
# Core selection helpers
# ----------------------------
@torch.no_grad()
def _block_energy_grid(X: torch.Tensor, b: int) -> torch.Tensor:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        pad = nb * b - n
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
        n = nb*b
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()  # (nb,nb,b,b)
    E = (Xb * Xb).sum(dim=(2,3))  # (nb,nb)
    return E

@torch.no_grad()
def _pick_top_blocks_from_energy(Eg: torch.Tensor, tot_energy: float, b: int, target: float, max_blocks: int) -> Tuple[List[Tuple[int,int,int,int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    kept = float(flat[order[:K]].sum().item())
    blocks = []
    for idx in pick:
        bi = idx // nb
        bj = idx % nb
        blocks.append((bi*b, bj*b, b, b))
    return blocks, kept / max(tot_energy, 1e-12)

@torch.no_grad()
def choose_core_blocks(
    X_list: List[torch.Tensor],
    mode: str,
    agg: str,
    block: int,
    target: float,
    max_blocks: int
):
    """
    Returns:
      {
        mode,
        blocks_shared: [(i0,j0,h,w), ...]    # if shared
        blocks_per_expert: [list_of_blocks]  # if perexpert
        energy_fracs: [float per expert] or [float] for shared
      }
    """
    if mode == "none":
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}

    n = X_list[0].shape[0]
    b = int(block)
    if b <= 0:
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}

    if mode == "blockdiag":
        nb = (n + b - 1) // b
        diagE = torch.zeros(nb, dtype=DTYPE_ACC, device=X_list[0].device)
        tots = []
        for X in X_list:
            tots.append(float((X*X).sum().item()))
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            diagE += torch.diagonal(Eg, 0)
        tot = float(np.mean(tots))
        diagE /= max(1, len(X_list))

        order = torch.argsort(diagE, descending=True)
        csum = torch.cumsum(diagE[order], dim=0)
        frac = csum / max(tot, 1e-12)
        need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else nb)
        K = min(need, max_blocks, nb)

        blocks = []
        for bi in order[:K].tolist():
            i0 = bi * b
            blocks.append((i0, i0, b, b))
        ef = float(frac[K-1].item()) if K > 0 else 0.0
        return {"mode": "blockdiag", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    if mode == "blocktopk":
        Eg_all = []
        tots = []
        for X in X_list:
            tots.append(float((X*X).sum().item()))
            Eg_all.append(_block_energy_grid(X, b).to(DTYPE_ACC))
        tot = float(np.mean(tots))
        if agg == "max":
            Eg = torch.stack(Eg_all, dim=0).amax(dim=0)
        else:
            Eg = torch.stack(Eg_all, dim=0).mean(dim=0)
        blocks, ef = _pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
        return {"mode": "blocktopk", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    if mode == "blocktopk_perexpert":
        blocks_per = []
        efracs = []
        for X in X_list:
            tot = float((X*X).sum().item())
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            blocks, ef = _pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
            blocks_per.append(blocks)
            efracs.append(ef)
        return {"mode": "blocktopk_perexpert", "blocks_shared": [], "blocks_per_expert": blocks_per, "energy_fracs": efracs}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

# ----------------------------
# Residual: greedy blocks
# ----------------------------
@torch.no_grad()
def greedy_blocks(R: torch.Tensor, t: int, b: int) -> List[Tuple[int,int,torch.Tensor]]:
    n = R.shape[0]
    used = torch.zeros(n, n, dtype=torch.bool, device=R.device)
    out = []
    if t <= 0:
        return out
    b = int(b)
    for _ in range(t):
        best = None
        bestE = -1.0
        for i0 in range(0, n, b):
            for j0 in range(0, n, b):
                i1 = min(n, i0 + b)
                j1 = min(n, j0 + b)
                if used[i0:i1, j0:j1].any():
                    continue
                Eblk = float((R[i0:i1, j0:j1] ** 2).sum().item())
                if Eblk > bestE:
                    bestE = Eblk
                    best = (i0, j0, i1, j1)
        if best is None:
            break
        i0, j0, i1, j1 = best
        used[i0:i1, j0:j1] = True
        out.append((i0, j0, R[i0:i1, j0:j1].clone()))
    return out

# ----------------------------
# Payload building per cluster
# ----------------------------
@torch.no_grad()
def build_payload_for_cluster(Ws: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor):
    # X = U^T W V
    X_list = [(U.t() @ Ws[e] @ V).contiguous() for e in idx]

    core = choose_core_blocks(
        X_list,
        cfg.CORE_MODE,
        cfg.CORE_AGG,
        cfg.CORE_BLOCK,
        cfg.CORE_TARGET,
        cfg.CORE_MAX_BLOCKS
    )

    # Materialize per-expert block tensors
    core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
    if core["blocks_per_expert"] is not None:
        for X, blocks in zip(X_list, core["blocks_per_expert"]):
            lst = []
            for (i0, j0, h, w) in blocks:
                h = min(h, X.shape[0] - i0)
                w = min(w, X.shape[1] - j0)
                if h > 0 and w > 0:
                    lst.append((i0, j0, X[i0:i0+h, j0:j0+w].clone()))
            core_blocks.append(lst)
    else:
        blocks = core["blocks_shared"]
        for X in X_list:
            lst = []
            for (i0, j0, h, w) in blocks:
                h = min(h, X.shape[0] - i0)
                w = min(w, X.shape[1] - j0)
                if h > 0 and w > 0:
                    lst.append((i0, j0, X[i0:i0+h, j0:j0+w].clone()))
            core_blocks.append(lst)

    # Residuals after core
    R_list = []
    for X, cb in zip(X_list, core_blocks):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    # Shared low-rank basis from mean residual
    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    U_s, _, Vh_s = torch.linalg.svd(Rmean, full_matrices=False)
    r = min(cfg.RES_RANK, U_s.shape[1])
    DL = U_s[:, :r].contiguous()
    DR = Vh_s.t()[:, :r].contiguous()

    gam_list = []
    blocks_list = []
    for Rm in R_list:
        g = torch.sum(DL * (Rm @ DR), dim=0)  # (r,)
        gam_list.append(g.contiguous())
        R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        blks = greedy_blocks(R2, cfg.RES_BLOCKS, cfg.RES_BSIZE)
        blocks_list.append(blks)

    gam = torch.stack(gam_list, dim=0)

    return {
        "U": U, "V": V,
        "core": core,
        "core_blocks": core_blocks,
        "DL": DL, "DR": DR,
        "gam": gam,
        "blocks": blocks_list,
        "idx": idx,
    }

class KTXRuntime(nn.Module):
    def __init__(self, payloads: List[Optional[dict]], basis_of_e: Dict[int, Tuple[int,int]]):
        super().__init__()
        self.payloads = payloads
        self.map = basis_of_e
        self.n = None
        for p in payloads:
            if p is not None:
                self.n = int(p["U"].shape[0])
                break
        if self.n is None:
            raise RuntimeError("No payloads provided.")

    @torch.no_grad()
    def forward(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        used: Dict[int, List[Tuple[float,int]]] = {}
        for a, e in zip(gates.tolist(), routed):
            if e in self.map:
                m, j = self.map[e]
                used.setdefault(m, []).append((a, j))

        for m, items in used.items():
            P = self.payloads[m]
            if P is None:
                continue
            U = P["U"]; V = P["V"]
            DL = P["DL"]; DR = P["DR"]

            z = x @ U
            u_acc = torch.zeros_like(z)

            for (a, j) in items:
                u = torch.zeros_like(z)

                # core blocks
                for (i0, j0, Bc) in P["core_blocks"][j]:
                    h, w = Bc.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bc

                # low-rank residual
                g = P["gam"][j]
                u += ((z @ DL) * g.view(1, -1)) @ DR.t()

                # greedy residual blocks
                for (i0, j0, Bb) in P["blocks"][j]:
                    h, w = Bb.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb

                u_acc += float(a) * u

            y += u_acc @ V.t()

        return y

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def dense_apply(Ws: torch.Tensor, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
    n = Ws.shape[-1]
    Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=x.device)
    for a, e in zip(gates, routed):
        Wsum += float(a.item()) * Ws[e]
    return x @ Wsum

@torch.no_grad()
def eval_runtime(rt: KTXRuntime, Ws: torch.Tensor, trials: int, routed_k: int, batch: int):
    E = Ws.shape[0]
    errs = []
    for _ in range(trials):
        x = torch.randn(batch, rt.n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(routed_k, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)

        y_hat = rt(x, routed, gates)
        y_ref = dense_apply(Ws, x, routed, gates)

        err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)

    log(f"[eval] forward rel-error = {float(np.mean(errs)):.4f} ± {float(np.std(errs)):.4f}  (trials={trials})")

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X Optimized (offline) v5 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  (RIDGE_DAMP={cfg.RIDGE_DAMP})")
    log(f"CORE_MODE:   {cfg.CORE_MODE}  agg={cfg.CORE_AGG} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK}  blocks={cfg.RES_BLOCKS}  bsize={cfg.RES_BSIZE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} subm={cfg.SUBM} batchE={cfg.BATCH_E} lr={cfg.LR_UV} min_cluster={cfg.TRAIN_MIN_CLUSTER}")
    log(f"TRAIN_OBJ:   {cfg.TRAIN_OBJ}  lam_block={cfg.TRAIN_LAM_BLOCK}  block_penalty_uses_core_block={cfg.TRAIN_BLOCK_USE_CORE_BLOCK}")
    log(f"CLUSTER:     M={cfg.M_CLUSTERS or '(auto)'} iters={cfg.CLUSTER_ITERS} feat_d={cfg.CLUSTER_FEAT_D} min_size={cfg.CLUSTER_MIN_SIZE}")
    log(f"Hadamard:    {cfg.USE_HADAMARD} seed={cfg.HAD_SEED}")
    log(f"Threads:     {NTHREADS}  Torch={torch.__version__} device={DEVICE}")
    log(f"Cache:       {cfg.CACHE_MODE}  path={cache_path()}  force_rebuild={cfg.FORCE_REBUILD_WS}")
    log("")

def main():
    banner()

    eids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} experts={eids[:8]}{'...' if len(eids)>8 else ''}")

    # Choose cluster count
    if cfg.M_CLUSTERS > 0:
        M = min(cfg.M_CLUSTERS, E)
    else:
        M = int(round(2.0 * math.sqrt(E)))
        M = max(6, min(M, E))
    log(f"[cluster] M={M}")

    # Cluster
    Xfeat = random_proj_features(Ws, d=cfg.CLUSTER_FEAT_D)
    labels, centers = kmeans_torch(Xfeat, k=M, iters=cfg.CLUSTER_ITERS)
    labels = enforce_min_cluster_size(labels, centers, Xfeat, min_size=cfg.CLUSTER_MIN_SIZE)

    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    log(f"[cluster] sizes: {[len(c) for c in clusters]}")

    # Init U,V per cluster
    U_par: List[Optional[OrthoParam]] = []
    V_par: List[Optional[OrthoParam]] = []
    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            U_par.append(None); V_par.append(None)
            continue
        Wm = Ws[idx].mean(dim=0)
        U0, V0 = svd_init(Wm)
        U_par.append(OrthoParam(U0))
        V_par.append(OrthoParam(V0))

    # Train U,V
    if cfg.TRAIN_STEPS > 0:
        params = [uv.M for uv in (U_par + V_par) if uv is not None]
        opt = torch.optim.Adam(params, lr=cfg.LR_UV)
        t0 = time.perf_counter()

        core_block_for_train = cfg.CORE_BLOCK if cfg.TRAIN_BLOCK_USE_CORE_BLOCK else max(16, cfg.SUBM // 4)

        with torch.enable_grad():
            for step in range(1, cfg.TRAIN_STEPS + 1):
                S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]
                L_total = None
                n_terms = 0

                for m, idx in enumerate(clusters):
                    if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                        continue

                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()

                    if cfg.BATCH_E > 0 and cfg.BATCH_E < len(idx):
                        pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                        idx_step = [idx[p] for p in pick]
                    else:
                        idx_step = idx

                    Ws_batch = Ws[idx_step]  # constant
                    Xs = slice_X_batch(Ws_batch, Uo, Vo, S)

                    off = offdiag_abs_mean(Xs)
                    diag = diag_abs_mean(Xs).clamp_min(1e-6)

                    if cfg.TRAIN_OBJ == "ratio":
                        loss = off / diag
                    else:
                        loss = torch.log(off + 1e-6) - torch.log(diag)

                    if cfg.TRAIN_LAM_BLOCK > 0 and cfg.CORE_MODE.startswith("block"):
                        loss = loss + cfg.TRAIN_LAM_BLOCK * block_group_sparsity_penalty(Xs, core_block_for_train)

                    L_total = loss if (L_total is None) else (L_total + loss)
                    n_terms += 1

                if L_total is None:
                    log("[train] skipped (no clusters >= TRAIN_MIN_CLUSTER)")
                    break

                L_total = L_total / max(1, n_terms)
                if not L_total.requires_grad:
                    raise RuntimeError("Training graph broken: loss has no grad_fn.")

                opt.zero_grad(set_to_none=True)
                L_total.backward()
                if cfg.GRAD_CLIP > 0:
                    torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                opt.step()

                if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    with torch.no_grad():
                        for uv in (U_par + V_par):
                            if uv is not None:
                                uv.M.copy_(uv.orthogonal())

                if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    t1 = time.perf_counter()
                    log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} | loss={float(L_total.item()):.4f}  (+{t1-t0:.1f}s)")
                    t0 = t1

    # Build payloads
    payloads: List[Optional[dict]] = []
    basis_of_e: Dict[int, Tuple[int,int]] = {}
    log("[build] payloads ...")

    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            payloads.append(None)
            continue

        Uo = U_par[m].orthogonal().detach()
        Vo = V_par[m].orthogonal().detach()
        P = build_payload_for_cluster(Ws, idx, Uo, Vo)
        payloads.append(P)

        for j, e in enumerate(idx):
            basis_of_e[e] = (m, j)

        efr = P["core"]["energy_fracs"]
        if len(efr) == 0:
            ef_mean = ef_min = 0.0
        else:
            ef_mean = float(np.mean(efr))
            ef_min = float(np.min(efr))
        nb_mean = float(np.mean([len(x) for x in P["core_blocks"]])) if len(P["core_blocks"]) else 0.0

        log(f"  - basis {m}: E={len(idx)} core={P['core']['mode']} "
            f"core_energy(mean/min)≈{ef_mean:.3f}/{ef_min:.3f} "
            f"core_blocks(mean)≈{nb_mean:.1f} rank={P['DL'].shape[1]} res_blocks={cfg.RES_BLOCKS}")

    rt = KTXRuntime(payloads, basis_of_e)
    eval_runtime(rt, Ws, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    log("\n✅ Done.")
    log("Practical next steps:")
    log(" - If any basis is capped at CORE_MAX_BLOCKS with low core_energy: increase M_CLUSTERS and/or CORE_BLOCK.")
    log(" - For real DeepSeek: use CALIB_PATH (true layer inputs) + ROUTER_PATH with RIDGE_WEIGHTED=1 and FORCE_REBUILD_WS=1.")
    log(" - If still capped: CORE_MAX_BLOCKS 256→384 or RES_RANK/RES_BLOCKS up after fixing core/cluster.")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X Optimized (offline) v5 ==
Time:        2026-01-01 17:03:56
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES=512
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
LIN_MODE:    ridge  (RIDGE_DAMP=0.001)
CORE_MODE:   blocktopk_perexpert  agg=mean block=64 target=0.75 max_blocks=256
RESIDUAL:    rank=1024  blocks=128  bsize=64
TRAIN:       steps=24 subm=256 batchE=4 lr=0.05 min_cluster=2
TRAIN_OBJ:   logratio  lam_block=0.2  block_penalty_uses_core_block=True
CLUSTER:     M=(auto) iters=60 feat_d=64 min_size=1
Hadamard:    False seed=1234
Threads:     8  Torch=2.4.1+cpu device=cpu
Cache:       readwrite  path=/home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v5.npz  force_rebuild=False

[found] layer 1 experts: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4

Build Ws:   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote: /home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v5.npz  (268.44 MB)
[Ws] shape=(16, 2048, 2048) experts=[0, 1, 2, 3, 4, 5, 6, 7]...
[cluster] M=8
[cluster] sizes: [2, 1, 1, 1, 1, 1, 7, 2]
[train] step   1/24 | loss=-0.7963  (+2.7s)
[train] step   4/24 | loss=4.7157  (+11.2s)
[train] step   8/24 | loss=4.2578  (+13.9s)
[train] step  12/24 | loss=3.7568  (+13.9s)
[train] step  16/24 | loss=3.4231  (+14.0s)
[train] step  20/24 | loss=4.2579  (+14.0s)
[train] step  24/24 | loss=4.3112  (+14.0s)
[build] payloads ...
  - basis 0: E=2 core=blocktopk_perexpert core_energy(mean/min)≈0.601/0.600 core_blocks(mean)≈256.0 rank=1024 res_blocks=128
  - basis 1: E=1 core=blocktopk_perexpert core_energy(mean/min)≈0.837/0.837 core_blocks(mean)≈1.0 rank=1024 res_blocks=128
  - basis 2: E=1 core=blocktopk_perexpert core_energy(mean/min)≈0.814/0.814 core_blocks(mean)≈3.0 rank=1024 res_blocks=128
  - basis 3: E=1 core=blocktopk_perexpert core_energy(mean/min)≈0.820/0.820 core_

In [19]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X Optimized (offline) — v6.1 (bugfix + safer relabel)
#
# Fixes your crash:
#   - hierarchical_split() used uniq.tolist() (python ints) but called .item()
#   - ALSO fixes a subtle in-place relabel collision bug by relabeling via a fresh tensor.
#
# Everything else is the same v6 “giant improved” runner:
#   - offline shard load via index.json
#   - build square Ws via ridge / weff_gate / weff
#   - hierarchical cluster splitting (avoid mega-clusters)
#   - UV training with scheduled block/group penalty + guidance mask penalty
#   - payload core: blockdiag / blocktopk / blocktopk_perexpert
#   - residual: shared low-rank + greedy blocks
#   - eval vs dense mixture
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise RuntimeError("This script requires PyTorch.") from e

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("This script requires safetensors (pip install safetensors).") from e

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

# ----------------------------
# Threading / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))

DTYPE_W   = torch.float16
DTYPE_ACC = torch.float32

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    # Model slice
    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "16"))

    # Calibration: npz with key 'X' shape (N,H)
    CALIB_PATH: str = os.environ.get("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "512"))

    # Router: npz with key 'P' shape (N, E_total) or (N, E_used)
    ROUTER_PATH: str = os.environ.get("ROUTER_PATH", "").strip()
    ROUTER_EIDS_ARE_GLOBAL: bool = os.environ.get("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    # Build square mats
    LIN_MODE: str = os.environ.get("LIN_MODE", "ridge").strip().lower()  # ridge | weff_gate | weff
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))
    RIDGE_WEIGHTED: bool = os.environ.get("RIDGE_WEIGHTED", "0") == "1"

    # Optional similarity rotation (applies after Ws build)
    USE_HADAMARD: bool = os.environ.get("USE_HADAMARD", "0") == "1"
    HAD_SEED: int = int(os.environ.get("HAD_SEED", "1234"))

    # Cache
    CACHE_MODE: str = os.environ.get("CACHE_MODE", "readwrite").strip().lower()  # off|read|write|readwrite
    CACHE_NAME: str = os.environ.get("CACHE_NAME", "").strip()
    FORCE_REBUILD_WS: bool = os.environ.get("FORCE_REBUILD_WS", "0") == "1"

    # Clustering base
    M_CLUSTERS: int = int(os.environ.get("M_CLUSTERS", "0"))  # 0 => auto M0
    M_MAX: int = int(os.environ.get("M_MAX", "16"))           # hierarchical split max clusters
    CLUSTER_ITERS: int = int(os.environ.get("CLUSTER_ITERS", "60"))
    CLUSTER_FEAT_D: int = int(os.environ.get("CLUSTER_FEAT_D", "64"))
    CLUSTER_MIN_SIZE: int = int(os.environ.get("CLUSTER_MIN_SIZE", "1"))  # reassign small clusters if >1

    # Hierarchical split controls
    CLUSTER_MAX_SIZE: int = int(os.environ.get("CLUSTER_MAX_SIZE", "3"))   # split clusters bigger than this
    SPLIT_ITERS: int = int(os.environ.get("SPLIT_ITERS", "50"))            # kmeans iters for splits

    # Training (U,V)
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "24"))
    TRAIN_WARMUP: int = int(os.environ.get("TRAIN_WARMUP", "6"))
    SUBM: int = int(os.environ.get("SUBM", "256"))
    BATCH_E: int = int(os.environ.get("BATCH_E", "4"))
    LR_UV: float = float(os.environ.get("LR_UV", "5e-2"))
    REORTHO_EVERY: int = int(os.environ.get("REORTHO_EVERY", "4"))
    REPORT_EVERY: int = int(os.environ.get("REPORT_EVERY", "4"))
    TRAIN_MIN_CLUSTER: int = int(os.environ.get("TRAIN_MIN_CLUSTER", "2"))
    GRAD_CLIP: float = float(os.environ.get("GRAD_CLIP", "1.0"))

    # Training objective
    TRAIN_OBJ: str = os.environ.get("TRAIN_OBJ", "logratio").strip().lower()  # logratio|ratio
    TRAIN_LAM_BLOCK: float = float(os.environ.get("TRAIN_LAM_BLOCK", "0.10"))
    TRAIN_LAM_GUIDE: float = float(os.environ.get("TRAIN_LAM_GUIDE", "1.0"))
    TRAIN_GUIDE_EVERY: int = int(os.environ.get("TRAIN_GUIDE_EVERY", "2"))
    TRAIN_GUIDE_TARGET: float = float(os.environ.get("TRAIN_GUIDE_TARGET", "0.75"))
    TRAIN_GUIDE_MAX_BLOCKS: int = int(os.environ.get("TRAIN_GUIDE_MAX_BLOCKS", "256"))

    # Core payload
    CORE_MODE: str = os.environ.get("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_AGG: str = os.environ.get("CORE_AGG", "mean").strip().lower()  # mean|max (for blocktopk shared)
    CORE_BLOCK: int = int(os.environ.get("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(os.environ.get("CORE_TARGET", "0.80"))
    CORE_MAX_BLOCKS: int = int(os.environ.get("CORE_MAX_BLOCKS", "256"))

    # Residual
    RES_RANK: int = int(os.environ.get("RES_RANK", "1024"))
    RES_BLOCKS: int = int(os.environ.get("RES_BLOCKS", "128"))
    RES_BSIZE: int = int(os.environ.get("RES_BSIZE", "64"))

    # Eval
    EVAL_TRIALS: int = int(os.environ.get("EVAL_TRIALS", "5"))
    ROUTED_K: int = int(os.environ.get("ROUTED_K", "8"))
    EVAL_BATCH: int = int(os.environ.get("EVAL_BATCH", "2"))

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def cache_path():
    if cfg.CACHE_NAME:
        return os.path.join(cfg.OUTPUT_DIR, cfg.CACHE_NAME)
    tag = f"deepseek_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_{cfg.LIN_MODE}_Ws_cache_v6_1.npz"
    return os.path.join(cfg.OUTPUT_DIR, tag)

def _meta_dict():
    return dict(
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        max_experts=cfg.MAX_EXPERTS,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES,
        router_path=cfg.ROUTER_PATH or "",
        router_global=cfg.ROUTER_EIDS_ARE_GLOBAL,
        hadamard=cfg.USE_HADAMARD,
        had_seed=cfg.HAD_SEED,
    )

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    out = {}
    if up: out["up"] = up
    if gate: out["gate"] = gate
    if down: out["down"] = down
    return out

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration / router
# ----------------------------
def load_calib_X(H: int) -> torch.Tensor:
    if not cfg.CALIB_PATH:
        X = torch.randn(cfg.CALIB_SAMPLES, H, dtype=DTYPE_W, device=DEVICE)
        log(f"[calib] RANDOM X: {tuple(X.shape)}  (set CALIB_PATH npz key 'X' for realistic ridge)")
        return X
    if not os.path.isfile(cfg.CALIB_PATH):
        raise FileNotFoundError(f"CALIB_PATH not found: {cfg.CALIB_PATH}")
    z = np.load(cfg.CALIB_PATH, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Has keys: {list(z.files)}")
    X = torch.from_numpy(z["X"]).to(DTYPE_W).to(DEVICE)
    z.close()
    if X.ndim != 2 or X.shape[0] < 2:
        raise RuntimeError(f"Bad X shape: {tuple(X.shape)}")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    log(f"[calib] Loaded X: {tuple(X.shape)}")
    return X

def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    if not os.path.isfile(cfg.ROUTER_PATH):
        raise FileNotFoundError(f"ROUTER_PATH not found: {cfg.ROUTER_PATH}")
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Has keys: {list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    log(f"[router] Loaded P: {tuple(P.shape)}")
    return P

# ----------------------------
# Hadamard rotation (optional)
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

@torch.no_grad()
def hadamard_orth(n: int, seed: int) -> torch.Tensor:
    if not is_power_of_two(n):
        raise ValueError(f"Hadamard requires power-of-two n, got {n}")
    H = torch.tensor([[1.0]], dtype=DTYPE_ACC, device=DEVICE)
    while H.shape[0] < n:
        H = torch.cat([torch.cat([H,  H], dim=1),
                       torch.cat([H, -H], dim=1)], dim=0)
    H = H / math.sqrt(n)
    g = torch.Generator(device="cpu").manual_seed(seed)
    signs = (torch.randint(0, 2, (n,), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    return (H * signs.view(1, n)).contiguous()

# ----------------------------
# Build expert square Ws
# ----------------------------
@torch.no_grad()
def forward_mlp_no_grad(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

@torch.no_grad()
def build_Ws_square() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer {cfg.LAYER} experts: {all_eids}  (using {len(eids)})")

    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if "up" not in kk or "down" not in kk or "gate" not in kk:
            raise RuntimeError(f"Expert {eid} missing keys: {kk}")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]].to(DTYPE_W)
    dff, H = W_up0.shape
    log(f"[shape] H={H} d_ff={dff}")

    X = load_calib_X(H)
    P = load_router_P()

    R = hadamard_orth(H, cfg.HAD_SEED) if cfg.USE_HADAMARD else None

    Xf = X.to(DTYPE_ACC)
    XtX = (Xf.t() @ Xf)
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)

    cholG = None
    if cfg.LIN_MODE == "ridge" and not (cfg.RIDGE_WEIGHTED and (P is not None)):
        G = XtX + lam * I
        cholG = torch.linalg.cholesky(G)

    Ws = []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws")):
        W_up   = T[per_e[eid]["up"]].to(DTYPE_W).to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DTYPE_W).to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DTYPE_W).to(DEVICE)

        if cfg.LIN_MODE == "weff":
            W = (W_down.to(DTYPE_ACC) @ W_up.to(DTYPE_ACC))

        elif cfg.LIN_MODE == "weff_gate":
            gate_act = (Xf @ W_gate.to(DTYPE_ACC).t())
            m = F.silu(gate_act).mean(dim=0)  # (dff,)
            W = (W_down.to(DTYPE_ACC) * m.view(1, -1)) @ W_up.to(DTYPE_ACC)

        elif cfg.LIN_MODE == "ridge":
            Y = forward_mlp_no_grad(X, W_gate, W_up, W_down).to(DTYPE_ACC)  # (N,H)

            if cfg.RIDGE_WEIGHTED and (P is not None):
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                else:
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw
                XtX_e = Xw.t() @ Xw
                lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
                G_e = XtX_e + lam_e * I
                chol = torch.linalg.cholesky(G_e)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)
                W = Wt.t().contiguous()
            else:
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)
                W = Wt.t().contiguous()
        else:
            raise ValueError("LIN_MODE must be ridge|weff_gate|weff")

        if R is not None:
            W = (R.t() @ W @ R).contiguous()

        fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
        Ws.append((W / fn).contiguous())

    Ws = torch.stack(Ws, dim=0).to(DTYPE_ACC)  # (E,H,H)
    return eids, Ws

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    p = cache_path()
    do_read = cfg.CACHE_MODE in ("read", "readwrite")
    do_write = cfg.CACHE_MODE in ("write", "readwrite")

    if do_read and (not cfg.FORCE_REBUILD_WS) and os.path.isfile(p):
        z = try_load_npz(p)
        if z and ("expert_ids" in z) and ("Ws" in z):
            ok = True
            if "meta" in z:
                meta = _decode_meta(z["meta"])
                ok = (meta == _meta_dict())
            if ok:
                eids = [int(x) for x in z["expert_ids"].tolist()]
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws: {p}  Ws={tuple(Ws.shape)}")
                return eids, Ws
            log("[cache] meta mismatch -> rebuilding Ws.")
        else:
            log("[cache] invalid cache -> rebuilding Ws.")

    eids, Ws = build_Ws_square()
    if do_write:
        save_npz(p, {
            "meta": _encode_meta(_meta_dict()),
            "expert_ids": np.array(eids, dtype=np.int32),
            "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        })
        log(f"[cache] wrote: {p}  ({os.path.getsize(p)/1e6:.2f} MB)")
    return eids, Ws

# ----------------------------
# Clustering: features + kmeans + hierarchical splitting
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R).unsqueeze(0), (col @ R).unsqueeze(0)], dim=1))
    return torch.cat(feats, dim=0)  # (E, 2d)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int) -> Tuple[torch.Tensor, torch.Tensor]:
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(X.shape[0], generator=g)[:k]
    C = X[idx].clone()
    for _ in range(iters):
        dist = torch.cdist(X, C)
        lab = dist.argmin(dim=1)
        for j in range(k):
            m = (lab == j)
            if m.any():
                C[j] = X[m].mean(dim=0)
    return lab.to(torch.int64), C

@torch.no_grad()
def enforce_min_cluster_size(labels: torch.Tensor, centers: torch.Tensor, X: torch.Tensor, min_size: int) -> torch.Tensor:
    if min_size <= 1:
        return labels.to(torch.int64)
    labels = labels.to(torch.int64)
    k = int(centers.shape[0])
    counts = torch.bincount(labels, minlength=k)
    big = (counts >= min_size).nonzero(as_tuple=False).flatten()
    if big.numel() == 0:
        return labels
    small = (counts < min_size).nonzero(as_tuple=False).flatten()
    if small.numel() == 0:
        return labels
    big_centers = centers[big]
    for c in small.tolist():
        idxs = (labels == int(c)).nonzero(as_tuple=False).flatten()
        if idxs.numel() == 0:
            continue
        d = torch.cdist(X[idxs], big_centers)
        nn = d.argmin(dim=1)
        labels[idxs] = big[nn].to(labels.dtype)
    return labels

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    """
    Safe relabel to contiguous 0..K-1 without in-place collision.
    """
    labels = labels.to(torch.int64)
    uniq = torch.unique(labels)
    uniq_list = uniq.tolist()  # python ints
    out = labels.clone()
    for new, old in enumerate(uniq_list):
        out[labels == int(old)] = int(new)
    return out.to(torch.int64)

@torch.no_grad()
def hierarchical_split(
    Xfeat: torch.Tensor,
    labels: torch.Tensor,
    max_size: int,
    max_k: int,
    split_iters: int
) -> torch.Tensor:
    """
    Repeatedly split the largest cluster using k=2 kmeans until:
      - all cluster sizes <= max_size OR
      - number of clusters reaches max_k OR
      - cannot split (degenerate)
    """
    if max_size <= 0:
        return relabel_contiguous(labels)

    labels = relabel_contiguous(labels)
    while True:
        K = int(labels.max().item()) + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        biggest = int(torch.argmax(counts).item())
        bigsz = int(counts[biggest].item())
        if bigsz <= max_size:
            break

        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2:
            break

        Xsub = Xfeat[idxs]
        sub_lab, _ = kmeans_torch(Xsub, k=2, iters=split_iters)

        a = idxs[sub_lab == 0]
        b = idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0:
            break

        labels[b] = K  # new label id

    return relabel_contiguous(labels)

# ----------------------------
# Ortho params + training helpers
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, M_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(M_init.clone().to(DEVICE))

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def svd_init(W_mean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(W_mean, full_matrices=False)
    V = Vh.t().contiguous()
    return U.contiguous(), V.contiguous()

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S = U[:, S]                # (n,s)
    V_S = V[:, S]                # (n,s)
    T = Ws_batch @ V_S           # (Eb,n,s)
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)  # -> (Eb,s,s)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    nb = s // b
    if nb <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3,4))  # (Eb,nb,nb)
    P = Eblk.mean(dim=0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(
    Xs: torch.Tensor,
    block: int,
    target: float,
    max_blocks: int
) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0

    nb = s // b
    if nb <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0

    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()

    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3,4)).mean(dim=0)  # (nb,nb)

    tot = float((X * X).sum().item()) / max(1, Eb)
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot, 1e-12)

    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K]

    mask = torch.zeros(s2, s2, dtype=DTYPE_ACC, device=Xs.device)
    for idx in pick.tolist():
        bi = idx // nb
        bj = idx % nb
        i0 = bi * b
        j0 = bj * b
        mask[i0:i0+b, j0:j0+b] = 1.0

    if s2 < s:
        full = torch.zeros(s, s, dtype=DTYPE_ACC, device=Xs.device)
        full[:s2, :s2] = mask
        mask = full

    ef = float(frac[K-1].item()) if K > 0 else 0.0
    return mask, ef, int(K)

def schedule(step: int, warmup: int, total: int) -> float:
    if total <= 0:
        return 1.0
    if step <= warmup:
        return 0.0
    return float(min(1.0, max(0.0, (step - warmup) / max(1, (total - warmup)))))

# ----------------------------
# Core selection + residual payload
# ----------------------------
@torch.no_grad()
def _block_energy_grid(X: torch.Tensor, b: int) -> torch.Tensor:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        pad = nb * b - n
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
        n = nb*b
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()  # (nb,nb,b,b)
    E = (Xb * Xb).sum(dim=(2,3))  # (nb,nb)
    return E

@torch.no_grad()
def _pick_top_blocks_from_energy(Eg: torch.Tensor, tot_energy: float, b: int, target: float, max_blocks: int) -> Tuple[List[Tuple[int,int,int,int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    kept = float(flat[order[:K]].sum().item())
    blocks = []
    for idx in pick:
        bi = idx // nb
        bj = idx % nb
        blocks.append((bi*b, bj*b, b, b))
    return blocks, kept / max(tot_energy, 1e-12)

@torch.no_grad()
def choose_core_blocks(
    X_list: List[torch.Tensor],
    mode: str,
    agg: str,
    block: int,
    target: float,
    max_blocks: int
):
    if mode == "none":
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}

    n = X_list[0].shape[0]
    b = int(block)
    if b <= 0:
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}

    if mode == "blockdiag":
        nb = (n + b - 1) // b
        diagE = torch.zeros(nb, dtype=DTYPE_ACC, device=X_list[0].device)
        tots = []
        for X in X_list:
            tots.append(float((X*X).sum().item()))
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            diagE += torch.diagonal(Eg, 0)
        tot = float(np.mean(tots))
        diagE /= max(1, len(X_list))

        order = torch.argsort(diagE, descending=True)
        csum = torch.cumsum(diagE[order], dim=0)
        frac = csum / max(tot, 1e-12)
        need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else nb)
        K = min(need, max_blocks, nb)

        blocks = []
        for bi in order[:K].tolist():
            i0 = bi * b
            blocks.append((i0, i0, b, b))
        ef = float(frac[K-1].item()) if K > 0 else 0.0
        return {"mode": "blockdiag", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    if mode == "blocktopk":
        Eg_all = []
        tots = []
        for X in X_list:
            tots.append(float((X*X).sum().item()))
            Eg_all.append(_block_energy_grid(X, b).to(DTYPE_ACC))
        tot = float(np.mean(tots))
        if agg == "max":
            Eg = torch.stack(Eg_all, dim=0).amax(dim=0)
        else:
            Eg = torch.stack(Eg_all, dim=0).mean(dim=0)
        blocks, ef = _pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
        return {"mode": "blocktopk", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    if mode == "blocktopk_perexpert":
        blocks_per = []
        efracs = []
        for X in X_list:
            tot = float((X*X).sum().item())
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            blocks, ef = _pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
            blocks_per.append(blocks)
            efracs.append(ef)
        return {"mode": "blocktopk_perexpert", "blocks_shared": [], "blocks_per_expert": blocks_per, "energy_fracs": efracs}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

@torch.no_grad()
def greedy_blocks(R: torch.Tensor, t: int, b: int) -> List[Tuple[int,int,torch.Tensor]]:
    n = R.shape[0]
    used = torch.zeros(n, n, dtype=torch.bool, device=R.device)
    out = []
    if t <= 0:
        return out
    b = int(b)
    for _ in range(t):
        best = None
        bestE = -1.0
        for i0 in range(0, n, b):
            for j0 in range(0, n, b):
                i1 = min(n, i0 + b)
                j1 = min(n, j0 + b)
                if used[i0:i1, j0:j1].any():
                    continue
                Eblk = float((R[i0:i1, j0:j1] ** 2).sum().item())
                if Eblk > bestE:
                    bestE = Eblk
                    best = (i0, j0, i1, j1)
        if best is None:
            break
        i0, j0, i1, j1 = best
        used[i0:i1, j0:j1] = True
        out.append((i0, j0, R[i0:i1, j0:j1].clone()))
    return out

@torch.no_grad()
def build_payload_for_cluster(Ws: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor):
    X_list = [(U.t() @ Ws[e] @ V).contiguous() for e in idx]

    core = choose_core_blocks(
        X_list,
        cfg.CORE_MODE,
        cfg.CORE_AGG,
        cfg.CORE_BLOCK,
        cfg.CORE_TARGET,
        cfg.CORE_MAX_BLOCKS
    )

    core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
    if core["blocks_per_expert"] is not None:
        for X, blocks in zip(X_list, core["blocks_per_expert"]):
            lst = []
            for (i0, j0, h, w) in blocks:
                h = min(h, X.shape[0] - i0)
                w = min(w, X.shape[1] - j0)
                if h > 0 and w > 0:
                    lst.append((i0, j0, X[i0:i0+h, j0:j0+w].clone()))
            core_blocks.append(lst)
    else:
        blocks = core["blocks_shared"]
        for X in X_list:
            lst = []
            for (i0, j0, h, w) in blocks:
                h = min(h, X.shape[0] - i0)
                w = min(w, X.shape[1] - j0)
                if h > 0 and w > 0:
                    lst.append((i0, j0, X[i0:i0+h, j0:j0+w].clone()))
            core_blocks.append(lst)

    R_list = []
    for X, cb in zip(X_list, core_blocks):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    U_s, _, Vh_s = torch.linalg.svd(Rmean, full_matrices=False)
    r = min(cfg.RES_RANK, U_s.shape[1])
    DL = U_s[:, :r].contiguous()
    DR = Vh_s.t()[:, :r].contiguous()

    gam_list = []
    blocks_list = []
    for Rm in R_list:
        g = torch.sum(DL * (Rm @ DR), dim=0)
        gam_list.append(g.contiguous())
        R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        blocks_list.append(greedy_blocks(R2, cfg.RES_BLOCKS, cfg.RES_BSIZE))

    gam = torch.stack(gam_list, dim=0)

    return {
        "U": U, "V": V,
        "core": core,
        "core_blocks": core_blocks,
        "DL": DL, "DR": DR,
        "gam": gam,
        "blocks": blocks_list,
        "idx": idx,
    }

class KTXRuntime(nn.Module):
    def __init__(self, payloads: List[Optional[dict]], basis_of_e: Dict[int, Tuple[int,int]]):
        super().__init__()
        self.payloads = payloads
        self.map = basis_of_e
        self.n = None
        for p in payloads:
            if p is not None:
                self.n = int(p["U"].shape[0])
                break
        if self.n is None:
            raise RuntimeError("No payloads provided.")

    @torch.no_grad()
    def forward(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        used: Dict[int, List[Tuple[float,int]]] = {}
        for a, e in zip(gates.tolist(), routed):
            if e in self.map:
                m, j = self.map[e]
                used.setdefault(m, []).append((a, j))

        for m, items in used.items():
            P = self.payloads[m]
            if P is None:
                continue

            U = P["U"]; V = P["V"]
            DL = P["DL"]; DR = P["DR"]

            z = x @ U
            u_acc = torch.zeros_like(z)

            for (a, j) in items:
                u = torch.zeros_like(z)

                for (i0, j0, Bc) in P["core_blocks"][j]:
                    h, w = Bc.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bc

                g = P["gam"][j]
                u += ((z @ DL) * g.view(1, -1)) @ DR.t()

                for (i0, j0, Bb) in P["blocks"][j]:
                    h, w = Bb.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb

                u_acc += float(a) * u

            y += u_acc @ V.t()

        return y

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def dense_apply(Ws: torch.Tensor, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
    n = Ws.shape[-1]
    Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=x.device)
    for a, e in zip(gates, routed):
        Wsum += float(a.item()) * Ws[e]
    return x @ Wsum

@torch.no_grad()
def eval_runtime(rt: KTXRuntime, Ws: torch.Tensor, trials: int, routed_k: int, batch: int):
    E = Ws.shape[0]
    errs = []
    for _ in range(trials):
        x = torch.randn(batch, rt.n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(routed_k, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)

        y_hat = rt(x, routed, gates)
        y_ref = dense_apply(Ws, x, routed, gates)

        err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)

    log(f"[eval] forward rel-error = {float(np.mean(errs)):.4f} ± {float(np.std(errs)):.4f}  (trials={trials})")

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X Optimized (offline) v6.1 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  (RIDGE_DAMP={cfg.RIDGE_DAMP})")
    log(f"CORE_MODE:   {cfg.CORE_MODE} agg={cfg.CORE_AGG} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK} blocks={cfg.RES_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} warmup={cfg.TRAIN_WARMUP} subm={cfg.SUBM} batchE={cfg.BATCH_E} lr={cfg.LR_UV} min_cluster={cfg.TRAIN_MIN_CLUSTER}")
    log(f"TRAIN_OBJ:   {cfg.TRAIN_OBJ} lam_block={cfg.TRAIN_LAM_BLOCK} lam_guide={cfg.TRAIN_LAM_GUIDE} guide_every={cfg.TRAIN_GUIDE_EVERY}")
    log(f"CLUSTER:     M0={cfg.M_CLUSTERS or '(auto)'} M_MAX={cfg.M_MAX} max_size={cfg.CLUSTER_MAX_SIZE} iters={cfg.CLUSTER_ITERS} feat_d={cfg.CLUSTER_FEAT_D} min_size={cfg.CLUSTER_MIN_SIZE}")
    log(f"Hadamard:    {cfg.USE_HADAMARD} seed={cfg.HAD_SEED}")
    log(f"Threads:     {NTHREADS} Torch={torch.__version__} device={DEVICE}")
    log(f"Cache:       {cfg.CACHE_MODE} path={cache_path()} force_rebuild={cfg.FORCE_REBUILD_WS}")
    log("")

def main():
    banner()

    eids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} experts={eids[:8]}{'...' if len(eids)>8 else ''}")

    # Initial cluster count M0
    if cfg.M_CLUSTERS > 0:
        M0 = min(cfg.M_CLUSTERS, E)
    else:
        M0 = int(round(2.0 * math.sqrt(E)))
        M0 = max(6, min(M0, E))
    M0 = max(2, min(M0, E))
    log(f"[cluster] M0={M0}")

    # Features
    Xfeat = random_proj_features(Ws, d=cfg.CLUSTER_FEAT_D)

    # Base kmeans
    labels, centers = kmeans_torch(Xfeat, k=M0, iters=cfg.CLUSTER_ITERS)
    labels = enforce_min_cluster_size(labels, centers, Xfeat, min_size=cfg.CLUSTER_MIN_SIZE)

    # Hierarchical split
    labels = hierarchical_split(
        Xfeat=Xfeat,
        labels=labels,
        max_size=cfg.CLUSTER_MAX_SIZE,
        max_k=min(cfg.M_MAX, E),
        split_iters=cfg.SPLIT_ITERS
    )

    M = int(labels.max().item()) + 1
    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    log(f"[cluster] M={M}")
    log(f"[cluster] sizes: {[len(c) for c in clusters]}")

    # Init U,V per cluster
    U_par: List[Optional[OrthoParam]] = []
    V_par: List[Optional[OrthoParam]] = []
    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            U_par.append(None); V_par.append(None)
            continue
        Wm = Ws[idx].mean(dim=0)
        U0, V0 = svd_init(Wm)
        U_par.append(OrthoParam(U0))
        V_par.append(OrthoParam(V0))

    # Training
    guidance_masks: Dict[int, torch.Tensor] = {}
    guidance_stats: Dict[int, Tuple[float,int]] = {}

    if cfg.TRAIN_STEPS > 0:
        params = [uv.M for uv in (U_par + V_par) if uv is not None]
        opt = torch.optim.Adam(params, lr=cfg.LR_UV)
        t0 = time.perf_counter()

        with torch.enable_grad():
            for step in range(1, cfg.TRAIN_STEPS + 1):
                S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]

                # Update guidance masks periodically
                if cfg.TRAIN_LAM_GUIDE > 0 and (step % max(1, cfg.TRAIN_GUIDE_EVERY) == 0 or step == 1):
                    with torch.no_grad():
                        guidance_masks.clear()
                        guidance_stats.clear()
                        for m, idx in enumerate(clusters):
                            if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                                continue
                            Uo = U_par[m].orthogonal()
                            Vo = V_par[m].orthogonal()
                            if cfg.BATCH_E > 0 and cfg.BATCH_E < len(idx):
                                pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                                idx_step = [idx[p] for p in pick]
                            else:
                                idx_step = idx
                            Xs_ng = slice_X_batch(Ws[idx_step], Uo, Vo, S).detach()
                            mask, ef, kblk = make_guidance_mask_from_Xs(
                                Xs_ng.to(DTYPE_ACC),
                                block=cfg.CORE_BLOCK,
                                target=cfg.TRAIN_GUIDE_TARGET,
                                max_blocks=cfg.TRAIN_GUIDE_MAX_BLOCKS
                            )
                            guidance_masks[m] = mask
                            guidance_stats[m] = (ef, kblk)

                lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
                lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
                lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp

                L_total = None
                n_terms = 0

                for m, idx in enumerate(clusters):
                    if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                        continue

                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()

                    if cfg.BATCH_E > 0 and cfg.BATCH_E < len(idx):
                        pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                        idx_step = [idx[p] for p in pick]
                    else:
                        idx_step = idx

                    Xs = slice_X_batch(Ws[idx_step], Uo, Vo, S).to(DTYPE_ACC)

                    off = offdiag_abs_mean(Xs)
                    diag = diag_abs_mean(Xs).clamp_min(1e-6)

                    if cfg.TRAIN_OBJ == "ratio":
                        base = off / diag
                    else:
                        base = torch.log(off + 1e-6) - torch.log(diag)

                    if lam_block > 0 and cfg.CORE_MODE.startswith("block"):
                        base = base + lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)

                    if lam_guide > 0 and (m in guidance_masks):
                        Mmask = guidance_masks[m]
                        Etot = (Xs * Xs).mean().clamp_min(1e-12)
                        Eout = ((Xs * (1.0 - Mmask)) ** 2).mean()
                        base = base + lam_guide * (Eout / Etot)

                    L_total = base if (L_total is None) else (L_total + base)
                    n_terms += 1

                if L_total is None:
                    log("[train] skipped (no clusters >= TRAIN_MIN_CLUSTER)")
                    break

                L_total = L_total / max(1, n_terms)
                if not L_total.requires_grad:
                    raise RuntimeError("Training graph broken: loss has no grad_fn.")

                opt.zero_grad(set_to_none=True)
                L_total.backward()
                if cfg.GRAD_CLIP > 0:
                    torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                opt.step()

                if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    with torch.no_grad():
                        for uv in (U_par + V_par):
                            if uv is not None:
                                uv.M.copy_(uv.orthogonal())

                if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    t1 = time.perf_counter()
                    if len(guidance_stats) > 0:
                        ef_mean = float(np.mean([v[0] for v in guidance_stats.values()]))
                        kb_mean = float(np.mean([v[1] for v in guidance_stats.values()]))
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} | loss={float(L_total.item()):.4f} "
                            f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} "
                            f"guide_energy≈{ef_mean:.3f} guide_blocks≈{kb_mean:.1f} (+{t1-t0:.1f}s)")
                    else:
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} | loss={float(L_total.item()):.4f} "
                            f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} (+{t1-t0:.1f}s)")
                    t0 = t1

    # Build payloads
    payloads: List[Optional[dict]] = []
    basis_of_e: Dict[int, Tuple[int,int]] = {}
    log("[build] payloads ...")

    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            payloads.append(None)
            continue

        Uo = U_par[m].orthogonal().detach()
        Vo = V_par[m].orthogonal().detach()
        P = build_payload_for_cluster(Ws, idx, Uo, Vo)
        payloads.append(P)

        for j, e in enumerate(idx):
            basis_of_e[e] = (m, j)

        efr = P["core"]["energy_fracs"]
        ef_mean = float(np.mean(efr)) if len(efr) else 0.0
        ef_min  = float(np.min(efr)) if len(efr) else 0.0
        nb_mean = float(np.mean([len(x) for x in P["core_blocks"]])) if len(P["core_blocks"]) else 0.0

        log(f"  - basis {m}: E={len(idx)} core={P['core']['mode']} "
            f"core_energy(mean/min)≈{ef_mean:.3f}/{ef_min:.3f} "
            f"core_blocks(mean)≈{nb_mean:.1f} rank={P['DL'].shape[1]} res_blocks={cfg.RES_BLOCKS}")

    rt = KTXRuntime(payloads, basis_of_e)
    eval_runtime(rt, Ws, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    log("\n✅ Done.")
    log("If core_blocks(mean) is pegged at CORE_MAX_BLOCKS with low core_energy:")
    log(" - Split more: set CLUSTER_MAX_SIZE=2 and M_MAX=16.")
    log(" - Stronger guide: TRAIN_LAM_GUIDE=2.0 and TRAIN_GUIDE_EVERY=1.")
    log(" - Alternative block geometry: CORE_BLOCK=128 and CORE_MAX_BLOCKS=128.")
    log("Real unlock later: CALIB_PATH + ROUTER_PATH with RIDGE_WEIGHTED=1 and FORCE_REBUILD_WS=1.")

if __name__ == "__main__":
    banner()
    main()


== DeepSeek KT++-X Optimized (offline) v6.1 ==
Time:        2026-01-01 17:17:15
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES=512
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
LIN_MODE:    ridge  (RIDGE_DAMP=0.001)
CORE_MODE:   blocktopk_perexpert agg=mean block=64 target=0.8 max_blocks=256
RESIDUAL:    rank=1024 blocks=128 bsize=64
TRAIN:       steps=24 warmup=6 subm=256 batchE=4 lr=0.05 min_cluster=2
TRAIN_OBJ:   logratio lam_block=0.1 lam_guide=1.0 guide_every=2
CLUSTER:     M0=(auto) M_MAX=16 max_size=3 iters=60 feat_d=64 min_size=1
Hadamard:    False seed=1234
Threads:     8 Torch=2.4.1+cpu device=cpu
Cache:       readwrite path=/home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v6_1.npz force_rebuild=False

== DeepSeek KT++-X Optimized (offline) v6.1 ==
Time:        2026-01-01 17:17:15
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs

Build Ws:   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote: /home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v6_1.npz  (268.44 MB)
[Ws] shape=(16, 2048, 2048) experts=[0, 1, 2, 3, 4, 5, 6, 7]...
[cluster] M0=8
[cluster] M=10
[cluster] sizes: [2, 1, 1, 1, 1, 1, 1, 2, 3, 3]
[train] step   1/24 | loss=-3.8462 lam_block=0.000 lam_guide=0.000 guide_energy≈0.794 guide_blocks≈3.0 (+5.4s)
[train] step   4/24 | loss=-1.3576 lam_block=0.000 lam_guide=0.000 guide_energy≈0.769 guide_blocks≈11.5 (+18.4s)
[train] step   8/24 | loss=-1.2643 lam_block=0.011 lam_guide=0.111 guide_energy≈0.773 guide_blocks≈11.0 (+22.3s)
[train] step  12/24 | loss=-0.5981 lam_block=0.033 lam_guide=0.333 guide_energy≈0.771 guide_blocks≈11.0 (+22.1s)
[train] step  16/24 | loss=-0.5918 lam_block=0.056 lam_guide=0.556 guide_energy≈0.775 guide_blocks≈10.0 (+22.2s)
[train] step  20/24 | loss=0.5406 lam_block=0.078 lam_guide=0.778 guide_energy≈0.767 guide_blocks≈10.5 (+21.9s)
[train] step  24/24 | loss=0.7154 lam_block=0.100 lam_guide=1.000 guide_energy≈0.7

In [ ]:
#BEST RESULTS

In [21]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X Optimized (offline) v7.2 (single file)
#
# OFFLINE + terminal-first:
#  - reads local safetensors shards via model.safetensors.index.json
#  - builds square expert mats Ws[e] (H x H) for one MoE layer
#  - learns per-cluster orthogonal bases U,V (QR-param) with a stable autograd graph
#  - novelty on top:
#     * block seriation (RCM-like on block graph)
#     * per-expert diagonal balancing (Sinkhorn-like L2 scaling) in basis
#     * differentiable guide objective (diag-block energy fraction)
#     * within-cluster mixing loss on diagonals
#     * randomized SVD option for residual dictionary
#  - payload = core blocks (blocktopk_perexpert) + low-rank residual + greedy blocks
#  - runtime evaluates forward rel-error under random routing weights
#
# Dependencies: torch, numpy, safetensors, tqdm (optional)
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise RuntimeError("This script requires PyTorch.") from e

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("This script requires safetensors (pip install safetensors).") from e

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

# ----------------------------
# Threading / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))

DTYPE_W   = torch.float16
DTYPE_ACC = torch.float32

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ============================================================
# Config
# ============================================================

@dataclass
class Cfg:
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "16"))

    # Calibration (npz with key 'X': (N,H))
    CALIB_PATH: str = os.environ.get("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "512"))

    # Router probs (npz with key 'P': (N,E_total)), optional
    ROUTER_PATH: str = os.environ.get("ROUTER_PATH", "").strip()
    ROUTER_EIDS_ARE_GLOBAL: bool = os.environ.get("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    # Build square Ws
    LIN_MODE: str = os.environ.get("LIN_MODE", "ridge").strip().lower()  # ridge | weff_gate | weff
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))
    RIDGE_WEIGHTED: bool = os.environ.get("RIDGE_WEIGHTED", "0") == "1"

    # Optional similarity Hadamard on Ws (rarely needed here)
    USE_HADAMARD: bool = os.environ.get("USE_HADAMARD", "0") == "1"
    HAD_SEED: int = int(os.environ.get("HAD_SEED", "1234"))

    # Clustering
    M0: int = int(os.environ.get("M0", "0"))                 # 0 => auto
    M_MAX: int = int(os.environ.get("M_MAX", "16"))
    CLUSTER_MAX_SIZE: int = int(os.environ.get("CLUSTER_MAX_SIZE", "3"))
    CLUSTER_MIN_SIZE: int = int(os.environ.get("CLUSTER_MIN_SIZE", "1"))
    KMEANS_ITERS: int = int(os.environ.get("KMEANS_ITERS", "60"))
    FEAT_D: int = int(os.environ.get("FEAT_D", "64"))

    # Training
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "24"))
    WARMUP_STEPS: int = int(os.environ.get("WARMUP_STEPS", "6"))
    LR_UV: float = float(os.environ.get("LR_UV", "5e-2"))
    BATCH_E: int = int(os.environ.get("BATCH_E", "4"))
    SUBM_BLOCKS: int = int(os.environ.get("SUBM_BLOCKS", "4"))  # sample this many blocks per step
    REORTHO_EVERY: int = int(os.environ.get("REORTHO_EVERY", "4"))
    REPORT_EVERY: int = int(os.environ.get("REPORT_EVERY", "4"))

    TRAIN_OBJ: str = os.environ.get("TRAIN_OBJ", "logratio").strip().lower()  # ratio|logratio
    TRAIN_LAM_BLOCK: float = float(os.environ.get("TRAIN_LAM_BLOCK", "0.1"))  # block entropy penalty
    TRAIN_LAM_GUIDE: float = float(os.environ.get("TRAIN_LAM_GUIDE", "1.0"))  # diag-block energy guide
    TRAIN_GUIDE_EVERY: int = int(os.environ.get("TRAIN_GUIDE_EVERY", "2"))
    TRAIN_MIX_LAM: float = float(os.environ.get("TRAIN_MIX_LAM", "0.15"))      # within-cluster diag mixing
    TRAIN_MIX_K: int = int(os.environ.get("TRAIN_MIX_K", "6"))                 # how many diag coords to mix (subsample)

    # Core selection
    CORE_MODE: str = os.environ.get("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_BLOCK: int = int(os.environ.get("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(os.environ.get("CORE_TARGET", "0.8"))
    CORE_MAX_BLOCKS: int = int(os.environ.get("CORE_MAX_BLOCKS", "256"))
    DIAG_BONUS: float = float(os.environ.get("DIAG_BONUS", "0.05"))  # boosts diagonal blocks in scoring

    # Seriation (novel)
    SERIATION: bool = os.environ.get("SERIATION", "1") == "1"
    SERIATION_TOPK_NEI: int = int(os.environ.get("SERIATION_TOPK_NEI", "8"))

    # Per-expert diagonal balancing (novel)
    BALANCE_DIAG: bool = os.environ.get("BALANCE_DIAG", "1") == "1"
    BALANCE_ITERS: int = int(os.environ.get("BALANCE_ITERS", "1"))
    BALANCE_EPS: float = float(os.environ.get("BALANCE_EPS", "1e-6"))

    # Residual model
    RES_RANK: int = int(os.environ.get("RES_RANK", "1024"))
    RES_BLOCKS: int = int(os.environ.get("RES_BLOCKS", "128"))
    RES_BSIZE: int = int(os.environ.get("RES_BSIZE", "64"))
    RES_SVD_MODE: str = os.environ.get("RES_SVD_MODE", "rand").strip().lower()  # svd|rand
    RSVD_OVERSAMPLE: int = int(os.environ.get("RSVD_OVERSAMPLE", "32"))
    RSVD_ITERS: int = int(os.environ.get("RSVD_ITERS", "1"))

    # Cache
    CACHE_MODE: str = os.environ.get("CACHE_MODE", "readwrite").strip().lower()  # off|read|write|readwrite
    CACHE_NAME: str = os.environ.get("CACHE_NAME", "").strip()
    FORCE_REBUILD_WS: bool = os.environ.get("FORCE_REBUILD_WS", "0") == "1"

    # Eval
    EVAL_TRIALS: int = int(os.environ.get("EVAL_TRIALS", "5"))
    ROUTED_K: int = int(os.environ.get("ROUTED_K", "8"))
    EVAL_BATCH: int = int(os.environ.get("EVAL_BATCH", "2"))

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def cache_path():
    if cfg.CACHE_NAME:
        return os.path.join(cfg.OUTPUT_DIR, cfg.CACHE_NAME)
    tag = f"deepseek_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_{cfg.LIN_MODE}_Ws_cache_v7_2.npz"
    return os.path.join(cfg.OUTPUT_DIR, tag)

# ============================================================
# Offline shard loading
# ============================================================

def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    """
    For DeepSeek-like MoE MLP experts.
    We need up, gate, down. DeepSeek often uses: up_proj, gate_proj, down_proj.
    Some checkpoints use w1/w2/w3 naming.
    """
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."

    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None

    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    out = {}
    if up: out["up"] = up
    if gate: out["gate"] = gate
    if down: out["down"] = down
    return out

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ============================================================
# Calibration + router
# ============================================================

def load_calib_X(H: int) -> torch.Tensor:
    if not cfg.CALIB_PATH:
        X = torch.randn(cfg.CALIB_SAMPLES, H, dtype=DTYPE_W)
        log(f"[calib] RANDOM X: {tuple(X.shape)}  (set CALIB_PATH npz key 'X' for realistic ridge)")
        return X
    if not os.path.isfile(cfg.CALIB_PATH):
        raise FileNotFoundError(f"CALIB_PATH not found: {cfg.CALIB_PATH}")
    z = np.load(cfg.CALIB_PATH, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Has keys: {list(z.files)}")
    X = torch.from_numpy(z["X"]).to(DTYPE_W)
    z.close()
    if X.ndim != 2 or X.shape[1] != H:
        raise RuntimeError(f"Bad X shape: {tuple(X.shape)} expected (*,{H})")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    log(f"[calib] Loaded X: {tuple(X.shape)}")
    return X

def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    if not os.path.isfile(cfg.ROUTER_PATH):
        raise FileNotFoundError(f"ROUTER_PATH not found: {cfg.ROUTER_PATH}")
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Has keys: {list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    log(f"[router] Loaded P: {tuple(P.shape)}")
    return P

# ============================================================
# Optional Hadamard similarity
# ============================================================

def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

@torch.no_grad()
def hadamard_orth(n: int, seed: int) -> torch.Tensor:
    if not is_power_of_two(n):
        raise ValueError(f"Hadamard requires power-of-two n, got {n}")
    H = torch.tensor([[1.0]], dtype=DTYPE_ACC, device="cpu")
    while H.shape[0] < n:
        H = torch.cat([torch.cat([H,  H], dim=1),
                       torch.cat([H, -H], dim=1)], dim=0)
    H = H / math.sqrt(n)
    g = torch.Generator(device="cpu").manual_seed(seed)
    signs = (torch.randint(0, 2, (n,), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC)
    return (H * signs.view(1, n)).contiguous()

# ============================================================
# Build square expert matrices Ws[e] (H x H)
# ============================================================

@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

@torch.no_grad()
def build_Ws_square() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer {cfg.LAYER} experts: {all_eids}  (using {len(eids)})")

    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if "up" not in kk or "down" not in kk or "gate" not in kk:
            raise RuntimeError(f"Expert {eid} missing keys: {kk}")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]].to(DTYPE_W)
    dff, H = W_up0.shape
    log(f"[shape] H={H} d_ff={dff}")

    X = load_calib_X(H)
    P = load_router_P()

    R = hadamard_orth(H, cfg.HAD_SEED) if cfg.USE_HADAMARD else None

    Xf = X.to(DTYPE_ACC)
    I = torch.eye(H, dtype=DTYPE_ACC)
    # Precompute shared ridge factor if unweighted
    cholG = None
    if cfg.LIN_MODE == "ridge" and not (cfg.RIDGE_WEIGHTED and (P is not None)):
        XtX = Xf.t() @ Xf
        lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
        G = XtX + lam * I
        cholG = torch.linalg.cholesky(G)

    Ws = []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws")):
        W_up   = T[per_e[eid]["up"]].to(DTYPE_W)
        W_down = T[per_e[eid]["down"]].to(DTYPE_W)
        W_gate = T[per_e[eid]["gate"]].to(DTYPE_W)

        if cfg.LIN_MODE == "weff":
            W = (W_down.to(DTYPE_ACC) @ W_up.to(DTYPE_ACC))

        elif cfg.LIN_MODE == "weff_gate":
            gate_act = (Xf @ W_gate.to(DTYPE_ACC).t())
            m = F.silu(gate_act).mean(dim=0)  # (dff,)
            W = (W_down.to(DTYPE_ACC) * m.view(1, -1)) @ W_up.to(DTYPE_ACC)

        elif cfg.LIN_MODE == "ridge":
            Y = forward_mlp(X, W_gate, W_up, W_down).to(DTYPE_ACC)  # (N,H)
            if cfg.RIDGE_WEIGHTED and (P is not None):
                # sample weights from router probs
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).clamp_min(0.0)
                else:
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).clamp_min(0.0)
                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw
                XtX = Xw.t() @ Xw
                lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
                G = XtX + lam * I
                chol = torch.linalg.cholesky(G)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)
                W = Wt.t().contiguous()
            else:
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)
                W = Wt.t().contiguous()
        else:
            raise ValueError("LIN_MODE must be ridge|weff_gate|weff")

        if R is not None:
            # similarity transform: W' = R^T W R
            W = (R.t() @ W @ R).contiguous()

        fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
        W = (W / fn).contiguous()
        Ws.append(W)

    Ws = torch.stack(Ws, dim=0)  # (E,H,H) float32
    return eids, Ws

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    p = cache_path()
    do_read = cfg.CACHE_MODE in ("read", "readwrite")
    do_write = cfg.CACHE_MODE in ("write", "readwrite")

    if do_read and (not cfg.FORCE_REBUILD_WS) and os.path.isfile(p):
        z = try_load_npz(p)
        if z and ("expert_ids" in z) and ("Ws" in z):
            eids = [int(x) for x in z["expert_ids"].tolist()]
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC)
            log(f"[cache] loaded Ws: {p}  Ws={tuple(Ws.shape)}")
            return eids, Ws
        log("[cache] invalid cache -> rebuilding.")

    eids, Ws = build_Ws_square()
    if do_write:
        save_npz(p, {
            "expert_ids": np.array(eids, dtype=np.int32),
            "Ws": Ws.cpu().numpy().astype(np.float32),
        })
        log(f"[cache] wrote: {p}  ({os.path.getsize(p)/1e6:.2f} MB)")
    return eids, Ws

# ============================================================
# Features + kmeans + cluster splitting
# ============================================================

@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R).unsqueeze(0), (col @ R).unsqueeze(0)], dim=1))
    return torch.cat(feats, dim=0)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int) -> Tuple[torch.Tensor, torch.Tensor]:
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(X.shape[0], generator=g)[:k]
    C = X[idx].clone()
    for _ in range(iters):
        dist = torch.cdist(X, C)
        lab = dist.argmin(dim=1)
        for j in range(k):
            m = (lab == j)
            if m.any():
                C[j] = X[m].mean(dim=0)
    return lab, C

@torch.no_grad()
def enforce_min_cluster_size(labels: torch.Tensor, centers: torch.Tensor, X: torch.Tensor, min_size: int) -> torch.Tensor:
    if min_size <= 1:
        return labels
    K = int(centers.shape[0])
    counts = torch.bincount(labels, minlength=K)
    # For each tiny cluster, reassign its points to nearest non-tiny center
    good = (counts >= min_size)
    if good.all():
        return labels
    good_ids = torch.nonzero(good, as_tuple=False).flatten()
    if good_ids.numel() == 0:
        return labels  # degenerate
    good_centers = centers[good_ids]
    for k in range(K):
        if counts[k] >= min_size:
            continue
        pts = torch.nonzero(labels == k, as_tuple=False).flatten()
        if pts.numel() == 0:
            continue
        dist = torch.cdist(X[pts], good_centers)
        nn = dist.argmin(dim=1)
        labels[pts] = good_ids[nn]
    return labels

@torch.no_grad()
def hierarchical_split(Xfeat: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    """
    Split clusters larger than max_size by running kmeans with k=2 inside them,
    until no cluster exceeds max_size or we hit max_k total clusters.
    """
    if max_size <= 0:
        return labels
    # ensure contiguous 0..K-1 first
    uniq = torch.unique(labels)
    remap = {int(u): i for i, u in enumerate(uniq.tolist())}
    for old, new in remap.items():
        labels[labels == old] = new

    while True:
        K = int(labels.max().item()) + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        big = torch.nonzero(counts > max_size, as_tuple=False).flatten()
        if big.numel() == 0:
            break
        # split the largest one
        k_big = int(big[counts[big].argmax()].item())
        pts = torch.nonzero(labels == k_big, as_tuple=False).flatten()
        if pts.numel() <= max_size:
            break
        # run kmeans 2-way on these points
        subX = Xfeat[pts]
        sublab, subcent = kmeans_torch(subX, k=2, iters=split_iters)
        new_label = K
        # keep 0 -> old label, 1 -> new label
        labels[pts[sublab == 1]] = new_label
    # final remap contiguous
    uniq = torch.unique(labels)
    remap = {int(u): i for i, u in enumerate(uniq.tolist())}
    for old, new in remap.items():
        labels[labels == old] = new
    return labels

# ============================================================
# Ortho params
# ============================================================

class OrthoParam(nn.Module):
    def __init__(self, M_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(M_init.clone())

    def orthogonal(self) -> torch.Tensor:
        # differentiable QR
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def svd_init(W_mean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(W_mean, full_matrices=False)
    return U.contiguous(), Vh.t().contiguous()

# ============================================================
# Block utilities (vectorized)
# ============================================================

def _pad_to_block(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, int]:
    n = X.shape[0]
    nb = (n + b - 1) // b
    pad = nb * b - n
    if pad == 0:
        return X, n
    Z = torch.zeros(nb * b, nb * b, dtype=X.dtype, device=X.device)
    Z[:n, :n] = X
    return Z, n

def block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, int, int]:
    """
    Return energies per (bi,bj) block in an nb x nb grid.
    Works with padding if n not divisible by b.
    """
    Xp, n0 = _pad_to_block(X, b)
    n = Xp.shape[0]
    nb = n // b
    # (nb,b,nb,b)
    G = Xp.view(nb, b, nb, b)
    E = (G * G).sum(dim=(1, 3))  # (nb, nb)
    return E, nb, n0

def pick_top_blocks_from_energy(E: torch.Tensor, target: float, max_blocks: int, diag_bonus: float) -> List[Tuple[int,int]]:
    nb = E.shape[0]
    Eb = E.clone()
    if diag_bonus != 0.0:
        diag = torch.diagonal(Eb, 0)
        Eb = Eb.clone()
        Eb[torch.arange(nb), torch.arange(nb)] = diag * (1.0 + diag_bonus)
    flat = Eb.reshape(-1)
    order = torch.argsort(flat, descending=True)
    tot = float(E.sum().item()) + 1e-12
    picked = []
    acc = 0.0
    used = 0
    for idx in order.tolist():
        bi = idx // nb
        bj = idx - bi * nb
        picked.append((bi, bj))
        acc += float(E[bi, bj].item())
        used += 1
        if used >= max_blocks:
            break
        if (acc / tot) >= target:
            break
    return picked

def blocks_to_slices(blocks: List[Tuple[int,int]], b: int, n0: int) -> List[Tuple[int,int,int,int]]:
    out = []
    for (bi, bj) in blocks:
        i0 = bi * b
        j0 = bj * b
        i1 = min(i0 + b, n0)
        j1 = min(j0 + b, n0)
        if i0 < n0 and j0 < n0 and i1 > i0 and j1 > j0:
            out.append((i0, i1, j0, j1))
    return out

# ============================================================
# Seriation (novel): RCM-like on block graph
# ============================================================

@torch.no_grad()
def rcm_order_from_block_graph(A: torch.Tensor, topk: int) -> List[int]:
    """
    Reverse Cuthill-McKee-like ordering on a weighted adjacency A (nb x nb).
    We build neighbors by top-k weights per node (excluding self).
    """
    nb = A.shape[0]
    # degrees from neighbor lists
    neighbors = []
    for i in range(nb):
        w = A[i].clone()
        w[i] = -1.0
        idx = torch.argsort(w, descending=True)[:max(1, min(topk, nb-1))]
        nei = [int(j.item()) for j in idx if w[j] > 0]
        neighbors.append(nei)
    deg = [len(neighbors[i]) for i in range(nb)]

    visited = [False] * nb
    order = []

    def bfs(start: int):
        q = [start]
        visited[start] = True
        while q:
            v = q.pop(0)
            order.append(v)
            # visit neighbors sorted by increasing degree
            cand = [u for u in neighbors[v] if not visited[u]]
            cand.sort(key=lambda x: deg[x])
            for u in cand:
                visited[u] = True
                q.append(u)

    while len(order) < nb:
        # pick unvisited node with smallest degree
        unv = [i for i in range(nb) if not visited[i]]
        unv.sort(key=lambda x: deg[x])
        bfs(unv[0])

    # reverse for RCM
    order = order[::-1]
    return order

@torch.no_grad()
def seriation_perm_for_basis(Ws: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor, b: int, diag_bonus: float, topk_nei: int) -> torch.Tensor:
    """
    Build a block graph from mean |X| block energies of Xm = U^T mean(W) V,
    then RCM-order the blocks and expand to a dimension permutation pi.
    """
    Wm = Ws[idx].mean(dim=0)
    Xm = (U.t() @ Wm @ V).to(DTYPE_ACC)
    Eblk, nb, n0 = block_energy_grid(Xm.abs(), b)
    # adjacency: use off-diagonal energies; add diag bonus to strengthen diagonal cohesion
    A = Eblk.clone()
    if diag_bonus != 0.0:
        A[torch.arange(nb), torch.arange(nb)] *= (1.0 + diag_bonus)
    # symmetrize
    A = 0.5 * (A + A.t())
    block_order = rcm_order_from_block_graph(A, topk=topk_nei)
    # expand to index permutation
    pi = []
    for bi in block_order:
        i0 = bi * b
        i1 = min(i0 + b, n0)
        pi.extend(list(range(i0, i1)))
    return torch.tensor(pi, dtype=torch.long)

# ============================================================
# Basis-space diagonal balancing (novel)
#   We transform X -> Xb = D^{-1} X D, store D per expert,
#   and apply it correctly at runtime: (z/D) @ Xb then *D.
# ============================================================

@torch.no_grad()
def balance_diag_l2(X: torch.Tensor, iters: int, eps: float) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Returns (Xb, d) where d is the diagonal scaling vector (n,)
    such that Xb = diag(1/d) X diag(d).
    One or a few L2 "balancing" iterations.
    """
    n = X.shape[0]
    d = torch.ones(n, dtype=DTYPE_ACC, device=X.device)
    Xb = X
    for _ in range(max(0, iters)):
        r = torch.sqrt((Xb * Xb).sum(dim=1) + eps)  # row L2
        c = torch.sqrt((Xb * Xb).sum(dim=0) + eps)  # col L2
        # update so rows/cols get closer
        # d <- d * sqrt(c/r)
        upd = torch.sqrt(c / r).clamp(1e-3, 1e3)
        d = d * upd
        Xb = (X / d.view(-1, 1)) * d.view(1, -1)
    return Xb.contiguous(), d.contiguous()

# ============================================================
# Residual dictionary: exact SVD or randomized SVD
# ============================================================

@torch.no_grad()
def rsvd_top_uv(A: torch.Tensor, rank: int, oversample: int, n_iter: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Randomized SVD: return (U[:, :rank], V[:, :rank]) for matrix A (n x n).
    """
    n = A.shape[0]
    r = min(rank, n)
    p = max(0, oversample)
    k = min(n, r + p)
    G = torch.randn(n, k, dtype=DTYPE_ACC, device=A.device)
    Y = A @ G
    for _ in range(max(0, n_iter)):
        Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Ub, _, Vhb = torch.linalg.svd(B, full_matrices=False)
    U = (Q @ Ub)[:, :r].contiguous()
    V = Vhb.t()[:, :r].contiguous()
    return U, V

@torch.no_grad()
def top_uv(A: torch.Tensor, rank: int) -> Tuple[torch.Tensor, torch.Tensor]:
    n = A.shape[0]
    r = min(rank, n)
    if cfg.RES_SVD_MODE == "svd":
        U, _, Vh = torch.linalg.svd(A, full_matrices=False)
        return U[:, :r].contiguous(), Vh.t()[:, :r].contiguous()
    # default: randomized
    return rsvd_top_uv(A, r, cfg.RSVD_OVERSAMPLE, cfg.RSVD_ITERS)

# ============================================================
# Core selection + greedy residual blocks (block grid)
# ============================================================

@torch.no_grad()
def core_blocks_perexpert(X: torch.Tensor, block: int, target: float, max_blocks: int, diag_bonus: float) -> Tuple[List[Tuple[int,int,torch.Tensor]], float]:
    """
    Select blocktopk blocks for ONE expert, return list of (i0,j0,B) and achieved energy fraction.
    """
    Eblk, nb, n0 = block_energy_grid(X, block)
    blocks = pick_top_blocks_from_energy(Eblk, target=target, max_blocks=max_blocks, diag_bonus=diag_bonus)
    slices = blocks_to_slices(blocks, block, n0)
    tot = float((X * X).sum().item()) + 1e-12
    acc = 0.0
    out = []
    for (i0, i1, j0, j1) in slices:
        B = X[i0:i1, j0:j1].clone().to(torch.float16)
        out.append((i0, j0, B))
        acc += float((X[i0:i1, j0:j1] ** 2).sum().item())
    frac = acc / tot
    return out, frac

@torch.no_grad()
def subtract_core(X: torch.Tensor, core: List[Tuple[int,int,torch.Tensor]]) -> torch.Tensor:
    if not core:
        return X
    R = X.clone()
    for (i0, j0, B) in core:
        h, w = B.shape
        R[i0:i0+h, j0:j0+w] -= B.to(DTYPE_ACC)
    return R

@torch.no_grad()
def greedy_blocks_from_residual(R: torch.Tensor, block: int, max_blocks: int) -> List[Tuple[int,int,torch.Tensor]]:
    """
    Pick top energy blocks on the block grid (can overlap in rows/cols, but unique (bi,bj)).
    """
    Eblk, nb, n0 = block_energy_grid(R, block)
    flat = Eblk.reshape(-1)
    order = torch.argsort(flat, descending=True)
    out = []
    used = set()
    for idx in order.tolist():
        if len(out) >= max_blocks:
            break
        bi = idx // nb
        bj = idx - bi * nb
        if (bi, bj) in used:
            continue
        i0 = bi * block
        j0 = bj * block
        i1 = min(i0 + block, n0)
        j1 = min(j0 + block, n0)
        if i0 >= n0 or j0 >= n0 or i1 <= i0 or j1 <= j0:
            continue
        B = R[i0:i1, j0:j1].clone().to(torch.float16)
        # skip near-zero blocks
        if float((B.float() * B.float()).sum().item()) < 1e-12:
            continue
        out.append((i0, j0, B))
        used.add((bi, bj))
    return out

# ============================================================
# Build payload for one basis/cluster (two-pass to reduce memory)
# ============================================================

@torch.no_grad()
def build_payload_for_cluster(Ws: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor) -> dict:
    """
    Returns a payload:
      - U,V
      - per-expert balancing vectors d (optional)
      - core blocks per expert
      - low-rank residual dictionary DL,DR and per-expert gam
      - greedy residual blocks per expert
    """
    n = U.shape[0]
    E = len(idx)
    b = cfg.CORE_BLOCK
    # pass 1: choose per-expert core, accumulate mean residual
    core_blocks = []
    bal_d = []

    Rsum = torch.zeros(n, n, dtype=DTYPE_ACC)

    core_energy_fracs = []
    for e in idx:
        X = (U.t() @ Ws[e] @ V).to(DTYPE_ACC)
        if cfg.BALANCE_DIAG:
            Xb, d = balance_diag_l2(X, iters=cfg.BALANCE_ITERS, eps=cfg.BALANCE_EPS)
        else:
            Xb, d = X, torch.ones(n, dtype=DTYPE_ACC)

        cb, frac = core_blocks_perexpert(
            Xb, block=cfg.CORE_BLOCK, target=cfg.CORE_TARGET,
            max_blocks=cfg.CORE_MAX_BLOCKS, diag_bonus=cfg.DIAG_BONUS
        )
        core_blocks.append(cb)
        bal_d.append(d.to(torch.float16))
        core_energy_fracs.append(frac)

        R = subtract_core(Xb, cb)
        Rsum += R

    Rmean = (Rsum / max(1, E)).contiguous()
    r = min(cfg.RES_RANK, n)
    DL, DR = top_uv(Rmean, rank=r)
    DL = DL.to(torch.float16)
    DR = DR.to(torch.float16)

    # pass 2: compute gam and greedy blocks from residual after low-rank subtraction
    gam = torch.empty(E, r, dtype=torch.float16)
    res_blocks = []

    DLf = DL.to(DTYPE_ACC)
    DRf = DR.to(DTYPE_ACC)

    for j, e in enumerate(idx):
        X = (U.t() @ Ws[e] @ V).to(DTYPE_ACC)
        if cfg.BALANCE_DIAG:
            d = bal_d[j].to(DTYPE_ACC)
            Xb = (X / d.view(-1, 1)) * d.view(1, -1)
        else:
            Xb = X

        R0 = subtract_core(Xb, core_blocks[j])

        # gam = diag( DL^T R DR ) implemented as sum(DL * (R @ DR)) over rows
        g = torch.sum(DLf * (R0 @ DRf), dim=0)  # (r,)
        gam[j] = g.to(torch.float16)

        # subtract low-rank approx
        R1 = (R0 - (DLf * g.view(1, -1)) @ DRf.t()).contiguous()
        blks = greedy_blocks_from_residual(R1, block=cfg.RES_BSIZE, max_blocks=cfg.RES_BLOCKS)
        res_blocks.append(blks)

    core_mean = float(np.mean(core_energy_fracs)) if core_energy_fracs else 0.0
    core_min = float(np.min(core_energy_fracs)) if core_energy_fracs else 0.0
    core_blk_mean = float(np.mean([len(x) for x in core_blocks])) if core_blocks else 0.0

    return {
        "U": U.to(torch.float16),
        "V": V.to(torch.float16),
        "idx": idx,
        "core_blocks": core_blocks,
        "core_energy_mean": core_mean,
        "core_energy_min": core_min,
        "core_blocks_mean": core_blk_mean,
        "bal_d": bal_d,  # list of float16 vectors
        "DL": DL, "DR": DR,
        "gam": gam,      # (E, r) float16
        "blocks": res_blocks,
    }

# ============================================================
# Runtime
# ============================================================

class KTXRuntime(nn.Module):
    def __init__(self, payloads: List[Optional[dict]], basis_of_e: Dict[int, Tuple[int,int]]):
        super().__init__()
        self.payloads = payloads
        self.map = basis_of_e
        self.n = None
        for p in payloads:
            if p is not None:
                self.n = int(p["U"].shape[0])
                break
        if self.n is None:
            raise RuntimeError("No payloads provided.")

    @torch.no_grad()
    def forward(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        used: Dict[int, List[Tuple[float,int]]] = {}
        for a, e in zip(gates.tolist(), routed):
            if e in self.map:
                m, j = self.map[e]
                used.setdefault(m, []).append((float(a), int(j)))

        for m, items in used.items():
            P = self.payloads[m]
            if P is None:
                continue

            U = P["U"].to(DTYPE_ACC)
            V = P["V"].to(DTYPE_ACC)
            DL = P["DL"].to(DTYPE_ACC)
            DR = P["DR"].to(DTYPE_ACC)

            z = x @ U  # (B,n)
            u_acc = torch.zeros_like(z)

            for (a, j) in items:
                d = P["bal_d"][j].to(DTYPE_ACC)
                z1 = z / d.view(1, -1) if cfg.BALANCE_DIAG else z

                u = torch.zeros_like(z)

                # core
                for (i0, j0, Bc16) in P["core_blocks"][j]:
                    Bc = Bc16.to(DTYPE_ACC)
                    h, w = Bc.shape
                    u[:, j0:j0+w] += z1[:, i0:i0+h] @ Bc

                # low-rank
                g = P["gam"][j].to(DTYPE_ACC)  # (r,)
                u += ((z1 @ DL) * g.view(1, -1)) @ DR.t()

                # residual blocks
                for (i0, j0, Bb16) in P["blocks"][j]:
                    Bb = Bb16.to(DTYPE_ACC)
                    h, w = Bb.shape
                    u[:, j0:j0+w] += z1[:, i0:i0+h] @ Bb

                if cfg.BALANCE_DIAG:
                    u = u * d.view(1, -1)

                u_acc += a * u

            y += u_acc @ V.t()

        return y

# ============================================================
# Eval helpers
# ============================================================

@torch.no_grad()
def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def dense_apply(Ws: torch.Tensor, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
    n = Ws.shape[-1]
    Wsum = torch.zeros(n, n, dtype=DTYPE_ACC)
    for a, e in zip(gates, routed):
        Wsum += float(a.item()) * Ws[e]
    return x @ Wsum

@torch.no_grad()
def eval_runtime(rt: KTXRuntime, Ws: torch.Tensor, trials: int, routed_k: int, batch: int):
    E = Ws.shape[0]
    errs = []
    for _ in range(trials):
        x = torch.randn(batch, rt.n, dtype=DTYPE_ACC)
        routed = random.sample(range(E), k=min(routed_k, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC)
        gates = gates / gates.sum().clamp_min(1e-12)
        y_hat = rt(x, routed, gates)
        y_ref = dense_apply(Ws, x, routed, gates)
        err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)
    log(f"[eval] forward rel-error = {float(np.mean(errs)):.4f} ± {float(np.std(errs)):.4f}  (trials={trials})")

# ============================================================
# Training objective (fully differentiable)
# ============================================================

def sample_block_indices(n: int, block: int, n_blocks: int) -> torch.Tensor:
    """
    Sample block indices (contiguous block ranges), return a 1D index tensor S.
    """
    nb = (n + block - 1) // block
    pick = torch.randperm(nb)[:max(1, min(n_blocks, nb))]
    idxs = []
    for bi in pick.tolist():
        i0 = bi * block
        i1 = min(i0 + block, n)
        idxs.extend(list(range(i0, i1)))
    return torch.tensor(idxs, dtype=torch.long)

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    """
    Xs[e] = U_S^T Ws[e] V_S, where U_S=U[:,S], V_S=V[:,S]
    Returns (Eb, s, s)
    """
    U_S = U[:, S]            # (n,s)
    V_S = V[:, S]            # (n,s)
    T = Ws_batch @ V_S       # (Eb,n,s)
    # (Eb,s,n) @ (Eb,n,s) -> (Eb,s,s)
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

def diag_mix_loss(Xs: torch.Tensor, k: int) -> torch.Tensor:
    """
    Novel: encourage within-cluster alignment by reducing variance of diag entries
    across experts for a small subsample of diag coords.
    Xs: (Eb,s,s)
    """
    D = torch.diagonal(Xs, dim1=1, dim2=2)  # (Eb,s)
    s = D.shape[1]
    if k <= 0 or s <= 1:
        return torch.zeros((), dtype=D.dtype, device=D.device)
    kk = min(k, s)
    cols = torch.randperm(s, device=D.device)[:kk]
    return D[:, cols].std(dim=0).mean()

def block_entropy_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    """
    Differentiable block concentration penalty: entropy of block energies (lower is better).
    Works on the *mean* matrix over experts in the batch.
    """
    Xm = Xs.mean(dim=0)  # (s,s)
    s = Xm.shape[0]
    b = min(block, s)
    # pad to b
    nb = (s + b - 1) // b
    pad = nb * b - s
    if pad:
        Z = torch.zeros(nb*b, nb*b, dtype=Xm.dtype, device=Xm.device)
        Z[:s, :s] = Xm
        Xm = Z
        s = nb * b
    G = Xm.view(nb, b, nb, b)
    E = (G * G).sum(dim=(1, 3)).reshape(-1)  # (nb*nb,)
    E = E.clamp_min(1e-12)
    p = E / E.sum()
    H = -(p * torch.log(p)).sum()
    # normalize by log(num_blocks) to keep scale ~[0,1]
    H = H / math.log(float(p.numel()) + 1e-12)
    return H

def diag_block_guide(Xs: torch.Tensor, block: int, diag_bonus: float) -> torch.Tensor:
    """
    Differentiable guide: maximize fraction of energy in diagonal blocks (coarse).
    """
    Xm = Xs.mean(dim=0)
    s = Xm.shape[0]
    b = min(block, s)
    nb = (s + b - 1) // b
    pad = nb * b - s
    if pad:
        Z = torch.zeros(nb*b, nb*b, dtype=Xm.dtype, device=Xm.device)
        Z[:s, :s] = Xm
        Xm = Z
        s = nb * b
    G = Xm.view(nb, b, nb, b)
    Eblk = (G * G).sum(dim=(1, 3))  # (nb,nb)
    tot = Eblk.sum().clamp_min(1e-12)
    diag = torch.diagonal(Eblk, 0).sum()
    if diag_bonus != 0.0:
        diag = diag * (1.0 + diag_bonus)
    frac = diag / tot
    # want frac high -> penalty (1-frac)
    return (1.0 - frac)

# ============================================================
# Main
# ============================================================

def banner():
    log("== DeepSeek KT++-X Optimized (offline) v7.2 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {(cfg.CALIB_PATH if cfg.CALIB_PATH else '(none)')}  CALIB_SAMPLES={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {(cfg.ROUTER_PATH if cfg.ROUTER_PATH else '(none)')}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  (RIDGE_DAMP={cfg.RIDGE_DAMP})")
    log(f"CORE_MODE:   {cfg.CORE_MODE}  block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"SERIATION:   {cfg.SERIATION} (diag_bonus={cfg.DIAG_BONUS})")
    log(f"BALANCE:     {cfg.BALANCE_DIAG} (iters={cfg.BALANCE_ITERS})")
    log(f"RESIDUAL:    rank={cfg.RES_RANK} blocks={cfg.RES_BLOCKS} bsize={cfg.RES_BSIZE} svd={cfg.RES_SVD_MODE}")
    if cfg.RES_SVD_MODE == "rand":
        log(f"  rSVD: oversample={cfg.RSVD_OVERSAMPLE} iters={cfg.RSVD_ITERS}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} warmup={cfg.WARMUP_STEPS} lr={cfg.LR_UV} batchE={cfg.BATCH_E} subm_blocks={cfg.SUBM_BLOCKS}")
    log(f"TRAIN_OBJ:   {cfg.TRAIN_OBJ} lam_block={cfg.TRAIN_LAM_BLOCK} lam_guide={cfg.TRAIN_LAM_GUIDE} guide_every={cfg.TRAIN_GUIDE_EVERY} mix_lam={cfg.TRAIN_MIX_LAM} mix_k={cfg.TRAIN_MIX_K}")
    log(f"CLUSTER:     M0={(cfg.M0 if cfg.M0>0 else '(auto)')} M_MAX={cfg.M_MAX} max_size={cfg.CLUSTER_MAX_SIZE} min_size={cfg.CLUSTER_MIN_SIZE} iters={cfg.KMEANS_ITERS} feat_d={cfg.FEAT_D}")
    log(f"Hadamard:    {cfg.USE_HADAMARD} seed={cfg.HAD_SEED}")
    log(f"Threads:     {NTHREADS} Torch={torch.__version__} device={DEVICE}")
    log(f"Cache:       {cfg.CACHE_MODE} path={cache_path()} force_rebuild={cfg.FORCE_REBUILD_WS}")
    log("")

def main():
    banner()

    eids, Ws = load_or_build_Ws()
    Ws = Ws.to(DTYPE_ACC).contiguous()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} experts={eids[:8]}{'...' if len(eids)>8 else ''}")

    # initial cluster count
    if cfg.M0 > 0:
        M0 = min(cfg.M0, E)
    else:
        # heuristic: start moderate
        M0 = min(8, max(2, E // 2))
    log(f"[cluster] M0={M0}")

    Xfeat = random_proj_features(Ws, d=cfg.FEAT_D)
    labels, centers = kmeans_torch(Xfeat, k=M0, iters=cfg.KMEANS_ITERS)
    labels = enforce_min_cluster_size(labels, centers, Xfeat, min_size=cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xfeat, labels, max_size=cfg.CLUSTER_MAX_SIZE, max_k=min(cfg.M_MAX, E), split_iters=20)

    M = int(labels.max().item()) + 1
    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    log(f"[cluster] M={M}")
    log(f"[cluster] sizes: {[len(c) for c in clusters]}")

    # init U,V per cluster
    U_par: List[Optional[OrthoParam]] = []
    V_par: List[Optional[OrthoParam]] = []
    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            U_par.append(None); V_par.append(None)
            continue
        Wm = Ws[idx].mean(dim=0)
        U0, V0 = svd_init(Wm)
        U_par.append(OrthoParam(U0))
        V_par.append(OrthoParam(V0))

    # ===========================
    # TRAIN (force grad enabled)
    # ===========================
    if cfg.TRAIN_STEPS > 0:
        params = [uv.M for uv in (U_par + V_par) if uv is not None]
        opt = torch.optim.Adam(params, lr=cfg.LR_UV)

        t0 = time.perf_counter()

        # Force autograd ON even if user environment has disabled grads
        with torch.enable_grad():
            torch.set_grad_enabled(True)

            for step in range(1, cfg.TRAIN_STEPS + 1):
                # sample block-index subset (much cheaper + aligns with block objectives)
                S = sample_block_indices(n=n, block=cfg.CORE_BLOCK, n_blocks=cfg.SUBM_BLOCKS).to(torch.long)

                L_total = None
                guide_energy = 0.0

                for m, idx in enumerate(clusters):
                    if len(idx) == 0:
                        continue

                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()

                    # sample experts
                    if cfg.BATCH_E > 0 and cfg.BATCH_E < len(idx):
                        pick = torch.randperm(len(idx))[:cfg.BATCH_E].tolist()
                        idx_step = [idx[p] for p in pick]
                    else:
                        idx_step = idx

                    Ws_batch = Ws[idx_step]  # constant, but graph flows through Uo/Vo

                    Xs = slice_X_batch(Ws_batch, Uo, Vo, S)  # (Eb,s,s)

                    off = offdiag_abs_mean(Xs)
                    diag = diag_abs_mean(Xs).clamp_min(1e-8)

                    if cfg.TRAIN_OBJ == "ratio":
                        loss_main = off / diag
                    else:
                        # logratio
                        loss_main = torch.log(off.clamp_min(1e-12)) - torch.log(diag)

                    # block entropy penalty (encourage concentration)
                    lam_block = 0.0
                    if step > cfg.WARMUP_STEPS:
                        # linear ramp after warmup
                        lam_block = cfg.TRAIN_LAM_BLOCK * float(step - cfg.WARMUP_STEPS) / float(max(1, cfg.TRAIN_STEPS - cfg.WARMUP_STEPS))
                    Lb = block_entropy_penalty(Xs, block=min(cfg.CORE_BLOCK, Xs.shape[-1]))

                    # guide (every few steps) to encourage diag-block energy
                    lam_guide = 0.0
                    Lg = torch.zeros((), dtype=DTYPE_ACC, device=loss_main.device)
                    if (step % cfg.TRAIN_GUIDE_EVERY) == 0 and step > cfg.WARMUP_STEPS:
                        lam_guide = cfg.TRAIN_LAM_GUIDE * float(step - cfg.WARMUP_STEPS) / float(max(1, cfg.TRAIN_STEPS - cfg.WARMUP_STEPS))
                        Lg = diag_block_guide(Xs, block=min(cfg.CORE_BLOCK, Xs.shape[-1]), diag_bonus=cfg.DIAG_BONUS)
                        guide_energy += float((1.0 - float(Lg.detach().item())))

                    # mixing loss (novel): align diagonals among experts in the cluster
                    Lmix = torch.zeros((), dtype=DTYPE_ACC, device=loss_main.device)
                    if cfg.TRAIN_MIX_LAM > 0 and Xs.shape[0] >= 2:
                        Lmix = diag_mix_loss(Xs, k=cfg.TRAIN_MIX_K)

                    loss = loss_main + lam_block * Lb + lam_guide * Lg + cfg.TRAIN_MIX_LAM * Lmix

                    L_total = loss if (L_total is None) else (L_total + loss)

                if L_total is None:
                    log("[train] no clusters? skipping training.")
                    break

                opt.zero_grad(set_to_none=True)
                L_total.backward()

                # re-orthogonalize explicitly (keeps params stable)
                if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    with torch.no_grad():
                        for uv in (U_par + V_par):
                            if uv is not None:
                                uv.M.copy_(uv.orthogonal())

                opt.step()

                if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    t1 = time.perf_counter()
                    ge = guide_energy / max(1, sum(1 for _ in clusters)) if guide_energy > 0 else 0.0
                    log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} | loss={float(L_total.detach().item()):.4f}  guide≈{ge:.3f} (+{t1-t0:.1f}s)")
                    t0 = t1

    # ===========================
    # Build payloads (with optional seriation)
    # ===========================
    payloads: List[Optional[dict]] = []
    basis_of_e: Dict[int, Tuple[int,int]] = {}
    log("[build] payloads ...")

    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            payloads.append(None)
            continue
        Uo = U_par[m].orthogonal().detach()
        Vo = V_par[m].orthogonal().detach()

        if cfg.SERIATION:
            pi = seriation_perm_for_basis(
                Ws=Ws, idx=idx, U=Uo, V=Vo, b=cfg.CORE_BLOCK,
                diag_bonus=cfg.DIAG_BONUS, topk_nei=cfg.SERIATION_TOPK_NEI
            )
            Uo = Uo[:, pi].contiguous()
            Vo = Vo[:, pi].contiguous()

        P = build_payload_for_cluster(Ws, idx, Uo, Vo)
        payloads.append(P)

        for j, e in enumerate(idx):
            basis_of_e[e] = (m, j)

        log(
            f"  - basis {m}: E={len(idx)} core={cfg.CORE_MODE} "
            f"core_energy(mean/min)≈{P['core_energy_mean']:.3f}/{P['core_energy_min']:.3f} "
            f"core_blocks(mean)≈{P['core_blocks_mean']:.1f} "
            f"rank={P['DL'].shape[1]} res_blocks={cfg.RES_BLOCKS}"
        )

    rt = KTXRuntime(payloads, basis_of_e)
    eval_runtime(rt, Ws, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    log("\n✅ Done.")
    log("Next levers (highest ROI):")
    log(" - Use real CALIB_PATH (true layer inputs) for LIN_MODE=ridge.")
    log(" - If you have ROUTER_PATH, set RIDGE_WEIGHTED=1 and FORCE_REBUILD_WS=1.")
    log(" - If core_blocks(mean) is pegged at CORE_MAX_BLOCKS with low core_energy: increase M_MAX / lower CLUSTER_MAX_SIZE / change CORE_BLOCK.")
    log(" - If CPU time too high: RES_SVD_MODE=rand (default) and reduce RES_RANK first (e.g., 512).")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X Optimized (offline) v7.2 ==
Time:        2026-01-01 17:39:38
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES=512
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
LIN_MODE:    ridge  (RIDGE_DAMP=0.001)
CORE_MODE:   blocktopk_perexpert  block=64 target=0.8 max_blocks=256
SERIATION:   True (diag_bonus=0.05)
BALANCE:     True (iters=1)
RESIDUAL:    rank=1024 blocks=128 bsize=64 svd=rand
  rSVD: oversample=32 iters=1
TRAIN:       steps=24 warmup=6 lr=0.05 batchE=4 subm_blocks=4
TRAIN_OBJ:   logratio lam_block=0.1 lam_guide=1.0 guide_every=2 mix_lam=0.15 mix_k=6
CLUSTER:     M0=(auto) M_MAX=16 max_size=3 min_size=1 iters=60 feat_d=64
Hadamard:    False seed=1234
Threads:     8 Torch=2.4.1+cpu device=cpu
Cache:       readwrite path=/home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v7_2.npz force_rebuild=False

[found] layer 1 experts: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 

Build Ws:   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote: /home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v7_2.npz  (268.44 MB)
[Ws] shape=(16, 2048, 2048) experts=[0, 1, 2, 3, 4, 5, 6, 7]...
[cluster] M0=8
[cluster] M=10
[cluster] sizes: [2, 1, 1, 1, 1, 1, 1, 2, 3, 3]
[train] step   1/24 | loss=-103.6701  guide≈0.000 (+9.3s)
[train] step   4/24 | loss=-3.5203  guide≈0.000 (+30.0s)
[train] step   8/24 | loss=-11.4434  guide≈0.325 (+38.4s)
[train] step  12/24 | loss=-21.2216  guide≈0.799 (+38.0s)
[train] step  16/24 | loss=-9.9240  guide≈0.498 (+37.8s)
[train] step  20/24 | loss=-16.2158  guide≈0.750 (+37.8s)
[train] step  24/24 | loss=-13.9165  guide≈0.607 (+37.8s)
[build] payloads ...
  - basis 0: E=2 core=blocktopk_perexpert core_energy(mean/min)≈0.566/0.564 core_blocks(mean)≈256.0 rank=1024 res_blocks=128
  - basis 1: E=1 core=blocktopk_perexpert core_energy(mean/min)≈0.801/0.801 core_blocks(mean)≈57.0 rank=1024 res_blocks=128
  - basis 2: E=1 core=blocktopk_perexpert core_energy(mean/min)≈0.595/0.595 core_bl

In [22]:
#!/usr/bin/env python3
"""
EQuAL-LLM+ (Dynamic): Extreme Quantization with Adaptive Layers & Dynamic Payload Modulation.
This version extends the static EQuAL-LLM with a lightweight, trainable memory module.
The memory key is a function of the routed expert IDs, and the value modulates the
de-quantized payload, allowing runtime adaptation.
"""
import os, math, time, random
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Any
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------ Configuration ------------------------
@dataclass
class DynamicConfig:
    # --- Core EQuAL-LLM Config (from your setup) ---
    model_dim: int = 2048
    num_experts: int = 16
    num_clusters: int = 8
    payload_bits: Tuple[int, int, int] = (4, 4, 3)  # (diag, low-rank, blocks)
    # --- Dynamic Modulation Config ---
    use_dynamic_modulation: bool = True
    mem_key_dim: int = 64          # Dimension for routing-based key
    mem_value_dim: int = 32        # Dimension of modulation vector
    mem_num_slots: int = 128       # Capacity of memory
    mem_temp: float = 0.1          # Softmax temperature for addressing
    # How modulation is applied: 'scale' (per-expert), 'shift', 'scale_shift'
    modulation_mode: str = 'scale_shift'
    # If True, key is hash of expert IDs; if False, learned embedding of IDs
    key_from_routing_hash: bool = True

cfg = DynamicConfig()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(1234)

# ------------------------ Dynamic Modulation Memory ------------------------
class RoutingAwareMemory(nn.Module):
    """
    A lightweight memory that generates a modulation vector based on routed experts.
    Key Idea: The combination of activated experts (routing pattern) may hint at
    the type of token or task, allowing specialized payload adjustment.
    """
    def __init__(self, config: DynamicConfig):
        super().__init__()
        self.cfg = config
        # Memory slots: values are modulation vectors
        self.mem_slots = nn.Parameter(torch.randn(config.mem_num_slots, config.mem_value_dim) * 0.02)
        # If not using hash, learn an embedding for each expert
        if not config.key_from_routing_hash:
            self.expert_embed = nn.Embedding(config.num_experts, config.mem_key_dim)
        # Projection to generate query from potentially embedded expert IDs
        self.query_proj = nn.Linear(config.mem_key_dim, config.mem_key_dim, bias=False)
        # Linear layer to transform retrieved memory to final modulation parameters
        # Output dim depends on mode: 'scale'(val_dim), 'shift'(val_dim), 'scale_shift'(2*val_dim)
        out_dim = config.mem_value_dim
        if config.modulation_mode == 'scale_shift':
            out_dim = 2 * config.mem_value_dim
        self.modulation_proj = nn.Linear(config.mem_value_dim, out_dim)

    def _expert_set_to_key(self, expert_ids: List[int]) -> torch.Tensor:
        """Convert a set of expert IDs to a fixed-size key vector."""
        if self.cfg.key_from_routing_hash:
            # Deterministic hash-based key (simpler, no learned params)
            key_vec = torch.zeros(self.cfg.mem_key_dim, device=DEVICE)
            for eid in expert_ids:
                # Simple hash: use expert ID to seed a random vector, sum
                g = torch.Generator(device='cpu').manual_seed(int(eid) % 10000)
                vec = torch.randn(self.cfg.mem_key_dim, generator=g, device=DEVICE)
                key_vec += vec
            key_vec = key_vec / (len(expert_ids) ** 0.5)
            return key_vec.unsqueeze(0)  # (1, key_dim)
        else:
            # Learned embedding-based key
            embeds = self.expert_embed(torch.tensor(expert_ids, device=DEVICE))  # (K, key_dim)
            return embeds.mean(dim=0, keepdim=True)  # (1, key_dim)

    def forward(self, routed_experts: List[int]) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
        """
        Args:
            routed_experts: list of expert IDs activated for the current token/batch.
        Returns:
            scale_mod: modulation vector for scaling (or None)
            shift_mod: modulation vector for shifting (or None)
        """
        if not self.cfg.use_dynamic_modulation or len(routed_experts) == 0:
            return None, None

        # 1. Create query from routed expert set
        query = self._expert_set_to_key(routed_experts)  # (1, key_dim)
        query = self.query_proj(query)                    # (1, key_dim)

        # 2. Soft-address memory slots based on similarity
        #    Using simple dot-product attention
        attn_logits = torch.matmul(query, self.mem_slots.t()) / (self.cfg.mem_key_dim ** 0.5)  # (1, num_slots)
        attn_weights = F.softmax(attn_logits / self.cfg.mem_temp, dim=-1)  # (1, num_slots)

        # 3. Retrieve modulation basis vector
        retrieved = torch.matmul(attn_weights, self.mem_slots)  # (1, value_dim)
        modulation = self.modulation_proj(retrieved).squeeze(0)  # (out_dim,)

        # 4. Split into scale and shift if needed
        scale_mod, shift_mod = None, None
        if self.cfg.modulation_mode == 'scale':
            scale_mod = torch.sigmoid(modulation)  # in (0,1)
        elif self.cfg.modulation_mode == 'shift':
            shift_mod = torch.tanh(modulation) * 0.1  # small shift
        elif self.cfg.modulation_mode == 'scale_shift':
            half = self.cfg.mem_value_dim
            scale_mod = torch.sigmoid(modulation[:half])
            shift_mod = torch.tanh(modulation[half:]) * 0.1
        return scale_mod, shift_mod

# ------------------------ Quantized Payload with Dynamic Modulation ------------------------
class QuantizedDynamicPayload(nn.Module):
    """
    Represents the quantized payload of one expert within a cluster,
    now with optional dynamic modulation.
    """
    def __init__(self, payload_dim: int, rank: int, num_blocks: int, block_size: int, config: DynamicConfig):
        super().__init__()
        self.cfg = config
        self.payload_dim = payload_dim
        # --- Core quantized parameters (from original EQuAL-LLM) ---
        # Diagonal (quantized)
        self.diag_val = nn.Parameter(torch.randn(payload_dim), requires_grad=False)
        self.diag_scale = nn.Parameter(torch.tensor(1.0), requires_grad=False)
        # Low-rank factors (quantized)
        self.lowrank_left = nn.Parameter(torch.randn(payload_dim, rank), requires_grad=False)
        self.lowrank_right = nn.Parameter(torch.randn(rank, payload_dim), requires_grad=False)
        self.lr_scale = nn.Parameter(torch.tensor(1.0), requires_grad=False)
        # Sparse blocks (quantized)
        self.block_coords = []  # list of (i, j)
        self.block_vals = []    # list of tensors
        for _ in range(num_blocks):
            i = random.randint(0, payload_dim - block_size)
            j = random.randint(0, payload_dim - block_size)
            self.block_coords.append((i, j))
            self.block_vals.append(nn.Parameter(torch.randn(block_size, block_size), requires_grad=False))
        self.block_scales = nn.Parameter(torch.ones(num_blocks), requires_grad=False)

        # --- Dynamic modulation parameters ---
        self.enable_dynamic = config.use_dynamic_modulation
        if self.enable_dynamic:
            # Projection to mix dynamic modulation into the payload dimensions
            self.dynamic_proj_scale = nn.Linear(config.mem_value_dim, payload_dim, bias=False)
            self.dynamic_proj_shift = nn.Linear(config.mem_value_dim, payload_dim, bias=False)
            # Learnable gating to blend static and dynamic
            self.dynamic_gate = nn.Parameter(torch.tensor(0.5))

    def apply_dynamic_modulation(self, scale_mod: Optional[torch.Tensor], shift_mod: Optional[torch.Tensor]):
        """
        Modifies the dequantized payload on-the-fly.
        This is a simplified illustration: in practice, modulation might apply
        differently to diag, low-rank, and blocks.
        """
        if not self.enable_dynamic or (scale_mod is None and shift_mod is None):
            return

        # Project the compact modulation vector to the payload dimension
        scale_full, shift_full = None, None
        if scale_mod is not None:
            scale_full = torch.sigmoid(self.dynamic_proj_scale(scale_mod.unsqueeze(0)))  # (1, payload_dim)
        if shift_mod is not None:
            shift_full = self.dynamic_proj_shift(shift_mod.unsqueeze(0))  # (1, payload_dim)

        # Store for use during forward pass
        self._current_scale = scale_full
        self._current_shift = shift_full

    def forward(self):
        """Reconstruct the approximated matrix with optional dynamic modulation."""
        # 1. Dequantize core payload (simplified, without actual quant/dequant ops)
        out = torch.diag(self.diag_val * self.diag_scale)
        out = out + self.lowrank_left @ (self.lowrank_right * self.lr_scale)
        for idx, (i, j) in enumerate(self.block_coords):
            bsz = self.block_vals[idx].shape[0]
            out[i:i+bsz, j:j+bsz] += self.block_vals[idx] * self.block_scales[idx]

        # 2. Apply dynamic modulation if available
        if self.enable_dynamic and hasattr(self, '_current_scale'):
            gate = torch.sigmoid(self.dynamic_gate)
            if self._current_scale is not None:
                out = out * (1 + gate * (self._current_scale - 1))
            if self._current_shift is not None:
                out = out + gate * self._current_shift
        return out

# ------------------------ Main Wrapper ------------------------
class EQualLLMDynamic(nn.Module):
    """
    Top-level module integrating cluster-shared rotations, quantized payloads,
    and dynamic routing-aware memory.
    """
    def __init__(self, config: DynamicConfig):
        super().__init__()
        self.cfg = config
        # Shared orthogonal bases per cluster
        self.cluster_bases_U = nn.ParameterList()
        self.cluster_bases_V = nn.ParameterList()
        for _ in range(config.num_clusters):
            U = torch.qr(torch.randn(config.model_dim, config.model_dim, device=DEVICE))[0]
            V = torch.qr(torch.randn(config.model_dim, config.model_dim, device=DEVICE))[0]
            self.cluster_bases_U.append(nn.Parameter(U.detach(), requires_grad=True))
            self.cluster_bases_V.append(nn.Parameter(V.detach(), requires_grad=True))

        # Per-expert payloads (placeholder: in reality, map expert -> cluster -> payload)
        self.expert_payloads = nn.ModuleList()
        for _ in range(config.num_experts):
            self.expert_payloads.append(
                QuantizedDynamicPayload(config.model_dim, rank=256, num_blocks=8,
                                        block_size=64, config=config)
            )

        # Routing-aware memory shared across all experts
        self.routing_memory = RoutingAwareMemory(config)

    def forward(self, x: torch.Tensor, routed_experts: List[int], expert_weights: List[float]):
        """
        Args:
            x: input tensor (batch, model_dim)
            routed_experts: list of expert IDs being activated
            expert_weights: corresponding gating weights
        """
        # 1. Retrieve dynamic modulation based on routing pattern
        scale_mod, shift_mod = self.routing_memory(routed_experts)

        # 2. For each activated expert, apply modulation and compute contribution
        total_output = torch.zeros_like(x)
        for eid, weight in zip(routed_experts, expert_weights):
            payload = self.expert_payloads[eid]
            # Instruct the payload to adjust based on current context
            payload.apply_dynamic_modulation(scale_mod, shift_mod)
            # Reconstruct expert matrix in rotated space
            # (In full implementation, you'd use the cluster's U, V)
            W_approx = payload()  # (model_dim, model_dim)
            total_output += weight * (x @ W_approx)

        return total_output

# ------------------------ Training Loop Snippet ------------------------
def train_dynamic_adaptation_step(model: EQualLLMDynamic, batch_data, router_logits):
    """
    Illustrative training step that updates both the quantization parameters
    and the dynamic memory based on routing fidelity loss.
    """
    model.train()
    x, y_true = batch_data

    # Simulate router to get expert choices
    probs = F.softmax(router_logits, dim=-1)
    topk_vals, topk_ids = torch.topk(probs, k=2, dim=-1)

    # Forward with dynamic modulation
    routed_experts = topk_ids[0].tolist()
    expert_weights = topk_vals[0].tolist()
    y_pred = model(x, routed_experts, expert_weights)

    # Loss: fidelity of routed mixture + regularization for dynamic modulation
    fidelity_loss = F.mse_loss(y_pred, y_true)
    # Encourage the memory to produce moderate modulations (avoid extreme values)
    reg_loss = 0.0
    if model.cfg.use_dynamic_modulation:
        # Access a sample of memory slots
        mem_slots = model.routing_memory.mem_slots
        reg_loss = 0.001 * torch.mean(mem_slots ** 2)

    total_loss = fidelity_loss + reg_loss
    total_loss.backward()
    # ... optimizer step would follow ...
    return total_loss.item()

# ------------------------ Quick Test ------------------------
if __name__ == "__main__":
    print("Testing EQuAL-LLM+ (Dynamic) with config:")
    print(f"  Model Dim: {cfg.model_dim}, Experts: {cfg.num_experts}, Clusters: {cfg.num_clusters}")
    print(f"  Dynamic Modulation: {cfg.use_dynamic_modulation}, Mode: {cfg.modulation_mode}")

    model = EQualLLMDynamic(cfg).to(DEVICE)
    dummy_input = torch.randn(4, cfg.model_dim, device=DEVICE)
    dummy_routed = [0, 5, 10]  # example expert IDs
    dummy_weights = [0.5, 0.3, 0.2]

    output = model(dummy_input, dummy_routed, dummy_weights)
    print(f"  Output shape: {output.shape}")
    print("  ✅ Dynamic adaptation module integrated successfully.")
    print("\n  Key Novelty: The model now conditionally adjusts its quantized")
    print("  payload based on the combination of experts activated (routing pattern),")
    print("  moving beyond static one-size-fits-all compression.")

Testing EQuAL-LLM+ (Dynamic) with config:
  Model Dim: 2048, Experts: 16, Clusters: 8
  Dynamic Modulation: True, Mode: scale_shift


/tmp/ipykernel_2771869/2804265587.py:204: UserWarning: torch.qr is deprecated in favor of torch.linalg.qr and will be removed in a future PyTorch release.
The boolean parameter 'some' has been replaced with a string parameter 'mode'.
Q, R = torch.qr(A, some)
should be replaced with
Q, R = torch.linalg.qr(A, 'reduced' if some else 'complete') (Triggered internally at ../aten/src/ATen/native/BatchLinearAlgebra.cpp:2416.)
  U = torch.qr(torch.randn(config.model_dim, config.model_dim, device=DEVICE))[0]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x64 and 32x128)

In [2]:
# %% [markdown]
# # Routed-mixture RelErr evaluator for PTQ 5/5/5, PTQ 4/4/3, QAT-lite 4/4/3
# Expected saved batch dict keys:
#   x: [N, H]
#   topk_idx: [N, K] (int64)
#   topk_alpha: [N, K] (float)
#   W: [E, H, H]  (experts)
#
# Optional (only if you also want to evaluate attention/KV quant):
#   k_cache: [...]
#   v_cache: [...]
#
# Metric:
#   RelErr = ||ŷ - y||_F / ||y||_F  over all tokens in the batch

# %%
import math
import torch
from dataclasses import dataclass
from typing import Optional, Dict, Any

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

# %%
def rel_fro_error(y_hat: torch.Tensor, y: torch.Tensor, eps: float = 1e-12) -> float:
    num = torch.linalg.norm((y_hat - y).reshape(-1), ord=2)
    den = torch.linalg.norm(y.reshape(-1), ord=2).clamp_min(eps)
    return (num / den).item()

# %%
def quant_dequant_symmetric(
    x: torch.Tensor,
    n_bits: int,
    per_channel: bool = False,
    ch_axis: int = -1,
    clip: Optional[float] = None,
) -> torch.Tensor:
    """
    Simple symmetric uniform PTQ: round(x/scale) * scale with int range [-qmax, qmax]
    - per_channel=True: separate scale per channel on ch_axis
    - clip: optional absolute clip value before quant
    Returns dequantized float tensor (same dtype as input).
    """
    if n_bits >= 16:
        return x

    if clip is not None:
        x = x.clamp(-clip, clip)

    qmax = (2 ** (n_bits - 1)) - 1

    if per_channel:
        # Move channel axis to last for easy broadcasting
        x_perm = x.movedim(ch_axis, -1)
        # scale per channel from max abs
        max_abs = x_perm.abs().amax(dim=tuple(range(x_perm.ndim - 1)), keepdim=True).clamp_min(1e-12)
        scale = max_abs / qmax
        q = torch.round(x_perm / scale).clamp(-qmax, qmax)
        dq = (q * scale).movedim(-1, ch_axis)
        return dq.to(dtype=x.dtype)
    else:
        max_abs = x.abs().max().clamp_min(1e-12)
        scale = max_abs / qmax
        q = torch.round(x / scale).clamp(-qmax, qmax)
        dq = q * scale
        return dq.to(dtype=x.dtype)

# %%
def moe_mixture_output(
    x: torch.Tensor,          # [N, H]
    W: torch.Tensor,          # [E, H, H] or [E, H_in, H_out]
    topk_idx: torch.Tensor,   # [N, K]
    topk_alpha: torch.Tensor, # [N, K]
    W_layout: str = "EHH",    # "EHH" means [E,H,H] with y = x @ W[e]
) -> torch.Tensor:
    """
    Compute y = sum_k alpha[n,k] * (x[n] @ W[idx[n,k]])
    Returns y: [N, H] (or [N, H_out] if your W is not square)
    """
    assert x.ndim == 2
    N, H = x.shape
    K = topk_idx.shape[1]
    y_accum = None

    for k in range(K):
        e_idx = topk_idx[:, k]  # [N]
        alpha = topk_alpha[:, k].unsqueeze(1)  # [N,1]

        Wk = W.index_select(0, e_idx)  # [N, H, H] (if square)

        if W_layout == "EHH":
            # yk[n] = x[n] @ Wk[n]
            yk = torch.einsum("nh,nhm->nm", x, Wk)  # [N, H]
        else:
            # If your W is [E, H_in, H_out], then Wk: [N, H_in, H_out]
            yk = torch.einsum("ni,nio->no", x, Wk)  # [N, H_out]

        yk = yk * alpha
        y_accum = yk if y_accum is None else (y_accum + yk)

    return y_accum

# %%
@dataclass
class QuantConfig:
    name: str
    w_bits: int
    a_bits: int
    kv_bits: Optional[int] = None  # kept for naming parity; often not used in MoE FFN eval

# %%
def eval_config_ptq(
    batch: Dict[str, torch.Tensor],
    cfg: QuantConfig,
    per_channel_w: bool = True,
    per_channel_a: bool = False,
    w_ch_axis: int = -1,   # for W [E,H,H], per-channel on last dim is common
    a_ch_axis: int = -1,
    W_layout: str = "EHH",
) -> float:
    """
    PTQ baseline: quantize weights + activations with simple symmetric uniform quant.
    """
    x = batch["x"]
    W = batch["W"]
    topk_idx = batch["topk_idx"]
    topk_alpha = batch["topk_alpha"]

    # Teacher output (FP)
    y = moe_mixture_output(x, W, topk_idx, topk_alpha, W_layout=W_layout)

    # Quantized student
    x_q = quant_dequant_symmetric(x, cfg.a_bits, per_channel=per_channel_a, ch_axis=a_ch_axis)
    W_q = quant_dequant_symmetric(W, cfg.w_bits, per_channel=per_channel_w, ch_axis=w_ch_axis)

    y_hat = moe_mixture_output(x_q, W_q, topk_idx, topk_alpha, W_layout=W_layout)
    return rel_fro_error(y_hat, y)

# %%
# --- QAT-lite: learn a simple clipping value for x and W to minimize teacher mismatch on a calibration batch ---
# This is NOT full QAT; it's a small, quick calibration loop so you can justify "QAT-lite".

class QATLiteCalibrator:
    def __init__(self, W: torch.Tensor, w_bits: int, a_bits: int):
        self.W_fp = W.detach()
        self.w_bits = w_bits
        self.a_bits = a_bits

        # Learnable clip parameters (scalar). Start from max abs.
        self.w_clip = torch.tensor(float(W.abs().max().item()), device=W.device, dtype=torch.float32, requires_grad=True)
        self.a_clip = None  # set when x is known

    @staticmethod
    def _fake_quant(x: torch.Tensor, n_bits: int, clip: torch.Tensor) -> torch.Tensor:
        # clamp to [-clip, clip], then symmetric quant-dequant
        clipv = torch.clamp(clip, min=1e-6)
        x = x.clamp(-clipv, clipv)
        qmax = (2 ** (n_bits - 1)) - 1
        scale = clipv / qmax
        q = torch.round(x / scale).clamp(-qmax, qmax)
        return (q * scale).to(dtype=x.dtype)

    def run(self, batch: Dict[str, torch.Tensor], steps: int = 200, lr: float = 5e-2, W_layout: str = "EHH") -> Dict[str, float]:
        x = batch["x"]
        topk_idx = batch["topk_idx"]
        topk_alpha = batch["topk_alpha"]

        # init a_clip from x
        self.a_clip = torch.tensor(float(x.abs().max().item()), device=x.device, dtype=torch.float32, requires_grad=True)

        # teacher
        with torch.no_grad():
            y = moe_mixture_output(x, self.W_fp, topk_idx, topk_alpha, W_layout=W_layout)

        # optimize clips
        opt = torch.optim.Adam([self.w_clip, self.a_clip], lr=lr)
        torch.set_grad_enabled(True)
        for t in range(steps):
            opt.zero_grad(set_to_none=True)

            W_q = self._fake_quant(self.W_fp, self.w_bits, self.w_clip)
            x_q = self._fake_quant(x, self.a_bits, self.a_clip)
            y_hat = moe_mixture_output(x_q, W_q, topk_idx, topk_alpha, W_layout=W_layout)

            # minimize MSE to teacher
            loss = torch.mean((y_hat - y) ** 2)
            loss.backward()
            opt.step()

            # keep clips reasonable
            with torch.no_grad():
                self.w_clip.clamp_(min=1e-6)
                self.a_clip.clamp_(min=1e-6)

        torch.set_grad_enabled(False)

        # final RelErr
        with torch.no_grad():
            W_q = self._fake_quant(self.W_fp, self.w_bits, self.w_clip)
            x_q = self._fake_quant(x, self.a_bits, self.a_clip)
            y_hat = moe_mixture_output(x_q, W_q, topk_idx, topk_alpha, W_layout=W_layout)
            relerr = rel_fro_error(y_hat, y)

        return {
            "relerr": relerr,
            "w_clip": float(self.w_clip.item()),
            "a_clip": float(self.a_clip.item()),
        }

# %%
# === Load your calibration batch ===
# Save it as a .pt dict with keys: x, W, topk_idx, topk_alpha
#
# Example:
#   torch.save({"x": x, "W": W, "topk_idx": idx, "topk_alpha": alpha}, "calib_batch.pt")

BATCH_PATH = "calib_batch.pt"   # <-- change
batch: Dict[str, Any] = torch.load(BATCH_PATH, map_location="cpu")

# Move tensors to device
for k, v in batch.items():
    if torch.is_tensor(v):
        batch[k] = v.to(device)

# Basic sanity
print({k: (tuple(v.shape), v.dtype) for k, v in batch.items() if torch.is_tensor(v)})

# %%
# === Define configs matching your poster labels ===
cfgs = [
    QuantConfig(name="PTQ 5/5/5 (W5A5KV5)", w_bits=5, a_bits=5, kv_bits=5),
    QuantConfig(name="PTQ 4/4/3 (W4A4KV3)", w_bits=4, a_bits=4, kv_bits=3),
]

# %%
# === Run PTQ eval ===
ptq_results = {}
for cfg in cfgs:
    err = eval_config_ptq(batch, cfg, per_channel_w=True, per_channel_a=False, W_layout="EHH")
    ptq_results[cfg.name] = err
    print(f"{cfg.name}: RelErr = {err:.6f}")

# %%
# === Run QAT-lite on 4/4/3 ===
qat_cfg = QuantConfig(name="QAT-lite 4/4/3 (W4A4KV3)", w_bits=4, a_bits=4, kv_bits=3)

cal = QATLiteCalibrator(batch["W"], w_bits=qat_cfg.w_bits, a_bits=qat_cfg.a_bits)
out = cal.run(batch, steps=250, lr=5e-2, W_layout="EHH")

print(f"{qat_cfg.name}: RelErr = {out['relerr']:.6f} (w_clip={out['w_clip']:.4g}, a_clip={out['a_clip']:.4g})")

# %%
# === Print LaTeX macro lines to paste into your poster ===
print("\nPaste into LaTeX:")
print(r"\newcommand{\ErrPTQFive}{" + f"{ptq_results[cfgs[0].name]:.6f}" + "}")
print(r"\newcommand{\ErrPTQFour}{" + f"{ptq_results[cfgs[1].name]:.6f}" + "}")
print(r"\newcommand{\ErrQATLite}{" + f"{out['relerr']:.6f}" + "}")


/tmp/ipykernel_2747619/1796690535.py:219: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  batch: Dict[str, Any] = torch.load(BATCH_PATH, map_location="cpu")


FileNotFoundError: [Errno 2] No such file or directory: 'calib_batch.pt'

In [7]:
#!/usr/bin/env python3
"""
moe_routed_fidelity_eval.py

Compute routed mixture fidelity errors for MoE expert banks.

Expected batch file (.pt) format:
  {
    "x":         Tensor [N, H]               (activations / token inputs)
    "W":         Tensor [E, H, H]            (expert weight matrices)
    "topk_idx":  LongTensor [N, K]           (expert indices per token)
    "topk_alpha":Tensor [N, K]               (gate weights per token per expert)
  }

This script computes:
  y  = sum_{k} alpha[n,k] * (x[n] @ W[e_idx[n,k]])
  ŷ = same but with quantized / reconstructed weights (and optional quantized x)
  RelErr_Frob = ||ŷ - y||_F / ||y||_F
  Per-token RelErr mean/std: mean_n ( ||diff[n]|| / ||y[n]|| ), std_n (...)

It can run built-in schemes:
  - baseline
  - ptq555  -> W_bits=5, A_bits=5, simple symmetric quant
  - ptq443  -> W_bits=4, A_bits=4, "3" used as activation clip preset (more aggressive)
  - qat443  -> "calibration-lite": per-column weight clip search + activation clip calibration

It can also evaluate user-provided variants via:
  --variant NAME:PATH
where PATH is a .pt containing either:
  - {"W_hat": Tensor [E,H,H]}  OR
  - {"W": Tensor [E,H,H]}      OR
  - directly a Tensor [E,H,H]

Outputs:
  - prints a table
  - optionally saves JSON
  - optionally saves a bar chart PNG

Run:
  python moe_routed_fidelity_eval.py --batch /path/to/calib_batch.pt --device cuda --plot out.png --save_json results.json
  python moe_routed_fidelity_eval.py --batch calib_batch.pt --variant "KTppv5:/path/to/W_hat_v5.pt" --variant "PTQ555:/path/to/W_hat_ptq555.pt"

Notes on "4/4/3":
  In your poster code, "4/4/3" is ambiguous (could mean component bits in a structured payload).
  Since we do not have your exact payload quantization semantics here, this script uses:
    W_bits=4, A_bits=4, and sets activation clipping to a more aggressive preset tied to "3".
  If you want "4/4/3" to mean something else, edit SCHEMES below (2 minutes).

Security:
  Uses torch.load(weights_only=True) when supported to reduce pickle risk.
"""

from __future__ import annotations

import argparse
import json
import math
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch


# ----------------------------
# Utility: safe-ish torch.load
# ----------------------------
def safe_torch_load(path: str, map_location: str = "cpu") -> Any:
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        # older torch versions
        return torch.load(path, map_location=map_location)


# ----------------------------
# Quantization helpers
# ----------------------------
def _qmax(bits: int) -> int:
    # symmetric signed int range: [-qmax, qmax]
    return (1 << (bits - 1)) - 1


def quant_dequant_symmetric(
    t: torch.Tensor,
    bits: int,
    per_channel_dim: Optional[int] = None,
    clip: Optional[float] = None,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    Symmetric quant-dequant:
      scale = max_abs / qmax
      q = round(t/scale).clamp(-qmax, qmax)
      dq = q * scale

    per_channel_dim:
      - if None: per-tensor scale
      - else: per-channel scale along that dim (keeps dim)
    clip:
      - if set (0 < clip <= 1): clip max_abs = clip * max_abs (aggressive)
      - if None: no additional clipping
    """
    if bits >= 16:
        return t

    qmax = _qmax(bits)

    if per_channel_dim is None:
        max_abs = t.abs().amax()
        if clip is not None:
            max_abs = max_abs * float(clip)
        scale = (max_abs / qmax).clamp_min(eps)
        q = torch.round(t / scale).clamp(-qmax, qmax)
        return q * scale

    # per-channel
    max_abs = t.abs().amax(dim=per_channel_dim, keepdim=True)
    if clip is not None:
        max_abs = max_abs * float(clip)
    scale = (max_abs / qmax).clamp_min(eps)
    q = torch.round(t / scale).clamp(-qmax, qmax)
    return q * scale


def find_best_clip_mse_per_column(
    W: torch.Tensor,
    bits: int,
    candidate_clips: List[float],
    col_dim: int = -1,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    "Calibration-lite" for weights:
    For each column (per-channel), try several clip multipliers and choose the one
    minimizing MSE between W and quant-dequant(W).

    Returns quant-dequant(W) using best clip per column.

    This is expensive-ish but OK for E=16, H=2048 if candidate_clips small.
    """
    if bits >= 16:
        return W

    qmax = _qmax(bits)
    # max_abs per column
    max_abs = W.abs().amax(dim=col_dim, keepdim=True).clamp_min(eps)

    best_mse = None
    best_dq = None

    for c in candidate_clips:
        scale = ((max_abs * c) / qmax).clamp_min(eps)
        q = torch.round(W / scale).clamp(-qmax, qmax)
        dq = q * scale
        mse = (dq - W).pow(2).mean()
        if best_mse is None or mse < best_mse:
            best_mse = mse
            best_dq = dq

    assert best_dq is not None
    return best_dq


def calibrate_activation_clip(
    x: torch.Tensor,
    percentile: float = 0.999,
) -> float:
    """
    Activation clip calibration: choose clip value based on percentile of |x|.
    Returns a multiplier relative to max_abs: clip = clip_val / max_abs.
    """
    # Flatten to 1D
    v = x.abs().flatten()
    if v.numel() == 0:
        return 1.0
    # quantile on CPU for stability
    v_cpu = v.detach().float().cpu()
    clip_val = torch.quantile(v_cpu, torch.tensor(percentile)).item()
    max_abs = v_cpu.max().item()
    if max_abs <= 0:
        return 1.0
    clip = float(clip_val / max_abs)
    # keep it sane
    return max(0.05, min(1.0, clip))


# ----------------------------
# MoE routed mixture compute
# ----------------------------
@torch.no_grad()
def routed_mixture_output(
    x: torch.Tensor,              # [N,H]
    W: torch.Tensor,              # [E,H,H]
    topk_idx: torch.Tensor,       # [N,K] long
    topk_alpha: torch.Tensor,     # [N,K]
) -> torch.Tensor:
    """
    Compute y[n] = sum_k alpha[n,k] * (x[n] @ W[e_idx[n,k]])
    Efficiently groups by expert.
    """
    device = x.device
    N, H = x.shape
    E = W.shape[0]
    K = topk_idx.shape[1]

    y = torch.zeros((N, H), device=device, dtype=torch.float32)

    # Flatten routes: (n, k) -> expert e
    n_idx = torch.arange(N, device=device).unsqueeze(1).expand(N, K).reshape(-1)  # [N*K]
    e_idx = topk_idx.reshape(-1).to(torch.long)                                   # [N*K]
    alpha = topk_alpha.reshape(-1).to(torch.float32)                              # [N*K]

    # For each expert, process tokens routed to it
    for e in range(E):
        mask = (e_idx == e)
        if not mask.any():
            continue
        ns = n_idx[mask]                     # token indices for this expert
        a = alpha[mask].unsqueeze(1)         # [M,1]
        xs = x[ns].to(torch.float32)         # [M,H]
        # matmul
        ys = xs @ W[e].to(torch.float32)     # [M,H]
        y.index_add_(0, ns, a * ys)

    return y


@torch.no_grad()
def routed_relerr_stats(y_hat: torch.Tensor, y: torch.Tensor, eps: float = 1e-12) -> Dict[str, float]:
    """
    Computes:
      - RelErr_Frob = ||y_hat - y||_F / ||y||_F
      - Per-token relative error mean/std
    """
    diff = (y_hat - y).to(torch.float32)
    y_f = y.to(torch.float32)

    num = torch.linalg.norm(diff)
    den = torch.linalg.norm(y_f).clamp_min(eps)
    rel_frob = (num / den).item()

    # Per-token
    diff_n = torch.linalg.norm(diff, dim=1)
    y_n = torch.linalg.norm(y_f, dim=1).clamp_min(eps)
    rel_n = (diff_n / y_n)
    mean_n = rel_n.mean().item()
    std_n = rel_n.std(unbiased=False).item()

    return {
        "relerr_frob": rel_frob,
        "relerr_token_mean": mean_n,
        "relerr_token_std": std_n,
    }


# ----------------------------
# Scheme definitions
# ----------------------------
@dataclass
class Scheme:
    name: str
    w_bits: int = 16
    a_bits: int = 16
    # quantization behavior
    w_per_column: bool = True       # per-column weight scales
    w_clip: Optional[float] = None  # clip multiplier for weights (relative to max_abs)
    a_clip: Optional[float] = None  # clip multiplier for activations
    # "qat-lite" calibration-lite options
    use_weight_mse_clip_search: bool = False
    weight_clip_candidates: Tuple[float, ...] = (1.00, 0.99, 0.98, 0.97, 0.95, 0.90)
    act_clip_percentile: Optional[float] = None  # if set, calibrate a_clip from x


def get_builtin_schemes() -> List[Scheme]:
    """
    Edit here if your "5/5/5" / "4/4/3" semantics differ.
    """
    return [
        Scheme(name="baseline_fp", w_bits=16, a_bits=16),

        # PTQ 5/5/5  (interpreted as W5 A5)
        Scheme(name="ptq_5_5_5", w_bits=5, a_bits=5, w_per_column=True, w_clip=None, a_clip=None),

        # PTQ 4/4/3 (interpreted as W4 A4 + more aggressive activation clip preset tied to "3")
        # If you want a different meaning, change a_clip here.
        Scheme(name="ptq_4_4_3", w_bits=4, a_bits=4, w_per_column=True, w_clip=None, a_clip=0.80),

        # QAT-lite 4/4/3 (implemented as calibration-lite):
        # - weight per-column clip search to minimize MSE
        # - activation clip calibrated from x at percentile (ties to "3" loosely)
        Scheme(
            name="qat_lite_4_4_3",
            w_bits=4,
            a_bits=4,
            w_per_column=True,
            use_weight_mse_clip_search=True,
            act_clip_percentile=0.999,  # you can try 0.995 / 0.9995 etc
        ),
    ]


# ----------------------------
# Variant loading (user-provided W_hat)
# ----------------------------
def load_variant_weights(path: str, device: torch.device) -> torch.Tensor:
    obj = safe_torch_load(path, map_location="cpu")
    if isinstance(obj, dict):
        if "W_hat" in obj:
            W_hat = obj["W_hat"]
        elif "W" in obj:
            W_hat = obj["W"]
        else:
            raise KeyError(f"{path}: dict must contain 'W_hat' or 'W'. Keys={list(obj.keys())}")
    elif torch.is_tensor(obj):
        W_hat = obj
    else:
        raise TypeError(f"{path}: expected dict or Tensor, got {type(obj)}")

    if W_hat.dim() != 3:
        raise ValueError(f"{path}: expected [E,H,H], got shape {tuple(W_hat.shape)}")
    return W_hat.to(device)


# ----------------------------
# Compression size estimate (weights only)
# ----------------------------
def estimate_weight_mb(E: int, H: int, w_bits: int, per_column_overhead: bool) -> float:
    """
    Approximate storage for weight-only quant:
      E * H * H * w_bits bits
      plus overhead for scales:
        per-column: E * H scales (float16 ~ 16 bits)  [very rough]
    """
    w_bits_total = E * H * H * w_bits
    overhead_bits = 0
    if per_column_overhead and w_bits < 16:
        overhead_bits += E * H * 16  # assume fp16 scale per column
    total_bytes = (w_bits_total + overhead_bits) / 8.0
    return total_bytes / (1024.0 * 1024.0)


# ----------------------------
# Main eval
# ----------------------------
@torch.no_grad()
def eval_scheme(
    scheme: Scheme,
    x: torch.Tensor,
    W: torch.Tensor,
    topk_idx: torch.Tensor,
    topk_alpha: torch.Tensor,
    y_ref: torch.Tensor,
) -> Dict[str, Any]:
    device = x.device
    E, H, _ = W.shape

    # activation quant-dequant
    x_use = x
    a_clip = scheme.a_clip

    if scheme.act_clip_percentile is not None:
        # calibrate clip multiplier from x
        a_clip = calibrate_activation_clip(x, percentile=float(scheme.act_clip_percentile))

    if scheme.a_bits < 16:
        x_use = quant_dequant_symmetric(
            x_use.to(torch.float32),
            bits=scheme.a_bits,
            per_channel_dim=None,   # per-tensor for activations
            clip=a_clip,
        ).to(x.dtype)

    # weight quant-dequant (per expert)
    # We'll build W_qdq on the fly per expert inside routed_mixture_output by constructing a temporary W_hat
    # But routed_mixture_output expects full W tensor. For simplicity and reproducibility, construct W_hat once.
    if scheme.w_bits >= 16:
        W_hat = W
    else:
        W_hat = torch.empty_like(W, dtype=torch.float32, device=device)
        for e in range(E):
            We = W[e].to(torch.float32)

            if scheme.use_weight_mse_clip_search:
                W_hat[e] = find_best_clip_mse_per_column(
                    We,
                    bits=scheme.w_bits,
                    candidate_clips=list(scheme.weight_clip_candidates),
                    col_dim=-1,
                )
            else:
                W_hat[e] = quant_dequant_symmetric(
                    We,
                    bits=scheme.w_bits,
                    per_channel_dim=(-1 if scheme.w_per_column else None),
                    clip=scheme.w_clip,
                )
        W_hat = W_hat.to(device)

    y_hat = routed_mixture_output(x_use, W_hat, topk_idx, topk_alpha)
    stats = routed_relerr_stats(y_hat, y_ref)

    est_mb = estimate_weight_mb(E, H, scheme.w_bits, per_column_overhead=scheme.w_per_column)

    out = {
        "name": scheme.name,
        "w_bits": scheme.w_bits,
        "a_bits": scheme.a_bits,
        "w_per_column": scheme.w_per_column,
        "w_clip": scheme.w_clip,
        "a_clip": a_clip,
        "act_clip_percentile": scheme.act_clip_percentile,
        "use_weight_mse_clip_search": scheme.use_weight_mse_clip_search,
        "est_weight_mb": est_mb,
        **stats,
    }
    return out


@torch.no_grad()
def eval_variant(
    name: str,
    W_hat: torch.Tensor,
    x: torch.Tensor,
    topk_idx: torch.Tensor,
    topk_alpha: torch.Tensor,
    y_ref: torch.Tensor,
) -> Dict[str, Any]:
    y_hat = routed_mixture_output(x, W_hat, topk_idx, topk_alpha)
    stats = routed_relerr_stats(y_hat, y_ref)
    E, H, _ = W_hat.shape
    est_mb = (W_hat.numel() * 2) / (1024.0 * 1024.0)  # if stored fp16; purely informational
    return {
        "name": name,
        "variant_source": "user_W_hat",
        "est_weight_mb_if_fp16": est_mb,
        **stats,
    }


def print_table(results: List[Dict[str, Any]]) -> None:
    # simple aligned print
    headers = [
        "name", "relerr_frob", "token_mean", "token_std",
        "w_bits", "a_bits", "est_weight_mb"
    ]
    print("\n=== Results ===")
    print("{:>18}  {:>12}  {:>12}  {:>12}  {:>5}  {:>5}  {:>12}".format(*headers))
    for r in results:
        print("{:>18}  {:12.6f}  {:12.6f}  {:12.6f}  {:>5}  {:>5}  {:12.3f}".format(
            str(r.get("name", ""))[:18],
            float(r["relerr_frob"]),
            float(r["relerr_token_mean"]),
            float(r["relerr_token_std"]),
            int(r.get("w_bits", -1)) if r.get("w_bits") is not None else -1,
            int(r.get("a_bits", -1)) if r.get("a_bits") is not None else -1,
            float(r.get("est_weight_mb", r.get("est_weight_mb_if_fp16", float("nan")))),
        ))


def maybe_save_plot(results: List[Dict[str, Any]], out_path: str) -> None:
    import matplotlib.pyplot as plt

    names = [r["name"] for r in results]
    vals = [r["relerr_frob"] for r in results]
    errs = [r["relerr_token_std"] for r in results]  # token std as errorbar proxy

    plt.figure(figsize=(10, 4.5))
    x = list(range(len(names)))
    plt.bar(x, vals, yerr=errs, capsize=4)
    plt.xticks(x, names, rotation=20, ha="right")
    plt.ylabel("RelErr (Frobenius) ↓")
    plt.title("Routed mixture fidelity error by variant")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"Saved plot: {out_path}")


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--batch", type=str, required=True, help="Path to calib_batch.pt")
    p.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    p.add_argument("--dtype", type=str, default="float16", choices=["float16", "float32"])
    p.add_argument("--builtins", type=str, default="baseline_fp,ptq_5_5_5,ptq_4_4_3,qat_lite_4_4_3",
                   help="Comma-separated built-in schemes to run (by name).")
    p.add_argument("--variant", type=str, action="append", default=[],
                   help="Extra variant as NAME:PATH (PATH contains W_hat or W)")
    p.add_argument("--save_json", type=str, default=None)
    p.add_argument("--plot", type=str, default=None)
    args = p.parse_args()

    device = torch.device(args.device)
    dtype = torch.float16 if args.dtype == "float16" else torch.float32

    batch = safe_torch_load(args.batch, map_location="cpu")
    if not isinstance(batch, dict):
        raise ValueError(f"Batch must be dict. Got {type(batch)}")

    for k in ["x", "W", "topk_idx", "topk_alpha"]:
        if k not in batch:
            raise KeyError(f"Batch missing key '{k}'. Keys={list(batch.keys())}")

    x = batch["x"].to(device=device, dtype=dtype)
    W = batch["W"].to(device=device, dtype=dtype)
    topk_idx = batch["topk_idx"].to(device=device)
    topk_alpha = batch["topk_alpha"].to(device=device, dtype=dtype)

    # basic checks
    if x.dim() != 2:
        raise ValueError(f"x must be [N,H], got {tuple(x.shape)}")
    if W.dim() != 3:
        raise ValueError(f"W must be [E,H,H], got {tuple(W.shape)}")
    if topk_idx.dim() != 2 or topk_alpha.dim() != 2:
        raise ValueError("topk_idx and topk_alpha must be [N,K]")
    if topk_idx.shape != topk_alpha.shape:
        raise ValueError(f"topk_idx shape {tuple(topk_idx.shape)} != topk_alpha shape {tuple(topk_alpha.shape)}")
    if topk_idx.shape[0] != x.shape[0]:
        raise ValueError(f"N mismatch: x has {x.shape[0]} tokens but topk has {topk_idx.shape[0]}")
    if W.shape[1] != x.shape[1] or W.shape[2] != x.shape[1]:
        raise ValueError(f"H mismatch: x is H={x.shape[1]} but W is {tuple(W.shape)}")

    # Reference output (baseline FP with given W)
    print("Computing reference y (baseline)...")
    y_ref = routed_mixture_output(x, W, topk_idx, topk_alpha)

    schemes = get_builtin_schemes()
    scheme_map = {s.name: s for s in schemes}

    to_run = [n.strip() for n in args.builtins.split(",") if n.strip()]
    results: List[Dict[str, Any]] = []

    for name in to_run:
        if name not in scheme_map:
            raise KeyError(f"Unknown builtin scheme '{name}'. Available: {list(scheme_map.keys())}")
        s = scheme_map[name]
        print(f"Running scheme: {s.name}")
        r = eval_scheme(s, x, W, topk_idx, topk_alpha, y_ref)
        results.append(r)

    # user variants
    for spec in args.variant:
        if ":" not in spec:
            raise ValueError(f"--variant must be NAME:PATH, got '{spec}'")
        vname, vpath = spec.split(":", 1)
        vpath = vpath.strip()
        print(f"Loading variant '{vname}' from {vpath}")
        W_hat = load_variant_weights(vpath, device=device).to(dtype)
        r = eval_variant(vname, W_hat, x, topk_idx, topk_alpha, y_ref)
        results.append(r)

    # Print and save
    print_table(results)

    if args.save_json:
        outp = Path(args.save_json)
        outp.parent.mkdir(parents=True, exist_ok=True)
        with open(outp, "w") as f:
            json.dump(results, f, indent=2)
        print(f"Saved JSON: {args.save_json}")

    if args.plot:
        maybe_save_plot(results, args.plot)


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] --batch BATCH [--device DEVICE]
                             [--dtype {float16,float32}] [--builtins BUILTINS]
                             [--variant VARIANT] [--save_json SAVE_JSON]
                             [--plot PLOT]
ipykernel_launcher.py: error: the following arguments are required: --batch


SystemExit: 2

In [6]:
!python /home/daniyar/notebooks/moe_routed_fidelity_eval.py \
  --batch /ABS/PATH/TO/calib_batch.pt \
  --device cuda \
  --plot routed_errors.png \
  --save_json routed_errors.json


In [5]:
%%writefile /home/daniyar/notebooks/moe_routed_fidelity_eval.py
# PASTE YOUR ENTIRE SCRIPT HERE (starting from #!/usr/bin/env python3)


Writing /home/daniyar/notebooks/moe_routed_fidelity_eval.py


In [1]:
# ============================================================
# DeepSeek KT++-X Optimized (offline) — "everything included" v7
#
# Goals:
#   - Runs end-to-end even when CALIB_PATH is missing (no hard crash)
#   - Still supports "strict" mode for meaningful ridge-based Ws
#   - Better residual model: FULL per-expert r×r coefficients (RES_COEF_MODE=full)
#   - Faster sparse residual: vectorized top-k blocks by energy
#   - Compression-realistic basis option: BASIS_MODE=hadamard_perm (implicit, tiny)
#
# IMPORTANT (accuracy reality check):
#   - 99–100% accuracy on real DeepSeek routing requires REAL CALIB_PATH (true layer inputs)
#     and usually ROUTER_PATH (RIDGE_WEIGHTED=1). Without them, any metric is at best proxy.
#
# Dependencies: torch, safetensors, numpy (tqdm optional)
# Optional: transformers (only if you enable CAPTURE_CALIB=1)
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise RuntimeError("This script requires PyTorch.") from e

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("This script requires safetensors (pip install safetensors).") from e

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Threading / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))

DTYPE_W   = torch.float16   # stored blocks/bases dtype (unless int8 quant)
DTYPE_ACC = torch.float32   # compute dtype

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    # Model slice
    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "16"))

    # Calibration: npz with key 'X' shape (N,H)
    CALIB_PATH: str = os.environ.get("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "2048"))

    # If STRICT_CALIB=1 and ridge needs calib but missing -> error.
    STRICT_CALIB: bool = os.environ.get("STRICT_CALIB", "0") == "1"

    # If ridge needs calib but missing: allow random (proxy only).
    ALLOW_RANDOM_CALIB: bool = os.environ.get("ALLOW_RANDOM_CALIB", "1") == "1"

    # Optional: auto-capture calib via transformers (best-effort)
    CAPTURE_CALIB: bool = os.environ.get("CAPTURE_CALIB", "0") == "1"
    CAPTURE_OUT_X: str = os.environ.get("CAPTURE_OUT_X", "").strip()  # if empty -> OUTPUT_DIR/calib_layer{L}.npz
    CAPTURE_OUT_P: str = os.environ.get("CAPTURE_OUT_P", "").strip()  # if empty -> OUTPUT_DIR/router_layer{L}.npz
    CAPTURE_TEXT: str = os.environ.get("CAPTURE_TEXT", "Hello world.").strip()
    CAPTURE_REPEAT: int = int(os.environ.get("CAPTURE_REPEAT", "128"))
    CAPTURE_BATCH_TOKENS: int = int(os.environ.get("CAPTURE_BATCH_TOKENS", "256"))

    # Router: npz with key 'P' shape (N, E_total) or (N, E_used)
    ROUTER_PATH: str = os.environ.get("ROUTER_PATH", "").strip()
    ROUTER_EIDS_ARE_GLOBAL: bool = os.environ.get("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    # Build square mats
    LIN_MODE: str = os.environ.get("LIN_MODE", "ridge").strip().lower()  # ridge|weff_gate|weff
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))      # relative to trace(XtX)/H
    RIDGE_WEIGHTED: bool = os.environ.get("RIDGE_WEIGHTED", "0") == "1"  # use router P as weights

    # If ridge/weff_gate needs X but X missing and STRICT_CALIB=0:
    #   - ridge falls back to weff unless ALLOW_RANDOM_CALIB=1
    AUTO_FALLBACK_LIN: bool = os.environ.get("AUTO_FALLBACK_LIN", "1") == "1"

    # For weff_gate when calib missing: approximate gate mean using weight-stat Monte Carlo
    WEFF_GATE_MC_SAMPLES: int = int(os.environ.get("WEFF_GATE_MC_SAMPLES", "64"))

    # Optional similarity rotation (applies after Ws build)
    USE_HADAMARD: bool = os.environ.get("USE_HADAMARD", "0") == "1"
    HAD_SEED: int = int(os.environ.get("HAD_SEED", "1234"))

    # Normalize each expert W to Frobenius norm 1 (keeps eval stable)
    NORMALIZE_W: bool = os.environ.get("NORMALIZE_W", "1") == "1"

    # Cache
    CACHE_MODE: str = os.environ.get("CACHE_MODE", "readwrite").strip().lower()  # off|read|write|readwrite
    CACHE_NAME: str = os.environ.get("CACHE_NAME", "").strip()
    FORCE_REBUILD_WS: bool = os.environ.get("FORCE_REBUILD_WS", "0") == "1"

    # Basis mode (compression-critical)
    BASIS_MODE: str = os.environ.get("BASIS_MODE", "dense_train").strip().lower()  # dense_train | hadamard_perm

    # Clustering
    M_CLUSTERS: int = int(os.environ.get("M_CLUSTERS", "0"))  # 0 => auto
    CLUSTER_ITERS: int = int(os.environ.get("CLUSTER_ITERS", "80"))
    CLUSTER_RESTARTS: int = int(os.environ.get("CLUSTER_RESTARTS", "4"))
    CLUSTER_FEAT_D: int = int(os.environ.get("CLUSTER_FEAT_D", "64"))
    CLUSTER_TARGET_SIZE: int = int(os.environ.get("CLUSTER_TARGET_SIZE", "4"))  # auto-M aims for this size
    CLUSTER_MIN_SIZE: int = int(os.environ.get("CLUSTER_MIN_SIZE", "2"))        # reassign tiny clusters

    # Training (U,V) only for BASIS_MODE=dense_train
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "0"))
    SUBM: int = int(os.environ.get("SUBM", "256"))
    BATCH_E: int = int(os.environ.get("BATCH_E", "4"))
    LR_UV: float = float(os.environ.get("LR_UV", "1e-2"))
    REORTHO_EVERY: int = int(os.environ.get("REORTHO_EVERY", "1"))
    REPORT_EVERY: int = int(os.environ.get("REPORT_EVERY", "5"))
    TRAIN_MIN_CLUSTER: int = int(os.environ.get("TRAIN_MIN_CLUSTER", "3"))
    GRAD_CLIP: float = float(os.environ.get("GRAD_CLIP", "1.0"))

    # Training objective knobs
    TRAIN_OBJ: str = os.environ.get("TRAIN_OBJ", "energy_ratio").strip().lower()  # energy_ratio|logratio
    TRAIN_LAM_BLOCK: float = float(os.environ.get("TRAIN_LAM_BLOCK", "0.1"))
    TRAIN_BLOCK_USE_CORE_BLOCK: bool = os.environ.get("TRAIN_BLOCK_USE_CORE_BLOCK", "1") == "1"

    # Core model
    CORE_MODE: str = os.environ.get("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_AGG: str = os.environ.get("CORE_AGG", "mean").strip().lower()  # mean|max for shared blocktopk
    CORE_BLOCK: int = int(os.environ.get("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(os.environ.get("CORE_TARGET", "0.90"))
    CORE_MAX_BLOCKS: int = int(os.environ.get("CORE_MAX_BLOCKS", "512"))

    # Residual
    RES_RANK: int = int(os.environ.get("RES_RANK", "64"))
    RES_COEF_MODE: str = os.environ.get("RES_COEF_MODE", "full").strip().lower()  # full|diag
    RES_BLOCKS: int = int(os.environ.get("RES_BLOCKS", "128"))
    RES_BSIZE: int = int(os.environ.get("RES_BSIZE", "64"))

    # Quantization (payload size)
    QMODE: str = os.environ.get("QMODE", "none").strip().lower()  # none | int8

    # Eval
    EVAL_TRIALS: int = int(os.environ.get("EVAL_TRIALS", "8"))
    ROUTED_K: int = int(os.environ.get("ROUTED_K", "8"))
    EVAL_BATCH: int = int(os.environ.get("EVAL_BATCH", "4"))
    EVAL_PER_EXPERT: bool = os.environ.get("EVAL_PER_EXPERT", "1") == "1"

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def cache_path():
    if cfg.CACHE_NAME:
        return os.path.join(cfg.OUTPUT_DIR, cfg.CACHE_NAME)
    tag = f"deepseek_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_{cfg.LIN_MODE}_Ws_cache_v7.npz"
    return os.path.join(cfg.OUTPUT_DIR, tag)

def _meta_dict():
    return dict(
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        max_experts=cfg.MAX_EXPERTS,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES,
        router_path=cfg.ROUTER_PATH or "",
        router_global=cfg.ROUTER_EIDS_ARE_GLOBAL,
        hadamard=cfg.USE_HADAMARD,
        had_seed=cfg.HAD_SEED,
        normalize_w=cfg.NORMALIZE_W,
    )

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        # try to disambiguate if fallback collided
        gate2 = pick(["gate_proj.weight"])
        if gate2:
            gate = gate2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Router
# ----------------------------
def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    if not os.path.isfile(cfg.ROUTER_PATH):
        raise FileNotFoundError(f"ROUTER_PATH not found: {cfg.ROUTER_PATH}")
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Has keys: {list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    log(f"[router] Loaded P: {tuple(P.shape)}")
    return P

# ----------------------------
# Hadamard utilities
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

def fwht(x: torch.Tensor) -> torch.Tensor:
    """
    Fast Walsh–Hadamard Transform on last dimension (unscaled).
    Works for any prefix dims, requires last dim to be power-of-two.
    """
    n = x.shape[-1]
    if not is_power_of_two(n):
        raise ValueError(f"FWHT requires power-of-two, got {n}")
    orig_shape = x.shape
    y = x.reshape(-1, n).contiguous()
    h = 1
    while h < n:
        y = y.reshape(-1, n // (2 * h), 2, h)
        a = y[:, :, 0, :]
        b = y[:, :, 1, :]
        y[:, :, 0, :] = a + b
        y[:, :, 1, :] = a - b
        y = y.reshape(-1, n)
        h *= 2
    return y.reshape(orig_shape)

def fwht_ortho(x: torch.Tensor) -> torch.Tensor:
    return fwht(x) / math.sqrt(x.shape[-1])

# ----------------------------
# Quantization (optional)
# ----------------------------
class QTensor:
    __slots__ = ("q", "scale", "shape", "per_row")
    def __init__(self, q: torch.Tensor, scale: torch.Tensor, shape: Tuple[int, ...], per_row: bool):
        self.q = q
        self.scale = scale
        self.shape = shape
        self.per_row = per_row

def quantize_int8(t: torch.Tensor, per_row: bool = False) -> Any:
    if cfg.QMODE != "int8":
        return t
    tt = t.to(torch.float32)
    if per_row and tt.ndim == 2:
        mx = tt.abs().amax(dim=1).clamp_min(1e-8)  # (m,)
        scale = (mx / 127.0).to(torch.float32)
        q = torch.round(tt / scale.unsqueeze(1)).clamp(-127, 127).to(torch.int8)
        return QTensor(q=q, scale=scale, shape=tuple(tt.shape), per_row=True)
    else:
        mx = tt.abs().max().clamp_min(1e-8)
        scale = (mx / 127.0).to(torch.float32)
        q = torch.round(tt / scale).clamp(-127, 127).to(torch.int8)
        return QTensor(q=q, scale=scale, shape=tuple(tt.shape), per_row=False)

def dequantize(x: Any, device: torch.device, dtype: torch.dtype = DTYPE_ACC) -> torch.Tensor:
    if isinstance(x, QTensor):
        if x.per_row:
            t = x.q.to(torch.float32) * x.scale.to(torch.float32).unsqueeze(1)
        else:
            t = x.q.to(torch.float32) * x.scale.to(torch.float32)
        return t.to(device=device, dtype=dtype)
    return x.to(device=device, dtype=dtype)

# ----------------------------
# Optional: capture calib X + router P from transformers (best-effort)
# ----------------------------
def _default_capture_paths() -> Tuple[str, str]:
    xout = cfg.CAPTURE_OUT_X or os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    pout = cfg.CAPTURE_OUT_P or os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return xout, pout

def try_capture_calib_and_router(H: int, E_total_guess: int) -> Tuple[Optional[str], Optional[str]]:
    """
    Best-effort, optional. Requires transformers installed + model loadable from MODEL_DIR.
    Produces:
      - X npz with key 'X' shape (N,H)
      - P npz with key 'P' shape (N,E_total_guess) if router module found
    """
    if not cfg.CAPTURE_CALIB:
        return None, None
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except Exception as e:
        log(f"[capture] transformers not available: {e}")
        return None, None

    xout, pout = _default_capture_paths()
    log(f"[capture] trying transformers capture -> {xout} and {pout}")

    tok = AutoTokenizer.from_pretrained(cfg.MODEL_DIR, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(cfg.MODEL_DIR, torch_dtype=torch.float16, device_map=None)
    model.eval()
    model.to(DEVICE)

    # Hook layer input (mlp input). We try common paths; if not found, fallback to module name search.
    layer = None
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        if cfg.LAYER < len(model.model.layers):
            layer = model.model.layers[cfg.LAYER]
    if layer is None:
        log("[capture] could not locate model.model.layers[LAYER]; capture aborted.")
        return None, None

    # Find MLP module
    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        log("[capture] layer has no .mlp; capture aborted.")
        return None, None

    X_chunks: List[torch.Tensor] = []
    P_chunks: List[torch.Tensor] = []

    # Try to locate a router/gate module inside mlp (best effort)
    gate_mod = None
    gate_name = None
    for name, mod in mlp.named_modules():
        # heuristic: linear with out_features >= experts
        if isinstance(mod, nn.Linear) and getattr(mod, "out_features", 0) >= E_total_guess:
            if ("gate" in name.lower()) or ("router" in name.lower()):
                gate_mod = mod
                gate_name = name
                break

    if gate_mod is not None:
        log(f"[capture] found gate-like module: mlp.{gate_name} (Linear out={gate_mod.out_features})")
    else:
        log("[capture] no gate-like Linear found; will capture X only.")

    # Hook mlp input
    def mlp_pre_hook(_module, inputs):
        hs = inputs[0]
        if hs is None:
            return
        hs = hs.detach().to(torch.float32)
        X_chunks.append(hs.reshape(-1, hs.shape[-1]).cpu())

    # Hook gate output if found
    def gate_hook(_module, inputs, output):
        out = output
        if isinstance(out, (tuple, list)):
            out = out[0]
        if torch.is_tensor(out):
            logits = out.detach().to(torch.float32)
            # softmax -> probs
            probs = torch.softmax(logits, dim=-1)
            P_chunks.append(probs.reshape(-1, probs.shape[-1]).cpu())

    h1 = mlp.register_forward_pre_hook(mlp_pre_hook)
    h2 = gate_mod.register_forward_hook(gate_hook) if gate_mod is not None else None

    # Build input batch
    text = (cfg.CAPTURE_TEXT + "\n") * cfg.CAPTURE_REPEAT
    enc = tok(text, return_tensors="pt", truncation=True, max_length=cfg.CAPTURE_BATCH_TOKENS)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        _ = model(**enc)

    # remove hooks
    h1.remove()
    if h2 is not None:
        h2.remove()

    if not X_chunks:
        log("[capture] no X collected; capture failed.")
        return None, None

    X = torch.cat(X_chunks, dim=0)
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    np.savez(xout, X=X.numpy().astype(np.float32))
    log(f"[capture] wrote X: {xout}  shape={tuple(X.shape)}")

    p_path_written = None
    if P_chunks:
        P = torch.cat(P_chunks, dim=0)
        # pad/truncate to E_total_guess
        if P.shape[1] >= E_total_guess:
            P = P[:, :E_total_guess]
        else:
            pad = torch.zeros(P.shape[0], E_total_guess - P.shape[1])
            P = torch.cat([P, pad], dim=1)
        if P.shape[0] > X.shape[0]:
            P = P[:X.shape[0]]
        elif P.shape[0] < X.shape[0]:
            X = X[:P.shape[0]]
        np.savez(pout, P=P.numpy().astype(np.float32))
        log(f"[capture] wrote P: {pout}  shape={tuple(P.shape)}")
        p_path_written = pout

    return xout, p_path_written

# ----------------------------
# Calibration X
# ----------------------------
def load_calib_X(H: int) -> Optional[torch.Tensor]:
    if cfg.CALIB_PATH and os.path.isfile(cfg.CALIB_PATH):
        z = np.load(cfg.CALIB_PATH, allow_pickle=False)
        if "X" not in z.files:
            raise KeyError(f"CALIB npz missing key 'X'. Has keys: {list(z.files)}")
        X = torch.from_numpy(z["X"]).to(DTYPE_ACC).to(DEVICE)
        z.close()
        if X.ndim != 2 or X.shape[1] != H:
            raise RuntimeError(f"Bad X shape: {tuple(X.shape)} expected (*,{H})")
        if X.shape[0] > cfg.CALIB_SAMPLES:
            X = X[:cfg.CALIB_SAMPLES]
        log(f"[calib] Loaded X: {tuple(X.shape)}")
        return X

    # Attempt capture if enabled
    if cfg.CAPTURE_CALIB:
        # E_total_guess doesn't matter much; use 256 as safe upper guess
        xout, pout = try_capture_calib_and_router(H, E_total_guess=256)
        if xout:
            cfg.CALIB_PATH = xout
        if pout and not cfg.ROUTER_PATH:
            cfg.ROUTER_PATH = pout
        if cfg.CALIB_PATH and os.path.isfile(cfg.CALIB_PATH):
            return load_calib_X(H)

    # No calib available
    return None

# ----------------------------
# MLP forward (for ridge)
# ----------------------------
@torch.no_grad()
def forward_mlp_no_grad(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

# ----------------------------
# Gate-mean approximation when X missing (for weff_gate)
# ----------------------------
@torch.no_grad()
def gate_mean_from_weight_stats(W_gate: torch.Tensor, nsamp: int) -> torch.Tensor:
    """
    Approximate m_j = E[silu(z_j)] where z_j ~ N(0, sigma_j^2).
    sigma_j estimated from row-norm of gate weight assuming x ~ N(0, I).
    Vectorized Monte Carlo over d_ff rows.
    """
    Wg = W_gate.to(DTYPE_ACC)
    sigma = torch.linalg.norm(Wg, dim=1).clamp_min(1e-8)  # (d_ff,)
    g = torch.Generator(device=Wg.device).manual_seed(SEED + 999)
    z = torch.randn(nsamp, sigma.shape[0], generator=g, device=Wg.device, dtype=DTYPE_ACC) * sigma.view(1, -1)
    m = F.silu(z).mean(dim=0)  # (d_ff,)
    return m

# ----------------------------
# Build expert square Ws
# ----------------------------
@torch.no_grad()
def build_Ws_square() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer {cfg.LAYER} experts: {all_eids}  (using {len(eids)})")

    per_e: Dict[int, Dict[str, str]] = {}
    need_keys: List[str] = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk or ("up" not in kk) or ("down" not in kk) or ("gate" not in kk):
            raise RuntimeError(f"Expert {eid} missing keys: {kk}")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]].to(DTYPE_W)
    dff, H = W_up0.shape
    log(f"[shape] H={H} d_ff={dff}")

    # Load X only if needed
    X: Optional[torch.Tensor] = None
    if cfg.LIN_MODE in ("ridge", "weff_gate"):
        X = load_calib_X(H)

    # Handle missing calib
    lin_mode = cfg.LIN_MODE
    if X is None and cfg.LIN_MODE == "ridge":
        if cfg.STRICT_CALIB:
            raise RuntimeError("CALIB_PATH is missing and STRICT_CALIB=1. Provide real X or disable strict.")
        if cfg.ALLOW_RANDOM_CALIB:
            log("[calib] CALIB_PATH missing -> using RANDOM X (proxy only; not meaningful for real accuracy).")
            X = torch.randn(cfg.CALIB_SAMPLES, H, dtype=DTYPE_ACC, device=DEVICE)
        elif cfg.AUTO_FALLBACK_LIN:
            log("[calib] CALIB_PATH missing and ALLOW_RANDOM_CALIB=0 -> falling back LIN_MODE=weff (proxy).")
            lin_mode = "weff"
        else:
            raise RuntimeError("CALIB_PATH missing; set ALLOW_RANDOM_CALIB=1 or AUTO_FALLBACK_LIN=1 or provide CALIB_PATH.")

    if X is None and cfg.LIN_MODE == "weff_gate":
        if cfg.STRICT_CALIB:
            raise RuntimeError("CALIB_PATH missing and STRICT_CALIB=1, but LIN_MODE=weff_gate needs X (or stats).")
        log("[calib] CALIB_PATH missing -> weff_gate will use WEIGHT-STATS Monte Carlo for gate mean.")

    # Router P optional
    P = load_router_P() if cfg.ROUTER_PATH else None

    use_post_had = cfg.USE_HADAMARD
    if use_post_had and not is_power_of_two(H):
        log(f"[hadamard] H={H} not power-of-two; disabling USE_HADAMARD.")
        use_post_had = False

    Xf = X.to(DTYPE_ACC) if X is not None else None
    cholG = None
    lam = None
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)

    if lin_mode == "ridge":
        assert Xf is not None
        XtX = (Xf.t() @ Xf)
        lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
        if not (cfg.RIDGE_WEIGHTED and (P is not None)):
            G = XtX + lam * I
            cholG = torch.linalg.cholesky(G)

    Ws: List[torch.Tensor] = []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws", leave=False)):
        W_up   = T[per_e[eid]["up"]].to(DTYPE_W).to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DTYPE_W).to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DTYPE_W).to(DEVICE)

        if lin_mode == "weff":
            W = (W_down.to(DTYPE_ACC) @ W_up.to(DTYPE_ACC))

        elif lin_mode == "weff_gate":
            if Xf is not None:
                gate_act = (Xf @ W_gate.to(DTYPE_ACC).t())
                m = F.silu(gate_act).mean(dim=0)  # (dff,)
            else:
                m = gate_mean_from_weight_stats(W_gate, nsamp=cfg.WEFF_GATE_MC_SAMPLES)  # (dff,)
            W = (W_down.to(DTYPE_ACC) * m.view(1, -1)) @ W_up.to(DTYPE_ACC)

        elif lin_mode == "ridge":
            assert X is not None and Xf is not None
            Y = forward_mlp_no_grad(X, W_gate, W_up, W_down).to(DTYPE_ACC)  # (N,H)

            # weighted ridge if router exists
            if cfg.RIDGE_WEIGHTED and (P is not None):
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    if eid >= P.shape[1]:
                        raise RuntimeError(f"ROUTER P has {P.shape[1]} experts but eid={eid} requested.")
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                else:
                    if i >= P.shape[1]:
                        raise RuntimeError(f"ROUTER P has {P.shape[1]} columns but i={i} requested.")
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)

                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw

                XtX_e = Xw.t() @ Xw
                lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
                G_e = XtX_e + lam_e * I
                chol = torch.linalg.cholesky(G_e)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)  # (H,H)
                W = Wt.t().contiguous()
            else:
                assert cholG is not None
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)  # (H,H)
                W = Wt.t().contiguous()
        else:
            raise ValueError("LIN_MODE must be ridge|weff_gate|weff")

        if use_post_had:
            # W <- H W H (orthonormal H); apply right then left via transpose trick
            W = fwht_ortho(W)              # right-multiply by H^T == H (on rows) is not correct
            # Correct: multiply on RIGHT => transform last dim:
            # Already done above (fwht on last dim). Now left multiply by H:
            W = fwht_ortho(W.t().contiguous()).t().contiguous()

        if cfg.NORMALIZE_W:
            fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
            W = (W / fn).contiguous()

        Ws.append(W.contiguous())

    WsT = torch.stack(Ws, dim=0).to(DTYPE_ACC)  # (E,H,H)
    return eids, WsT

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    p = cache_path()
    do_read = cfg.CACHE_MODE in ("read", "readwrite")
    do_write = cfg.CACHE_MODE in ("write", "readwrite")

    if do_read and (not cfg.FORCE_REBUILD_WS) and os.path.isfile(p):
        z = try_load_npz(p)
        if z and ("expert_ids" in z) and ("Ws" in z):
            ok = True
            if "meta" in z:
                meta = _decode_meta(z["meta"])
                ok = (meta == _meta_dict())
            if ok:
                eids = [int(x) for x in z["expert_ids"].tolist()]
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws: {p}  Ws={tuple(Ws.shape)}")
                return eids, Ws
            log("[cache] meta mismatch -> rebuilding Ws.")
        else:
            log("[cache] invalid cache -> rebuilding Ws.")

    eids, Ws = build_Ws_square()
    if do_write:
        save_npz(p, {
            "meta": _encode_meta(_meta_dict()),
            "expert_ids": np.array(eids, dtype=np.int32),
            "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        })
        log(f"[cache] wrote: {p}  ({os.path.getsize(p)/1e6:.2f} MB)")
    return eids, Ws

# ----------------------------
# Basis (dense or implicit)
# ----------------------------
class ImplicitHadamardPerm:
    """
    Orthonormal U:
      x @ U = FWHT( (x[:, perm] * sign) ) / sqrt(n)
    """
    def __init__(self, n: int, seed: int):
        if not is_power_of_two(n):
            raise ValueError(f"hadamard_perm requires power-of-two n, got {n}")
        g = torch.Generator(device="cpu").manual_seed(seed)
        perm = torch.randperm(n, generator=g)
        inv = torch.empty_like(perm)
        inv[perm] = torch.arange(n)
        sign = (torch.randint(0, 2, (n,), generator=g, dtype=torch.int8) * 2 - 1).to(torch.int8)
        self.n = n
        self.perm = perm
        self.inv_perm = inv
        self.sign = sign

    def apply(self, x: torch.Tensor) -> torch.Tensor:
        xp = x.index_select(-1, self.perm.to(x.device))
        xs = xp * self.sign.to(x.device).to(x.dtype)
        return fwht_ortho(xs)

    def apply_T(self, x: torch.Tensor) -> torch.Tensor:
        y = fwht_ortho(x)
        y = y * self.sign.to(y.device).to(y.dtype)
        return y.index_select(-1, self.inv_perm.to(y.device))

    def project_matrix(self, W: torch.Tensor, V: "ImplicitHadamardPerm") -> torch.Tensor:
        # X = U^T W V
        WTU = self.apply(W.t().contiguous()).t().contiguous()
        return V.apply(WTU.contiguous())

    def reconstruct_matrix(self, X: torch.Tensor, V: "ImplicitHadamardPerm") -> torch.Tensor:
        # W = U X V^T
        XTUT = self.apply_T(X.t().contiguous()).t().contiguous()
        return V.apply_T(XTUT.contiguous())

class DenseBasis(nn.Module):
    def __init__(self, U_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(U_init.clone().to(DEVICE))

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

# ----------------------------
# Clustering
# ----------------------------
@torch.no_grad()
def expert_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    d = int(d)
    g = torch.Generator(device="cpu").manual_seed(SEED)
    Rsign = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)

    feats = []
    for e in range(E):
        W = Ws[e]
        rowE = (W * W).sum(dim=1)  # (n,)
        colE = (W * W).sum(dim=0)  # (n,)
        f = torch.cat([(rowE @ Rsign), (colE @ Rsign)], dim=0)
        feats.append(f.unsqueeze(0))
    X = torch.cat(feats, dim=0)  # (E, 2d)
    X = X / (X.norm(dim=1, keepdim=True).clamp_min(1e-12))
    return X

@torch.no_grad()
def kmeans_pp_init(X: torch.Tensor, k: int, g: torch.Generator) -> torch.Tensor:
    n = X.shape[0]
    centers = []
    first = torch.randint(0, n, (1,), generator=g).item()
    centers.append(X[first].clone())
    for _ in range(1, k):
        C = torch.stack(centers, dim=0)
        dist2 = torch.cdist(X, C).pow(2).amin(dim=1)
        probs = dist2 / dist2.sum().clamp_min(1e-12)
        idx = torch.multinomial(probs, 1, generator=g).item()
        centers.append(X[idx].clone())
    return torch.stack(centers, dim=0)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> Tuple[torch.Tensor, torch.Tensor]:
    best_inertia = None
    best_lab = None
    best_C = None
    for r in range(max(1, restarts)):
        g = torch.Generator(device="cpu").manual_seed(SEED + 1000 + r)
        C = kmeans_pp_init(X, k=k, g=g)

        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(dim=1)
            for j in range(k):
                m = (lab == j)
                if m.any():
                    C[j] = X[m].mean(dim=0)
                else:
                    far = dist.max(dim=1).indices[0].item()
                    C[j] = X[far].clone()

        dist = torch.cdist(X, C)
        lab = dist.argmin(dim=1)
        inertia = float((dist.gather(1, lab.view(-1, 1)).pow(2)).sum().item())
        if (best_inertia is None) or (inertia < best_inertia):
            best_inertia = inertia
            best_lab = lab.clone()
            best_C = C.clone()
    return best_lab, best_C  # type: ignore

@torch.no_grad()
def enforce_min_cluster_size(labels: torch.Tensor, centers: torch.Tensor, X: torch.Tensor, min_size: int) -> torch.Tensor:
    if min_size <= 1:
        return labels
    k = centers.shape[0]
    counts = torch.bincount(labels, minlength=k)
    big = (counts >= min_size).nonzero(as_tuple=False).flatten()
    small = (counts < min_size).nonzero(as_tuple=False).flatten()
    if big.numel() == 0 or small.numel() == 0:
        return labels
    big_centers = centers[big]
    for c in small.tolist():
        idxs = (labels == c).nonzero(as_tuple=False).flatten()
        if idxs.numel() == 0:
            continue
        d = torch.cdist(X[idxs], big_centers)
        nn = d.argmin(dim=1)
        labels[idxs] = big[nn]
    return labels

# ----------------------------
# Core selection
# ----------------------------
@torch.no_grad()
def _block_energy_grid(X: torch.Tensor, b: int) -> torch.Tensor:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        pad = nb * b - n
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
        n = nb*b
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()  # (nb,nb,b,b)
    E = (Xb * Xb).sum(dim=(2,3))  # (nb,nb)
    return E

@torch.no_grad()
def _pick_top_blocks_from_energy(Eg: torch.Tensor, tot_energy: float, b: int, target: float, max_blocks: int) -> Tuple[List[Tuple[int,int,int,int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    kept = float(flat[order[:K]].sum().item())
    blocks = []
    for idx in pick:
        bi = idx // nb
        bj = idx % nb
        blocks.append((bi*b, bj*b, b, b))
    return blocks, kept / max(tot_energy, 1e-12)

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor], mode: str, agg: str, block: int, target: float, max_blocks: int):
    if mode == "none":
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}
    n = X_list[0].shape[0]
    b = int(block)
    if b <= 0:
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}

    if mode == "blocktopk":
        Eg_all = []
        tots = []
        for X in X_list:
            tots.append(float((X*X).sum().item()))
            Eg_all.append(_block_energy_grid(X, b).to(DTYPE_ACC))
        tot = float(np.mean(tots))
        Eg = torch.stack(Eg_all, dim=0).amax(dim=0) if agg == "max" else torch.stack(Eg_all, dim=0).mean(dim=0)
        blocks, ef = _pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
        return {"mode": "blocktopk", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    if mode == "blocktopk_perexpert":
        blocks_per = []
        efracs = []
        for X in X_list:
            tot = float((X*X).sum().item())
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            blocks, ef = _pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
            blocks_per.append(blocks)
            efracs.append(ef)
        return {"mode": "blocktopk_perexpert", "blocks_shared": [], "blocks_per_expert": blocks_per, "energy_fracs": efracs}

    if mode == "blockdiag":
        nb = (n + b - 1) // b
        diagE = torch.zeros(nb, dtype=DTYPE_ACC, device=X_list[0].device)
        tots = []
        for X in X_list:
            tots.append(float((X*X).sum().item()))
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            diagE += torch.diagonal(Eg, 0)
        tot = float(np.mean(tots))
        diagE /= max(1, len(X_list))
        order = torch.argsort(diagE, descending=True)
        csum = torch.cumsum(diagE[order], dim=0)
        frac = csum / max(tot, 1e-12)
        need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else nb)
        K = min(need, max_blocks, nb)
        blocks = []
        for bi in order[:K].tolist():
            i0 = bi * b
            blocks.append((i0, i0, b, b))
        ef = float(frac[K-1].item()) if K > 0 else 0.0
        return {"mode": "blockdiag", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

# ----------------------------
# Residual blocks: vectorized top-k
# ----------------------------
@torch.no_grad()
def topk_blocks(R: torch.Tensor, t: int, b: int) -> List[Tuple[int,int,torch.Tensor]]:
    if t <= 0 or b <= 0:
        return []
    Eg = _block_energy_grid(R, b).to(DTYPE_ACC)
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    K = min(int(t), flat.numel())
    idx = torch.argsort(flat, descending=True)[:K]
    blocks: List[Tuple[int,int,torch.Tensor]] = []
    for k in idx.tolist():
        bi = k // nb
        bj = k % nb
        i0 = bi*b
        j0 = bj*b
        i1 = min(R.shape[0], i0+b)
        j1 = min(R.shape[1], j0+b)
        blocks.append((i0, j0, R[i0:i1, j0:j1].clone()))
    return blocks

# ----------------------------
# Payload build
# ----------------------------
@torch.no_grad()
def svd_init(W_mean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(W_mean, full_matrices=False)
    return U.contiguous(), Vh.t().contiguous()

@torch.no_grad()
def build_payload_from_X_list(X_list: List[torch.Tensor], idx: List[int], basis: dict, n: int) -> dict:
    core = choose_core_blocks(X_list, cfg.CORE_MODE, cfg.CORE_AGG, cfg.CORE_BLOCK, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)

    # core blocks per expert
    core_blocks: List[List[Tuple[int,int,Any]]] = []
    if core["blocks_per_expert"] is not None:
        for X, bl in zip(X_list, core["blocks_per_expert"]):
            lst = []
            for (i0, j0, h, w) in bl:
                h = min(h, X.shape[0]-i0); w = min(w, X.shape[1]-j0)
                if h > 0 and w > 0:
                    B = X[i0:i0+h, j0:j0+w].to(DTYPE_W).contiguous()
                    lst.append((i0, j0, quantize_int8(B)))
            core_blocks.append(lst)
    else:
        bl = core["blocks_shared"]
        for X in X_list:
            lst = []
            for (i0, j0, h, w) in bl:
                h = min(h, X.shape[0]-i0); w = min(w, X.shape[1]-j0)
                if h > 0 and w > 0:
                    B = X[i0:i0+h, j0:j0+w].to(DTYPE_W).contiguous()
                    lst.append((i0, j0, quantize_int8(B)))
            core_blocks.append(lst)

    # residual after core
    R_list = []
    for X, cb in zip(X_list, core_blocks):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bq) in cb:
            B = dequantize(Bq, device=X.device, dtype=DTYPE_ACC)
            h, w = B.shape
            Xc[i0:i0+h, j0:j0+w] = B
        R_list.append((X - Xc).contiguous())

    # shared low-rank bases DL/DR from concatenated residuals
    r = min(int(cfg.RES_RANK), X_list[0].shape[0])
    DLq = DRq = None
    coef = None
    if r > 0:
        Rcat = torch.cat(R_list, dim=0)  # (E*n, n)
        RcatT = torch.cat([R.t() for R in R_list], dim=0)

        q = min(Rcat.shape[1], r + 16)
        try:
            _, _, V = torch.svd_lowrank(Rcat, q=q, niter=2)
            DR = V[:, :r].contiguous()
        except Exception:
            _, _, Vh = torch.linalg.svd(Rcat, full_matrices=False)
            DR = Vh.t()[:, :r].contiguous()

        q2 = min(RcatT.shape[1], r + 16)
        try:
            _, _, V2 = torch.svd_lowrank(RcatT, q=q2, niter=2)
            DL = V2[:, :r].contiguous()
        except Exception:
            _, _, Vh2 = torch.linalg.svd(RcatT, full_matrices=False)
            DL = Vh2.t()[:, :r].contiguous()

        # per-expert coefficients
        if cfg.RES_COEF_MODE == "diag":
            gam = []
            for Rm in R_list:
                D = torch.diagonal(DL.t() @ Rm @ DR, 0).contiguous()  # (r,)
                gam.append(D)
            coef = torch.stack(gam, dim=0).contiguous()  # (Ecl,r)
        else:
            As = []
            for Rm in R_list:
                A = (DL.t() @ Rm @ DR).contiguous()  # (r,r)
                As.append(A)
            coef = torch.stack(As, dim=0).contiguous()  # (Ecl,r,r)

        DLq = quantize_int8(DL.to(DTYPE_W).contiguous())
        DRq = quantize_int8(DR.to(DTYPE_W).contiguous())

    # sparse residual blocks after low-rank
    blocks_list: List[List[Tuple[int,int,Any]]] = []
    for j, Rm in enumerate(R_list):
        R2 = Rm
        if r > 0 and DLq is not None and DRq is not None and coef is not None:
            DL = dequantize(DLq, device=Rm.device, dtype=DTYPE_ACC)
            DR = dequantize(DRq, device=Rm.device, dtype=DTYPE_ACC)
            if cfg.RES_COEF_MODE == "diag":
                g = coef[j].to(DTYPE_ACC).to(Rm.device)
                R2 = R2 - (DL * g.view(1, -1)) @ DR.t()
            else:
                A = coef[j].to(DTYPE_ACC).to(Rm.device)
                R2 = R2 - (DL @ A) @ DR.t()
            R2 = R2.contiguous()

        blks = topk_blocks(R2, cfg.RES_BLOCKS, cfg.RES_BSIZE)
        out_blks = []
        for (i0, j0, Bb) in blks:
            out_blks.append((i0, j0, quantize_int8(Bb.to(DTYPE_W).contiguous())))
        blocks_list.append(out_blks)

    out = {
        "n": n,
        "idx": idx,
        "basis": basis,            # either {"U":..., "V":...} or {"Ub":..., "Vb":...}
        "core": core,
        "core_blocks": core_blocks,
        "DL": DLq, "DR": DRq,
        "coef": coef,
        "blocks": blocks_list,
    }
    return out

@torch.no_grad()
def build_payload_for_cluster_dense(Ws: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor) -> dict:
    X_list = [(U.t() @ Ws[e] @ V).contiguous() for e in idx]
    basis = {
        "type": "dense",
        "U": quantize_int8(U.to(DTYPE_W).contiguous()),
        "V": quantize_int8(V.to(DTYPE_W).contiguous()),
    }
    return build_payload_from_X_list(X_list, idx, basis=basis, n=U.shape[0])

@torch.no_grad()
def build_payload_for_cluster_hadamard(Ws: torch.Tensor, idx: List[int], Ub: ImplicitHadamardPerm, Vb: ImplicitHadamardPerm) -> dict:
    X_list = [Ub.project_matrix(Ws[e], Vb).contiguous() for e in idx]
    basis = {"type": "hadamard_perm", "Ub": Ub, "Vb": Vb}
    return build_payload_from_X_list(X_list, idx, basis=basis, n=Ub.n)

# ----------------------------
# Runtime
# ----------------------------
class KTXRuntime(nn.Module):
    def __init__(self, payloads: List[Optional[dict]], basis_of_e: Dict[int, Tuple[int,int]], n: int):
        super().__init__()
        self.payloads = payloads
        self.map = basis_of_e
        self.n = n

    @torch.no_grad()
    def forward(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        used: Dict[int, List[Tuple[float,int]]] = {}
        for a, e in zip(gates.tolist(), routed):
            if e in self.map:
                m, j = self.map[e]
                used.setdefault(m, []).append((a, j))

        for m, items in used.items():
            P = self.payloads[m]
            if P is None:
                continue

            basis = P["basis"]
            if basis["type"] == "hadamard_perm":
                Ub: ImplicitHadamardPerm = basis["Ub"]
                Vb: ImplicitHadamardPerm = basis["Vb"]
                z = Ub.apply(x)
            else:
                U = dequantize(basis["U"], device=x.device, dtype=DTYPE_ACC)
                V = dequantize(basis["V"], device=x.device, dtype=DTYPE_ACC)
                z = x @ U

            u_acc = torch.zeros_like(z)

            DLq = P.get("DL", None)
            DRq = P.get("DR", None)
            coef = P.get("coef", None)
            has_lr = (DLq is not None) and (DRq is not None) and (coef is not None)

            if has_lr:
                DL = dequantize(DLq, device=x.device, dtype=DTYPE_ACC)
                DR = dequantize(DRq, device=x.device, dtype=DTYPE_ACC)
            else:
                DL = DR = None

            for (a, j) in items:
                u = torch.zeros_like(z)

                # core blocks
                for (i0, j0, Bq) in P["core_blocks"][j]:
                    Bc = dequantize(Bq, device=x.device, dtype=DTYPE_ACC)
                    h, w = Bc.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bc

                # low-rank residual
                if has_lr and DL is not None and DR is not None:
                    t = (z @ DL)  # (B,r)
                    if cfg.RES_COEF_MODE == "diag":
                        g = coef[j].to(DTYPE_ACC).to(x.device)
                        t = t * g.view(1, -1)
                    else:
                        A = coef[j].to(DTYPE_ACC).to(x.device)
                        t = t @ A
                    u += t @ DR.t()

                # sparse residual blocks
                for (i0, j0, Bbq) in P["blocks"][j]:
                    Bb = dequantize(Bbq, device=x.device, dtype=DTYPE_ACC)
                    h, w = Bb.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb

                u_acc += float(a) * u

            if basis["type"] == "hadamard_perm":
                y += Vb.apply_T(u_acc)
            else:
                V = dequantize(basis["V"], device=x.device, dtype=DTYPE_ACC)
                y += u_acc @ V.t()

        return y

# ----------------------------
# Reconstruction (per-expert error)
# ----------------------------
@torch.no_grad()
def reconstruct_W_for_expert(payload: dict, j: int) -> torch.Tensor:
    n = int(payload["n"])
    X = torch.zeros(n, n, dtype=DTYPE_ACC, device=DEVICE)

    for (i0, j0, Bq) in payload["core_blocks"][j]:
        B = dequantize(Bq, device=DEVICE, dtype=DTYPE_ACC)
        h, w = B.shape
        X[i0:i0+h, j0:j0+w] = B

    DLq = payload.get("DL", None)
    DRq = payload.get("DR", None)
    coef = payload.get("coef", None)
    if DLq is not None and DRq is not None and coef is not None:
        DL = dequantize(DLq, device=DEVICE, dtype=DTYPE_ACC)
        DR = dequantize(DRq, device=DEVICE, dtype=DTYPE_ACC)
        if cfg.RES_COEF_MODE == "diag":
            g = coef[j].to(DTYPE_ACC).to(DEVICE)
            X = X + (DL * g.view(1, -1)) @ DR.t()
        else:
            A = coef[j].to(DTYPE_ACC).to(DEVICE)
            X = X + (DL @ A) @ DR.t()

    for (i0, j0, Bbq) in payload["blocks"][j]:
        Bb = dequantize(Bbq, device=DEVICE, dtype=DTYPE_ACC)
        h, w = Bb.shape
        X[i0:i0+h, j0:j0+w] += Bb

    basis = payload["basis"]
    if basis["type"] == "hadamard_perm":
        Ub: ImplicitHadamardPerm = basis["Ub"]
        Vb: ImplicitHadamardPerm = basis["Vb"]
        W = Ub.reconstruct_matrix(X, Vb)
    else:
        U = dequantize(basis["U"], device=DEVICE, dtype=DTYPE_ACC)
        V = dequantize(basis["V"], device=DEVICE, dtype=DTYPE_ACC)
        W = (U @ X @ V.t()).contiguous()
    return W

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def dense_apply(Ws: torch.Tensor, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
    n = Ws.shape[-1]
    Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=x.device)
    for a, e in zip(gates, routed):
        Wsum += float(a.item()) * Ws[e]
    return x @ Wsum

@torch.no_grad()
def pick_routes_from_router(P: np.ndarray, eids: List[int], trial: int, k: int) -> Tuple[List[int], torch.Tensor]:
    E = len(eids)
    nrows = P.shape[0]
    r = (SEED + 777 + trial) % max(1, nrows)
    p = P[r].astype(np.float32, copy=False)

    if cfg.ROUTER_EIDS_ARE_GLOBAL:
        probs = np.array([p[eid] if eid < p.shape[0] else 0.0 for eid in eids], dtype=np.float32)
    else:
        probs = p[:E].copy()

    probs = np.maximum(probs, 0.0)
    if probs.sum() <= 0:
        routed = random.sample(range(E), k=min(k, E))
        g = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        g = g / g.sum().clamp_min(1e-12)
        return routed, g

    top = np.argsort(-probs)[:min(k, E)]
    w = probs[top]
    w = w / max(w.sum(), 1e-12)
    return top.tolist(), torch.from_numpy(w).to(DTYPE_ACC).to(DEVICE)

@torch.no_grad()
def eval_runtime(rt: KTXRuntime, Ws: torch.Tensor, eids: List[int], trials: int, routed_k: int, batch: int):
    P = load_router_P() if cfg.ROUTER_PATH else None
    errs = []
    for t in range(trials):
        x = torch.randn(batch, rt.n, dtype=DTYPE_ACC, device=DEVICE)
        if P is not None:
            routed, gates = pick_routes_from_router(P, eids, t, routed_k)
        else:
            routed = random.sample(range(Ws.shape[0]), k=min(routed_k, Ws.shape[0]))
            gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
            gates = gates / gates.sum().clamp_min(1e-12)

        y_hat = rt(x, routed, gates)
        y_ref = dense_apply(Ws, x, routed, gates)
        err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)

    log(f"[eval] routed forward rel-error = {float(np.mean(errs)):.6f} ± {float(np.std(errs)):.6f}  (trials={trials})")

@torch.no_grad()
def eval_per_expert(payloads: List[Optional[dict]], basis_of_e: Dict[int, Tuple[int,int]], Ws: torch.Tensor):
    errs = []
    for e in range(Ws.shape[0]):
        m, j = basis_of_e[e]
        P = payloads[m]
        if P is None:
            continue
        What = reconstruct_W_for_expert(P, j)
        err = float((frob(What - Ws[e]) / (frob(Ws[e]) + 1e-12)).item())
        errs.append(err)

    if errs:
        mean_err = float(np.mean(errs))
        p95_err  = float(np.percentile(errs, 95))
        max_err  = float(np.max(errs))
        log(f"[eval] per-expert W rel-error  mean={mean_err:.6f}  p95={p95_err:.6f}  max={max_err:.6f}")
    else:
        log("[eval] per-expert W rel-error  (no experts?)")

# ----------------------------
# Rough payload size estimate
# ----------------------------
def sizeof_payload(payloads: List[Optional[dict]]) -> float:
    def tensor_bytes(t: Any) -> int:
        if t is None:
            return 0
        if isinstance(t, QTensor):
            bq = t.q.numel() * t.q.element_size()
            bs = t.scale.numel() * t.scale.element_size()
            return bq + bs
        if torch.is_tensor(t):
            return t.numel() * t.element_size()
        return 0

    total = 0
    for P in payloads:
        if P is None:
            continue
        basis = P["basis"]
        if basis["type"] == "dense":
            total += tensor_bytes(basis["U"]) + tensor_bytes(basis["V"])
        else:
            Ub: ImplicitHadamardPerm = basis["Ub"]
            Vb: ImplicitHadamardPerm = basis["Vb"]
            total += Ub.perm.numel() * 4 + Ub.inv_perm.numel() * 4 + Ub.sign.numel() * 1
            total += Vb.perm.numel() * 4 + Vb.inv_perm.numel() * 4 + Vb.sign.numel() * 1

        for perE in P["core_blocks"]:
            for (_, _, Bq) in perE:
                total += tensor_bytes(Bq)
        for perE in P["blocks"]:
            for (_, _, Bq) in perE:
                total += tensor_bytes(Bq)

        total += tensor_bytes(P.get("DL")) + tensor_bytes(P.get("DR"))
        coef = P.get("coef", None)
        if torch.is_tensor(coef):
            total += coef.numel() * coef.element_size()

    return total / 1e6

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X Optimized (offline) v7 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES={cfg.CALIB_SAMPLES}  STRICT_CALIB={cfg.STRICT_CALIB}  ALLOW_RANDOM_CALIB={cfg.ALLOW_RANDOM_CALIB}")
    log(f"CAPTURE_CALIB={cfg.CAPTURE_CALIB}  (optional transformers capture)")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  (RIDGE_DAMP={cfg.RIDGE_DAMP})  AUTO_FALLBACK_LIN={cfg.AUTO_FALLBACK_LIN}")
    log(f"BASIS_MODE:  {cfg.BASIS_MODE}")
    log(f"CORE_MODE:   {cfg.CORE_MODE}  agg={cfg.CORE_AGG} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK}  coef={cfg.RES_COEF_MODE}  blocks={cfg.RES_BLOCKS}  bsize={cfg.RES_BSIZE}")
    log(f"QUANT:       QMODE={cfg.QMODE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} (dense_train only) lr={cfg.LR_UV} subm={cfg.SUBM} batchE={cfg.BATCH_E}")
    log(f"CLUSTER:     M={cfg.M_CLUSTERS or '(auto)'} iters={cfg.CLUSTER_ITERS} restarts={cfg.CLUSTER_RESTARTS} feat_d={cfg.CLUSTER_FEAT_D} target_size={cfg.CLUSTER_TARGET_SIZE} min_size={cfg.CLUSTER_MIN_SIZE}")
    log(f"Hadamard(post-W): USE_HADAMARD={cfg.USE_HADAMARD} seed={cfg.HAD_SEED}")
    log(f"Threads:     {NTHREADS}  Torch={torch.__version__} device={DEVICE}")
    log(f"Cache:       {cfg.CACHE_MODE}  path={cache_path()}  force_rebuild={cfg.FORCE_REBUILD_WS}")
    log("")

def main():
    banner()

    eids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} experts={eids[:8]}{'...' if len(eids)>8 else ''}")

    # Choose cluster count
    if cfg.M_CLUSTERS > 0:
        M = min(cfg.M_CLUSTERS, E)
    else:
        M = max(1, min(E, int(round(E / max(1, cfg.CLUSTER_TARGET_SIZE)))))
    log(f"[cluster] M={M}")

    Xfeat = expert_features(Ws, d=cfg.CLUSTER_FEAT_D)
    labels, centers = kmeans_torch(Xfeat, k=M, iters=cfg.CLUSTER_ITERS, restarts=cfg.CLUSTER_RESTARTS)
    labels = enforce_min_cluster_size(labels, centers, Xfeat, min_size=cfg.CLUSTER_MIN_SIZE)

    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    log(f"[cluster] sizes: {[len(c) for c in clusters]}")

    payloads: List[Optional[dict]] = []
    basis_of_e: Dict[int, Tuple[int,int]] = {}

    log("[build] payloads ...")

    if cfg.BASIS_MODE == "dense_train":
        # Init U,V per cluster via SVD(mean W)
        U_par: List[Optional[DenseBasis]] = []
        V_par: List[Optional[DenseBasis]] = []
        for m, idx in enumerate(clusters):
            if len(idx) == 0:
                U_par.append(None); V_par.append(None)
                continue
            Wm = Ws[idx].mean(dim=0)
            U0, V0 = svd_init(Wm)
            U_par.append(DenseBasis(U0))
            V_par.append(DenseBasis(V0))

        # Optional training (off by default). Best used ONLY with real CALIB_PATH.
        if cfg.TRAIN_STEPS > 0:
            params = [uv.M for uv in (U_par + V_par) if uv is not None]
            opt = torch.optim.AdamW(params, lr=cfg.LR_UV, weight_decay=0.0)

            core_block_for_train = cfg.CORE_BLOCK if cfg.TRAIN_BLOCK_USE_CORE_BLOCK else max(16, cfg.SUBM // 4)

            def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
                Eb, s, _ = Xs.shape
                b = int(block)
                nb = s // b
                if b <= 0 or nb <= 0:
                    return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
                s2 = nb * b
                X = Xs[:, :s2, :s2].contiguous()
                Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
                Eblk = (Xb * Xb).sum(dim=(3,4))
                Pm = Eblk.mean(dim=0)
                return torch.sqrt(Pm + 1e-12).sum() / (Pm.sum() + 1e-12)

            def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
                U_S = U[:, S]
                V_S = V[:, S]
                T = Ws_batch @ V_S
                return torch.matmul(U_S.t().unsqueeze(0), T)

            t0 = time.perf_counter()
            with torch.enable_grad():
                for step in range(1, cfg.TRAIN_STEPS + 1):
                    S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]
                    L_total = None
                    n_terms = 0

                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                            continue
                        Uo = U_par[m].orthogonal()
                        Vo = V_par[m].orthogonal()

                        if 0 < cfg.BATCH_E < len(idx):
                            pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                            idx_step = [idx[p] for p in pick]
                        else:
                            idx_step = idx

                        Ws_batch = Ws[idx_step]
                        Xs = slice_X_batch(Ws_batch, Uo, Vo, S)  # (Eb,s,s)

                        diag = torch.diagonal(Xs, dim1=1, dim2=2)
                        diagE = (diag * diag).mean().clamp_min(1e-12)
                        off = Xs - torch.diag_embed(diag)
                        offE = (off * off).mean()

                        if cfg.TRAIN_OBJ == "logratio":
                            loss = torch.log(offE + 1e-12) - torch.log(diagE + 1e-12)
                        else:
                            loss = offE / diagE

                        if cfg.TRAIN_LAM_BLOCK > 0 and cfg.CORE_MODE.startswith("block"):
                            loss = loss + cfg.TRAIN_LAM_BLOCK * block_group_sparsity_penalty(Xs, core_block_for_train)

                        L_total = loss if (L_total is None) else (L_total + loss)
                        n_terms += 1

                    if L_total is None:
                        log("[train] skipped (no clusters >= TRAIN_MIN_CLUSTER)")
                        break

                    L_total = L_total / max(1, n_terms)
                    opt.zero_grad(set_to_none=True)
                    L_total.backward()
                    if cfg.GRAD_CLIP > 0:
                        torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                    opt.step()

                    if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                        with torch.no_grad():
                            for uv in (U_par + V_par):
                                if uv is not None:
                                    uv.M.copy_(uv.orthogonal())

                    if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                        t1 = time.perf_counter()
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} | loss={float(L_total.item()):.6f}  (+{t1-t0:.1f}s)")
                        t0 = t1

        for m, idx in enumerate(clusters):
            if len(idx) == 0:
                payloads.append(None)
                continue
            Uo = U_par[m].orthogonal().detach()
            Vo = V_par[m].orthogonal().detach()
            Pld = build_payload_for_cluster_dense(Ws, idx, Uo, Vo)
            payloads.append(Pld)
            for j, e in enumerate(idx):
                basis_of_e[e] = (m, j)

            efr = Pld["core"]["energy_fracs"]
            ef_mean = float(np.mean(efr)) if efr else 0.0
            ef_min  = float(np.min(efr)) if efr else 0.0
            nb_mean = float(np.mean([len(x) for x in Pld["core_blocks"]])) if Pld["core_blocks"] else 0.0
            log(f"  - basis {m}: E={len(idx)} core={Pld['core']['mode']} core_energy(mean/min)≈{ef_mean:.3f}/{ef_min:.3f} core_blocks(mean)≈{nb_mean:.1f}")

    elif cfg.BASIS_MODE == "hadamard_perm":
        if not is_power_of_two(n):
            raise RuntimeError(f"BASIS_MODE=hadamard_perm requires H power-of-two; got H={n}")
        for m, idx in enumerate(clusters):
            if len(idx) == 0:
                payloads.append(None)
                continue
            Ub = ImplicitHadamardPerm(n, seed=SEED + 10000 + 13*m)
            Vb = ImplicitHadamardPerm(n, seed=SEED + 20000 + 17*m)
            Pld = build_payload_for_cluster_hadamard(Ws, idx, Ub, Vb)
            payloads.append(Pld)
            for j, e in enumerate(idx):
                basis_of_e[e] = (m, j)

            efr = Pld["core"]["energy_fracs"]
            ef_mean = float(np.mean(efr)) if efr else 0.0
            ef_min  = float(np.min(efr)) if efr else 0.0
            nb_mean = float(np.mean([len(x) for x in Pld["core_blocks"]])) if Pld["core_blocks"] else 0.0
            log(f"  - basis {m}: E={len(idx)} core={Pld['core']['mode']} core_energy(mean/min)≈{ef_mean:.3f}/{ef_min:.3f} core_blocks(mean)≈{nb_mean:.1f}")

    else:
        raise ValueError("BASIS_MODE must be dense_train or hadamard_perm")

    rt = KTXRuntime(payloads, basis_of_e, n=n)

    mb = sizeof_payload(payloads)
    log(f"[size] approx payload bytes ≈ {mb:.2f} MB  (dense_train stores full U/V -> huge; hadamard_perm avoids that)")

    if cfg.EVAL_PER_EXPERT:
        eval_per_expert(payloads, basis_of_e, Ws)

    eval_runtime(rt, Ws, eids, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    log("\n✅ Done.")
    log("Practical next steps for real accuracy:")
    log("  1) Provide real CALIB_PATH (true layer inputs) and (ideally) ROUTER_PATH, then set RIDGE_WEIGHTED=1.")
    log("  2) For real compression (payload size), use BASIS_MODE=hadamard_perm (implicit basis).")
    log("  3) To push accuracy: CORE_TARGET↑, CORE_MAX_BLOCKS↑, RES_RANK↑, RES_BLOCKS↑.")

# Run
main()


== DeepSeek KT++-X Optimized (offline) v7 ==
Time:        2026-01-13 09:38:03
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES=2048  STRICT_CALIB=False  ALLOW_RANDOM_CALIB=True
CAPTURE_CALIB=False  (optional transformers capture)
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
LIN_MODE:    ridge  (RIDGE_DAMP=0.001)  AUTO_FALLBACK_LIN=True
BASIS_MODE:  dense_train
CORE_MODE:   blocktopk_perexpert  agg=mean block=64 target=0.9 max_blocks=512
RESIDUAL:    rank=64  coef=full  blocks=128  bsize=64
QUANT:       QMODE=none
TRAIN:       steps=0 (dense_train only) lr=0.01 subm=256 batchE=4
CLUSTER:     M=(auto) iters=80 restarts=4 feat_d=64 target_size=4 min_size=2
Hadamard(post-W): USE_HADAMARD=False seed=1234
Threads:     8  Torch=2.4.1+cpu device=cpu
Cache:       readwrite  path=/home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v7.npz  force_rebuild=False

[cache] loaded Ws: /home/da